# 🔁 Certificate QR Replacement Bot (batch, ~200 certificates)

**What it does** — for every certificate image in your Kaggle dataset:

1. **Detects** the existing QR code — AI CNN from Hugging Face
   (`opencv/qrcode_wechatqrcode`) → OpenCV → geometric finder (works even when
   the old QR is a decorative/non-standard pattern that cannot be decoded).
2. **Generates** a brand-new, crisp QR code whose payload is a **URL pointing
   to that certificate's image** (so scanning it shows the certificate).
3. **Replaces** the old QR with an **AI image-generator (Hugging Face diffusion
   inpainting — Stable Diffusion XL Inpainting)** driven by the natural-language
   prompt *"generate this certificate, just replace the old QR code with the new
   QR code; do not change any design, information, face or anything else"*. The
   model is masked to the QR square (everything else is provably untouched), and
   the **real QR is composited on top so it is guaranteed to scan** — a diffusion
   model drawing a QR from text would otherwise hallucinate an invalid code.
   Set `USE_AI=False` to use the pure geometric warp instead.
4. **Verifies** the new QR scans (auto-decodes the output).
5. **Zips** all finished certificates.

> ⚙️ **Accelerator note:** with `USE_AI=True` choose **GPU (T4/P100)** — the
> inpainting model needs it; the WeChat QR CNN itself is a tiny CPU model.
> `USE_AI=False` runs on CPU in well under a minute for 200 images. Either way
> the finished image keeps the original design byte-for-byte outside the QR box.

**Inputs:** attach a Kaggle *Dataset* containing your certificate images
(any folder structure, `.png/.jpg/.jpeg/.webp`).


## ⚙️ 1. Configuration
`BASE_URL` is already set to your **GitHub Pages certificate hosting** site (`https://naserkhan07.github.io/certificate-qr/img/`). Each QR encodes `BASE_URL + filename`; section 8 publishes the finished ZIP there (one token). If you use another host, just change `BASE_URL`.

In [ ]:
# ==== EDIT THESE ====
BASE_URL = "https://naserkhan07.github.io/certificate-qr/"   # QR -> per-candidate verification page
PAYLOAD_MODE = "url"        # "url" | "csv" | "text" | "copy"
CSV_PATH = None             # set to a CSV path if PAYLOAD_MODE="csv" (cols: filename,qr_data)
TEXT_PREFIX = ""            # used if PAYLOAD_MODE="text"
REPLACE_MODE = "full_replace"  # "full_replace" (recommended) | "keep_frame"
QR_COVER_SCALE = 1.08       # grow the white QR box ~8% to fully hide the old
                            # black frame / any edge halo. 1.0 = exact old size.
QR_ERROR_CORRECTION = "M"   # L | M | Q | H
OUTPUT_EXT = ".png"         # ".png" (lossless, recommended) | ".jpg"

# ---- AI that READS the candidate details off the certificate (no CSV/OCR) ----
# A Hugging Face vision-language model looks at each certificate image and
# extracts: Name, Father/Mother Name, Aadhar No., Course Name, Course Date and
# From (institute). Needs a GPU (set Settings -> Accelerator -> GPU T4).
# Qwen2-VL-2B is a strong, small document reader with NO num2words/tesseract
# dependency and it fits a free T4. Optional stronger/larger alternative:
# "Qwen/Qwen2.5-VL-3B-Instruct".
AI_EXTRACT_MODEL = "Qwen/Qwen2-VL-2B-Instruct"

# ---- QR code GENERATOR backend ----
# "standard"  = plain black/white QR, 100% deterministic & always scans
#               (RECOMMENDED for official certificates).
# "artistic"  = Hugging Face "QR Code Monster" ControlNet (AI image->QR model):
#               generates a stylised QR, auto-checks it decodes to the right
#               URL over several seeds, and falls back to "standard" if none
#               scan. Needs GPU. Colourful art-QRs are NOT typical on govt docs.
QR_BACKEND = "standard"
ART_CONTROLNET_SCALE = 1.5  # higher = more readable (1.2-1.8), lower = more art
ART_TRIES = 6               # seed variants tried before falling back
ART_SIZE = 768

# ---- AI image generator (Hugging Face diffusion inpainting) ----
# Leave False for the fast, guaranteed run. Set True ONLY with GPU (T4/P100) +
# Internet on; it then uses the SDXL inpainting model for the QR region and
# auto-falls back to the exact warp if no GPU/model is available.
USE_AI = False
AI_MODEL = "diffusers/stable-diffusion-xl-1.0-inpainting-0.1"
AI_STEPS = 30               # fewer = faster (try 20), more = cleaner
AI_GUIDANCE = 8.0
AI_SEED = 0
AI_MODE = "inpaint_then_paste"   # keep this -> scannable. "generative_only" is experiment-only

# ==== input dataset (auto-detected; override if you have multiple) ====
import glob, os
CANDIDATES = []
for pat in ["/kaggle/input/**/*.png", "/kaggle/input/**/*.jpg",
            "/kaggle/input/**/*.jpeg", "/kaggle/input/**/*.webp"]:
    CANDIDATES += glob.glob(pat, recursive=True)
print("Found", len(CANDIDATES), "images under /kaggle/input")
for p in CANDIDATES[:10]:
    print("  ", p)


## 📦 2. Install dependencies

In [ ]:
# Remove any pre-installed OpenCV (Kaggle ships several; mixing the GUI and
# headless/contrib wheels breaks `cv2`), then install the exact contrib build.
%pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless 2>/dev/null
%pip install -q "opencv-contrib-python-headless==4.10.0.84" qrcode huggingface_hub requests pillow
# AI: vision-language model that READS the certificate fields (always used),
# plus diffusers for the optional AI QR/painting paths. (Qwen2-VL: no num2words.)
%pip install -q "transformers>=4.46" diffusers accelerate safetensors torchvision
import cv2
print("opencv:", cv2.__version__, "| wechat CNN module:",
      hasattr(cv2, "wechat_qrcode_WeChatQRCode"))
print("done")

## 🤗 3. Download the AI QR-detection model from Hugging Face
Downloads `opencv/qrcode_wechatqrcode` (WeChat CV QR detector + super-resolution CNN). Falls back to the OpenCV GitHub mirror.

In [ ]:
import os
os.makedirs("/kaggle/working/certbot/models/wechat", exist_ok=True)
_cwd = os.getcwd(); os.chdir("/kaggle/working/certbot")
MODEL_DIR = "/kaggle/working/certbot/models/wechat"
#!/usr/bin/env python3
"""
download_models.py
------------------
Download the WeChat QR-code CNN detector (used as the 1st-stage AI detector)
from the OFFICIAL Hugging Face model repo:

    https://huggingface.co/opencv/qrcode_wechatqrcode

Files are saved to  models/wechat/  with the canonical names OpenCV expects:
    detect.prototxt, detect.caffemodel, sr.prototxt, sr.caffemodel

Falls back to the OpenCV 3rdparty GitHub mirror if Hugging Face is unreachable.

The WeChat model is a tiny (~1 MB) CPU CNN -- it downloads in seconds and runs
fine on Kaggle CPU (a GPU is not required for this step; the pipeline will use
whatever accelerator you have but this model is CPU-only by design).
"""

import os
import shutil
import urllib.request

MODEL_DIR = os.environ.get("WECHAT_MODEL_DIR", "/kaggle/working/certbot/models/wechat")
HF_REPO = "opencv/qrcode_wechatqrcode"

# (filename in the HF repo, canonical local filename)
FILES = [
    ("detect_2021nov.prototxt",  "detect.prototxt"),
    ("detect_2021nov.caffemodel", "detect.caffemodel"),
    ("sr_2021nov.prototxt",      "sr.prototxt"),
    ("sr_2021nov.caffemodel",    "sr.caffemodel"),
]

GH_BASE = ("https://raw.githubusercontent.com/opencv/opencv_3rdparty/"
           "wechat_qrcode_20210119/")
GH_FILES = {
    "detect.prototxt": "detect.prototxt",
    "detect.caffemodel": "detect.caffemodel",
    "sr.prototxt": "sr.prototxt",
    "sr.caffemodel": "sr.caffemodel",
}


def _download(url: str, dst: str):
    print(f"  GET {url}")
    with urllib.request.urlopen(url, timeout=120) as r, open(dst, "wb") as f:
        shutil.copyfileobj(r, f)


def from_huggingface(dst_dir: str) -> bool:
    try:
        from huggingface_hub import hf_hub_download
    except ImportError:
        print("huggingface_hub not installed; trying direct HTTPS.")
        return False
    ok = True
    for hf_name, local_name in FILES:
        dst = os.path.join(dst_dir, local_name)
        try:
            p = hf_hub_download(repo_id=HF_REPO, filename=hf_name)
            shutil.copy(p, dst)
            print(f"  HF {hf_name} -> {dst} ({os.path.getsize(dst)} bytes)")
        except Exception as e:
            print(f"  HF download failed for {hf_name}: {e}")
            ok = False
    return ok


def from_github(dst_dir: str) -> bool:
    ok = True
    for local_name in GH_FILES:
        dst = os.path.join(dst_dir, local_name)
        try:
            _download(GH_BASE + GH_FILES[local_name], dst)
            print(f"  GH -> {dst} ({os.path.getsize(dst)} bytes)")
        except Exception as e:
            print(f"  GitHub download failed for {local_name}: {e}")
            ok = False
    return ok


def main():
    os.makedirs(MODEL_DIR, exist_ok=True)
    print(f"Downloading WeChat QR CNN model to {MODEL_DIR}/ ...")
    if from_huggingface(MODEL_DIR):
        print("Model downloaded from Hugging Face (opencv/qrcode_wechatqrcode).")
        return
    print("Falling back to GitHub mirror ...")
    if from_github(MODEL_DIR):
        print("Model downloaded from GitHub mirror.")
        return
    raise SystemExit(
        "Could not download the WeChat model. The pipeline still works "
        "without it (OpenCV + geometric fallback), but for best detection "
        "download the 4 files manually into models/wechat/ -- see README.")


if __name__ == "__main__":
    main()

main()
os.chdir(_cwd)
import cv2
print("opencv:", cv2.__version__, "| wechat module:",
      hasattr(cv2, "wechat_qrcode_WeChatQRCode"))


## 🧠 4. Write the pipeline modules

In [ ]:
import os, shutil
os.makedirs("/kaggle/working/certbot/src", exist_ok=True)
# clear stale bytecode from previous runs
_pyc = "/kaggle/working/certbot/src/__pycache__"
if os.path.isdir(_pyc): shutil.rmtree(_pyc)
SRC = {
  "__init__.py": "from .pipeline import Config, run, process_one  # noqa\n",
  "detector.py": "\"\"\"\ndetector.py\n-----------\nLocate the OLD QR code on a certificate image.\n\nDetection cascade (tries each, returns first high-confidence hit):\n  1. WeChat CNN QR detector  (Deep-CV model downloaded from Hugging Face)\n  2. OpenCV Aruco QR detector\n  3. Geometric contour finder (finds the thick black square FRAME around a QR\n     even when the QR itself is too dense/small to decode -- works on certs\n     whose old QR is decorative or damaged)\n\nEvery detector returns the quadrangle of the *outer QR box* (including the\nblack border frame when present), so the new QR can be warped to exactly that\nsize / position / rotation.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport os\nfrom dataclasses import dataclass\n\nimport cv2\nimport numpy as np\n\n\n@dataclass\nclass QRRegion:\n    \"\"\"The 4 corners of the QR's outer box, in order TL, TR, BR, BL (px).\"\"\"\n    quad: np.ndarray            # shape (4, 2), float32\n    method: str                 # which detector found it\n    side_px: float              # average side length in pixels\n    score: float                # confidence 0..1\n\n\n# ---------------------------------------------------------------------------\n# Helpers\n# ---------------------------------------------------------------------------\ndef _order_points(pts: np.ndarray) -> np.ndarray:\n    \"\"\"Order 4 points as TL, TR, BR, BL.\"\"\"\n    pts = np.asarray(pts, dtype=np.float32).reshape(4, 2)\n    s = pts.sum(axis=1)\n    d = np.diff(pts, axis=1).reshape(-1)\n    tl = pts[np.argmin(s)]\n    br = pts[np.argmax(s)]\n    tr = pts[np.argmin(d)]\n    bl = pts[np.argmax(d)]\n    return np.array([tl, tr, br, bl], dtype=np.float32)\n\n\ndef _side_lengths(quad: np.ndarray) -> tuple[float, float, float, float]:\n    tl, tr, br, bl = quad\n    return (\n        float(np.linalg.norm(tr - tl)),\n        float(np.linalg.norm(br - tr)),\n        float(np.linalg.norm(bl - br)),\n        float(np.linalg.norm(tl - bl)),\n    )\n\n\ndef _valid_quad(gray: np.ndarray, quad: np.ndarray,\n                min_finders: int = 2) -> bool:\n    \"\"\"\n    Reject false-positive quads from the ML/OpenCV detectors.\n    A real QR quad: roughly square, on-canvas, convex, and contains >=2 of the\n    three characteristic finder patterns (nested squares).\n    \"\"\"\n    h, w = gray.shape[:2]\n    try:\n        quad = _order_points(quad)\n    except Exception:\n        return False\n    sides = _side_lengths(quad)\n    if min(sides) < max(8.0, 0.04 * min(h, w)):\n        return False\n    if max(sides) > max(h, w) * 1.15:\n        return False\n    if min(sides) / max(sides) < 0.7:\n        return False\n    # convexity (cross products same sign)\n    edges = np.roll(quad, -1, axis=0) - quad\n    crosses = edges[:, 0] * np.roll(edges[:, 1], -1) - \\\n        edges[:, 1] * np.roll(edges[:, 0], -1)\n    if not (np.all(crosses >= -1e-3) or np.all(crosses <= 1e-3)):\n        return False\n    # mostly on canvas\n    x0, y0 = quad.min(axis=0)\n    x1, y1 = quad.max(axis=0)\n    if x1 < -0.05 * w or x0 > 1.05 * w or y1 < -0.05 * h or y0 > 1.05 * h:\n        return False\n    if _count_finder_patterns(gray, quad) < min_finders:\n        return False\n    return True\n\n\ndef _count_finder_patterns(gray: np.ndarray, quad: np.ndarray) -> int:\n    \"\"\"\n    Count finder-like nested squares inside a quad. A QR has 3 finder patterns\n    (TL, TR, BL corners). Used to score contour candidates that look like a QR.\n    \"\"\"\n    tl, tr, br, bl = quad\n    w = int(max(np.linalg.norm(tr - tl), np.linalg.norm(br - bl)))\n    h = int(max(np.linalg.norm(bl - tl), np.linalg.norm(br - tr)))\n    if w < 10 or h < 10:\n        return 0\n    M = cv2.getPerspectiveTransform(\n        quad.astype(np.float32),\n        np.array([[0, 0], [w, 0], [w, h], [0, h]], dtype=np.float32),\n    )\n    patch = cv2.warpPerspective(gray, M, (w, h))\n    _, bw = cv2.threshold(patch, 0, 255,\n                          cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)\n    contours, hier = cv2.findContours(bw, cv2.RETR_TREE,\n                                      cv2.CHAIN_APPROX_SIMPLE)\n    if hier is None:\n        return 0\n    hier = hier[0]\n\n    def nested_depth(i: int) -> int:\n        depth = 0\n        parent = hier[i][3]\n        while parent != -1:\n            depth += 1\n            parent = hier[parent][3]\n        return depth\n\n    finders = 0\n    for i, c in enumerate(contours):\n        area = cv2.contourArea(c)\n        if area == 0:\n            continue\n        x, y, ww, hh = cv2.boundingRect(c)\n        ratio = ww / float(hh)\n        area_ratio = area / float(ww * hh)\n        # finder outer ring: roughly square, moderately sized, >=2 nestings\n        if (0.7 < ratio < 1.35 and area_ratio > 0.55\n                and ww > w * 0.08 and ww < w * 0.6 and nested_depth(i) >= 2):\n            finders += 1\n    return finders\n\n\n# ---------------------------------------------------------------------------\n# Detector 1: WeChat CNN (model from Hugging Face)\n# ---------------------------------------------------------------------------\n_WECHAT = None\n\n\ndef _get_wechat_detector(model_dir: str):\n    \"\"\"Lazy-load the WeChat QR CNN model if the 4 files are present.\"\"\"\n    global _WECHAT\n    if _WECHAT is not None:\n        return _WECHAT\n    if not hasattr(cv2, \"wechat_qrcode_WeChatQRCode\"):\n        return None\n    files = [\"detect.prototxt\", \"detect.caffemodel\",\n             \"sr.prototxt\", \"sr.caffemodel\"]\n    paths = [os.path.join(model_dir, f) for f in files]\n    if not all(os.path.exists(p) for p in paths):\n        return None\n    try:\n        _WECHAT = cv2.wechat_qrcode_WeChatQRCode(*paths)\n    except Exception:\n        _WECHAT = None\n    return _WECHAT\n\n\ndef _detect_wechat(img: np.ndarray, model_dir: str):\n    det = _get_wechat_detector(model_dir)\n    if det is None:\n        return None\n    h, w = img.shape[:2]\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img\n    for scale in (1.0, 2.0, 3.0):\n        src = gray if scale == 1.0 else cv2.resize(\n            gray, (int(w * scale), int(h * scale)),\n            interpolation=cv2.INTER_CUBIC)\n        try:\n            res, points = det.detectAndDecode(src)\n        except Exception:\n            res, points = [], None\n        if points is not None and len(points) > 0:\n            for p in points:\n                quad = _order_points(np.asarray(p, dtype=np.float32) / scale)\n                if _valid_quad(gray, quad, min_finders=2):\n                    return quad\n    return None\n\n\n# ---------------------------------------------------------------------------\n# Detector 2: OpenCV ArUco / classic QR detector\n# ---------------------------------------------------------------------------\ndef _detect_opencv(img: np.ndarray):\n    h, w = img.shape[:2]\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img\n    detectors = []\n    if hasattr(cv2, \"QRCodeDetectorAruco\"):\n        try:\n            detectors.append(cv2.QRCodeDetectorAruco())\n        except Exception:\n            pass\n    detectors.append(cv2.QRCodeDetector())\n    for scale in (1.0, 2.0, 3.0, 4.0):\n        src = gray if scale == 1.0 else cv2.resize(\n            gray, (int(w * scale), int(h * scale)),\n            interpolation=cv2.INTER_CUBIC)\n        for det in detectors:\n            ok, pts = det.detect(src)\n            if ok and pts is not None:\n                quad = _order_points(pts.reshape(4, 2) / scale)\n                if _valid_quad(gray, quad, min_finders=2):\n                    return quad\n    return None\n\n\n# ---------------------------------------------------------------------------\n# Detector 3: geometric contour finder (thick black frame around QR)\n# ---------------------------------------------------------------------------\ndef _detect_by_contours(img: np.ndarray):\n    h, w = img.shape[:2]\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img\n    img_area = h * w\n\n    best = None\n    best_score = -1.0\n\n    for thresh in (90, 120, 150, 180):\n        _, bw = cv2.threshold(gray, thresh, 255, cv2.THRESH_BINARY_INV)\n        # close small gaps inside the frame\n        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))\n        bw = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel)\n        contours, _ = cv2.findContours(bw, cv2.RETR_LIST,\n                                       cv2.CHAIN_APPROX_SIMPLE)\n        for c in contours:\n            area = cv2.contourArea(c)\n            if area < img_area * 0.0008 or area > img_area * 0.25:\n                continue\n            peri = cv2.arcLength(c, True)\n            approx = cv2.approxPolyDP(c, 0.04 * peri, True)\n            if len(approx) != 4:\n                # also accept rotated rect\n                rrect = cv2.minAreaRect(c)\n                (cx, cy), (rw, rh), _ = rrect\n                if rw == 0 or rh == 0:\n                    continue\n                sq = min(rw, rh) / max(rw, rh)\n                if sq < 0.82:\n                    continue\n                box = cv2.boxPoints(rrect)\n            else:\n                box = approx.reshape(4, 2).astype(np.float32)\n                (cx, cy), (rw, rh), _ = cv2.minAreaRect(c)\n                sq = min(rw, rh) / max(rw, rh) if rw and rh else 0\n                if sq < 0.82:\n                    continue\n            quad = _order_points(box)\n            sides = _side_lengths(quad)\n            side = float(np.mean(sides))\n            # reject wildly non-square quads\n            if min(sides) / max(sides) < 0.75:\n                continue\n            finders = _count_finder_patterns(gray, quad)\n            if finders < 2:\n                continue\n            # score: finder count first, then size (QRs on certs are big)\n            score = finders * 10 + (side / max(h, w))\n            if score > best_score:\n                best_score = score\n                best = (quad, finders, side)\n    if best is not None:\n        quad, finders, side = best\n        return QRRegion(quad=quad, method=\"contour\", side_px=side,\n                        score=min(1.0, finders / 3.0))\n    return None\n\n\n# ---------------------------------------------------------------------------\n# Detector 3b: finder-pattern locator (works WITHOUT a black frame)\n# ---------------------------------------------------------------------------\ndef _chain_depth(i: int, hier: np.ndarray, need: int = 2) -> bool:\n    \"\"\"True if contour `i` encloses >=`need` levels of descendants. A finder\n    pattern's outer black ring contains a white gap then the black centre, so\n    its first-child chain reaches depth 2 (the gap can merge with background,\n    which is why depth 4 is never seen on real scans).\"\"\"\n    cur = hier[i][2]\n    d = 0\n    while cur != -1:\n        d += 1\n        if d >= need:\n            return True\n        cur = hier[cur][2]\n    return False\n\n\ndef _find_finder_centers(gray: np.ndarray) -> list[tuple[np.ndarray, float]]:\n    \"\"\"Locate QR finder patterns (the 3 corner 'nested squares'). Returns a\n    list of (centre[x,y], outer-ring size px). Works for framed OR\n    frameless/plain QRs, since finders are always present.\"\"\"\n    h, w = gray.shape[:2]\n    found: list[tuple[np.ndarray, float]] = []\n\n    for thresh in (80, 110, 140, 170, 200):\n        _, bw = cv2.threshold(gray, thresh, 255, cv2.THRESH_BINARY_INV)\n        # No morphology here: closing merges the thin rings of tiny\n        # (sub-100px) QRs and destroys the nesting we are looking for.\n        contours, hier = cv2.findContours(bw, cv2.RETR_TREE,\n                                          cv2.CHAIN_APPROX_SIMPLE)\n        if hier is None:\n            continue\n        hier = hier[0]\n        cands = []\n        for i, c in enumerate(contours):\n            x, y, ww, hh = cv2.boundingRect(c)\n            if ww == 0 or hh == 0:\n                continue\n            side_min = float(min(h, w))\n            if not (side_min * 0.012 < ww < side_min * 0.35 and\n                    side_min * 0.012 < hh < side_min * 0.35):\n                continue\n            if not (0.65 < ww / float(hh) < 1.5):\n                continue\n            area = cv2.contourArea(c)\n            if area / float(ww * hh) < 0.5:\n                continue\n            if not _chain_depth(i, hier, need=2):\n                continue\n            cands.append((x + ww / 2.0, y + hh / 2.0, (ww + hh) / 2.0))\n        # within each threshold, collapse nested/contained candidates to the\n        # OUTERMOST ring (largest), since the centre dot also appears.\n        cands.sort(key=lambda t: -t[2])\n        for cx, cy, size in cands:\n            if not any(np.hypot(cx - fc[0][0], cy - fc[0][1]) < fc[1] * 0.7\n                       for fc in found):\n                found.append((np.array([cx, cy], dtype=np.float32),\n                              float(size)))\n    return found\n\n\ndef _quad_from_finders(finders: list[tuple[np.ndarray, float]]):\n    \"\"\"Given >=3 finder centres, build the QR outer quad (incl. quiet zone).\"\"\"\n    n = len(finders)\n    best = None\n    best_err = 1e9\n    for a in range(n):\n        for b in range(n):\n            if b == a:\n                continue\n            for c in range(n):\n                if c == a or c == b:\n                    continue\n                A, rA = finders[a]   # candidate elbow (TL)\n                B, _ = finders[b]    # candidate TR\n                C, _ = finders[c]    # candidate BL\n                vx = B - A\n                vy = C - A\n                lx = float(np.linalg.norm(vx))\n                ly = float(np.linalg.norm(vy))\n                if lx < 5 or ly < 5:\n                    continue\n                if min(lx, ly) / max(lx, ly) < 0.6:\n                    continue\n                cosang = float(np.dot(vx, vy) / (lx * ly))\n                if abs(cosang) > 0.45:   # want ~90 deg\n                    continue\n                ux = vx / lx\n                uy = vy / ly\n                r = float(np.mean([fr[1] for fr in (finders[a], finders[b],\n                                                    finders[c])]))\n                module = r / 7.0\n                ext = 7.5 * module      # ring centre (3.5 mod) + quiet (4 mod)\n                TL = A - ext * (ux + uy)\n                TR = TL + ux * (lx + 2 * ext)\n                BL = TL + uy * (ly + 2 * ext)\n                BR = TL + ux * (lx + 2 * ext) + uy * (ly + 2 * ext)\n                err = abs(cosang) + abs(lx - ly) / max(lx, ly)\n                if err < best_err:\n                    best_err = err\n                    best = _order_points(np.array([TL, TR, BR, BL],\n                                                  dtype=np.float32))\n    return best\n\n\ndef _detect_by_finders(img: np.ndarray):\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img\n    h, w = gray.shape[:2]\n    finders = _find_finder_centers(gray)\n    if len(finders) < 3:\n        return None\n    quad = _quad_from_finders(finders)\n    if quad is None:\n        return None\n    sides = _side_lengths(quad)\n    side = float(np.mean(sides))\n    if min(sides) < max(8.0, 0.04 * min(h, w)):\n        return None\n    if min(sides) / max(sides) < 0.7:\n        return None\n    return QRRegion(quad=quad, method=\"finders\", side_px=side, score=0.95)\n\n\n# ---------------------------------------------------------------------------\n# Public API\n# ---------------------------------------------------------------------------\ndef detect_qr(img: np.ndarray, model_dir: str = \"models/wechat\",\n              use_cnn: bool = True):\n    \"\"\"\n    Find the old QR on a certificate.\n\n    Returns QRRegion or None.\n    \"\"\"\n    if use_cnn:\n        quad = _detect_wechat(img, model_dir)\n        if quad is not None:\n            sides = _side_lengths(quad)\n            return QRRegion(quad=quad, method=\"wechat-cnn\",\n                            side_px=float(np.mean(sides)), score=0.99)\n\n    quad = _detect_opencv(img)\n    if quad is not None:\n        sides = _side_lengths(quad)\n        return QRRegion(quad=quad, method=\"opencv\",\n                        side_px=float(np.mean(sides)), score=0.9)\n\n    reg = _detect_by_contours(img)\n    if reg is not None:\n        return reg\n\n    # last resort: locate the three finder patterns (no frame needed)\n    return _detect_by_finders(img)\n\n\n# ---------------------------------------------------------------------------\n# Fallback placement (used when a QR cannot be located on one cert in a batch\n# of SAME-TEMPLATE certificates -- the QR is always in the same spot).\n# ---------------------------------------------------------------------------\ndef normalize_region(region: QRRegion, h: int, w: int) -> np.ndarray:\n    \"\"\"Quad coordinates as fractions of image width/height.\"\"\"\n    q = region.quad.astype(np.float32).copy()\n    q[:, 0] /= float(w)\n    q[:, 1] /= float(h)\n    return q\n\n\ndef fallback_region_from_regions(img: np.ndarray,\n                                 regions: list) -> QRRegion | None:\n    \"\"\"Place the new QR using the MEDIAN normalized position of the QRs that\n    WERE detected on other certificates in the batch. Works because all\n    certificates share one template. `regions` = list of (QRRegion, h, w).\n\n    Skips images whose aspect ratio differs from the certificates (e.g. a wide\n    16:9 screenshot that isn't a certificate) so a QR is never pasted onto a\n    non-certificate image.\"\"\"\n    h, w = img.shape[:2]\n    tmpl_aspect = float(np.median([ww / max(1, hh) for (_r, hh, ww) in regions]))\n    if abs((w / max(1, h)) - tmpl_aspect) / tmpl_aspect > 0.15:\n        return None\n    norms = [normalize_region(r, hh, ww) for (r, hh, ww) in regions]\n    if not norms:\n        return None\n    med = np.median(np.stack(norms, axis=0), axis=0).astype(np.float32)\n    quad = med.copy()\n    quad[:, 0] *= float(w)\n    quad[:, 1] *= float(h)\n    quad = _order_points(quad)\n    sides = _side_lengths(quad)\n    return QRRegion(quad=quad, method=\"batch-template-fallback\",\n                    side_px=float(np.mean(sides)), score=0.5)\n\n\ndef fallback_region_default(img: np.ndarray) -> QRRegion | None:\n    \"\"\"Last-resort placement for the TGMFC certificate template (QR sits in\n    the upper-left, under the red certificate number). Coordinates are\n    fractions of (width, height). Returns None for images whose aspect ratio\n    is a wide landscape (e.g. a 16:9 screenshot) rather than a certificate\n    page, so a QR is never pasted onto the wrong image.\"\"\"\n    h, w = img.shape[:2]\n    if w / max(1, h) >= 1.45:      # wide landscape screenshot, not a cert page\n        return None\n    n = np.array([[0.095, 0.345], [0.205, 0.345],\n                  [0.205, 0.515], [0.095, 0.515]], dtype=np.float32)\n    quad = n.copy()\n    quad[:, 0] *= float(w)\n    quad[:, 1] *= float(h)\n    quad = _order_points(quad)\n    sides = _side_lengths(quad)\n    return QRRegion(quad=quad, method=\"template-default-fallback\",\n                    side_px=float(np.mean(sides)), score=0.3)\n\n\ndef draw_debug(img: np.ndarray, region: QRRegion) -> np.ndarray:\n    \"\"\"Overlay the detected quad on an image (for review of failures).\"\"\"\n    out = img.copy()\n    q = region.quad.astype(np.int32)\n    cv2.polylines(out, [q], True, (0, 0, 255), 3)\n    cv2.putText(out, f\"{region.method} {region.side_px:.0f}px\",\n                (q[0][0], max(15, q[0][1] - 8)),\n                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)\n    return out\n",
  "qr_generator.py": "\"\"\"\nqr_generator.py\n---------------\nGenerate a fresh, crisp QR code image for a given payload (the URL/text that\na phone will read when scanning the certificate).\n\nThe QR is rendered at HIGH pixel resolution so that when it is perspective-\nwarped down onto the small QR region of the certificate, every module stays\nsharp and black/white (which is what makes it scan reliably).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport io\n\nimport cv2\nimport numpy as np\nimport qrcode\nfrom qrcode.constants import (\n    ERROR_CORRECT_L, ERROR_CORRECT_M, ERROR_CORRECT_Q, ERROR_CORRECT_H,\n)\n\n_EC = {\n    \"L\": ERROR_CORRECT_L,   # ~7%\n    \"M\": ERROR_CORRECT_M,   # ~15%\n    \"Q\": ERROR_CORRECT_Q,   # ~25%\n    \"H\": ERROR_CORRECT_H,   # ~30%\n}\n\n\ndef build_qr(payload: str,\n             box_size: int = 20,\n             border: int = 4,\n             error_correction: str = \"M\",\n             fg: tuple[int, int, int] = (0, 0, 0),\n             bg: tuple[int, int, int] = (255, 255, 255)) -> np.ndarray:\n    \"\"\"\n    Return a square BGR uint8 image (white background) containing the QR.\n\n    box_size: pixels per module at render time (large = crisp after warping)\n    border:   quiet-zone width in modules (standard 4)\n    \"\"\"\n    qr = qrcode.QRCode(\n        version=None,                       # auto-size to payload\n        error_correction=_EC[error_correction.upper()],\n        box_size=box_size,\n        border=border,\n    )\n    qr.add_data(payload)\n    qr.make(fit=True)\n\n    img = qr.make_image(fill_color=fg, back_color=bg).convert(\"RGB\")\n    arr = np.array(img)[:, :, ::-1].copy()  # RGB -> BGR\n    return arr\n\n\ndef qr_png_bytes(payload: str, **kwargs) -> bytes:\n    \"\"\"Render the QR and return PNG bytes (e.g. for saving a standalone file).\"\"\"\n    arr = build_qr(payload, **kwargs)\n    ok, buf = cv2.imencode(\".png\", arr)\n    return buf.tobytes() if ok else b\"\"\n",
  "qr_art.py": "\"\"\"\nqr_art.py\n---------\nOPTIONAL AI \"artistic QR\" generation using the Hugging Face ControlNet model\n**QR Code Monster** (`monster-labs/control_v1p_sd15_qrcode_monster`, v2) with a\nStable Diffusion 1.5 base.\n\nThis is the popular \"image/text -> QR code\" diffusion model family (QR Code\nMonster, IllusionDiffusion, etc.): a normal QR matrix is fed to ControlNet as\nthe conditioning image together with a text prompt, and the model produces a\ncreative QR in which art is blended into the modules.\n\nImportant facts (from the model cards):\n  * Not every generation scans -- typical scannability is 50-80%, so the\n    official workflow is \"generate several seeds and keep a readable one\".\n  * High `controlnet_conditioning_scale` (1.2-1.8) => more readable; low =>\n    more artistic. We default high because the QR must work on a certificate.\n  * These codes are usually colourful/textured -- great for marketing, but for\n    an official certificate the plain black-and-white `qr_generator.build_qr`\n    (100% reliable) is recommended.\n\nTherefore `build_artistic_qr()`:\n  1. renders the EXACT, verified QR matrix (error-correction H),\n  2. runs the ControlNet pipeline over several seeds,\n  3. DECODES every candidate and keeps one that reads back as `payload`,\n  4. returns None if none scan within `max_tries` (caller falls back to the\n     deterministic QR, so a batch never breaks).\n\nRuns on a CUDA GPU (Kaggle free T4). torch/diffusers are imported lazily, so\nimporting this module on a CPU-only box is harmless.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport cv2\nimport numpy as np\n\nfrom .qr_generator import build_qr\n\n# A prompt that pushes the model toward a clean, document-friendly result while\n# still using the AI generator. Override for creative/artistic codes.\nCLEAN_PROMPT = (\n    \"a clean flat black and white QR code on a plain white background, \"\n    \"crisp high contrast modules, minimalist, sharp edges, no decoration\"\n)\nCLEAN_NEGATIVE = (\n    \"colorful, painting, landscape, portrait, photo, texture, blurry, \"\n    \"watermark, text, logo, distorted, low contrast, cluttered background\"\n)\n\n_ART_PIPE = None\n_ART_DEVICE = None\n\n\ndef _device() -> str:\n    global _ART_DEVICE\n    if _ART_DEVICE is not None:\n        return _ART_DEVICE\n    try:\n        import torch\n        _ART_DEVICE = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    except Exception:\n        _ART_DEVICE = \"cpu\"\n    return _ART_DEVICE\n\n\ndef art_available() -> bool:\n    try:\n        import torch  # noqa: F401\n        import diffusers  # noqa: F401\n        return True\n    except Exception:\n        return False\n\n\ndef _condition_image(payload: str, module_px: int = 16,\n                     gray_bg: bool = False) -> np.ndarray:\n    \"\"\"Render the exact QR matrix as the ControlNet conditioning image.\n\n    QR Monster expects ~16px modules. v2 blends better on a gray (#808080)\n    background; for maximum readability we use white (gray_bg=False).\n    \"\"\"\n    qr = build_qr(payload, box_size=module_px, border=0,\n                  error_correction=\"H\")\n    if gray_bg:\n        # QR has white bg; replace outer non-module area with gray is complex,\n        # so we tint the whole background gray while keeping modules black.\n        gray = cv2.cvtColor(qr, cv2.COLOR_BGR2GRAY)\n        out = np.full_like(qr, 128)\n        dark = gray < 128\n        out[dark] = (0, 0, 0)\n        return out\n    return qr\n\n\ndef load_art_pipeline(\n        controlnet_id: str = \"monster-labs/control_v1p_sd15_qrcode_monster\",\n        base_model: str = \"stable-diffusion-v1-5/stable-diffusion-v1-5\"):\n    \"\"\"Lazily load and cache the ControlNet QR pipeline (fp16 on GPU).\"\"\"\n    global _ART_PIPE\n    if _ART_PIPE is not None:\n        return _ART_PIPE\n    import torch\n    from diffusers import (ControlNetModel,\n                           StableDiffusionControlNetPipeline,\n                           DPMSolverMultistepScheduler)\n\n    dtype = torch.float16 if _device() == \"cuda\" else torch.float32\n    controlnet = ControlNetModel.from_pretrained(\n        controlnet_id, torch_dtype=dtype)\n    pipe = StableDiffusionControlNetPipeline.from_pretrained(\n        base_model, controlnet=controlnet, torch_dtype=dtype,\n        safety_checker=None)\n    pipe.scheduler = DPMSolverMultistepScheduler.from_config(\n        pipe.scheduler.config)\n    if _device() == \"cuda\":\n        try:\n            pipe.enable_xformers_memory_efficient_attention()\n        except Exception:\n            pass\n        try:\n            pipe.enable_model_cpu_offload()\n        except Exception:\n            pipe.to(\"cuda\")\n    else:\n        pipe.to(\"cpu\")\n    _ART_PIPE = pipe\n    return pipe\n\n\ndef _decode_qr(img_bgr: np.ndarray, want: str) -> bool:\n    \"\"\"True if img contains a QR that decodes exactly to `want`.\"\"\"\n    from .pipeline import _decode_any_qr  # reuse strong verifier\n    h, w = img_bgr.shape[:2]\n    for scale in (1, 2):\n        im = img_bgr if scale == 1 else cv2.resize(\n            img_bgr, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)\n        got = _decode_any_qr(im)\n        if got and got.strip() == want.strip():\n            return True\n    return False\n\n\ndef build_artistic_qr(payload: str,\n                      prompt: str = CLEAN_PROMPT,\n                      negative_prompt: str = CLEAN_NEGATIVE,\n                      size: int = 768,\n                      controlnet_scale: float = 1.5,\n                      guidance_scale: float = 7.5,\n                      steps: int = 30,\n                      max_tries: int = 6,\n                      seed0: int = 1000,\n                      gray_bg: bool = False):\n    \"\"\"\n    Generate an AI (ControlNet) artistic QR for `payload`.\n\n    Returns a square BGR uint8 image that DECODES to `payload`, or None if no\n    seed produced a scannable code within `max_tries` (caller then falls back\n    to the deterministic QR).\n    \"\"\"\n    if not art_available() or _device() != \"cuda\":\n        return None\n    try:\n        import torch\n        from PIL import Image\n        pipe = load_art_pipeline()\n    except Exception as e:  # model download/OOM\n        print(f\"[qr_art] pipeline unavailable ({e}); using standard QR\")\n        return None\n\n    cond = _condition_image(payload, module_px=16, gray_bg=gray_bg)\n    cond_pil = Image.fromarray(cv2.cvtColor(cond, cv2.COLOR_BGR2RGB))\n    # make conditioning square at requested size\n    cond_pil = cond_pil.resize((size, size), Image.NEAREST)\n\n    for i in range(max_tries):\n        seed = seed0 + i\n        g = torch.Generator(device=\"cpu\").manual_seed(seed)\n        try:\n            result = pipe(\n                prompt=prompt, negative_prompt=negative_prompt,\n                image=cond_pil, width=size, height=size,\n                num_inference_steps=steps,\n                guidance_scale=guidance_scale,\n                controlnet_conditioning_scale=controlnet_scale,\n                generator=g,\n            ).images[0]\n        except Exception as e:\n            print(f\"[qr_art] generation failed (seed {seed}): {e}\")\n            continue\n        bgr = cv2.cvtColor(np.array(result), cv2.COLOR_RGB2BGR)\n        if _decode_qr(bgr, payload):\n            print(f\"[qr_art] scannable artistic QR at seed {seed} \"\n                  f\"(try {i+1}/{max_tries})\")\n            return bgr\n        print(f\"[qr_art] seed {seed} did not scan; retrying...\")\n    print(\"[qr_art] no scannable variant found; falling back to standard QR\")\n    return None\n",
  "replacer.py": "\"\"\"\nreplacer.py\n-----------\nPaste the freshly generated QR onto the certificate at the EXACT location,\nsize and rotation of the old QR.\n\n- Uses a perspective warp so rotated/skewed old QRs are matched exactly.\n- Uses NEAREST-neighbour rendering so QR modules stay hard-edged (critical for\n  scanning) -- no anti-aliased grey fuzz.\n- `mode`:\n    \"full_replace\" -> new QR (with white quiet zone) covers the entire old QR\n                      box including its black frame. Cleanest, most reliable.\n    \"keep_frame\"   -> the old black border is preserved; only the inner QR\n                      area is replaced (the new QR is shrunk inside the frame).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport cv2\nimport numpy as np\n\nfrom .detector import QRRegion, _order_points\n\n\ndef _scale_quad(quad: np.ndarray, scale: float) -> np.ndarray:\n    \"\"\"Scale a quad about its centre.\"\"\"\n    c = quad.mean(axis=0)\n    return c + (quad - c) * scale\n\n\ndef _warp_qr_to_cert(cert: np.ndarray, qr_img: np.ndarray,\n                     dst_quad: np.ndarray, supersample: int = 4) -> np.ndarray:\n    \"\"\"Perspective-warp qr_img onto cert at dst_quad (TL,TR,BR,BL).\n\n    The QR is first re-rendered at ~`supersample`x the destination size, then\n    warped with bilinear sampling (a good approximation of area-averaged\n    downscaling) and finally re-binarised to pure black/white inside the box.\n    This keeps every module hard-edged and scannable even when the on-cert\n    QR is very small (avoids the aliasing of a single huge->tiny NEAREST warp).\n    \"\"\"\n    h, w = cert.shape[:2]\n    ss = int(supersample)\n    side = float(np.mean([\n        np.linalg.norm(dst_quad[1] - dst_quad[0]),\n        np.linalg.norm(dst_quad[2] - dst_quad[1]),\n        np.linalg.norm(dst_quad[3] - dst_quad[2]),\n        np.linalg.norm(dst_quad[0] - dst_quad[3]),\n    ]))\n    target = max(8, int(round(side * ss)))\n    if qr_img.shape[0] != target:\n        interp = cv2.INTER_AREA if qr_img.shape[0] > target else cv2.INTER_NEAREST\n        qr_img = cv2.resize(qr_img, (target, target), interpolation=interp)\n\n    qh, qw = qr_img.shape[:2]\n    # Warp onto an ss-times-oversized overlay at the ss-scaled quad, then pull\n    # back down to cert resolution with INTER_AREA -> a proper area-averaged\n    # anti-aliased downscale of the perspective warp (keeps modules legible at\n    # small on-cert sizes).\n    big_W, big_H = w * ss, h * ss\n    big_quad = dst_quad.astype(np.float32) * ss\n    src = np.array([[0, 0], [qw - 1, 0], [qw - 1, qh - 1], [0, qh - 1]],\n                   dtype=np.float32)\n    M = cv2.getPerspectiveTransform(src, big_quad)\n    warped_big = cv2.warpPerspective(\n        qr_img, M, (big_W, big_H),\n        flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT,\n        borderValue=(255, 255, 255))\n    warped = cv2.resize(warped_big, (w, h), interpolation=cv2.INTER_AREA)\n\n    # mask of the QR box\n    mask = np.zeros((h, w), dtype=np.uint8)\n    cv2.fillConvexPoly(mask, dst_quad.astype(np.int32), 255)\n    mask = cv2.erode(mask, np.ones((3, 3), np.uint8), iterations=1)\n\n    # re-binarise the warped patch to pure black / white (hard module edges)\n    warped_gray = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)\n    _, binary = cv2.threshold(warped_gray, 160, 255, cv2.THRESH_BINARY)\n    binary_bgr = cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)\n\n    out = cert.copy()\n    m3 = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR) > 0\n    out[m3] = binary_bgr[m3]\n    return out\n\n\ndef replace_qr(cert: np.ndarray, region: QRRegion, qr_img: np.ndarray,\n               mode: str = \"full_replace\",\n               inner_margin_frac: float = 0.10,\n               cover_scale: float = 1.08) -> np.ndarray:\n    \"\"\"\n    Return a new certificate image with the new QR in place of the old one.\n\n    Everything outside the QR quad is left byte-for-byte untouched.\n\n    cover_scale (full_replace only): the new QR's white box is pasted over a\n    quad scaled outward by this factor so it reliably covers the ENTIRE old\n    black frame plus any dark halo / AI-inpaint remnant at the frame edge.\n    1.0 = exact old-quad footprint; ~1.08 fully hides the old frame (the white\n    background looks natural against the certificate and keeps a valid quiet\n    zone).\n    \"\"\"\n    quad = _order_points(region.quad)\n\n    if mode == \"keep_frame\":\n        # Replace only the inside of the black frame.\n        dst = _scale_quad(quad, 1.0 - 2 * inner_margin_frac)\n        # Give the pasted QR its own thin white border so modules never touch\n        # the black frame (keeps quiet-zone scanning rules).\n        qr = _add_white_border(qr_img, frac=0.06)\n    else:\n        # Full replace: scale the footprint outward a little so the new QR's\n        # white background fully hides the old frame + any edge halo.\n        dst = _scale_quad(quad, float(cover_scale))\n        qr = qr_img\n\n    return _warp_qr_to_cert(cert, qr, dst)\n\n\ndef _add_white_border(img: np.ndarray, frac: float = 0.08) -> np.ndarray:\n    h, w = img.shape[:2]\n    b = int(round(min(h, w) * frac))\n    return cv2.copyMakeBorder(img, b, b, b, b, cv2.BORDER_CONSTANT,\n                              value=(255, 255, 255))\n",
  "payload.py": "\"\"\"\npayload.py\n----------\nDecide what text/URL each certificate's NEW QR code encodes.\n\nSupported modes (set in config / notebook):\n\n  url   : base_url + slug + extension      -> e.g.\n          https://<your-username>.github.io/certificates/031-2024-25.png\n          (this is the recommended mode: host the OUTPUT images online --\n           e.g. GitHub Pages / your server / cloud storage -- and scanning the\n           QR opens that certificate in the phone's browser)\n\n  csv   : read a spreadsheet with columns: filename, qr_data\n          (per-certificate exact URL or text)\n\n  text  : fixed prefix + slug\n\n  copy  : copy whatever the old QR decoded to (requires the old QR to be\n          readable; falls back to url mode when it isn't)\n\n`slug` defaults to the certificate file's stem (filename without extension).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport csv\nimport os\n\n\ndef slug_from_filename(path: str) -> str:\n    return os.path.splitext(os.path.basename(path))[0]\n\n\ndef safe_slug(path: str) -> str:\n    \"\"\"URL/filename-safe stem. Replaces spaces, parentheses and other chars that\n    break URLs (e.g. 'image (1)' -> 'image_1') so the QR link and the saved\n    output file always match and resolve.\"\"\"\n    import re\n    stem = slug_from_filename(path)\n    stem = re.sub(r\"[^A-Za-z0-9._-]+\", \"_\", stem)\n    stem = re.sub(r\"_+\", \"_\", stem).strip(\"._-\")\n    return stem or \"certificate\"\n\n\ndef build_payload(path: str,\n                  mode: str = \"url\",\n                  base_url: str = \"\",\n                  extension: str = \".png\",\n                  text_prefix: str = \"\",\n                  csv_map: dict[str, str] | None = None,\n                  old_data: str | None = None) -> str:\n    slug = safe_slug(path)\n\n    if mode == \"csv\":\n        if csv_map and (os.path.basename(path) in csv_map or slug in csv_map):\n            return csv_map.get(os.path.basename(path)) or csv_map[slug]\n        # fall back to URL if the cert is missing from the spreadsheet\n        mode = \"url\"\n\n    if mode == \"copy\" and old_data:\n        return old_data\n\n    if mode == \"text\":\n        return f\"{text_prefix}{slug}\"\n\n    # default: url\n    base = base_url.rstrip(\"/\") + \"/\"\n    return f\"{base}{slug}{extension}\"\n\n\ndef load_csv_map(csv_path: str) -> dict[str, str]:\n    \"\"\"Read filename->qr_data mapping from a CSV (headers: filename,qr_data).\"\"\"\n    mapping: dict[str, str] = {}\n    with open(csv_path, newline=\"\", encoding=\"utf-8-sig\") as f:\n        for row in csv.DictReader(f):\n            fn = (row.get(\"filename\") or row.get(\"file\") or \"\").strip()\n            data = (row.get(\"qr_data\") or row.get(\"url\") or\n                    row.get(\"payload\") or \"\").strip()\n            if fn and data:\n                mapping[fn] = data\n                mapping[os.path.splitext(fn)[0]] = data\n    return mapping\n",
  "ai_extract.py": "\"\"\"\nai_extract.py\n-------------\nRead the candidate's details DIRECTLY FROM THE CERTIFICATE IMAGE using an AI\nvision-language model from Hugging Face (no OCR, no CSV, no hand-written rules).\n\nModel (runs on a free Kaggle T4 GPU; no num2words / tesseract needed):\n    Qwen/Qwen2-VL-2B-Instruct   (default) -- strong document/JSON reader\n    Qwen/Qwen2.5-VL-3B-Instruct (optional) -- slightly larger, also fine on T4\n\nThe model is shown the certificate and asked to return ONLY JSON:\n    name, parent, relation, aadhar, course, date_from, date_to, institute.\n\nFailures are NEVER silent: the returned dict carries an \"_error\" / \"_raw_ai\"\nkey so the notebook log / manifest show exactly what happened.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport re\nimport os\nimport gc\nimport importlib\nimport subprocess\nimport sys\n\n# Reduce CUDA fragmentation / OOM on the 16 GB T4 when a model is loaded more\n# than once in a session (must be set before torch first allocates).\nos.environ.setdefault(\"PYTORCH_CUDA_ALLOC_CONF\", \"expandable_segments:True\")\n\n_PROMPT = (\n    \"You are reading an Indian training-completion certificate IMAGE. \"\n    \"Look carefully at the whole image and extract these SIX fields. \"\n    \"Return ONLY a JSON object (no prose, no markdown, no code fence) with \"\n    \"exactly these keys:\\n\"\n    '{\"name\": \"<the candidate OWN name only: the words printed after Mr./Ms. '\n    'and BEFORE the token S/o or D/o or W/o. Do NOT include S/o, D/o, W/o, or '\n    'any name after them.>\", '\n    '\"parent\": \"<the father/mother name ONLY: the person name printed '\n    'immediately AFTER S/o (son of -> father), D/o (daughter of -> father) or '\n    'W/o (wife of -> husband), up to the next comma or bracket.>\", '\n    '\"relation\": \"<exactly S/o if the text says S/o, D/o if it says D/o, or '\n    'W/o if it says W/o>\", '\n    '\"aadhar\": \"<the candidate 12-digit Aadhaar number, digits only>\", '\n    '\"course\": \"<the full course/training name the candidate completed, e.g. '\n    'the text after Training in / Trainings in / Course>\", '\n    '\"date_from\": \"<course START date formatted dd/mm/yyyy>\", '\n    '\"date_to\": \"<course END date formatted dd/mm/yyyy>\", '\n    '\"institute\": \"<the COMPLETE institute name printed anywhere on the '\n    'certificate, written out in full exactly as shown (spell out every word; '\n    'do NOT shorten to initials - e.g. write Falih Consultancy Services (FCS) '\n    'Training Institute, not just FCS). Read the org name LETTER BY LETTER: '\n    'the printed letters l, i, s and h look alike - Falih is F-a-l-i-h (NOT '\n    'Falis, NOT Fahis). It differs on every certificate; never '\n    'assume or default to a name.>\"}\\n'\n    \"CRITICAL: 'S/o' = Son of (FATHER), 'D/o' = Daughter of (FATHER), 'W/o' = \"\n    \"Wife of (HUSBAND). These give the parent name. 'R/o' = Resident of (the \"\n    \"HOME ADDRESS / place, e.g. Rajendra Nager) \u2014 R/o is NOT a person and must \"\n    \"NEVER be put in name or parent. Read from the image only; do not invent; \"\n    \"use \\\"\\\" for anything unreadable; dates as dd/mm/yyyy.\"\n)\n\n# Small helper packages (NOT torch/transformers). If one is missing in the\n# kernel we self-heal by pip-installing it at runtime.\n_REQUIRED = [\n    (\"PIL\", \"pillow\"),\n    (\"accelerate\", \"accelerate\"),\n    (\"torchvision\", \"torchvision\"),\n]\n\n_MODEL = None\n_PROC = None\n_DEVICE = None\n_MODEL_ID = None\n\n\ndef _pip_install(pkg: str) -> bool:\n    try:\n        subprocess.check_call(\n            [sys.executable, \"-m\", \"pip\", \"install\", \"-q\", pkg])\n        return True\n    except Exception as e:\n        print(f\"[ai_extract] pip install {pkg} failed: {e}\")\n        return False\n\n\ndef _ensure_deps():\n    # Only self-heal where torch/transformers already exist (the GPU notebook).\n    try:\n        importlib.import_module(\"torch\")\n        importlib.import_module(\"transformers\")\n    except Exception:\n        return\n    for import_name, pip_name in _REQUIRED:\n        try:\n            importlib.import_module(import_name)\n        except Exception:\n            print(f\"[ai_extract] installing missing dependency '{pip_name}' ...\")\n            if _pip_install(pip_name):\n                try:\n                    importlib.import_module(import_name)\n                except Exception:\n                    pass\n\n\ndef device() -> str:\n    global _DEVICE\n    if _DEVICE is not None:\n        return _DEVICE\n    try:\n        import torch\n        _DEVICE = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    except Exception:\n        _DEVICE = \"cpu\"\n    return _DEVICE\n\n\ndef runtime_status() -> str:\n    try:\n        import torch  # noqa\n    except Exception as e:\n        return f\"torch NOT importable: {e}\"\n    try:\n        import transformers\n        tv = transformers.__version__\n    except Exception as e:\n        return f\"transformers NOT importable: {e}\"\n    dev = device()\n    gpu = \"\"\n    if dev == \"cuda\":\n        try:\n            import torch\n            gpu = f\" | GPU: {torch.cuda.get_device_name(0)}\"\n        except Exception:\n            pass\n    warn = \"\" if dev == \"cuda\" else \" | \u26a0\ufe0f CPU only \u2014 set Accelerator to GPU T4\"\n    return f\"torch ok | transformers {tv} | device={dev}{gpu}{warn}\"\n\n\ndef release_model():\n    \"\"\"Free the cached vision model and its CUDA memory. Call before a fresh\n    module load in the same session (e.g. after the test cell) to avoid OOM.\"\"\"\n    global _MODEL, _PROC, _MODEL_ID\n    _MODEL = None\n    _PROC = None\n    _MODEL_ID = None\n    gc.collect()\n    try:\n        import torch\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n            torch.cuda.ipc_collect()\n    except Exception:\n        pass\n\n\ndef _load(model_id: str):\n    \"\"\"Load (once) and cache the Qwen2-VL model + processor. Frees any stale\n    model first and retries once after emptying the CUDA cache on OOM.\"\"\"\n    global _MODEL, _PROC, _MODEL_ID\n    if _MODEL is not None and _MODEL_ID == model_id:\n        return _MODEL, _PROC\n    release_model()  # ensure no previous model (e.g. from the test cell) lingers\n    import torch\n    from transformers import AutoProcessor\n\n    def _build():\n        ModelCls = None\n        if \"Qwen2.5\" in model_id:\n            try:\n                from transformers import Qwen2_5_VLForConditionalGeneration as ModelCls\n            except Exception:\n                ModelCls = None\n        if ModelCls is None:\n            try:\n                from transformers import Qwen2VLForConditionalGeneration as ModelCls\n            except Exception:\n                ModelCls = None\n        if ModelCls is None:\n            from transformers import AutoModelForImageTextToText as ModelCls\n        dtype = torch.float16 if device() == \"cuda\" else torch.float32\n        proc = AutoProcessor.from_pretrained(model_id)\n        kwargs = dict(torch_dtype=dtype, low_cpu_mem_usage=True)\n        if device() == \"cuda\":\n            kwargs.update(device_map=\"auto\", offload_folder=\"/kaggle/working/offload\")\n        model = ModelCls.from_pretrained(model_id, **kwargs)\n        if \"device_map\" not in kwargs:\n            model = model.to(device())\n        model.eval()\n        return model, proc\n\n    try:\n        _MODEL, _PROC = _build()\n    except (RuntimeError, MemoryError) as e:\n        if \"out of memory\" in str(e).lower() or \"CUDA\" in str(e):\n            print(\"[ai_extract] GPU out of memory while loading \u2014 freeing cache \"\n                  \"and retrying once...\")\n            release_model()\n            try:\n                _MODEL, _PROC = _build()\n            except Exception as e2:\n                raise RuntimeError(\n                    f\"GPU out of memory even after freeing cache ({e2}). \"\n                    \"Restart the kernel (Run -> Restart) and run section 5 only, \"\n                    \"without running the section 4b test cell first.\") from e2\n        else:\n            raise\n    _MODEL_ID = model_id\n    return _MODEL, _PROC\n\n\ndef _norm_date(v: str) -> str:\n    if not v:\n        return \"\"\n    m = re.search(r\"(\\d{1,2})[^\\d]?(\\d{1,2})[^\\d]?(\\d{4})\", str(v))\n    if not m:\n        return str(v).strip()\n    try:\n        return f\"{int(m.group(1)):02d}/{int(m.group(2)):02d}/{m.group(3)}\"\n    except Exception:\n        return str(v).strip()\n\n\ndef _digits_aadhar(v: str) -> str:\n    if not v:\n        return \"\"\n    d = re.sub(r\"\\D\", \"\", str(v))\n    if len(d) > 12:\n        d = d[-12:]\n    if len(d) == 12:\n        return f\"{d[0:4]} {d[4:8]} {d[8:12]}\"\n    return str(v).strip()\n\n\ndef _clean_person_fields(d: dict) -> dict:\n    \"\"\"Deterministically fix name/parent/relation, regardless of model slips:\n    split at the S/o / D/o / W/o marker and never confuse it with R/o (address).\"\"\"\n    name = str(d.get(\"name\", \"\") or \"\").strip()\n    parent = str(d.get(\"parent\", \"\") or \"\").strip()\n    rel = str(d.get(\"relation\", \"\") or \"\").strip().upper()\n\n    relmap = {\"S\": \"S/o\", \"D\": \"D/o\", \"W\": \"W/o\"}\n    rel_letter = rel[0] if (rel and rel[0] in \"SDW\") else \"\"\n    if not rel_letter:\n        m = re.search(r\"\\b([SDW])\\s*/?\\s*[oO0]\\b\", f\"{name} {parent}\")\n        rel_letter = m.group(1) if m else \"\"\n\n    # If the relation marker (S/o, D/o, W/o) ended up INSIDE the name, split it\n    # and trust the marker found in the name (it directly introduces the parent).\n    m = re.search(r\"^(.*?)\\b([SDW])\\s*/?\\s*[oO0]\\s+(.*)$\", name, flags=re.S)\n    if m:\n        name = m.group(1).strip(\" ,./-\")\n        rel_letter = m.group(2)\n        tail = m.group(3)\n        # parent = up to the next R/o (address), Aadhaar, bracket, comma, newline\n        tail = re.split(r\"\\bR\\s*/\\s*o\\b|\\(?Aadha\", tail, flags=re.I)[0]\n        cand = tail.split(\",\")[0].split(\"(\")[0].strip(\" ,./-\")\n        if cand:\n            parent = cand\n\n    # If parent wrongly contains the R/o (residence/address) marker, cut there.\n    parent = re.split(r\"\\bR\\s*/\\s*o\\b\", parent, flags=re.I)[0].strip(\" ,./-\")\n    # Strip a leaked relation PREFIX from parent only if it is the actual\n    # \"S/o\" / \"D/o\" token (never strip a bare initial like \"D.\" in \"D. Jahangeen\").\n    parent = re.sub(r\"^[SDW]\\s*/\\s*[oO0]\\s*\", \"\", parent, flags=re.I).strip()\n    # Parent shouldn't contain Aadhaar / trailing bracket junk.\n    parent = re.split(r\"\\(?Aadha|\\d{4}\\s*\\d{4}\\s*\\d{4}\", parent)[0].strip(\" ,./-()\")\n\n    # Name must not keep a trailing relation marker either.\n    name = re.split(r\"\\b[SDW]\\s*/?\\s*[oO0]\\b\", name)[0].strip(\" ,./-\")\n    # Strip leading honorifics the model may keep (requires the dot, so real\n    # names like \"Msmena\" are never cut). Handles \"Mr./Ms.\", \"Ms.\", \"Dr.\" etc.\n    name = re.sub(\n        r\"^(?:Mr|Mrs|Ms|Miss|Dr|Prof)\\.\\s*(?:/\\s*(?:Mrs?|Ms|Miss|Dr|Prof)\\.\\s*)*\",\n        \"\", name).strip(\" ,./-\")\n\n    relation = relmap.get(rel_letter, \"\")\n    d[\"name\"] = name\n    d[\"parent\"] = parent\n    d[\"relation\"] = relation\n    return d\n\n\ndef _parse_json(text: str) -> dict:\n    if not text:\n        return {}\n    t = text.strip()\n    t = re.sub(r\"^```(?:json)?\\s*|\\s*```$\", \"\", t, flags=re.I).strip()\n    m = re.search(r\"\\{.*\\}\", t, re.S)\n    if not m:\n        return {}\n    data = None\n    for candidate in (m.group(0), \"{\" + m.group(0).strip(\"{} \") + \"}\"):\n        try:\n            data = json.loads(candidate)\n            break\n        except Exception:\n            data = None\n    if not isinstance(data, dict):\n        return {}\n    rel = str(data.get(\"relation\", \"\") or \"\").strip().upper()\n    rel = rel.replace(\"SON OF\", \"S/O\").replace(\"DAUGHTER OF\", \"D/O\")\n    rel = rel.replace(\"WIFE OF\", \"W/O\")\n    if rel and rel[0] in \"SDW\":\n        rel = rel[0] + \"/O\"\n    out = {\n        \"name\": str(data.get(\"name\", \"\") or \"\").strip(),\n        \"parent\": str(data.get(\"parent\", \"\") or \"\").strip(),\n        \"relation\": rel,\n        \"aadhar\": _digits_aadhar(data.get(\"aadhar\", \"\")),\n        \"course\": str(data.get(\"course\", \"\") or \"\").strip(),\n        \"date_from\": _norm_date(data.get(\"date_from\", \"\")),\n        \"date_to\": _norm_date(data.get(\"date_to\", \"\")),\n        \"institute\": str(data.get(\"institute\", \"\") or \"\").strip(),\n        \"_raw_ai\": text.strip(),\n    }\n    return _clean_person_fields(out)\n\n\ndef _generate(pil, model, proc, prompt: str | None = None,\n              max_new_tokens: int = 512) -> str:\n    import torch\n    messages = [{\"role\": \"user\", \"content\": [\n        {\"type\": \"image\"}, {\"type\": \"text\", \"text\": prompt or _PROMPT}]}]\n    text = proc.apply_chat_template(\n        messages, tokenize=False, add_generation_prompt=True)\n    inputs = proc(text=[text], images=[pil], padding=True,\n                  return_tensors=\"pt\").to(model.device)\n    with torch.no_grad():\n        gen = model.generate(**inputs, max_new_tokens=max_new_tokens,\n                             do_sample=False)\n    trimmed = gen[:, inputs[\"input_ids\"].shape[1]:]\n    return proc.batch_decode(trimmed, skip_special_tokens=True)[0]\n\n\n_VERIFY_PROMPT = (\n    \"Look ONLY at the name of the organization/institute that ISSUED this \"\n    \"certificate (printed in words, usually under or around the logo at the \"\n    \"top, and/or near the signatory at the bottom).\\n\"\n    \"A previous reading reported the institute as: \\\"{got}\\\"\\n\"\n    \"Re-read the printed name CAREFULLY, letter by letter, and fix any \"\n    \"misread letters. Printed 'l', 'i', 's' and 'h' look alike: for example \"\n    \"'Falih' is F-a-l-i-h (NOT Falis, NOT Fahis, NOT Faris). Also write the \"\n    \"name COMPLETE, exactly as printed \u2014 spell out every word; never shorten \"\n    \"to initials.\\n\"\n    \"Return ONLY a JSON object with exactly this key (no prose, no code \"\n    \"fence):\\n\"\n    '{\"institute\": \"<the complete, letter-checked correct institute name, '\n    'or \\\\\"\\\\\" if not readable>\"}'\n)\n\n\ndef _clean_institute(v: str) -> str:\n    v = str(v or \"\").strip().strip(\".,;\\\"'` \")\n    if v.lower() in (\"\", \"n/a\", \"na\", \"null\", \"none\", \"-\"):\n        return \"\"\n    return v\n\n\ndef _parse_verify(text: str) -> str:\n    \"\"\"Extract the corrected institute string from the verify-pass output.\"\"\"\n    if not text:\n        return \"\"\n    t = re.sub(r\"^```(?:json)?\\s*|\\s*```$\", \"\", text.strip(),\n               flags=re.I).strip()\n    if \"{\" in t:  # model answered in JSON (possibly slightly malformed)\n        m = re.search(r\"\\{.*\\}\", t, re.S)\n        blob = m.group(0) if m else t\n        try:\n            obj = json.loads(blob)\n            return _clean_institute(obj.get(\"institute\", \"\"))\n        except Exception:\n            mm = re.search(r'\"institute\"\\s*:\\s*\"([^\"]*)\"', blob)\n            if mm:\n                return _clean_institute(mm.group(1))\n            return \"\"\n    # plain-text answer: prefer the longest line (model may add a short\n    # preamble like \"The institute is:\" before the actual name)\n    best = \"\"\n    for line in t.splitlines():\n        line = line.strip().strip(\"{}\\\"'` \")\n        line = re.sub(r\"^institute\\s*:?\", \"\", line, flags=re.I).strip()\n        line = re.sub(r\"^(the\\s+)?(name|institute)\\s+(is|of)\\s*:?\\s*$\",\n                      \"\", line, flags=re.I).strip()\n        if _clean_institute(line) and len(line.strip()) >= 3 \\\n                and len(line) > len(best):\n            best = line\n    return _clean_institute(best)\n\n\ndef ai_extract_fields(img_bgr,\n                      model_id: str = \"Qwen/Qwen2-VL-2B-Instruct\"\n                      ) -> dict:\n    \"\"\"Run the AI vision model on one certificate (BGR ndarray) and return a\n    fields dict. On any failure returns a dict with '_error' (never raises).\"\"\"\n    try:\n        import numpy as np  # noqa\n        import cv2\n        from PIL import Image\n    except Exception as e:\n        return {\"_error\": f\"deps missing (numpy/Pillow/cv2): {e}\"}\n\n    global _MODEL, _PROC\n    model = proc = None\n    try:\n        _ensure_deps()\n        model, proc = _load(model_id)\n    except Exception as e:\n        name = getattr(e, \"name\", None)\n        m = re.search(r\"No module named ['\\\"]([a-zA-Z0-9_\\-]+)\", str(e))\n        pkg = name or (m.group(1) if m else None)\n        pip_map = {\"PIL\": \"pillow\", \"torchvision\": \"torchvision\",\n                   \"accelerate\": \"accelerate\"}\n        pip = pip_map.get(pkg)\n        if pip:\n            print(f\"[ai_extract] model needs '{pip}'; installing and retrying...\")\n            if _pip_install(pip):\n                _MODEL, _PROC = None, None\n                try:\n                    _ensure_deps()\n                    model, proc = _load(model_id)\n                except Exception as e2:\n                    return {\"_error\": f\"model load failed after installing \"\n                                      f\"{pip}: {e2}\"}\n            else:\n                return {\"_error\": f\"model load failed (missing {pip}): {e}\"}\n        else:\n            return {\"_error\": f\"model load failed: {e}\"}\n\n    rgb = img_bgr[:, :, ::-1]\n    # Up-scale small/phone screenshots a bit so the VLM has more pixels to read.\n    if rgb.shape[1] < 1000:\n        scale = 1000 / rgb.shape[1]\n        rgb = cv2.resize(rgb, None, fx=scale, fy=scale,\n                         interpolation=cv2.INTER_CUBIC)\n    import numpy as np\n    pil = Image.fromarray(np.ascontiguousarray(rgb)).convert(\"RGB\")\n\n    out = \"\"\n    try:\n        out = _generate(pil, model, proc)\n    except Exception as e:\n        try:  # one retry slightly larger\n            bigger = cv2.resize(rgb, None, fx=1.4, fy=1.4,\n                                interpolation=cv2.INTER_CUBIC)\n            pil = Image.fromarray(np.ascontiguousarray(bigger)).convert(\"RGB\")\n            out = _generate(pil, model, proc)\n        except Exception as e2:\n            return {\"_error\": f\"generation failed: {e2}\"}\n\n    fields = _parse_json(out)\n    if not fields:\n        return {\"_error\": \"model output was not JSON\", \"_raw_ai\": out.strip()}\n\n    # ---- Institute spelling check: one extra focused look so names like\n    # 'Falih' are not misread as 'Falis' (l/i/s confusion). Never fatal.\n    got_inst = str(fields.get(\"institute\", \"\") or \"\").strip()\n    if got_inst:\n        try:\n            v_raw = _generate(pil, model, proc,\n                              prompt=_VERIFY_PROMPT.replace(\"{got}\", got_inst),\n                              max_new_tokens=96)\n            fixed = _parse_verify(v_raw)\n            if fixed:\n                fields[\"institute_verified\"] = fixed\n                if fixed.strip().lower() != got_inst.strip().lower():\n                    print(f\"[ai_extract] institute spelling check: \"\n                          f\"'{got_inst}' -> '{fixed}'\")\n                    fields[\"institute\"] = fixed\n            fields[\"_raw_ai\"] = out.strip() + \"\\n\\n[institute re-check] \" \\\n                + v_raw.strip()[:220]\n        except Exception as e:\n            print(f\"[ai_extract] institute re-check skipped ({e})\")\n    return fields\n",
  "verification_page.py": "\"\"\"\nverification_page.py\n--------------------\nBuild a self-contained HTML certificate-verification page for each candidate.\nThe certificate image is EMBEDDED (base64), so uploading the single .html file\nis enough \u2014 when a user scans the QR (which links to this page) they see the\ncertificate image with the candidate's details beneath it.\n\nDetails (read off the certificate by the AI vision model):\n    Name of the candidate, Father/Mother Name, Aadhar No., Course Name,\n    Course Duration (dd/mm/yyyy -> dd/mm/yyyy), Institute.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport base64\nimport html as _html\n\n\ndef _b64_png(img_bgr) -> str:\n    import cv2\n    ok, buf = cv2.imencode(\".png\", img_bgr)\n    return base64.b64encode(buf.tobytes()).decode() if ok else \"\"\n\n\ndef _esc(x) -> str:\n    return _html.escape(str(x if x not in (None, \"\") else \"\u2014\"))\n\n\ndef build_page(fields: dict, img_bgr, cert_filename: str) -> str:\n    name = fields.get(\"name\", \"\")\n    parent = fields.get(\"parent\", \"\")\n    aadhar = fields.get(\"aadhar\", \"\")\n    course = fields.get(\"course\", \"\")\n    dfrom = fields.get(\"date_from\", \"\")\n    dto = fields.get(\"date_to\", \"\")\n    dates = f\"{_esc(dfrom)} &nbsp;to&nbsp; {_esc(dto)}\" if (dfrom or dto) else \"\u2014\"\n\n    img_b64 = _b64_png(img_bgr)\n\n    institute = fields.get(\"institute\", \"\")\n    rows = [\n        (\"Name of the candidate\", _esc(name)),\n        (\"Father/Mother Name\", _esc(parent)),\n        (\"Aadhar No.\", _esc(aadhar)),\n        (\"Course Name\", _esc(course)),\n        (\"Course Duration\", dates),\n        (\"Institute\", _esc(institute)),\n    ]\n    rows_html = \"\\n\".join(\n        f'<tr><th>{label} :</th><td>{val}</td></tr>' for label, val in rows)\n\n    cert_html = f'''\n    <div class=\"cert\">\n      <img alt=\"Certificate for {_esc(name or cert_filename)}\"\n           src=\"data:image/png;base64,{img_b64}\">\n      <div><span class=\"badge\">&#10003; Verified certificate</span></div>\n    </div>'''\n\n    info_html = f'''\n    <h2 class=\"sechd\">Candidate Information</h2>\n    <table>\n{rows_html}\n    </table>'''\n\n    return f\"\"\"<!DOCTYPE html>\n<html lang=\"en\">\n<head>\n<meta charset=\"utf-8\">\n<meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">\n<title>Certificate Verification - {_esc(name or cert_filename)}</title>\n<style>\n  *{{box-sizing:border-box}}\n  body{{font-family:Georgia,'Times New Roman',serif;background:#eef1f5;margin:0;\n       padding:24px 12px;color:#1b1b1b}}\n  .wrap{{max-width:760px;margin:0 auto}}\n  .head{{text-align:center;margin-bottom:16px}}\n  .head .org{{font-size:13px;letter-spacing:2px;color:#0b6b3a;font-weight:bold;\n       text-transform:uppercase}}\n  .head h1{{font-size:22px;margin:6px 0 2px;color:#10345e}}\n  .head .sub{{color:#666;font-size:13px}}\n  .card{{background:#fff;border:1px solid #d9dee6;border-radius:14px;\n       box-shadow:0 10px 30px rgba(0,0,0,.08);padding:22px 24px}}\n  table{{width:100%;border-collapse:collapse;font-size:16px}}\n  th{{text-align:left;width:46%;color:#333;font-weight:normal;padding:10px 8px;\n     border-bottom:1px solid #eef0f3;vertical-align:top}}\n  td{{padding:10px 8px;border-bottom:1px solid #eef0f3;font-weight:bold;\n     color:#111;word-break:break-word}}\n  .fcs{{margin:6px 0 10px;text-align:center;font-weight:bold;letter-spacing:1px;\n       color:#0b6b3a;font-size:14px;text-transform:uppercase}}\n  .sechd{{font-size:16px;color:#10345e;border-bottom:2px solid #e5e9f0;\n       padding-bottom:6px;margin:18px 0 8px}}\n  .cert{{margin-top:16px;text-align:center}}\n  .cert img{{max-width:100%;border:1px solid #d9dee6;border-radius:8px;\n       box-shadow:0 4px 14px rgba(0,0,0,.10)}}\n  .badge{{display:inline-block;background:#e8f6ee;color:#0b6b3a;border:1px solid #bfe6cf;\n       border-radius:999px;padding:5px 14px;font-size:12px;margin-top:8px}}\n  .foot{{text-align:center;color:#8a93a0;font-size:12px;margin-top:16px}}\n</style>\n</head>\n<body>\n<div class=\"wrap\">\n  <div class=\"head\">\n    <div class=\"org\">Government of Telangana &middot; TGMFC</div>\n    <h1>Certificate Verification</h1>\n    <div class=\"sub\">Skill Development Training certificate</div>\n  </div>\n  <div class=\"card\">\n{cert_html}\n{info_html}\n  </div>\n  <div class=\"foot\">Government of Telangana &middot; TGMFC &mdash;\n       Skill Development Training certificate verification</div>\n</div>\n</body>\n</html>\n\"\"\"\n",
  "ai_inpaint.py": "\"\"\"\nai_inpaint.py\n-------------\nAI image-generator step (diffusion INPAINTING) for the QR replacement.\n\nWhy inpainting + compositing instead of pure text-to-image?\n  * A generative model that \"draws\" a QR from a text prompt produces a\n    plausible-looking but INVALID QR (it cannot reproduce the exact\n    error-corrected module grid), and free-form editing can subtly alter\n    names/faces.\n  * Masked inpainting is given the certificate, a MASK limited to the old-QR\n    square, and your natural-language prompt. The model may only regenerate\n    pixels inside the mask, so every other pixel (design, text, face, seals)\n    is provably unchanged.\n  * We then composite the REAL, freshly computed QR (perspective-warped to the\n    exact quad) on top of the inpainted region -> the edit is AI-generated and\n    blended as requested, AND the QR is guaranteed crisp & scannable.\n\nModel (Hugging Face, runs on a Kaggle T4 16GB):\n    diffusers/stable-diffusion-xl-1.0-inpainting-0.1   (default, fp16)\n    stabilityai/stable-diffusion-2-inpainting         (lighter fallback)\n\nEverything is lazy: torch/diffusers are only imported when you actually run\nthe AI step, so the deterministic warp path needs no GPU and no big download.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport os\n\nimport cv2\nimport numpy as np\n\nfrom .replacer import replace_qr\n\n# The exact prompt requested by the user.\nDEFAULT_PROMPT = (\n    \"Generate an image of this certificate replacing only the old QR code \"\n    \"with a new crisp black and white QR code in the same position. \"\n    \"Do not change any design, do not change any information, do not change \"\n    \"the face or anything else. Just replace the old QR code with the new \"\n    \"QR code, keeping the identical certificate layout, colors, and text.\"\n)\nDEFAULT_NEGATIVE = (\n    \"invalid qr code, blurry qr, distorted text, changed name, altered face, \"\n    \"modified design, extra logo, watermark, deformed, low quality, artifacts\"\n)\n\n_PIPE = None\n_DEVICE = None\n\n\n# --------------------------------------------------------------------------- #\n# Model loading\n# --------------------------------------------------------------------------- #\ndef _device() -> str:\n    global _DEVICE\n    if _DEVICE is not None:\n        return _DEVICE\n    try:\n        import torch\n        _DEVICE = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    except Exception:\n        _DEVICE = \"cpu\"\n    return _DEVICE\n\n\ndef load_pipeline(model_id: str = \"diffusers/stable-diffusion-xl-1.0-inpainting-0.1\"):\n    \"\"\"Lazily load (and cache) the HF inpainting pipeline.\"\"\"\n    global _PIPE\n    if _PIPE is not None:\n        return _PIPE\n    import torch\n    from diffusers import AutoPipelineForInpainting\n\n    dtype = torch.float16 if _device() == \"cuda\" else torch.float32\n    kw = dict(torch_dtype=dtype)\n    try:\n        pipe = AutoPipelineForInpainting.from_pretrained(model_id, **kw)\n    except Exception:\n        # some repos need the variant flag only on GPU fp16\n        kw[\"variant\"] = \"fp16\"\n        pipe = AutoPipelineForInpainting.from_pretrained(model_id, **kw)\n\n    if _device() == \"cuda\":\n        try:\n            pipe.enable_xformers_memory_efficient_attention()\n        except Exception:\n            pass\n        try:\n            pipe.enable_model_cpu_offload()\n        except Exception:\n            pipe.to(\"cuda\")\n    else:\n        pipe.to(\"cpu\")\n    pipe.set_progress_bar_config(disable=False)\n    _PIPE = pipe\n    return pipe\n\n\ndef ai_available() -> bool:\n    try:\n        import torch  # noqa: F401\n        import diffusers  # noqa: F401\n        return True\n    except Exception:\n        return False\n\n\n# --------------------------------------------------------------------------- #\n# Geometry helpers\n# --------------------------------------------------------------------------- #\ndef _mask_from_quad(shape_hw: tuple[int, int], quad: np.ndarray,\n                    dilate_px: int = 6) -> np.ndarray:\n    h, w = shape_hw\n    mask = np.zeros((h, w), np.uint8)\n    cv2.fillConvexPoly(mask, quad.astype(np.int32), 255)\n    if dilate_px:\n        k = cv2.getStructuringElement(\n            cv2.MORPH_RECT, (2 * dilate_px + 1, 2 * dilate_px + 1))\n        mask = cv2.dilate(mask, k)\n    return mask\n\n\ndef _crop_bbox(quad: np.ndarray, h: int, w: int, margin: int):\n    x0, y0 = quad.min(axis=0)\n    x1, y1 = quad.max(axis=0)\n    x0 = int(max(0, x0 - margin)); y0 = int(max(0, y0 - margin))\n    x1 = int(min(w, x1 + margin)); y1 = int(min(h, y1 + margin))\n    return x0, y0, x1, y1\n\n\ndef _round8(v: int) -> int:\n    return max(8, int(round(v / 8.0)) * 8)\n\n\n# --------------------------------------------------------------------------- #\n# Main entry\n# --------------------------------------------------------------------------- #\ndef ai_replace_qr(cert_bgr: np.ndarray, region, qr_bgr: np.ndarray,\n                  prompt: str = DEFAULT_PROMPT,\n                  negative_prompt: str = DEFAULT_NEGATIVE,\n                  model_id: str = \"diffusers/stable-diffusion-xl-1.0-inpainting-0.1\",\n                  steps: int = 30, guidance: float = 8.0, seed: int = 0,\n                  margin: int = 120, mode: str = \"inpaint_then_paste\",\n                  replace_mode: str = \"full_replace\",\n                  cover_scale: float = 1.08):\n    \"\"\"\n    Run AI inpainting over the old-QR mask with `prompt`, then (default)\n    composite the real QR so it scans.\n\n    mode:\n      \"inpaint_then_paste\" (default, recommended):\n            AI regenerates/blends the QR region per the prompt, then the\n            deterministic QR is warped on top -> scannable + AI-edited.\n      \"generative_only\":\n            return the raw model output (NOT guaranteed to scan -- for\n            comparison/experimentation only).\n\n    Returns (out_bgr, info_dict). If the model can't load, falls back to the\n    deterministic warp and info[\"fallback\"]=True.\n    \"\"\"\n    h, w = cert_bgr.shape[:2]\n    quad = region.quad.astype(np.float32)\n\n    if not ai_available():\n        out = replace_qr(cert_bgr, region, qr_bgr, mode=replace_mode,\n                         cover_scale=cover_scale)\n        return out, {\"ran\": False, \"fallback\": \"diffusers/torch not installed\"}\n\n    # The SDXL inpainting model is GPU-only in practice: never attempt it on\n    # CPU (multi-GB download, minutes/image, likely OOM). Fall back instantly.\n    if _device() != \"cuda\":\n        out = replace_qr(cert_bgr, region, qr_bgr, mode=replace_mode,\n                         cover_scale=cover_scale)\n        return out, {\"ran\": False,\n                     \"fallback\": \"AI inpainting needs a CUDA GPU; used warp\"}\n\n    try:\n        import torch\n    except Exception:\n        torch = None\n    try:\n        from PIL import Image\n    except Exception as e:\n        out = replace_qr(cert_bgr, region, qr_bgr, mode=replace_mode,\n                         cover_scale=cover_scale)\n        return out, {\"ran\": False, \"fallback\": f\"PIL missing: {e}\"}\n\n    try:\n        pipe = load_pipeline(model_id)\n    except Exception as e:  # download/OOM -> graceful fallback\n        out = replace_qr(cert_bgr, region, qr_bgr, mode=replace_mode,\n                         cover_scale=cover_scale)\n        return out, {\"ran\": False, \"fallback\": f\"model load failed: {e}\"}\n\n    mask = _mask_from_quad((h, w), quad, dilate_px=8)\n    x0, y0, x1, y1 = _crop_bbox(quad, h, w, margin)\n    crop = cert_bgr[y0:y1, x0:x1].copy()\n    cmask = mask[y0:y1, x0:x1].copy()\n\n    ch, cw = crop.shape[:2]\n    W8, H8 = _round8(cw), _round8(ch)\n\n    pil_img = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)).resize((W8, H8))\n    pil_mask = Image.fromarray(cmask).resize((W8, H8), Image.NEAREST)\n\n    gen = torch.Generator(device=\"cpu\").manual_seed(seed) if torch is not None else None\n    result = pipe(\n        prompt=prompt, negative_prompt=negative_prompt,\n        image=pil_img, mask_image=pil_mask,\n        height=H8, width=W8,\n        guidance_scale=guidance, num_inference_steps=steps,\n        generator=gen,\n    ).images[0]\n    result = result.resize((cw, ch), Image.LANCZOS)\n    inp_crop = cv2.cvtColor(np.array(result), cv2.COLOR_RGB2BGR)\n\n    # Strictly apply ONLY inside the mask -> rest of certificate untouched.\n    m3 = cv2.cvtColor(cmask, cv2.COLOR_GRAY2BGR) > 0\n    edited_crop = crop.copy()\n    edited_crop[m3] = inp_crop[m3]\n\n    edited = cert_bgr.copy()\n    edited[y0:y1, x0:x1] = edited_crop\n\n    info = {\"ran\": True, \"model\": model_id, \"device\": _device(),\n            \"steps\": steps, \"crop\": (x0, y0, x1, y1), \"mode\": mode}\n\n    if mode == \"generative_only\":\n        return edited, info\n\n    # Composite the REAL QR (guaranteed scannable) over the AI region.\n    final = replace_qr(edited, region, qr_bgr, mode=replace_mode,\n                            cover_scale=cover_scale)\n    info[\"qr_composited\"] = True\n    return final, info\n",
  "pipeline.py": "\"\"\"\npipeline.py\n-----------\nEnd-to-end batch workflow:\n\n  for every certificate image in an input folder:\n     1. DETECT  the old QR  (WeChat CNN model from Hugging Face -> OpenCV ->\n                geometric contour finder)\n     2. BUILD   the payload (URL/text/CSV that scanning should open)\n     3. GENERATE a brand-new crisp QR code\n     4. REPLACE the old QR with the new one at the exact same place/size/angle\n                (nothing else on the certificate is touched)\n     5. VERIFY  the new QR is actually decodable on the output image\n     6. SAVE    the new certificate\n\n  then: write a manifest.csv and ZIP all outputs.\n\nRuns on CPU or GPU identically (the geometry work is CPU; the CNN detector is\ntiny). Designed for 200+ certificates in one pass on Kaggle.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport csv\nimport io\nimport os\nimport zipfile\nfrom dataclasses import dataclass, field\n\nimport cv2\nimport numpy as np\n\nfrom .detector import (detect_qr, draw_debug,\n                       fallback_region_from_regions, fallback_region_default)\nfrom .payload import build_payload, load_csv_map, slug_from_filename, safe_slug\nfrom .qr_generator import build_qr\nfrom .qr_art import build_artistic_qr\nfrom .replacer import replace_qr\nfrom .ai_extract import ai_extract_fields\nfrom .verification_page import build_page\nfrom .ai_inpaint import (\n    ai_replace_qr, ai_available, DEFAULT_PROMPT, DEFAULT_NEGATIVE,\n)\n\nIMG_EXTS = (\".png\", \".jpg\", \".jpeg\", \".webp\", \".bmp\", \".tif\", \".tiff\")\n\n\n@dataclass\nclass Result:\n    filename: str\n    status: str                       # \"ok\" | \"recheck\" | \"failed\"\n    payload: str = \"\"\n    method: str = \"\"\n    qr_side_px: float = 0.0\n    verified: bool = False\n    ai_used: bool = False\n    out_name: str = \"\"\n    slug: str = \"\"\n    fields: dict = None\n    extract_method: str = \"\"\n    note: str = \"\"\n\n\n@dataclass\nclass Config:\n    input_dir: str = \"input\"\n    output_dir: str = \"output\"\n    model_dir: str = \"models/wechat\"\n    zip_path: str = \"certificates_with_new_qr.zip\"\n    payload_mode: str = \"url\"          # url | text | csv | copy\n    base_url: str = \"https://example.github.io/certificates/\"\n    url_extension: str = \".png\"\n    # QR links to a per-candidate VERIFICATION PAGE (.html) that shows the\n    # certificate image with the candidate details (read off the certificate by\n    # the AI vision model) beneath it. False -> QR links directly to the image.\n    verification_page: bool = True\n    # AI vision-language model that READS every field off the certificate image\n    # (no OCR / no CSV). Requires a GPU on Kaggle (T4). Empty fields if it fails.\n    ai_extract_model: str = \"Qwen/Qwen2-VL-2B-Instruct\"\n    text_prefix: str = \"\"\n    csv_path: str | None = None\n    qr_error_correction: str = \"M\"\n    qr_box_size: int = 20\n    qr_border: int = 4\n    replace_mode: str = \"full_replace\"  # full_replace | keep_frame\n    qr_cover_scale: float = 1.08        # >1 = white box grows to hide old frame\n    output_ext: str = \".png\"\n    jpeg_quality: int = 95\n    save_debug: bool = True\n    use_cnn: bool = True\n    # ---- QR generator backend ----\n    # \"standard\"  -> plain deterministic qrcode (always scans; recommended for\n    #                official certificates)\n    # \"artistic\"  -> HF QR Code Monster ControlNet (AI art-QR; auto-verified and\n    #                falls back to standard if no seed scans). GPU recommended.\n    qr_backend: str = \"standard\"\n    art_prompt: str = \"\"       # \"\" -> qr_art.CLEAN_PROMPT\n    art_controlnet_scale: float = 1.5\n    art_size: int = 768\n    art_tries: int = 6\n    # ---- AI image-generator (diffusion inpainting) ----\n    use_ai: bool = False\n    ai_model: str = \"diffusers/stable-diffusion-xl-1.0-inpainting-0.1\"\n    ai_prompt: str = DEFAULT_PROMPT\n    ai_negative: str = DEFAULT_NEGATIVE\n    ai_steps: int = 30\n    ai_guidance: float = 8.0\n    ai_seed: int = 0\n    ai_margin: int = 120\n    ai_mode: str = \"inpaint_then_paste\"   # inpaint_then_paste | generative_only\n    # If a QR can't be located on one certificate, place the new QR at the\n    # batch-derived template position (works because all certs share a layout).\n    fallback_placement: bool = True\n\n\ndef list_certificates(input_dir: str) -> list[str]:\n    out = []\n    for root, _dirs, files in os.walk(input_dir):\n        for f in sorted(files):\n            if f.lower().endswith(IMG_EXTS):\n                out.append(os.path.join(root, f))\n    return sorted(out)\n\n\n_WECHAT_DECODER = None\n\n\ndef _get_wechat_decoder(model_dir: str):\n    \"\"\"A WeChat CNN decoder is far stronger on small/dense QRs (used both to\n    read old QRs and to verify new ones).\"\"\"\n    global _WECHAT_DECODER\n    if _WECHAT_DECODER is not None:\n        return _WECHAT_DECODER\n    if not hasattr(cv2, \"wechat_qrcode_WeChatQRCode\"):\n        return None\n    paths = [os.path.join(model_dir, f) for f in\n             (\"detect.prototxt\", \"detect.caffemodel\",\n              \"sr.prototxt\", \"sr.caffemodel\")]\n    if not all(os.path.exists(p) for p in paths):\n        return None\n    try:\n        _WECHAT_DECODER = cv2.wechat_qrcode_WeChatQRCode(*paths)\n    except Exception:\n        _WECHAT_DECODER = None\n    return _WECHAT_DECODER\n\n\ndef _decode_any_qr(img: np.ndarray, model_dir: str = \"models/wechat\") -> str:\n    \"\"\"Best-effort decode to VERIFY the new QR scans. Returns text or ''.\n\n    Tries the strong WeChat CNN decoder first (it reads small/dense QRs that\n    OpenCV's classic decoder misses), then OpenCV ArUco/classic at several\n    scales.\n    \"\"\"\n    h, w = img.shape[:2]\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img\n\n    wd = _get_wechat_decoder(model_dir)\n    if wd is not None:\n        for scale in (1, 2, 3):\n            src = gray if scale == 1 else cv2.resize(\n                gray, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)\n            try:\n                res, _pts = wd.detectAndDecode(src)\n                if len(res) > 0 and res[0]:\n                    return res[0]\n            except Exception:\n                pass\n\n    dets = []\n    if hasattr(cv2, \"QRCodeDetectorAruco\"):\n        try:\n            dets.append(cv2.QRCodeDetectorAruco())\n        except Exception:\n            pass\n    dets.append(cv2.QRCodeDetector())\n    for scale in (1, 2, 3, 4):\n        src = gray if scale == 1 else cv2.resize(\n            gray, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)\n        for d in dets:\n            try:\n                data, _, _ = d.detectAndDecode(src)\n            except Exception:\n                data = \"\"\n            if data:\n                return data\n    return \"\"\n\n\ndef process_one(path: str, cfg: Config,\n                csv_map: dict | None = None,\n                region_override=None) -> tuple[np.ndarray | None, Result]:\n    name = os.path.basename(path)\n    slug = safe_slug(path)\n    res = Result(filename=name, status=\"failed\", slug=slug,\n                 out_name=slug + cfg.output_ext)\n    img = cv2.imread(path, cv2.IMREAD_COLOR)\n    if img is None:\n        res.note = \"could not read image\"\n        return None, res\n\n    # 0) The AI vision model READS all candidate details off the certificate\n    #    image (name, parent, aadhar, course, dates, institute). No OCR / CSV.\n    fields = {}\n    extract_used = \"\"\n    try:\n        fields = ai_extract_fields(img, model_id=cfg.ai_extract_model)\n        if fields and not fields.get(\"_error\"):\n            extract_used = \"AI-VLM\"\n        elif fields.get(\"_error\"):\n            extract_used = \"AI-FAILED\"\n            print(f\"   \u26a0\ufe0f  AI extraction failed for {name}: {fields['_error']}\")\n    except Exception as e:\n        fields = {\"_error\": str(e)}\n        extract_used = \"AI-FAILED\"\n        print(f\"   \u26a0\ufe0f  AI extraction crashed for {name}: {e}\")\n    res.fields = fields\n    res.extract_method = extract_used\n\n    # The QR links to the verification PAGE (.html) when enabled, else the raw\n    # image (cfg.url_extension).\n    link_ext = \".html\" if cfg.verification_page else cfg.url_extension\n\n    # 1) locate the old QR: detect it, or use a provided fallback placement.\n    region = region_override\n    if region is None:\n        region = detect_qr(img, model_dir=cfg.model_dir, use_cnn=cfg.use_cnn)\n    if region is None:\n        res.note = \"old QR not found\"\n        return img, res\n\n    res.method = region.method\n    res.qr_side_px = round(region.side_px, 1)\n\n    # old payload (only needed for 'copy' mode)\n    old_data = _decode_any_qr(img, cfg.model_dir) if cfg.payload_mode == \"copy\" else None\n\n    # 2) payload (links to the verification page or the raw image)\n    payload = build_payload(\n        path, mode=cfg.payload_mode, base_url=cfg.base_url,\n        extension=link_ext, text_prefix=cfg.text_prefix,\n        csv_map=csv_map, old_data=old_data)\n    res.payload = payload\n\n    # 3) generate fresh QR at high resolution.\n    #    Optional AI \"artistic QR\" backend (HF QR Code Monster ControlNet):\n    #    returns an AI-styled QR that decodes to `payload`, else None -> we\n    #    transparently fall back to the deterministic standard QR.\n    qr_img = None\n    if cfg.qr_backend == \"artistic\":\n        from .qr_art import CLEAN_PROMPT\n        qr_img = build_artistic_qr(\n            payload,\n            prompt=cfg.art_prompt or CLEAN_PROMPT,\n            controlnet_scale=cfg.art_controlnet_scale,\n            size=cfg.art_size, max_tries=cfg.art_tries)\n        if qr_img is not None:\n            res.note = \"AI artistic QR (verified)\"\n    if qr_img is None:\n        qr_img = build_qr(payload, box_size=cfg.qr_box_size,\n                          border=cfg.qr_border,\n                          error_correction=cfg.qr_error_correction)\n\n    # 4) REPLACE the old QR.\n    #    - AI path: masked diffusion inpainting (prompt-driven), then the\n    #      real QR is composited so it is guaranteed scannable.\n    #    - deterministic path: exact perspective warp.\n    if cfg.use_ai:\n        out, ai_info = ai_replace_qr(\n            img, region, qr_img,\n            prompt=cfg.ai_prompt, negative_prompt=cfg.ai_negative,\n            model_id=cfg.ai_model, steps=cfg.ai_steps,\n            guidance=cfg.ai_guidance, seed=cfg.ai_seed,\n            margin=cfg.ai_margin, mode=cfg.ai_mode,\n            replace_mode=cfg.replace_mode,\n            cover_scale=cfg.qr_cover_scale)\n        res.ai_used = bool(ai_info.get(\"ran\"))\n        if not ai_info.get(\"ran\"):\n            res.method += \"+ai-fallback(warp)\"\n    else:\n        out = replace_qr(img, region, qr_img, mode=cfg.replace_mode,\n                         cover_scale=cfg.qr_cover_scale)\n        ai_info = {}\n\n    # 5) verify the new QR decodes on the output\n    decoded = _decode_any_qr(out, cfg.model_dir)\n    res.verified = bool(decoded)\n    art_tag = \"AI artistic QR; \" if (cfg.qr_backend == \"artistic\"\n                                    and \"artistic\" in res.note) else \"\"\n    if not decoded:\n        res.status = \"recheck\"\n        res.note = art_tag + \"new QR not auto-verified (usually still scans; check debug)\"\n    else:\n        res.status = \"ok\"\n        if res.ai_used:\n            res.note = art_tag + \"verified scannable (AI inpaint+QR)\"\n        else:\n            res.note = art_tag + \"verified scannable\"\n\n    return out, res\n\n\ndef run(cfg: Config) -> dict:\n    os.makedirs(cfg.output_dir, exist_ok=True)\n    debug_dir = os.path.join(cfg.output_dir, \"_debug\")\n    if cfg.save_debug:\n        os.makedirs(debug_dir, exist_ok=True)\n\n    csv_map = None\n    if cfg.payload_mode == \"csv\" and cfg.csv_path:\n        csv_map = load_csv_map(cfg.csv_path)\n\n    certs = list_certificates(cfg.input_dir)\n\n    if cfg.payload_mode == \"url\":\n        placeholder_markers = (\"YOUR-USERNAME\", \"example.github.io\",\n                               \"example.com\", \"REPLACE\", \"your-username\")\n        if any(m.lower() in (cfg.base_url or \"\").lower()\n               for m in placeholder_markers):\n            print(\"=\" * 70)\n            print(\"\u26a0\ufe0f  WARNING: BASE_URL still looks like a PLACEHOLDER.\")\n            print(f\"    base_url = {cfg.base_url}\")\n            print(\"    The QR codes WILL scan, but they point at a URL that\")\n            print(\"    does not exist yet -> your phone shows a 404 page.\")\n            print(\"    1) Set base_url to the address where you will upload the\")\n            print(\"       OUTPUT images (e.g. GitHub Pages: create a repo, turn\")\n            print(\"       on Settings->Pages, upload the finished PNGs).\")\n            print(\"    2) Re-run. See README 'Hosting' / notebook section 8.\")\n            print(\"=\" * 70)\n\n    # ---- detection pass: locate the QR on every certificate so that a cert\n    # where detection fails can inherit the position from its template (all\n    # certificates in a batch share a layout).\n    detected: dict[str, object] = {}\n    template: list = []\n    for path in certs:\n        img = cv2.imread(path, cv2.IMREAD_COLOR)\n        if img is None:\n            continue\n        reg = detect_qr(img, model_dir=cfg.model_dir, use_cnn=cfg.use_cnn)\n        h, w = img.shape[:2]\n        detected[path] = reg\n        if reg is not None and \"fallback\" not in reg.method:\n            template.append((reg, h, w))\n\n    results: list[Result] = []\n    ok = recheck = failed = 0\n    ai_ok = ai_fail = 0\n\n    zip_path = cfg.zip_path\n    with zipfile.ZipFile(zip_path, \"w\", zipfile.ZIP_DEFLATED) as zf:\n        for path in certs:\n            img = cv2.imread(path, cv2.IMREAD_COLOR)\n            override = None\n            if detected.get(path) is None and cfg.fallback_placement:\n                # inherit QR position from the other certificates / template\n                override = fallback_region_from_regions(img, template)\n                if override is None:\n                    override = fallback_region_default(img)\n            out, res = process_one(path, cfg, csv_map, region_override=override)\n            if res.extract_method == \"AI-VLM\":\n                ai_ok += 1\n            else:\n                ai_fail += 1\n            if override is not None and res.status in (\"ok\", \"recheck\"):\n                res.note += f\" | QR placed via {override.method}\"\n            results.append(res)\n            if res.status == \"ok\":\n                ok += 1\n            elif res.status == \"recheck\":\n                recheck += 1\n            else:\n                failed += 1\n\n            if out is not None and res.status in (\"ok\", \"recheck\"):\n                out_name = res.out_name\n                out_path = os.path.join(cfg.output_dir, out_name)\n                _write_image(out, out_path, cfg)\n                zf.write(out_path, arcname=\"img/\" + out_name)\n\n                # Per-candidate VERIFICATION PAGE (self-contained HTML).\n                if cfg.verification_page:\n                    page = build_page(res.fields or {}, out, out_name)\n                    page_name = res.slug + \".html\"\n                    page_path = os.path.join(cfg.output_dir, page_name)\n                    with open(page_path, \"w\", encoding=\"utf-8\") as fh:\n                        fh.write(page)\n                    zf.write(page_path, arcname=page_name)\n\n                if cfg.save_debug and res.status == \"recheck\":\n                    reg = detect_qr(out, model_dir=cfg.model_dir,\n                                    use_cnn=False)\n                    dbg = draw_debug(out, reg) if reg else out\n                    cv2.imwrite(os.path.join(debug_dir, \"dbg_\" + out_name), dbg)\n            elif out is not None and cfg.save_debug:\n                cv2.imwrite(os.path.join(\n                    debug_dir, \"FAILED_\" + safe_slug(path) + cfg.output_ext), out)\n\n            tag = \"AI\" if res.ai_used else (\"ai-fb\" if \"+ai-fallback\" in res.method else \"  \")\n            print(f\"[{res.status:7s}] {tag} {res.filename:36s} \"\n                  f\"{res.method:24s} {res.qr_side_px:6.1f}px  {res.payload}\")\n\n    # manifest\n    manifest = os.path.join(cfg.output_dir, \"manifest.csv\")\n    with open(manifest, \"w\", newline=\"\", encoding=\"utf-8\") as f:\n        wr = csv.writer(f)\n        wr.writerow([\"filename\", \"status\", \"ai_used\", \"method\", \"qr_side_px\",\n                     \"verified\", \"qr_payload\", \"name\", \"parent\", \"aadhar\",\n                     \"course\", \"date_from\", \"date_to\", \"institute\",\n                     \"extracted_by\", \"note\"])\n        for r in results:\n            f = r.fields or {}\n            wr.writerow([r.filename, r.status, r.ai_used, r.method,\n                         r.qr_side_px, r.verified, r.payload,\n                         f.get(\"name\", \"\"), f.get(\"parent\", \"\"),\n                         f.get(\"aadhar\", \"\"), f.get(\"course\", \"\"),\n                         f.get(\"date_from\", \"\"), f.get(\"date_to\", \"\"),\n                         f.get(\"institute\", \"\"), r.extract_method, r.note])\n\n    summary = {\"total\": len(certs), \"ok\": ok, \"recheck\": recheck,\n               \"failed\": failed, \"fields_extracted\": ai_ok,\n               \"fields_failed\": ai_fail,\n               \"zip\": os.path.abspath(zip_path),\n               \"manifest\": os.path.abspath(manifest)}\n    print(\"\\n===== SUMMARY =====\")\n    for k, v in summary.items():\n        print(f\"{k:16s}: {v}\")\n    if ai_fail:\n        print(\"\\n\u26a0\ufe0f  The AI could NOT read fields from \"\n              f\"{ai_fail} certificate(s) -> those pages show '\u2014'.\")\n        print(\"   Check: Accelerator = GPU T4; and run the 'Test AI extraction'\")\n        print(\"   cell to see the raw model output / error.\")\n    return summary\n\n\ndef _write_image(img: np.ndarray, path: str, cfg: Config):\n    ext = cfg.output_ext.lower()\n    if ext in (\".jpg\", \".jpeg\"):\n        cv2.imwrite(path, img, [cv2.IMWRITE_JPEG_QUALITY, cfg.jpeg_quality])\n    else:\n        cv2.imwrite(path, img)\n"
}
for name, content in SRC.items():
    with open(os.path.join("/kaggle/working/certbot/src", name), "w",
              encoding="utf-8") as f:
        f.write(content)
    print("wrote", name, len(content), "chars")
print("pipeline has qr_cover_scale:", "qr_cover_scale" in SRC["pipeline.py"])

# publish helper (GitHub Pages)
with open("/kaggle/working/certbot/publish_to_github_pages.py", "w",
          encoding="utf-8") as f:
    f.write("#!/usr/bin/env python3\n\"\"\"\npublish_to_github_pages.py\n--------------------------\nUpload the finished certificate images to a GitHub Pages repository so the QR\nlinks resolve (scan -> certificate opens in the browser). Self-verifying:\nafter each upload it polls the public Pages URL until the image is live.\n\nToken: a GitHub Personal Access Token (classic) with the `repo` scope.\nProvided via GITHUB_TOKEN / GH_TOKEN env var (on Kaggle, add a notebook Secret\nnamed GITHUB_TOKEN), or pasted at the prompt.\n\nCreate a token: https://github.com/settings/tokens  (Generate new token\n(classic) -> tick \"repo\").\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport base64\nimport os\nimport time\nimport zipfile\n\ntry:\n    import requests\nexcept Exception:  # pragma: no cover\n    requests = None\n\nAPI = \"https://api.github.com\"\n\n\ndef _get_token() -> str:\n    tok = os.environ.get(\"GITHUB_TOKEN\") or os.environ.get(\"GH_TOKEN\") or \"\"\n    if not tok:\n        tok = input(\"Paste your GitHub token (repo scope): \").strip()\n    return tok\n\n\ndef publish(zip_path: str, owner: str, repo: str, token: str,\n            subdir: str = \"img\", branch: str = \"main\",\n            verify: bool = True, verbose: bool = True):\n    if requests is None:\n        raise SystemExit(\"'requests' is required (pip install requests).\")\n\n    files = []   # (name, content, upload_dir)\n    with zipfile.ZipFile(zip_path) as z:\n        for name in z.namelist():\n            base_name = os.path.basename(name)\n            low = name.lower()\n            if low.endswith((\".png\", \".jpg\", \".jpeg\", \".webp\")):\n                # images stay under img/ (whether the zip root or img/ prefix)\n                files.append((base_name, z.read(name), subdir))\n            elif low.endswith(\".html\"):\n                # verification pages live at the repo root (QR URLs point there)\n                files.append((base_name, z.read(name), \"\"))\n\n    headers = {\"Authorization\": f\"Bearer {token}\",\n               \"Accept\": \"application/vnd.github+json\",\n               \"User-Agent\": \"cert-qr-bot-publisher\"}\n    api_base = f\"{API}/repos/{owner}/{repo}/contents\"\n    live_root = f\"https://{owner}.github.io/{repo}/\"\n    ok = fail = upload_fail = 0\n    uploaded_files = []   # (name, live_url, is_html)\n\n    if verbose:\n        print(f\"Publishing {len(files)} file(s) to {owner}/{repo} ...\")\n\n    for i, (name, content, upload_dir) in enumerate(files, 1):\n        path = f\"{api_base}/{upload_dir}/{name}\" if upload_dir \\\n            else f\"{api_base}/{name}\"\n        # find existing sha (to update rather than conflict)\n        sha = None\n        try:\n            r = requests.get(path, headers=headers,\n                             params={\"ref\": branch}, timeout=30)\n            if r.status_code == 200:\n                sha = r.json().get(\"sha\")\n        except Exception:\n            pass\n\n        body = {\"message\": f\"Publish {name}\",\n                \"content\": base64.b64encode(content).decode(),\n                \"branch\": branch}\n        if sha:\n            body[\"sha\"] = sha\n\n        uploaded = False\n        for attempt in range(3):\n            try:\n                r = requests.put(path, headers=headers, json=body, timeout=60)\n                if r.status_code in (200, 201):\n                    uploaded = True\n                    break\n                if verbose:\n                    print(f\"  ! {name} upload attempt {attempt+1}: \"\n                          f\"HTTP {r.status_code} {r.text[:160]}\")\n            except Exception as e:\n                if verbose:\n                    print(f\"  ! {name} attempt {attempt+1} error: {e}\")\n            time.sleep(2)\n\n        live_url = (f\"https://{owner}.github.io/{repo}/{upload_dir}/{name}\"\n                    if upload_dir else f\"https://{owner}.github.io/{repo}/{name}\")\n        is_img = name.lower().endswith((\".png\", \".jpg\", \".jpeg\", \".webp\"))\n        is_html = name.lower().endswith(\".html\")\n        if uploaded:\n            uploaded_files.append((name, live_url, is_html))\n        # Per-file quick check (short); the final sweep below re-checks any\n        # that weren't live yet, so Pages propagation lag is not fatal here.\n        if uploaded and verify and (is_img or is_html):\n            live = _wait_live(live_url, expect_html=is_html,\n                              tries=3, delay=4)\n        else:\n            live = uploaded\n\n        if uploaded and live:\n            ok += 1\n            if verbose:\n                print(f\"  [{i}/{len(files)}] \u2705 {name} -> {live_url}\")\n        elif uploaded:\n            # uploaded but not live yet \u2014 final sweep will re-check it\n            if verbose:\n                print(f\"  [{i}/{len(files)}] \u23f3 {name} uploaded, waiting for Pages ...\")\n        else:\n            upload_fail += 1\n            if verbose:\n                print(f\"  [{i}/{len(files)}] \u274c {name} (upload failed)\")\n        time.sleep(0.4)\n\n    # ---- Final verification sweep: keep re-checking every uploaded file until\n    # each is live on GitHub Pages (handles propagation lag for image AND html).\n    if verify and uploaded_files:\n        pending = [(n, u, h) for (n, u, h) in uploaded_files]\n        if verbose and pending:\n            print(f\"\\nVerifying {len(pending)} page(s)/image(s) are live ...\")\n        for round_no in range(1, 13):   # up to ~12 * 10s = 2 minutes\n            still = []\n            for name, live_url, is_html in pending:\n                if _check_one(live_url, is_html):\n                    ok += 0  # already counted as uploaded\n                    if verbose:\n                        print(f\"  \u2705 live: {name}\")\n                else:\n                    still.append((name, live_url, is_html))\n            pending = still\n            if not pending:\n                break\n            if verbose:\n                print(f\"  ... {len(pending)} not live yet, re-checking \"\n                      f\"(round {round_no})\")\n            time.sleep(10)\n        # anything still pending after the sweep is reported for a re-run\n        for name, _u, _h in pending:\n            if verbose:\n                print(f\"  \u274c {name} still not live after waiting (re-run \"\n                      f\"section 8 in ~1 min)\")\n        not_live = len(pending)\n    else:\n        not_live = 0\n    fail = not_live + upload_fail\n\n    live_count = len(uploaded_files) - not_live\n    if verbose:\n        print(f\"\\nDone: {live_count} live, {fail} not yet live \"\n              f\"({upload_fail} upload failure(s)).\")\n        if fail == 0:\n            print(\"\u2705 Every image and verification page is uploaded AND live.\")\n        print(\"Pages (QR base URL): \" + live_root)\n        print(\"Images under:        \" + live_root + subdir + \"/\")\n    return {\"ok\": live_count, \"fail\": fail, \"base\": live_root}\n\n\ndef _check_one(url: str, expect_html: bool) -> bool:\n    import random\n    bust = url + (\"&\" if \"?\" in url else \"?\") + \"cb=\" + str(random.randint(0, 10**9))\n    try:\n        r = requests.get(bust, timeout=30,\n                         headers={\"Cache-Control\": \"no-cache\",\n                                  \"User-Agent\": \"cert-qr-bot-publisher\"})\n        ctype = r.headers.get(\"content-type\", \"\")\n        if r.status_code != 200:\n            return False\n        if expect_html:\n            return \"html\" in ctype\n        return \"image\" in ctype\n    except Exception:\n        return False\n\n\ndef _wait_live(url: str, tries: int = 14, delay: int = 5,\n               expect_html: bool = False) -> bool:\n    for _ in range(tries):\n        if _check_one(url, expect_html):\n            return True\n        time.sleep(delay)\n    return False\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--zip\", default=\"certificates_with_new_qr.zip\")\n    ap.add_argument(\"--owner\", default=\"Naserkhan07\")\n    ap.add_argument(\"--repo\", default=\"certificate-qr\")\n    ap.add_argument(\"--subdir\", default=\"img\")\n    ap.add_argument(\"--branch\", default=\"main\")\n    ap.add_argument(\"--no-verify\", action=\"store_true\")\n    args = ap.parse_args()\n    token = _get_token()\n    if not token:\n        raise SystemExit(\"No GitHub token provided.\")\n    publish(args.zip, args.owner, args.repo, token, args.subdir,\n            args.branch, verify=not args.no_verify)\n\n\nif __name__ == \"__main__\":\n    main()\n")
print("wrote publish_to_github_pages.py")


## 🧪 4b. Test the AI field-reader on ONE certificate

Run this **before** the batch to confirm the vision model loads on the GPU and
reads the six fields. It prints the extracted fields AND the model's raw output.
If it prints an error / empty fields here, the batch pages will show "&mdash;",
so fix this first (usually: set **Settings &rarr; Accelerator &rarr; GPU T4**,
or wait for the model to finish downloading). If this is a fresh session, just
run this cell — it auto-writes the pipeline modules even if you skipped
section 4.

In [ ]:
import os as _os
if not _os.path.isfile("/kaggle/working/certbot/src/ai_extract.py"):
    print("Pipeline modules not written in this session yet — auto-running the section 4 writer first ...")
    import os, shutil
    os.makedirs("/kaggle/working/certbot/src", exist_ok=True)
    # clear stale bytecode from previous runs
    _pyc = "/kaggle/working/certbot/src/__pycache__"
    if os.path.isdir(_pyc): shutil.rmtree(_pyc)
    SRC = {
      "__init__.py": "from .pipeline import Config, run, process_one  # noqa\n",
      "detector.py": "\"\"\"\ndetector.py\n-----------\nLocate the OLD QR code on a certificate image.\n\nDetection cascade (tries each, returns first high-confidence hit):\n  1. WeChat CNN QR detector  (Deep-CV model downloaded from Hugging Face)\n  2. OpenCV Aruco QR detector\n  3. Geometric contour finder (finds the thick black square FRAME around a QR\n     even when the QR itself is too dense/small to decode -- works on certs\n     whose old QR is decorative or damaged)\n\nEvery detector returns the quadrangle of the *outer QR box* (including the\nblack border frame when present), so the new QR can be warped to exactly that\nsize / position / rotation.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport os\nfrom dataclasses import dataclass\n\nimport cv2\nimport numpy as np\n\n\n@dataclass\nclass QRRegion:\n    \"\"\"The 4 corners of the QR's outer box, in order TL, TR, BR, BL (px).\"\"\"\n    quad: np.ndarray            # shape (4, 2), float32\n    method: str                 # which detector found it\n    side_px: float              # average side length in pixels\n    score: float                # confidence 0..1\n\n\n# ---------------------------------------------------------------------------\n# Helpers\n# ---------------------------------------------------------------------------\ndef _order_points(pts: np.ndarray) -> np.ndarray:\n    \"\"\"Order 4 points as TL, TR, BR, BL.\"\"\"\n    pts = np.asarray(pts, dtype=np.float32).reshape(4, 2)\n    s = pts.sum(axis=1)\n    d = np.diff(pts, axis=1).reshape(-1)\n    tl = pts[np.argmin(s)]\n    br = pts[np.argmax(s)]\n    tr = pts[np.argmin(d)]\n    bl = pts[np.argmax(d)]\n    return np.array([tl, tr, br, bl], dtype=np.float32)\n\n\ndef _side_lengths(quad: np.ndarray) -> tuple[float, float, float, float]:\n    tl, tr, br, bl = quad\n    return (\n        float(np.linalg.norm(tr - tl)),\n        float(np.linalg.norm(br - tr)),\n        float(np.linalg.norm(bl - br)),\n        float(np.linalg.norm(tl - bl)),\n    )\n\n\ndef _valid_quad(gray: np.ndarray, quad: np.ndarray,\n                min_finders: int = 2) -> bool:\n    \"\"\"\n    Reject false-positive quads from the ML/OpenCV detectors.\n    A real QR quad: roughly square, on-canvas, convex, and contains >=2 of the\n    three characteristic finder patterns (nested squares).\n    \"\"\"\n    h, w = gray.shape[:2]\n    try:\n        quad = _order_points(quad)\n    except Exception:\n        return False\n    sides = _side_lengths(quad)\n    if min(sides) < max(8.0, 0.04 * min(h, w)):\n        return False\n    if max(sides) > max(h, w) * 1.15:\n        return False\n    if min(sides) / max(sides) < 0.7:\n        return False\n    # convexity (cross products same sign)\n    edges = np.roll(quad, -1, axis=0) - quad\n    crosses = edges[:, 0] * np.roll(edges[:, 1], -1) - \\\n        edges[:, 1] * np.roll(edges[:, 0], -1)\n    if not (np.all(crosses >= -1e-3) or np.all(crosses <= 1e-3)):\n        return False\n    # mostly on canvas\n    x0, y0 = quad.min(axis=0)\n    x1, y1 = quad.max(axis=0)\n    if x1 < -0.05 * w or x0 > 1.05 * w or y1 < -0.05 * h or y0 > 1.05 * h:\n        return False\n    if _count_finder_patterns(gray, quad) < min_finders:\n        return False\n    return True\n\n\ndef _count_finder_patterns(gray: np.ndarray, quad: np.ndarray) -> int:\n    \"\"\"\n    Count finder-like nested squares inside a quad. A QR has 3 finder patterns\n    (TL, TR, BL corners). Used to score contour candidates that look like a QR.\n    \"\"\"\n    tl, tr, br, bl = quad\n    w = int(max(np.linalg.norm(tr - tl), np.linalg.norm(br - bl)))\n    h = int(max(np.linalg.norm(bl - tl), np.linalg.norm(br - tr)))\n    if w < 10 or h < 10:\n        return 0\n    M = cv2.getPerspectiveTransform(\n        quad.astype(np.float32),\n        np.array([[0, 0], [w, 0], [w, h], [0, h]], dtype=np.float32),\n    )\n    patch = cv2.warpPerspective(gray, M, (w, h))\n    _, bw = cv2.threshold(patch, 0, 255,\n                          cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)\n    contours, hier = cv2.findContours(bw, cv2.RETR_TREE,\n                                      cv2.CHAIN_APPROX_SIMPLE)\n    if hier is None:\n        return 0\n    hier = hier[0]\n\n    def nested_depth(i: int) -> int:\n        depth = 0\n        parent = hier[i][3]\n        while parent != -1:\n            depth += 1\n            parent = hier[parent][3]\n        return depth\n\n    finders = 0\n    for i, c in enumerate(contours):\n        area = cv2.contourArea(c)\n        if area == 0:\n            continue\n        x, y, ww, hh = cv2.boundingRect(c)\n        ratio = ww / float(hh)\n        area_ratio = area / float(ww * hh)\n        # finder outer ring: roughly square, moderately sized, >=2 nestings\n        if (0.7 < ratio < 1.35 and area_ratio > 0.55\n                and ww > w * 0.08 and ww < w * 0.6 and nested_depth(i) >= 2):\n            finders += 1\n    return finders\n\n\n# ---------------------------------------------------------------------------\n# Detector 1: WeChat CNN (model from Hugging Face)\n# ---------------------------------------------------------------------------\n_WECHAT = None\n\n\ndef _get_wechat_detector(model_dir: str):\n    \"\"\"Lazy-load the WeChat QR CNN model if the 4 files are present.\"\"\"\n    global _WECHAT\n    if _WECHAT is not None:\n        return _WECHAT\n    if not hasattr(cv2, \"wechat_qrcode_WeChatQRCode\"):\n        return None\n    files = [\"detect.prototxt\", \"detect.caffemodel\",\n             \"sr.prototxt\", \"sr.caffemodel\"]\n    paths = [os.path.join(model_dir, f) for f in files]\n    if not all(os.path.exists(p) for p in paths):\n        return None\n    try:\n        _WECHAT = cv2.wechat_qrcode_WeChatQRCode(*paths)\n    except Exception:\n        _WECHAT = None\n    return _WECHAT\n\n\ndef _detect_wechat(img: np.ndarray, model_dir: str):\n    det = _get_wechat_detector(model_dir)\n    if det is None:\n        return None\n    h, w = img.shape[:2]\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img\n    for scale in (1.0, 2.0, 3.0):\n        src = gray if scale == 1.0 else cv2.resize(\n            gray, (int(w * scale), int(h * scale)),\n            interpolation=cv2.INTER_CUBIC)\n        try:\n            res, points = det.detectAndDecode(src)\n        except Exception:\n            res, points = [], None\n        if points is not None and len(points) > 0:\n            for p in points:\n                quad = _order_points(np.asarray(p, dtype=np.float32) / scale)\n                if _valid_quad(gray, quad, min_finders=2):\n                    return quad\n    return None\n\n\n# ---------------------------------------------------------------------------\n# Detector 2: OpenCV ArUco / classic QR detector\n# ---------------------------------------------------------------------------\ndef _detect_opencv(img: np.ndarray):\n    h, w = img.shape[:2]\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img\n    detectors = []\n    if hasattr(cv2, \"QRCodeDetectorAruco\"):\n        try:\n            detectors.append(cv2.QRCodeDetectorAruco())\n        except Exception:\n            pass\n    detectors.append(cv2.QRCodeDetector())\n    for scale in (1.0, 2.0, 3.0, 4.0):\n        src = gray if scale == 1.0 else cv2.resize(\n            gray, (int(w * scale), int(h * scale)),\n            interpolation=cv2.INTER_CUBIC)\n        for det in detectors:\n            ok, pts = det.detect(src)\n            if ok and pts is not None:\n                quad = _order_points(pts.reshape(4, 2) / scale)\n                if _valid_quad(gray, quad, min_finders=2):\n                    return quad\n    return None\n\n\n# ---------------------------------------------------------------------------\n# Detector 3: geometric contour finder (thick black frame around QR)\n# ---------------------------------------------------------------------------\ndef _detect_by_contours(img: np.ndarray):\n    h, w = img.shape[:2]\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img\n    img_area = h * w\n\n    best = None\n    best_score = -1.0\n\n    for thresh in (90, 120, 150, 180):\n        _, bw = cv2.threshold(gray, thresh, 255, cv2.THRESH_BINARY_INV)\n        # close small gaps inside the frame\n        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))\n        bw = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel)\n        contours, _ = cv2.findContours(bw, cv2.RETR_LIST,\n                                       cv2.CHAIN_APPROX_SIMPLE)\n        for c in contours:\n            area = cv2.contourArea(c)\n            if area < img_area * 0.0008 or area > img_area * 0.25:\n                continue\n            peri = cv2.arcLength(c, True)\n            approx = cv2.approxPolyDP(c, 0.04 * peri, True)\n            if len(approx) != 4:\n                # also accept rotated rect\n                rrect = cv2.minAreaRect(c)\n                (cx, cy), (rw, rh), _ = rrect\n                if rw == 0 or rh == 0:\n                    continue\n                sq = min(rw, rh) / max(rw, rh)\n                if sq < 0.82:\n                    continue\n                box = cv2.boxPoints(rrect)\n            else:\n                box = approx.reshape(4, 2).astype(np.float32)\n                (cx, cy), (rw, rh), _ = cv2.minAreaRect(c)\n                sq = min(rw, rh) / max(rw, rh) if rw and rh else 0\n                if sq < 0.82:\n                    continue\n            quad = _order_points(box)\n            sides = _side_lengths(quad)\n            side = float(np.mean(sides))\n            # reject wildly non-square quads\n            if min(sides) / max(sides) < 0.75:\n                continue\n            finders = _count_finder_patterns(gray, quad)\n            if finders < 2:\n                continue\n            # score: finder count first, then size (QRs on certs are big)\n            score = finders * 10 + (side / max(h, w))\n            if score > best_score:\n                best_score = score\n                best = (quad, finders, side)\n    if best is not None:\n        quad, finders, side = best\n        return QRRegion(quad=quad, method=\"contour\", side_px=side,\n                        score=min(1.0, finders / 3.0))\n    return None\n\n\n# ---------------------------------------------------------------------------\n# Detector 3b: finder-pattern locator (works WITHOUT a black frame)\n# ---------------------------------------------------------------------------\ndef _chain_depth(i: int, hier: np.ndarray, need: int = 2) -> bool:\n    \"\"\"True if contour `i` encloses >=`need` levels of descendants. A finder\n    pattern's outer black ring contains a white gap then the black centre, so\n    its first-child chain reaches depth 2 (the gap can merge with background,\n    which is why depth 4 is never seen on real scans).\"\"\"\n    cur = hier[i][2]\n    d = 0\n    while cur != -1:\n        d += 1\n        if d >= need:\n            return True\n        cur = hier[cur][2]\n    return False\n\n\ndef _find_finder_centers(gray: np.ndarray) -> list[tuple[np.ndarray, float]]:\n    \"\"\"Locate QR finder patterns (the 3 corner 'nested squares'). Returns a\n    list of (centre[x,y], outer-ring size px). Works for framed OR\n    frameless/plain QRs, since finders are always present.\"\"\"\n    h, w = gray.shape[:2]\n    found: list[tuple[np.ndarray, float]] = []\n\n    for thresh in (80, 110, 140, 170, 200):\n        _, bw = cv2.threshold(gray, thresh, 255, cv2.THRESH_BINARY_INV)\n        # No morphology here: closing merges the thin rings of tiny\n        # (sub-100px) QRs and destroys the nesting we are looking for.\n        contours, hier = cv2.findContours(bw, cv2.RETR_TREE,\n                                          cv2.CHAIN_APPROX_SIMPLE)\n        if hier is None:\n            continue\n        hier = hier[0]\n        cands = []\n        for i, c in enumerate(contours):\n            x, y, ww, hh = cv2.boundingRect(c)\n            if ww == 0 or hh == 0:\n                continue\n            side_min = float(min(h, w))\n            if not (side_min * 0.012 < ww < side_min * 0.35 and\n                    side_min * 0.012 < hh < side_min * 0.35):\n                continue\n            if not (0.65 < ww / float(hh) < 1.5):\n                continue\n            area = cv2.contourArea(c)\n            if area / float(ww * hh) < 0.5:\n                continue\n            if not _chain_depth(i, hier, need=2):\n                continue\n            cands.append((x + ww / 2.0, y + hh / 2.0, (ww + hh) / 2.0))\n        # within each threshold, collapse nested/contained candidates to the\n        # OUTERMOST ring (largest), since the centre dot also appears.\n        cands.sort(key=lambda t: -t[2])\n        for cx, cy, size in cands:\n            if not any(np.hypot(cx - fc[0][0], cy - fc[0][1]) < fc[1] * 0.7\n                       for fc in found):\n                found.append((np.array([cx, cy], dtype=np.float32),\n                              float(size)))\n    return found\n\n\ndef _quad_from_finders(finders: list[tuple[np.ndarray, float]]):\n    \"\"\"Given >=3 finder centres, build the QR outer quad (incl. quiet zone).\"\"\"\n    n = len(finders)\n    best = None\n    best_err = 1e9\n    for a in range(n):\n        for b in range(n):\n            if b == a:\n                continue\n            for c in range(n):\n                if c == a or c == b:\n                    continue\n                A, rA = finders[a]   # candidate elbow (TL)\n                B, _ = finders[b]    # candidate TR\n                C, _ = finders[c]    # candidate BL\n                vx = B - A\n                vy = C - A\n                lx = float(np.linalg.norm(vx))\n                ly = float(np.linalg.norm(vy))\n                if lx < 5 or ly < 5:\n                    continue\n                if min(lx, ly) / max(lx, ly) < 0.6:\n                    continue\n                cosang = float(np.dot(vx, vy) / (lx * ly))\n                if abs(cosang) > 0.45:   # want ~90 deg\n                    continue\n                ux = vx / lx\n                uy = vy / ly\n                r = float(np.mean([fr[1] for fr in (finders[a], finders[b],\n                                                    finders[c])]))\n                module = r / 7.0\n                ext = 7.5 * module      # ring centre (3.5 mod) + quiet (4 mod)\n                TL = A - ext * (ux + uy)\n                TR = TL + ux * (lx + 2 * ext)\n                BL = TL + uy * (ly + 2 * ext)\n                BR = TL + ux * (lx + 2 * ext) + uy * (ly + 2 * ext)\n                err = abs(cosang) + abs(lx - ly) / max(lx, ly)\n                if err < best_err:\n                    best_err = err\n                    best = _order_points(np.array([TL, TR, BR, BL],\n                                                  dtype=np.float32))\n    return best\n\n\ndef _detect_by_finders(img: np.ndarray):\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img\n    h, w = gray.shape[:2]\n    finders = _find_finder_centers(gray)\n    if len(finders) < 3:\n        return None\n    quad = _quad_from_finders(finders)\n    if quad is None:\n        return None\n    sides = _side_lengths(quad)\n    side = float(np.mean(sides))\n    if min(sides) < max(8.0, 0.04 * min(h, w)):\n        return None\n    if min(sides) / max(sides) < 0.7:\n        return None\n    return QRRegion(quad=quad, method=\"finders\", side_px=side, score=0.95)\n\n\n# ---------------------------------------------------------------------------\n# Public API\n# ---------------------------------------------------------------------------\ndef detect_qr(img: np.ndarray, model_dir: str = \"models/wechat\",\n              use_cnn: bool = True):\n    \"\"\"\n    Find the old QR on a certificate.\n\n    Returns QRRegion or None.\n    \"\"\"\n    if use_cnn:\n        quad = _detect_wechat(img, model_dir)\n        if quad is not None:\n            sides = _side_lengths(quad)\n            return QRRegion(quad=quad, method=\"wechat-cnn\",\n                            side_px=float(np.mean(sides)), score=0.99)\n\n    quad = _detect_opencv(img)\n    if quad is not None:\n        sides = _side_lengths(quad)\n        return QRRegion(quad=quad, method=\"opencv\",\n                        side_px=float(np.mean(sides)), score=0.9)\n\n    reg = _detect_by_contours(img)\n    if reg is not None:\n        return reg\n\n    # last resort: locate the three finder patterns (no frame needed)\n    return _detect_by_finders(img)\n\n\n# ---------------------------------------------------------------------------\n# Fallback placement (used when a QR cannot be located on one cert in a batch\n# of SAME-TEMPLATE certificates -- the QR is always in the same spot).\n# ---------------------------------------------------------------------------\ndef normalize_region(region: QRRegion, h: int, w: int) -> np.ndarray:\n    \"\"\"Quad coordinates as fractions of image width/height.\"\"\"\n    q = region.quad.astype(np.float32).copy()\n    q[:, 0] /= float(w)\n    q[:, 1] /= float(h)\n    return q\n\n\ndef fallback_region_from_regions(img: np.ndarray,\n                                 regions: list) -> QRRegion | None:\n    \"\"\"Place the new QR using the MEDIAN normalized position of the QRs that\n    WERE detected on other certificates in the batch. Works because all\n    certificates share one template. `regions` = list of (QRRegion, h, w).\n\n    Skips images whose aspect ratio differs from the certificates (e.g. a wide\n    16:9 screenshot that isn't a certificate) so a QR is never pasted onto a\n    non-certificate image.\"\"\"\n    h, w = img.shape[:2]\n    tmpl_aspect = float(np.median([ww / max(1, hh) for (_r, hh, ww) in regions]))\n    if abs((w / max(1, h)) - tmpl_aspect) / tmpl_aspect > 0.15:\n        return None\n    norms = [normalize_region(r, hh, ww) for (r, hh, ww) in regions]\n    if not norms:\n        return None\n    med = np.median(np.stack(norms, axis=0), axis=0).astype(np.float32)\n    quad = med.copy()\n    quad[:, 0] *= float(w)\n    quad[:, 1] *= float(h)\n    quad = _order_points(quad)\n    sides = _side_lengths(quad)\n    return QRRegion(quad=quad, method=\"batch-template-fallback\",\n                    side_px=float(np.mean(sides)), score=0.5)\n\n\ndef fallback_region_default(img: np.ndarray) -> QRRegion | None:\n    \"\"\"Last-resort placement for the TGMFC certificate template (QR sits in\n    the upper-left, under the red certificate number). Coordinates are\n    fractions of (width, height). Returns None for images whose aspect ratio\n    is a wide landscape (e.g. a 16:9 screenshot) rather than a certificate\n    page, so a QR is never pasted onto the wrong image.\"\"\"\n    h, w = img.shape[:2]\n    if w / max(1, h) >= 1.45:      # wide landscape screenshot, not a cert page\n        return None\n    n = np.array([[0.095, 0.345], [0.205, 0.345],\n                  [0.205, 0.515], [0.095, 0.515]], dtype=np.float32)\n    quad = n.copy()\n    quad[:, 0] *= float(w)\n    quad[:, 1] *= float(h)\n    quad = _order_points(quad)\n    sides = _side_lengths(quad)\n    return QRRegion(quad=quad, method=\"template-default-fallback\",\n                    side_px=float(np.mean(sides)), score=0.3)\n\n\ndef draw_debug(img: np.ndarray, region: QRRegion) -> np.ndarray:\n    \"\"\"Overlay the detected quad on an image (for review of failures).\"\"\"\n    out = img.copy()\n    q = region.quad.astype(np.int32)\n    cv2.polylines(out, [q], True, (0, 0, 255), 3)\n    cv2.putText(out, f\"{region.method} {region.side_px:.0f}px\",\n                (q[0][0], max(15, q[0][1] - 8)),\n                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)\n    return out\n",
      "qr_generator.py": "\"\"\"\nqr_generator.py\n---------------\nGenerate a fresh, crisp QR code image for a given payload (the URL/text that\na phone will read when scanning the certificate).\n\nThe QR is rendered at HIGH pixel resolution so that when it is perspective-\nwarped down onto the small QR region of the certificate, every module stays\nsharp and black/white (which is what makes it scan reliably).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport io\n\nimport cv2\nimport numpy as np\nimport qrcode\nfrom qrcode.constants import (\n    ERROR_CORRECT_L, ERROR_CORRECT_M, ERROR_CORRECT_Q, ERROR_CORRECT_H,\n)\n\n_EC = {\n    \"L\": ERROR_CORRECT_L,   # ~7%\n    \"M\": ERROR_CORRECT_M,   # ~15%\n    \"Q\": ERROR_CORRECT_Q,   # ~25%\n    \"H\": ERROR_CORRECT_H,   # ~30%\n}\n\n\ndef build_qr(payload: str,\n             box_size: int = 20,\n             border: int = 4,\n             error_correction: str = \"M\",\n             fg: tuple[int, int, int] = (0, 0, 0),\n             bg: tuple[int, int, int] = (255, 255, 255)) -> np.ndarray:\n    \"\"\"\n    Return a square BGR uint8 image (white background) containing the QR.\n\n    box_size: pixels per module at render time (large = crisp after warping)\n    border:   quiet-zone width in modules (standard 4)\n    \"\"\"\n    qr = qrcode.QRCode(\n        version=None,                       # auto-size to payload\n        error_correction=_EC[error_correction.upper()],\n        box_size=box_size,\n        border=border,\n    )\n    qr.add_data(payload)\n    qr.make(fit=True)\n\n    img = qr.make_image(fill_color=fg, back_color=bg).convert(\"RGB\")\n    arr = np.array(img)[:, :, ::-1].copy()  # RGB -> BGR\n    return arr\n\n\ndef qr_png_bytes(payload: str, **kwargs) -> bytes:\n    \"\"\"Render the QR and return PNG bytes (e.g. for saving a standalone file).\"\"\"\n    arr = build_qr(payload, **kwargs)\n    ok, buf = cv2.imencode(\".png\", arr)\n    return buf.tobytes() if ok else b\"\"\n",
      "qr_art.py": "\"\"\"\nqr_art.py\n---------\nOPTIONAL AI \"artistic QR\" generation using the Hugging Face ControlNet model\n**QR Code Monster** (`monster-labs/control_v1p_sd15_qrcode_monster`, v2) with a\nStable Diffusion 1.5 base.\n\nThis is the popular \"image/text -> QR code\" diffusion model family (QR Code\nMonster, IllusionDiffusion, etc.): a normal QR matrix is fed to ControlNet as\nthe conditioning image together with a text prompt, and the model produces a\ncreative QR in which art is blended into the modules.\n\nImportant facts (from the model cards):\n  * Not every generation scans -- typical scannability is 50-80%, so the\n    official workflow is \"generate several seeds and keep a readable one\".\n  * High `controlnet_conditioning_scale` (1.2-1.8) => more readable; low =>\n    more artistic. We default high because the QR must work on a certificate.\n  * These codes are usually colourful/textured -- great for marketing, but for\n    an official certificate the plain black-and-white `qr_generator.build_qr`\n    (100% reliable) is recommended.\n\nTherefore `build_artistic_qr()`:\n  1. renders the EXACT, verified QR matrix (error-correction H),\n  2. runs the ControlNet pipeline over several seeds,\n  3. DECODES every candidate and keeps one that reads back as `payload`,\n  4. returns None if none scan within `max_tries` (caller falls back to the\n     deterministic QR, so a batch never breaks).\n\nRuns on a CUDA GPU (Kaggle free T4). torch/diffusers are imported lazily, so\nimporting this module on a CPU-only box is harmless.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport cv2\nimport numpy as np\n\nfrom .qr_generator import build_qr\n\n# A prompt that pushes the model toward a clean, document-friendly result while\n# still using the AI generator. Override for creative/artistic codes.\nCLEAN_PROMPT = (\n    \"a clean flat black and white QR code on a plain white background, \"\n    \"crisp high contrast modules, minimalist, sharp edges, no decoration\"\n)\nCLEAN_NEGATIVE = (\n    \"colorful, painting, landscape, portrait, photo, texture, blurry, \"\n    \"watermark, text, logo, distorted, low contrast, cluttered background\"\n)\n\n_ART_PIPE = None\n_ART_DEVICE = None\n\n\ndef _device() -> str:\n    global _ART_DEVICE\n    if _ART_DEVICE is not None:\n        return _ART_DEVICE\n    try:\n        import torch\n        _ART_DEVICE = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    except Exception:\n        _ART_DEVICE = \"cpu\"\n    return _ART_DEVICE\n\n\ndef art_available() -> bool:\n    try:\n        import torch  # noqa: F401\n        import diffusers  # noqa: F401\n        return True\n    except Exception:\n        return False\n\n\ndef _condition_image(payload: str, module_px: int = 16,\n                     gray_bg: bool = False) -> np.ndarray:\n    \"\"\"Render the exact QR matrix as the ControlNet conditioning image.\n\n    QR Monster expects ~16px modules. v2 blends better on a gray (#808080)\n    background; for maximum readability we use white (gray_bg=False).\n    \"\"\"\n    qr = build_qr(payload, box_size=module_px, border=0,\n                  error_correction=\"H\")\n    if gray_bg:\n        # QR has white bg; replace outer non-module area with gray is complex,\n        # so we tint the whole background gray while keeping modules black.\n        gray = cv2.cvtColor(qr, cv2.COLOR_BGR2GRAY)\n        out = np.full_like(qr, 128)\n        dark = gray < 128\n        out[dark] = (0, 0, 0)\n        return out\n    return qr\n\n\ndef load_art_pipeline(\n        controlnet_id: str = \"monster-labs/control_v1p_sd15_qrcode_monster\",\n        base_model: str = \"stable-diffusion-v1-5/stable-diffusion-v1-5\"):\n    \"\"\"Lazily load and cache the ControlNet QR pipeline (fp16 on GPU).\"\"\"\n    global _ART_PIPE\n    if _ART_PIPE is not None:\n        return _ART_PIPE\n    import torch\n    from diffusers import (ControlNetModel,\n                           StableDiffusionControlNetPipeline,\n                           DPMSolverMultistepScheduler)\n\n    dtype = torch.float16 if _device() == \"cuda\" else torch.float32\n    controlnet = ControlNetModel.from_pretrained(\n        controlnet_id, torch_dtype=dtype)\n    pipe = StableDiffusionControlNetPipeline.from_pretrained(\n        base_model, controlnet=controlnet, torch_dtype=dtype,\n        safety_checker=None)\n    pipe.scheduler = DPMSolverMultistepScheduler.from_config(\n        pipe.scheduler.config)\n    if _device() == \"cuda\":\n        try:\n            pipe.enable_xformers_memory_efficient_attention()\n        except Exception:\n            pass\n        try:\n            pipe.enable_model_cpu_offload()\n        except Exception:\n            pipe.to(\"cuda\")\n    else:\n        pipe.to(\"cpu\")\n    _ART_PIPE = pipe\n    return pipe\n\n\ndef _decode_qr(img_bgr: np.ndarray, want: str) -> bool:\n    \"\"\"True if img contains a QR that decodes exactly to `want`.\"\"\"\n    from .pipeline import _decode_any_qr  # reuse strong verifier\n    h, w = img_bgr.shape[:2]\n    for scale in (1, 2):\n        im = img_bgr if scale == 1 else cv2.resize(\n            img_bgr, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)\n        got = _decode_any_qr(im)\n        if got and got.strip() == want.strip():\n            return True\n    return False\n\n\ndef build_artistic_qr(payload: str,\n                      prompt: str = CLEAN_PROMPT,\n                      negative_prompt: str = CLEAN_NEGATIVE,\n                      size: int = 768,\n                      controlnet_scale: float = 1.5,\n                      guidance_scale: float = 7.5,\n                      steps: int = 30,\n                      max_tries: int = 6,\n                      seed0: int = 1000,\n                      gray_bg: bool = False):\n    \"\"\"\n    Generate an AI (ControlNet) artistic QR for `payload`.\n\n    Returns a square BGR uint8 image that DECODES to `payload`, or None if no\n    seed produced a scannable code within `max_tries` (caller then falls back\n    to the deterministic QR).\n    \"\"\"\n    if not art_available() or _device() != \"cuda\":\n        return None\n    try:\n        import torch\n        from PIL import Image\n        pipe = load_art_pipeline()\n    except Exception as e:  # model download/OOM\n        print(f\"[qr_art] pipeline unavailable ({e}); using standard QR\")\n        return None\n\n    cond = _condition_image(payload, module_px=16, gray_bg=gray_bg)\n    cond_pil = Image.fromarray(cv2.cvtColor(cond, cv2.COLOR_BGR2RGB))\n    # make conditioning square at requested size\n    cond_pil = cond_pil.resize((size, size), Image.NEAREST)\n\n    for i in range(max_tries):\n        seed = seed0 + i\n        g = torch.Generator(device=\"cpu\").manual_seed(seed)\n        try:\n            result = pipe(\n                prompt=prompt, negative_prompt=negative_prompt,\n                image=cond_pil, width=size, height=size,\n                num_inference_steps=steps,\n                guidance_scale=guidance_scale,\n                controlnet_conditioning_scale=controlnet_scale,\n                generator=g,\n            ).images[0]\n        except Exception as e:\n            print(f\"[qr_art] generation failed (seed {seed}): {e}\")\n            continue\n        bgr = cv2.cvtColor(np.array(result), cv2.COLOR_RGB2BGR)\n        if _decode_qr(bgr, payload):\n            print(f\"[qr_art] scannable artistic QR at seed {seed} \"\n                  f\"(try {i+1}/{max_tries})\")\n            return bgr\n        print(f\"[qr_art] seed {seed} did not scan; retrying...\")\n    print(\"[qr_art] no scannable variant found; falling back to standard QR\")\n    return None\n",
      "replacer.py": "\"\"\"\nreplacer.py\n-----------\nPaste the freshly generated QR onto the certificate at the EXACT location,\nsize and rotation of the old QR.\n\n- Uses a perspective warp so rotated/skewed old QRs are matched exactly.\n- Uses NEAREST-neighbour rendering so QR modules stay hard-edged (critical for\n  scanning) -- no anti-aliased grey fuzz.\n- `mode`:\n    \"full_replace\" -> new QR (with white quiet zone) covers the entire old QR\n                      box including its black frame. Cleanest, most reliable.\n    \"keep_frame\"   -> the old black border is preserved; only the inner QR\n                      area is replaced (the new QR is shrunk inside the frame).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport cv2\nimport numpy as np\n\nfrom .detector import QRRegion, _order_points\n\n\ndef _scale_quad(quad: np.ndarray, scale: float) -> np.ndarray:\n    \"\"\"Scale a quad about its centre.\"\"\"\n    c = quad.mean(axis=0)\n    return c + (quad - c) * scale\n\n\ndef _warp_qr_to_cert(cert: np.ndarray, qr_img: np.ndarray,\n                     dst_quad: np.ndarray, supersample: int = 4) -> np.ndarray:\n    \"\"\"Perspective-warp qr_img onto cert at dst_quad (TL,TR,BR,BL).\n\n    The QR is first re-rendered at ~`supersample`x the destination size, then\n    warped with bilinear sampling (a good approximation of area-averaged\n    downscaling) and finally re-binarised to pure black/white inside the box.\n    This keeps every module hard-edged and scannable even when the on-cert\n    QR is very small (avoids the aliasing of a single huge->tiny NEAREST warp).\n    \"\"\"\n    h, w = cert.shape[:2]\n    ss = int(supersample)\n    side = float(np.mean([\n        np.linalg.norm(dst_quad[1] - dst_quad[0]),\n        np.linalg.norm(dst_quad[2] - dst_quad[1]),\n        np.linalg.norm(dst_quad[3] - dst_quad[2]),\n        np.linalg.norm(dst_quad[0] - dst_quad[3]),\n    ]))\n    target = max(8, int(round(side * ss)))\n    if qr_img.shape[0] != target:\n        interp = cv2.INTER_AREA if qr_img.shape[0] > target else cv2.INTER_NEAREST\n        qr_img = cv2.resize(qr_img, (target, target), interpolation=interp)\n\n    qh, qw = qr_img.shape[:2]\n    # Warp onto an ss-times-oversized overlay at the ss-scaled quad, then pull\n    # back down to cert resolution with INTER_AREA -> a proper area-averaged\n    # anti-aliased downscale of the perspective warp (keeps modules legible at\n    # small on-cert sizes).\n    big_W, big_H = w * ss, h * ss\n    big_quad = dst_quad.astype(np.float32) * ss\n    src = np.array([[0, 0], [qw - 1, 0], [qw - 1, qh - 1], [0, qh - 1]],\n                   dtype=np.float32)\n    M = cv2.getPerspectiveTransform(src, big_quad)\n    warped_big = cv2.warpPerspective(\n        qr_img, M, (big_W, big_H),\n        flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT,\n        borderValue=(255, 255, 255))\n    warped = cv2.resize(warped_big, (w, h), interpolation=cv2.INTER_AREA)\n\n    # mask of the QR box\n    mask = np.zeros((h, w), dtype=np.uint8)\n    cv2.fillConvexPoly(mask, dst_quad.astype(np.int32), 255)\n    mask = cv2.erode(mask, np.ones((3, 3), np.uint8), iterations=1)\n\n    # re-binarise the warped patch to pure black / white (hard module edges)\n    warped_gray = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)\n    _, binary = cv2.threshold(warped_gray, 160, 255, cv2.THRESH_BINARY)\n    binary_bgr = cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)\n\n    out = cert.copy()\n    m3 = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR) > 0\n    out[m3] = binary_bgr[m3]\n    return out\n\n\ndef replace_qr(cert: np.ndarray, region: QRRegion, qr_img: np.ndarray,\n               mode: str = \"full_replace\",\n               inner_margin_frac: float = 0.10,\n               cover_scale: float = 1.08) -> np.ndarray:\n    \"\"\"\n    Return a new certificate image with the new QR in place of the old one.\n\n    Everything outside the QR quad is left byte-for-byte untouched.\n\n    cover_scale (full_replace only): the new QR's white box is pasted over a\n    quad scaled outward by this factor so it reliably covers the ENTIRE old\n    black frame plus any dark halo / AI-inpaint remnant at the frame edge.\n    1.0 = exact old-quad footprint; ~1.08 fully hides the old frame (the white\n    background looks natural against the certificate and keeps a valid quiet\n    zone).\n    \"\"\"\n    quad = _order_points(region.quad)\n\n    if mode == \"keep_frame\":\n        # Replace only the inside of the black frame.\n        dst = _scale_quad(quad, 1.0 - 2 * inner_margin_frac)\n        # Give the pasted QR its own thin white border so modules never touch\n        # the black frame (keeps quiet-zone scanning rules).\n        qr = _add_white_border(qr_img, frac=0.06)\n    else:\n        # Full replace: scale the footprint outward a little so the new QR's\n        # white background fully hides the old frame + any edge halo.\n        dst = _scale_quad(quad, float(cover_scale))\n        qr = qr_img\n\n    return _warp_qr_to_cert(cert, qr, dst)\n\n\ndef _add_white_border(img: np.ndarray, frac: float = 0.08) -> np.ndarray:\n    h, w = img.shape[:2]\n    b = int(round(min(h, w) * frac))\n    return cv2.copyMakeBorder(img, b, b, b, b, cv2.BORDER_CONSTANT,\n                              value=(255, 255, 255))\n",
      "payload.py": "\"\"\"\npayload.py\n----------\nDecide what text/URL each certificate's NEW QR code encodes.\n\nSupported modes (set in config / notebook):\n\n  url   : base_url + slug + extension      -> e.g.\n          https://<your-username>.github.io/certificates/031-2024-25.png\n          (this is the recommended mode: host the OUTPUT images online --\n           e.g. GitHub Pages / your server / cloud storage -- and scanning the\n           QR opens that certificate in the phone's browser)\n\n  csv   : read a spreadsheet with columns: filename, qr_data\n          (per-certificate exact URL or text)\n\n  text  : fixed prefix + slug\n\n  copy  : copy whatever the old QR decoded to (requires the old QR to be\n          readable; falls back to url mode when it isn't)\n\n`slug` defaults to the certificate file's stem (filename without extension).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport csv\nimport os\n\n\ndef slug_from_filename(path: str) -> str:\n    return os.path.splitext(os.path.basename(path))[0]\n\n\ndef safe_slug(path: str) -> str:\n    \"\"\"URL/filename-safe stem. Replaces spaces, parentheses and other chars that\n    break URLs (e.g. 'image (1)' -> 'image_1') so the QR link and the saved\n    output file always match and resolve.\"\"\"\n    import re\n    stem = slug_from_filename(path)\n    stem = re.sub(r\"[^A-Za-z0-9._-]+\", \"_\", stem)\n    stem = re.sub(r\"_+\", \"_\", stem).strip(\"._-\")\n    return stem or \"certificate\"\n\n\ndef build_payload(path: str,\n                  mode: str = \"url\",\n                  base_url: str = \"\",\n                  extension: str = \".png\",\n                  text_prefix: str = \"\",\n                  csv_map: dict[str, str] | None = None,\n                  old_data: str | None = None) -> str:\n    slug = safe_slug(path)\n\n    if mode == \"csv\":\n        if csv_map and (os.path.basename(path) in csv_map or slug in csv_map):\n            return csv_map.get(os.path.basename(path)) or csv_map[slug]\n        # fall back to URL if the cert is missing from the spreadsheet\n        mode = \"url\"\n\n    if mode == \"copy\" and old_data:\n        return old_data\n\n    if mode == \"text\":\n        return f\"{text_prefix}{slug}\"\n\n    # default: url\n    base = base_url.rstrip(\"/\") + \"/\"\n    return f\"{base}{slug}{extension}\"\n\n\ndef load_csv_map(csv_path: str) -> dict[str, str]:\n    \"\"\"Read filename->qr_data mapping from a CSV (headers: filename,qr_data).\"\"\"\n    mapping: dict[str, str] = {}\n    with open(csv_path, newline=\"\", encoding=\"utf-8-sig\") as f:\n        for row in csv.DictReader(f):\n            fn = (row.get(\"filename\") or row.get(\"file\") or \"\").strip()\n            data = (row.get(\"qr_data\") or row.get(\"url\") or\n                    row.get(\"payload\") or \"\").strip()\n            if fn and data:\n                mapping[fn] = data\n                mapping[os.path.splitext(fn)[0]] = data\n    return mapping\n",
      "ai_extract.py": "\"\"\"\nai_extract.py\n-------------\nRead the candidate's details DIRECTLY FROM THE CERTIFICATE IMAGE using an AI\nvision-language model from Hugging Face (no OCR, no CSV, no hand-written rules).\n\nModel (runs on a free Kaggle T4 GPU; no num2words / tesseract needed):\n    Qwen/Qwen2-VL-2B-Instruct   (default) -- strong document/JSON reader\n    Qwen/Qwen2.5-VL-3B-Instruct (optional) -- slightly larger, also fine on T4\n\nThe model is shown the certificate and asked to return ONLY JSON:\n    name, parent, relation, aadhar, course, date_from, date_to, institute.\n\nFailures are NEVER silent: the returned dict carries an \"_error\" / \"_raw_ai\"\nkey so the notebook log / manifest show exactly what happened.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport re\nimport os\nimport gc\nimport importlib\nimport subprocess\nimport sys\n\n# Reduce CUDA fragmentation / OOM on the 16 GB T4 when a model is loaded more\n# than once in a session (must be set before torch first allocates).\nos.environ.setdefault(\"PYTORCH_CUDA_ALLOC_CONF\", \"expandable_segments:True\")\n\n_PROMPT = (\n    \"You are reading an Indian training-completion certificate IMAGE. \"\n    \"Look carefully at the whole image and extract these SIX fields. \"\n    \"Return ONLY a JSON object (no prose, no markdown, no code fence) with \"\n    \"exactly these keys:\\n\"\n    '{\"name\": \"<the candidate OWN name only: the words printed after Mr./Ms. '\n    'and BEFORE the token S/o or D/o or W/o. Do NOT include S/o, D/o, W/o, or '\n    'any name after them.>\", '\n    '\"parent\": \"<the father/mother name ONLY: the person name printed '\n    'immediately AFTER S/o (son of -> father), D/o (daughter of -> father) or '\n    'W/o (wife of -> husband), up to the next comma or bracket.>\", '\n    '\"relation\": \"<exactly S/o if the text says S/o, D/o if it says D/o, or '\n    'W/o if it says W/o>\", '\n    '\"aadhar\": \"<the candidate 12-digit Aadhaar number, digits only>\", '\n    '\"course\": \"<the full course/training name the candidate completed, e.g. '\n    'the text after Training in / Trainings in / Course>\", '\n    '\"date_from\": \"<course START date formatted dd/mm/yyyy>\", '\n    '\"date_to\": \"<course END date formatted dd/mm/yyyy>\", '\n    '\"institute\": \"<the COMPLETE institute name printed anywhere on the '\n    'certificate, written out in full exactly as shown (spell out every word; '\n    'do NOT shorten to initials - e.g. write Falih Consultancy Services (FCS) '\n    'Training Institute, not just FCS). Read the org name LETTER BY LETTER: '\n    'the printed letters l, i, s and h look alike - Falih is F-a-l-i-h (NOT '\n    'Falis, NOT Fahis). It differs on every certificate; never '\n    'assume or default to a name.>\"}\\n'\n    \"CRITICAL: 'S/o' = Son of (FATHER), 'D/o' = Daughter of (FATHER), 'W/o' = \"\n    \"Wife of (HUSBAND). These give the parent name. 'R/o' = Resident of (the \"\n    \"HOME ADDRESS / place, e.g. Rajendra Nager) \u2014 R/o is NOT a person and must \"\n    \"NEVER be put in name or parent. Read from the image only; do not invent; \"\n    \"use \\\"\\\" for anything unreadable; dates as dd/mm/yyyy.\"\n)\n\n# Small helper packages (NOT torch/transformers). If one is missing in the\n# kernel we self-heal by pip-installing it at runtime.\n_REQUIRED = [\n    (\"PIL\", \"pillow\"),\n    (\"accelerate\", \"accelerate\"),\n    (\"torchvision\", \"torchvision\"),\n]\n\n_MODEL = None\n_PROC = None\n_DEVICE = None\n_MODEL_ID = None\n\n\ndef _pip_install(pkg: str) -> bool:\n    try:\n        subprocess.check_call(\n            [sys.executable, \"-m\", \"pip\", \"install\", \"-q\", pkg])\n        return True\n    except Exception as e:\n        print(f\"[ai_extract] pip install {pkg} failed: {e}\")\n        return False\n\n\ndef _ensure_deps():\n    # Only self-heal where torch/transformers already exist (the GPU notebook).\n    try:\n        importlib.import_module(\"torch\")\n        importlib.import_module(\"transformers\")\n    except Exception:\n        return\n    for import_name, pip_name in _REQUIRED:\n        try:\n            importlib.import_module(import_name)\n        except Exception:\n            print(f\"[ai_extract] installing missing dependency '{pip_name}' ...\")\n            if _pip_install(pip_name):\n                try:\n                    importlib.import_module(import_name)\n                except Exception:\n                    pass\n\n\ndef device() -> str:\n    global _DEVICE\n    if _DEVICE is not None:\n        return _DEVICE\n    try:\n        import torch\n        _DEVICE = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    except Exception:\n        _DEVICE = \"cpu\"\n    return _DEVICE\n\n\ndef runtime_status() -> str:\n    try:\n        import torch  # noqa\n    except Exception as e:\n        return f\"torch NOT importable: {e}\"\n    try:\n        import transformers\n        tv = transformers.__version__\n    except Exception as e:\n        return f\"transformers NOT importable: {e}\"\n    dev = device()\n    gpu = \"\"\n    if dev == \"cuda\":\n        try:\n            import torch\n            gpu = f\" | GPU: {torch.cuda.get_device_name(0)}\"\n        except Exception:\n            pass\n    warn = \"\" if dev == \"cuda\" else \" | \u26a0\ufe0f CPU only \u2014 set Accelerator to GPU T4\"\n    return f\"torch ok | transformers {tv} | device={dev}{gpu}{warn}\"\n\n\ndef release_model():\n    \"\"\"Free the cached vision model and its CUDA memory. Call before a fresh\n    module load in the same session (e.g. after the test cell) to avoid OOM.\"\"\"\n    global _MODEL, _PROC, _MODEL_ID\n    _MODEL = None\n    _PROC = None\n    _MODEL_ID = None\n    gc.collect()\n    try:\n        import torch\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n            torch.cuda.ipc_collect()\n    except Exception:\n        pass\n\n\ndef _load(model_id: str):\n    \"\"\"Load (once) and cache the Qwen2-VL model + processor. Frees any stale\n    model first and retries once after emptying the CUDA cache on OOM.\"\"\"\n    global _MODEL, _PROC, _MODEL_ID\n    if _MODEL is not None and _MODEL_ID == model_id:\n        return _MODEL, _PROC\n    release_model()  # ensure no previous model (e.g. from the test cell) lingers\n    import torch\n    from transformers import AutoProcessor\n\n    def _build():\n        ModelCls = None\n        if \"Qwen2.5\" in model_id:\n            try:\n                from transformers import Qwen2_5_VLForConditionalGeneration as ModelCls\n            except Exception:\n                ModelCls = None\n        if ModelCls is None:\n            try:\n                from transformers import Qwen2VLForConditionalGeneration as ModelCls\n            except Exception:\n                ModelCls = None\n        if ModelCls is None:\n            from transformers import AutoModelForImageTextToText as ModelCls\n        dtype = torch.float16 if device() == \"cuda\" else torch.float32\n        proc = AutoProcessor.from_pretrained(model_id)\n        kwargs = dict(torch_dtype=dtype, low_cpu_mem_usage=True)\n        if device() == \"cuda\":\n            kwargs.update(device_map=\"auto\", offload_folder=\"/kaggle/working/offload\")\n        model = ModelCls.from_pretrained(model_id, **kwargs)\n        if \"device_map\" not in kwargs:\n            model = model.to(device())\n        model.eval()\n        return model, proc\n\n    try:\n        _MODEL, _PROC = _build()\n    except (RuntimeError, MemoryError) as e:\n        if \"out of memory\" in str(e).lower() or \"CUDA\" in str(e):\n            print(\"[ai_extract] GPU out of memory while loading \u2014 freeing cache \"\n                  \"and retrying once...\")\n            release_model()\n            try:\n                _MODEL, _PROC = _build()\n            except Exception as e2:\n                raise RuntimeError(\n                    f\"GPU out of memory even after freeing cache ({e2}). \"\n                    \"Restart the kernel (Run -> Restart) and run section 5 only, \"\n                    \"without running the section 4b test cell first.\") from e2\n        else:\n            raise\n    _MODEL_ID = model_id\n    return _MODEL, _PROC\n\n\ndef _norm_date(v: str) -> str:\n    if not v:\n        return \"\"\n    m = re.search(r\"(\\d{1,2})[^\\d]?(\\d{1,2})[^\\d]?(\\d{4})\", str(v))\n    if not m:\n        return str(v).strip()\n    try:\n        return f\"{int(m.group(1)):02d}/{int(m.group(2)):02d}/{m.group(3)}\"\n    except Exception:\n        return str(v).strip()\n\n\ndef _digits_aadhar(v: str) -> str:\n    if not v:\n        return \"\"\n    d = re.sub(r\"\\D\", \"\", str(v))\n    if len(d) > 12:\n        d = d[-12:]\n    if len(d) == 12:\n        return f\"{d[0:4]} {d[4:8]} {d[8:12]}\"\n    return str(v).strip()\n\n\ndef _clean_person_fields(d: dict) -> dict:\n    \"\"\"Deterministically fix name/parent/relation, regardless of model slips:\n    split at the S/o / D/o / W/o marker and never confuse it with R/o (address).\"\"\"\n    name = str(d.get(\"name\", \"\") or \"\").strip()\n    parent = str(d.get(\"parent\", \"\") or \"\").strip()\n    rel = str(d.get(\"relation\", \"\") or \"\").strip().upper()\n\n    relmap = {\"S\": \"S/o\", \"D\": \"D/o\", \"W\": \"W/o\"}\n    rel_letter = rel[0] if (rel and rel[0] in \"SDW\") else \"\"\n    if not rel_letter:\n        m = re.search(r\"\\b([SDW])\\s*/?\\s*[oO0]\\b\", f\"{name} {parent}\")\n        rel_letter = m.group(1) if m else \"\"\n\n    # If the relation marker (S/o, D/o, W/o) ended up INSIDE the name, split it\n    # and trust the marker found in the name (it directly introduces the parent).\n    m = re.search(r\"^(.*?)\\b([SDW])\\s*/?\\s*[oO0]\\s+(.*)$\", name, flags=re.S)\n    if m:\n        name = m.group(1).strip(\" ,./-\")\n        rel_letter = m.group(2)\n        tail = m.group(3)\n        # parent = up to the next R/o (address), Aadhaar, bracket, comma, newline\n        tail = re.split(r\"\\bR\\s*/\\s*o\\b|\\(?Aadha\", tail, flags=re.I)[0]\n        cand = tail.split(\",\")[0].split(\"(\")[0].strip(\" ,./-\")\n        if cand:\n            parent = cand\n\n    # If parent wrongly contains the R/o (residence/address) marker, cut there.\n    parent = re.split(r\"\\bR\\s*/\\s*o\\b\", parent, flags=re.I)[0].strip(\" ,./-\")\n    # Strip a leaked relation PREFIX from parent only if it is the actual\n    # \"S/o\" / \"D/o\" token (never strip a bare initial like \"D.\" in \"D. Jahangeen\").\n    parent = re.sub(r\"^[SDW]\\s*/\\s*[oO0]\\s*\", \"\", parent, flags=re.I).strip()\n    # Parent shouldn't contain Aadhaar / trailing bracket junk.\n    parent = re.split(r\"\\(?Aadha|\\d{4}\\s*\\d{4}\\s*\\d{4}\", parent)[0].strip(\" ,./-()\")\n\n    # Name must not keep a trailing relation marker either.\n    name = re.split(r\"\\b[SDW]\\s*/?\\s*[oO0]\\b\", name)[0].strip(\" ,./-\")\n    # Strip leading honorifics the model may keep (requires the dot, so real\n    # names like \"Msmena\" are never cut). Handles \"Mr./Ms.\", \"Ms.\", \"Dr.\" etc.\n    name = re.sub(\n        r\"^(?:Mr|Mrs|Ms|Miss|Dr|Prof)\\.\\s*(?:/\\s*(?:Mrs?|Ms|Miss|Dr|Prof)\\.\\s*)*\",\n        \"\", name).strip(\" ,./-\")\n\n    relation = relmap.get(rel_letter, \"\")\n    d[\"name\"] = name\n    d[\"parent\"] = parent\n    d[\"relation\"] = relation\n    return d\n\n\ndef _parse_json(text: str) -> dict:\n    if not text:\n        return {}\n    t = text.strip()\n    t = re.sub(r\"^```(?:json)?\\s*|\\s*```$\", \"\", t, flags=re.I).strip()\n    m = re.search(r\"\\{.*\\}\", t, re.S)\n    if not m:\n        return {}\n    data = None\n    for candidate in (m.group(0), \"{\" + m.group(0).strip(\"{} \") + \"}\"):\n        try:\n            data = json.loads(candidate)\n            break\n        except Exception:\n            data = None\n    if not isinstance(data, dict):\n        return {}\n    rel = str(data.get(\"relation\", \"\") or \"\").strip().upper()\n    rel = rel.replace(\"SON OF\", \"S/O\").replace(\"DAUGHTER OF\", \"D/O\")\n    rel = rel.replace(\"WIFE OF\", \"W/O\")\n    if rel and rel[0] in \"SDW\":\n        rel = rel[0] + \"/O\"\n    out = {\n        \"name\": str(data.get(\"name\", \"\") or \"\").strip(),\n        \"parent\": str(data.get(\"parent\", \"\") or \"\").strip(),\n        \"relation\": rel,\n        \"aadhar\": _digits_aadhar(data.get(\"aadhar\", \"\")),\n        \"course\": str(data.get(\"course\", \"\") or \"\").strip(),\n        \"date_from\": _norm_date(data.get(\"date_from\", \"\")),\n        \"date_to\": _norm_date(data.get(\"date_to\", \"\")),\n        \"institute\": str(data.get(\"institute\", \"\") or \"\").strip(),\n        \"_raw_ai\": text.strip(),\n    }\n    return _clean_person_fields(out)\n\n\ndef _generate(pil, model, proc, prompt: str | None = None,\n              max_new_tokens: int = 512) -> str:\n    import torch\n    messages = [{\"role\": \"user\", \"content\": [\n        {\"type\": \"image\"}, {\"type\": \"text\", \"text\": prompt or _PROMPT}]}]\n    text = proc.apply_chat_template(\n        messages, tokenize=False, add_generation_prompt=True)\n    inputs = proc(text=[text], images=[pil], padding=True,\n                  return_tensors=\"pt\").to(model.device)\n    with torch.no_grad():\n        gen = model.generate(**inputs, max_new_tokens=max_new_tokens,\n                             do_sample=False)\n    trimmed = gen[:, inputs[\"input_ids\"].shape[1]:]\n    return proc.batch_decode(trimmed, skip_special_tokens=True)[0]\n\n\n_VERIFY_PROMPT = (\n    \"Look ONLY at the name of the organization/institute that ISSUED this \"\n    \"certificate (printed in words, usually under or around the logo at the \"\n    \"top, and/or near the signatory at the bottom).\\n\"\n    \"A previous reading reported the institute as: \\\"{got}\\\"\\n\"\n    \"Re-read the printed name CAREFULLY, letter by letter, and fix any \"\n    \"misread letters. Printed 'l', 'i', 's' and 'h' look alike: for example \"\n    \"'Falih' is F-a-l-i-h (NOT Falis, NOT Fahis, NOT Faris). Also write the \"\n    \"name COMPLETE, exactly as printed \u2014 spell out every word; never shorten \"\n    \"to initials.\\n\"\n    \"Return ONLY a JSON object with exactly this key (no prose, no code \"\n    \"fence):\\n\"\n    '{\"institute\": \"<the complete, letter-checked correct institute name, '\n    'or \\\\\"\\\\\" if not readable>\"}'\n)\n\n\ndef _clean_institute(v: str) -> str:\n    v = str(v or \"\").strip().strip(\".,;\\\"'` \")\n    if v.lower() in (\"\", \"n/a\", \"na\", \"null\", \"none\", \"-\"):\n        return \"\"\n    return v\n\n\ndef _parse_verify(text: str) -> str:\n    \"\"\"Extract the corrected institute string from the verify-pass output.\"\"\"\n    if not text:\n        return \"\"\n    t = re.sub(r\"^```(?:json)?\\s*|\\s*```$\", \"\", text.strip(),\n               flags=re.I).strip()\n    if \"{\" in t:  # model answered in JSON (possibly slightly malformed)\n        m = re.search(r\"\\{.*\\}\", t, re.S)\n        blob = m.group(0) if m else t\n        try:\n            obj = json.loads(blob)\n            return _clean_institute(obj.get(\"institute\", \"\"))\n        except Exception:\n            mm = re.search(r'\"institute\"\\s*:\\s*\"([^\"]*)\"', blob)\n            if mm:\n                return _clean_institute(mm.group(1))\n            return \"\"\n    # plain-text answer: prefer the longest line (model may add a short\n    # preamble like \"The institute is:\" before the actual name)\n    best = \"\"\n    for line in t.splitlines():\n        line = line.strip().strip(\"{}\\\"'` \")\n        line = re.sub(r\"^institute\\s*:?\", \"\", line, flags=re.I).strip()\n        line = re.sub(r\"^(the\\s+)?(name|institute)\\s+(is|of)\\s*:?\\s*$\",\n                      \"\", line, flags=re.I).strip()\n        if _clean_institute(line) and len(line.strip()) >= 3 \\\n                and len(line) > len(best):\n            best = line\n    return _clean_institute(best)\n\n\ndef ai_extract_fields(img_bgr,\n                      model_id: str = \"Qwen/Qwen2-VL-2B-Instruct\"\n                      ) -> dict:\n    \"\"\"Run the AI vision model on one certificate (BGR ndarray) and return a\n    fields dict. On any failure returns a dict with '_error' (never raises).\"\"\"\n    try:\n        import numpy as np  # noqa\n        import cv2\n        from PIL import Image\n    except Exception as e:\n        return {\"_error\": f\"deps missing (numpy/Pillow/cv2): {e}\"}\n\n    global _MODEL, _PROC\n    model = proc = None\n    try:\n        _ensure_deps()\n        model, proc = _load(model_id)\n    except Exception as e:\n        name = getattr(e, \"name\", None)\n        m = re.search(r\"No module named ['\\\"]([a-zA-Z0-9_\\-]+)\", str(e))\n        pkg = name or (m.group(1) if m else None)\n        pip_map = {\"PIL\": \"pillow\", \"torchvision\": \"torchvision\",\n                   \"accelerate\": \"accelerate\"}\n        pip = pip_map.get(pkg)\n        if pip:\n            print(f\"[ai_extract] model needs '{pip}'; installing and retrying...\")\n            if _pip_install(pip):\n                _MODEL, _PROC = None, None\n                try:\n                    _ensure_deps()\n                    model, proc = _load(model_id)\n                except Exception as e2:\n                    return {\"_error\": f\"model load failed after installing \"\n                                      f\"{pip}: {e2}\"}\n            else:\n                return {\"_error\": f\"model load failed (missing {pip}): {e}\"}\n        else:\n            return {\"_error\": f\"model load failed: {e}\"}\n\n    rgb = img_bgr[:, :, ::-1]\n    # Up-scale small/phone screenshots a bit so the VLM has more pixels to read.\n    if rgb.shape[1] < 1000:\n        scale = 1000 / rgb.shape[1]\n        rgb = cv2.resize(rgb, None, fx=scale, fy=scale,\n                         interpolation=cv2.INTER_CUBIC)\n    import numpy as np\n    pil = Image.fromarray(np.ascontiguousarray(rgb)).convert(\"RGB\")\n\n    out = \"\"\n    try:\n        out = _generate(pil, model, proc)\n    except Exception as e:\n        try:  # one retry slightly larger\n            bigger = cv2.resize(rgb, None, fx=1.4, fy=1.4,\n                                interpolation=cv2.INTER_CUBIC)\n            pil = Image.fromarray(np.ascontiguousarray(bigger)).convert(\"RGB\")\n            out = _generate(pil, model, proc)\n        except Exception as e2:\n            return {\"_error\": f\"generation failed: {e2}\"}\n\n    fields = _parse_json(out)\n    if not fields:\n        return {\"_error\": \"model output was not JSON\", \"_raw_ai\": out.strip()}\n\n    # ---- Institute spelling check: one extra focused look so names like\n    # 'Falih' are not misread as 'Falis' (l/i/s confusion). Never fatal.\n    got_inst = str(fields.get(\"institute\", \"\") or \"\").strip()\n    if got_inst:\n        try:\n            v_raw = _generate(pil, model, proc,\n                              prompt=_VERIFY_PROMPT.replace(\"{got}\", got_inst),\n                              max_new_tokens=96)\n            fixed = _parse_verify(v_raw)\n            if fixed:\n                fields[\"institute_verified\"] = fixed\n                if fixed.strip().lower() != got_inst.strip().lower():\n                    print(f\"[ai_extract] institute spelling check: \"\n                          f\"'{got_inst}' -> '{fixed}'\")\n                    fields[\"institute\"] = fixed\n            fields[\"_raw_ai\"] = out.strip() + \"\\n\\n[institute re-check] \" \\\n                + v_raw.strip()[:220]\n        except Exception as e:\n            print(f\"[ai_extract] institute re-check skipped ({e})\")\n    return fields\n",
      "verification_page.py": "\"\"\"\nverification_page.py\n--------------------\nBuild a self-contained HTML certificate-verification page for each candidate.\nThe certificate image is EMBEDDED (base64), so uploading the single .html file\nis enough \u2014 when a user scans the QR (which links to this page) they see the\ncertificate image with the candidate's details beneath it.\n\nDetails (read off the certificate by the AI vision model):\n    Name of the candidate, Father/Mother Name, Aadhar No., Course Name,\n    Course Duration (dd/mm/yyyy -> dd/mm/yyyy), Institute.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport base64\nimport html as _html\n\n\ndef _b64_png(img_bgr) -> str:\n    import cv2\n    ok, buf = cv2.imencode(\".png\", img_bgr)\n    return base64.b64encode(buf.tobytes()).decode() if ok else \"\"\n\n\ndef _esc(x) -> str:\n    return _html.escape(str(x if x not in (None, \"\") else \"\u2014\"))\n\n\ndef build_page(fields: dict, img_bgr, cert_filename: str) -> str:\n    name = fields.get(\"name\", \"\")\n    parent = fields.get(\"parent\", \"\")\n    aadhar = fields.get(\"aadhar\", \"\")\n    course = fields.get(\"course\", \"\")\n    dfrom = fields.get(\"date_from\", \"\")\n    dto = fields.get(\"date_to\", \"\")\n    dates = f\"{_esc(dfrom)} &nbsp;to&nbsp; {_esc(dto)}\" if (dfrom or dto) else \"\u2014\"\n\n    img_b64 = _b64_png(img_bgr)\n\n    institute = fields.get(\"institute\", \"\")\n    rows = [\n        (\"Name of the candidate\", _esc(name)),\n        (\"Father/Mother Name\", _esc(parent)),\n        (\"Aadhar No.\", _esc(aadhar)),\n        (\"Course Name\", _esc(course)),\n        (\"Course Duration\", dates),\n        (\"Institute\", _esc(institute)),\n    ]\n    rows_html = \"\\n\".join(\n        f'<tr><th>{label} :</th><td>{val}</td></tr>' for label, val in rows)\n\n    cert_html = f'''\n    <div class=\"cert\">\n      <img alt=\"Certificate for {_esc(name or cert_filename)}\"\n           src=\"data:image/png;base64,{img_b64}\">\n      <div><span class=\"badge\">&#10003; Verified certificate</span></div>\n    </div>'''\n\n    info_html = f'''\n    <h2 class=\"sechd\">Candidate Information</h2>\n    <table>\n{rows_html}\n    </table>'''\n\n    return f\"\"\"<!DOCTYPE html>\n<html lang=\"en\">\n<head>\n<meta charset=\"utf-8\">\n<meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">\n<title>Certificate Verification - {_esc(name or cert_filename)}</title>\n<style>\n  *{{box-sizing:border-box}}\n  body{{font-family:Georgia,'Times New Roman',serif;background:#eef1f5;margin:0;\n       padding:24px 12px;color:#1b1b1b}}\n  .wrap{{max-width:760px;margin:0 auto}}\n  .head{{text-align:center;margin-bottom:16px}}\n  .head .org{{font-size:13px;letter-spacing:2px;color:#0b6b3a;font-weight:bold;\n       text-transform:uppercase}}\n  .head h1{{font-size:22px;margin:6px 0 2px;color:#10345e}}\n  .head .sub{{color:#666;font-size:13px}}\n  .card{{background:#fff;border:1px solid #d9dee6;border-radius:14px;\n       box-shadow:0 10px 30px rgba(0,0,0,.08);padding:22px 24px}}\n  table{{width:100%;border-collapse:collapse;font-size:16px}}\n  th{{text-align:left;width:46%;color:#333;font-weight:normal;padding:10px 8px;\n     border-bottom:1px solid #eef0f3;vertical-align:top}}\n  td{{padding:10px 8px;border-bottom:1px solid #eef0f3;font-weight:bold;\n     color:#111;word-break:break-word}}\n  .fcs{{margin:6px 0 10px;text-align:center;font-weight:bold;letter-spacing:1px;\n       color:#0b6b3a;font-size:14px;text-transform:uppercase}}\n  .sechd{{font-size:16px;color:#10345e;border-bottom:2px solid #e5e9f0;\n       padding-bottom:6px;margin:18px 0 8px}}\n  .cert{{margin-top:16px;text-align:center}}\n  .cert img{{max-width:100%;border:1px solid #d9dee6;border-radius:8px;\n       box-shadow:0 4px 14px rgba(0,0,0,.10)}}\n  .badge{{display:inline-block;background:#e8f6ee;color:#0b6b3a;border:1px solid #bfe6cf;\n       border-radius:999px;padding:5px 14px;font-size:12px;margin-top:8px}}\n  .foot{{text-align:center;color:#8a93a0;font-size:12px;margin-top:16px}}\n</style>\n</head>\n<body>\n<div class=\"wrap\">\n  <div class=\"head\">\n    <div class=\"org\">Government of Telangana &middot; TGMFC</div>\n    <h1>Certificate Verification</h1>\n    <div class=\"sub\">Skill Development Training certificate</div>\n  </div>\n  <div class=\"card\">\n{cert_html}\n{info_html}\n  </div>\n  <div class=\"foot\">Government of Telangana &middot; TGMFC &mdash;\n       Skill Development Training certificate verification</div>\n</div>\n</body>\n</html>\n\"\"\"\n",
      "ai_inpaint.py": "\"\"\"\nai_inpaint.py\n-------------\nAI image-generator step (diffusion INPAINTING) for the QR replacement.\n\nWhy inpainting + compositing instead of pure text-to-image?\n  * A generative model that \"draws\" a QR from a text prompt produces a\n    plausible-looking but INVALID QR (it cannot reproduce the exact\n    error-corrected module grid), and free-form editing can subtly alter\n    names/faces.\n  * Masked inpainting is given the certificate, a MASK limited to the old-QR\n    square, and your natural-language prompt. The model may only regenerate\n    pixels inside the mask, so every other pixel (design, text, face, seals)\n    is provably unchanged.\n  * We then composite the REAL, freshly computed QR (perspective-warped to the\n    exact quad) on top of the inpainted region -> the edit is AI-generated and\n    blended as requested, AND the QR is guaranteed crisp & scannable.\n\nModel (Hugging Face, runs on a Kaggle T4 16GB):\n    diffusers/stable-diffusion-xl-1.0-inpainting-0.1   (default, fp16)\n    stabilityai/stable-diffusion-2-inpainting         (lighter fallback)\n\nEverything is lazy: torch/diffusers are only imported when you actually run\nthe AI step, so the deterministic warp path needs no GPU and no big download.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport os\n\nimport cv2\nimport numpy as np\n\nfrom .replacer import replace_qr\n\n# The exact prompt requested by the user.\nDEFAULT_PROMPT = (\n    \"Generate an image of this certificate replacing only the old QR code \"\n    \"with a new crisp black and white QR code in the same position. \"\n    \"Do not change any design, do not change any information, do not change \"\n    \"the face or anything else. Just replace the old QR code with the new \"\n    \"QR code, keeping the identical certificate layout, colors, and text.\"\n)\nDEFAULT_NEGATIVE = (\n    \"invalid qr code, blurry qr, distorted text, changed name, altered face, \"\n    \"modified design, extra logo, watermark, deformed, low quality, artifacts\"\n)\n\n_PIPE = None\n_DEVICE = None\n\n\n# --------------------------------------------------------------------------- #\n# Model loading\n# --------------------------------------------------------------------------- #\ndef _device() -> str:\n    global _DEVICE\n    if _DEVICE is not None:\n        return _DEVICE\n    try:\n        import torch\n        _DEVICE = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    except Exception:\n        _DEVICE = \"cpu\"\n    return _DEVICE\n\n\ndef load_pipeline(model_id: str = \"diffusers/stable-diffusion-xl-1.0-inpainting-0.1\"):\n    \"\"\"Lazily load (and cache) the HF inpainting pipeline.\"\"\"\n    global _PIPE\n    if _PIPE is not None:\n        return _PIPE\n    import torch\n    from diffusers import AutoPipelineForInpainting\n\n    dtype = torch.float16 if _device() == \"cuda\" else torch.float32\n    kw = dict(torch_dtype=dtype)\n    try:\n        pipe = AutoPipelineForInpainting.from_pretrained(model_id, **kw)\n    except Exception:\n        # some repos need the variant flag only on GPU fp16\n        kw[\"variant\"] = \"fp16\"\n        pipe = AutoPipelineForInpainting.from_pretrained(model_id, **kw)\n\n    if _device() == \"cuda\":\n        try:\n            pipe.enable_xformers_memory_efficient_attention()\n        except Exception:\n            pass\n        try:\n            pipe.enable_model_cpu_offload()\n        except Exception:\n            pipe.to(\"cuda\")\n    else:\n        pipe.to(\"cpu\")\n    pipe.set_progress_bar_config(disable=False)\n    _PIPE = pipe\n    return pipe\n\n\ndef ai_available() -> bool:\n    try:\n        import torch  # noqa: F401\n        import diffusers  # noqa: F401\n        return True\n    except Exception:\n        return False\n\n\n# --------------------------------------------------------------------------- #\n# Geometry helpers\n# --------------------------------------------------------------------------- #\ndef _mask_from_quad(shape_hw: tuple[int, int], quad: np.ndarray,\n                    dilate_px: int = 6) -> np.ndarray:\n    h, w = shape_hw\n    mask = np.zeros((h, w), np.uint8)\n    cv2.fillConvexPoly(mask, quad.astype(np.int32), 255)\n    if dilate_px:\n        k = cv2.getStructuringElement(\n            cv2.MORPH_RECT, (2 * dilate_px + 1, 2 * dilate_px + 1))\n        mask = cv2.dilate(mask, k)\n    return mask\n\n\ndef _crop_bbox(quad: np.ndarray, h: int, w: int, margin: int):\n    x0, y0 = quad.min(axis=0)\n    x1, y1 = quad.max(axis=0)\n    x0 = int(max(0, x0 - margin)); y0 = int(max(0, y0 - margin))\n    x1 = int(min(w, x1 + margin)); y1 = int(min(h, y1 + margin))\n    return x0, y0, x1, y1\n\n\ndef _round8(v: int) -> int:\n    return max(8, int(round(v / 8.0)) * 8)\n\n\n# --------------------------------------------------------------------------- #\n# Main entry\n# --------------------------------------------------------------------------- #\ndef ai_replace_qr(cert_bgr: np.ndarray, region, qr_bgr: np.ndarray,\n                  prompt: str = DEFAULT_PROMPT,\n                  negative_prompt: str = DEFAULT_NEGATIVE,\n                  model_id: str = \"diffusers/stable-diffusion-xl-1.0-inpainting-0.1\",\n                  steps: int = 30, guidance: float = 8.0, seed: int = 0,\n                  margin: int = 120, mode: str = \"inpaint_then_paste\",\n                  replace_mode: str = \"full_replace\",\n                  cover_scale: float = 1.08):\n    \"\"\"\n    Run AI inpainting over the old-QR mask with `prompt`, then (default)\n    composite the real QR so it scans.\n\n    mode:\n      \"inpaint_then_paste\" (default, recommended):\n            AI regenerates/blends the QR region per the prompt, then the\n            deterministic QR is warped on top -> scannable + AI-edited.\n      \"generative_only\":\n            return the raw model output (NOT guaranteed to scan -- for\n            comparison/experimentation only).\n\n    Returns (out_bgr, info_dict). If the model can't load, falls back to the\n    deterministic warp and info[\"fallback\"]=True.\n    \"\"\"\n    h, w = cert_bgr.shape[:2]\n    quad = region.quad.astype(np.float32)\n\n    if not ai_available():\n        out = replace_qr(cert_bgr, region, qr_bgr, mode=replace_mode,\n                         cover_scale=cover_scale)\n        return out, {\"ran\": False, \"fallback\": \"diffusers/torch not installed\"}\n\n    # The SDXL inpainting model is GPU-only in practice: never attempt it on\n    # CPU (multi-GB download, minutes/image, likely OOM). Fall back instantly.\n    if _device() != \"cuda\":\n        out = replace_qr(cert_bgr, region, qr_bgr, mode=replace_mode,\n                         cover_scale=cover_scale)\n        return out, {\"ran\": False,\n                     \"fallback\": \"AI inpainting needs a CUDA GPU; used warp\"}\n\n    try:\n        import torch\n    except Exception:\n        torch = None\n    try:\n        from PIL import Image\n    except Exception as e:\n        out = replace_qr(cert_bgr, region, qr_bgr, mode=replace_mode,\n                         cover_scale=cover_scale)\n        return out, {\"ran\": False, \"fallback\": f\"PIL missing: {e}\"}\n\n    try:\n        pipe = load_pipeline(model_id)\n    except Exception as e:  # download/OOM -> graceful fallback\n        out = replace_qr(cert_bgr, region, qr_bgr, mode=replace_mode,\n                         cover_scale=cover_scale)\n        return out, {\"ran\": False, \"fallback\": f\"model load failed: {e}\"}\n\n    mask = _mask_from_quad((h, w), quad, dilate_px=8)\n    x0, y0, x1, y1 = _crop_bbox(quad, h, w, margin)\n    crop = cert_bgr[y0:y1, x0:x1].copy()\n    cmask = mask[y0:y1, x0:x1].copy()\n\n    ch, cw = crop.shape[:2]\n    W8, H8 = _round8(cw), _round8(ch)\n\n    pil_img = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)).resize((W8, H8))\n    pil_mask = Image.fromarray(cmask).resize((W8, H8), Image.NEAREST)\n\n    gen = torch.Generator(device=\"cpu\").manual_seed(seed) if torch is not None else None\n    result = pipe(\n        prompt=prompt, negative_prompt=negative_prompt,\n        image=pil_img, mask_image=pil_mask,\n        height=H8, width=W8,\n        guidance_scale=guidance, num_inference_steps=steps,\n        generator=gen,\n    ).images[0]\n    result = result.resize((cw, ch), Image.LANCZOS)\n    inp_crop = cv2.cvtColor(np.array(result), cv2.COLOR_RGB2BGR)\n\n    # Strictly apply ONLY inside the mask -> rest of certificate untouched.\n    m3 = cv2.cvtColor(cmask, cv2.COLOR_GRAY2BGR) > 0\n    edited_crop = crop.copy()\n    edited_crop[m3] = inp_crop[m3]\n\n    edited = cert_bgr.copy()\n    edited[y0:y1, x0:x1] = edited_crop\n\n    info = {\"ran\": True, \"model\": model_id, \"device\": _device(),\n            \"steps\": steps, \"crop\": (x0, y0, x1, y1), \"mode\": mode}\n\n    if mode == \"generative_only\":\n        return edited, info\n\n    # Composite the REAL QR (guaranteed scannable) over the AI region.\n    final = replace_qr(edited, region, qr_bgr, mode=replace_mode,\n                            cover_scale=cover_scale)\n    info[\"qr_composited\"] = True\n    return final, info\n",
      "pipeline.py": "\"\"\"\npipeline.py\n-----------\nEnd-to-end batch workflow:\n\n  for every certificate image in an input folder:\n     1. DETECT  the old QR  (WeChat CNN model from Hugging Face -> OpenCV ->\n                geometric contour finder)\n     2. BUILD   the payload (URL/text/CSV that scanning should open)\n     3. GENERATE a brand-new crisp QR code\n     4. REPLACE the old QR with the new one at the exact same place/size/angle\n                (nothing else on the certificate is touched)\n     5. VERIFY  the new QR is actually decodable on the output image\n     6. SAVE    the new certificate\n\n  then: write a manifest.csv and ZIP all outputs.\n\nRuns on CPU or GPU identically (the geometry work is CPU; the CNN detector is\ntiny). Designed for 200+ certificates in one pass on Kaggle.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport csv\nimport io\nimport os\nimport zipfile\nfrom dataclasses import dataclass, field\n\nimport cv2\nimport numpy as np\n\nfrom .detector import (detect_qr, draw_debug,\n                       fallback_region_from_regions, fallback_region_default)\nfrom .payload import build_payload, load_csv_map, slug_from_filename, safe_slug\nfrom .qr_generator import build_qr\nfrom .qr_art import build_artistic_qr\nfrom .replacer import replace_qr\nfrom .ai_extract import ai_extract_fields\nfrom .verification_page import build_page\nfrom .ai_inpaint import (\n    ai_replace_qr, ai_available, DEFAULT_PROMPT, DEFAULT_NEGATIVE,\n)\n\nIMG_EXTS = (\".png\", \".jpg\", \".jpeg\", \".webp\", \".bmp\", \".tif\", \".tiff\")\n\n\n@dataclass\nclass Result:\n    filename: str\n    status: str                       # \"ok\" | \"recheck\" | \"failed\"\n    payload: str = \"\"\n    method: str = \"\"\n    qr_side_px: float = 0.0\n    verified: bool = False\n    ai_used: bool = False\n    out_name: str = \"\"\n    slug: str = \"\"\n    fields: dict = None\n    extract_method: str = \"\"\n    note: str = \"\"\n\n\n@dataclass\nclass Config:\n    input_dir: str = \"input\"\n    output_dir: str = \"output\"\n    model_dir: str = \"models/wechat\"\n    zip_path: str = \"certificates_with_new_qr.zip\"\n    payload_mode: str = \"url\"          # url | text | csv | copy\n    base_url: str = \"https://example.github.io/certificates/\"\n    url_extension: str = \".png\"\n    # QR links to a per-candidate VERIFICATION PAGE (.html) that shows the\n    # certificate image with the candidate details (read off the certificate by\n    # the AI vision model) beneath it. False -> QR links directly to the image.\n    verification_page: bool = True\n    # AI vision-language model that READS every field off the certificate image\n    # (no OCR / no CSV). Requires a GPU on Kaggle (T4). Empty fields if it fails.\n    ai_extract_model: str = \"Qwen/Qwen2-VL-2B-Instruct\"\n    text_prefix: str = \"\"\n    csv_path: str | None = None\n    qr_error_correction: str = \"M\"\n    qr_box_size: int = 20\n    qr_border: int = 4\n    replace_mode: str = \"full_replace\"  # full_replace | keep_frame\n    qr_cover_scale: float = 1.08        # >1 = white box grows to hide old frame\n    output_ext: str = \".png\"\n    jpeg_quality: int = 95\n    save_debug: bool = True\n    use_cnn: bool = True\n    # ---- QR generator backend ----\n    # \"standard\"  -> plain deterministic qrcode (always scans; recommended for\n    #                official certificates)\n    # \"artistic\"  -> HF QR Code Monster ControlNet (AI art-QR; auto-verified and\n    #                falls back to standard if no seed scans). GPU recommended.\n    qr_backend: str = \"standard\"\n    art_prompt: str = \"\"       # \"\" -> qr_art.CLEAN_PROMPT\n    art_controlnet_scale: float = 1.5\n    art_size: int = 768\n    art_tries: int = 6\n    # ---- AI image-generator (diffusion inpainting) ----\n    use_ai: bool = False\n    ai_model: str = \"diffusers/stable-diffusion-xl-1.0-inpainting-0.1\"\n    ai_prompt: str = DEFAULT_PROMPT\n    ai_negative: str = DEFAULT_NEGATIVE\n    ai_steps: int = 30\n    ai_guidance: float = 8.0\n    ai_seed: int = 0\n    ai_margin: int = 120\n    ai_mode: str = \"inpaint_then_paste\"   # inpaint_then_paste | generative_only\n    # If a QR can't be located on one certificate, place the new QR at the\n    # batch-derived template position (works because all certs share a layout).\n    fallback_placement: bool = True\n\n\ndef list_certificates(input_dir: str) -> list[str]:\n    out = []\n    for root, _dirs, files in os.walk(input_dir):\n        for f in sorted(files):\n            if f.lower().endswith(IMG_EXTS):\n                out.append(os.path.join(root, f))\n    return sorted(out)\n\n\n_WECHAT_DECODER = None\n\n\ndef _get_wechat_decoder(model_dir: str):\n    \"\"\"A WeChat CNN decoder is far stronger on small/dense QRs (used both to\n    read old QRs and to verify new ones).\"\"\"\n    global _WECHAT_DECODER\n    if _WECHAT_DECODER is not None:\n        return _WECHAT_DECODER\n    if not hasattr(cv2, \"wechat_qrcode_WeChatQRCode\"):\n        return None\n    paths = [os.path.join(model_dir, f) for f in\n             (\"detect.prototxt\", \"detect.caffemodel\",\n              \"sr.prototxt\", \"sr.caffemodel\")]\n    if not all(os.path.exists(p) for p in paths):\n        return None\n    try:\n        _WECHAT_DECODER = cv2.wechat_qrcode_WeChatQRCode(*paths)\n    except Exception:\n        _WECHAT_DECODER = None\n    return _WECHAT_DECODER\n\n\ndef _decode_any_qr(img: np.ndarray, model_dir: str = \"models/wechat\") -> str:\n    \"\"\"Best-effort decode to VERIFY the new QR scans. Returns text or ''.\n\n    Tries the strong WeChat CNN decoder first (it reads small/dense QRs that\n    OpenCV's classic decoder misses), then OpenCV ArUco/classic at several\n    scales.\n    \"\"\"\n    h, w = img.shape[:2]\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img\n\n    wd = _get_wechat_decoder(model_dir)\n    if wd is not None:\n        for scale in (1, 2, 3):\n            src = gray if scale == 1 else cv2.resize(\n                gray, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)\n            try:\n                res, _pts = wd.detectAndDecode(src)\n                if len(res) > 0 and res[0]:\n                    return res[0]\n            except Exception:\n                pass\n\n    dets = []\n    if hasattr(cv2, \"QRCodeDetectorAruco\"):\n        try:\n            dets.append(cv2.QRCodeDetectorAruco())\n        except Exception:\n            pass\n    dets.append(cv2.QRCodeDetector())\n    for scale in (1, 2, 3, 4):\n        src = gray if scale == 1 else cv2.resize(\n            gray, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)\n        for d in dets:\n            try:\n                data, _, _ = d.detectAndDecode(src)\n            except Exception:\n                data = \"\"\n            if data:\n                return data\n    return \"\"\n\n\ndef process_one(path: str, cfg: Config,\n                csv_map: dict | None = None,\n                region_override=None) -> tuple[np.ndarray | None, Result]:\n    name = os.path.basename(path)\n    slug = safe_slug(path)\n    res = Result(filename=name, status=\"failed\", slug=slug,\n                 out_name=slug + cfg.output_ext)\n    img = cv2.imread(path, cv2.IMREAD_COLOR)\n    if img is None:\n        res.note = \"could not read image\"\n        return None, res\n\n    # 0) The AI vision model READS all candidate details off the certificate\n    #    image (name, parent, aadhar, course, dates, institute). No OCR / CSV.\n    fields = {}\n    extract_used = \"\"\n    try:\n        fields = ai_extract_fields(img, model_id=cfg.ai_extract_model)\n        if fields and not fields.get(\"_error\"):\n            extract_used = \"AI-VLM\"\n        elif fields.get(\"_error\"):\n            extract_used = \"AI-FAILED\"\n            print(f\"   \u26a0\ufe0f  AI extraction failed for {name}: {fields['_error']}\")\n    except Exception as e:\n        fields = {\"_error\": str(e)}\n        extract_used = \"AI-FAILED\"\n        print(f\"   \u26a0\ufe0f  AI extraction crashed for {name}: {e}\")\n    res.fields = fields\n    res.extract_method = extract_used\n\n    # The QR links to the verification PAGE (.html) when enabled, else the raw\n    # image (cfg.url_extension).\n    link_ext = \".html\" if cfg.verification_page else cfg.url_extension\n\n    # 1) locate the old QR: detect it, or use a provided fallback placement.\n    region = region_override\n    if region is None:\n        region = detect_qr(img, model_dir=cfg.model_dir, use_cnn=cfg.use_cnn)\n    if region is None:\n        res.note = \"old QR not found\"\n        return img, res\n\n    res.method = region.method\n    res.qr_side_px = round(region.side_px, 1)\n\n    # old payload (only needed for 'copy' mode)\n    old_data = _decode_any_qr(img, cfg.model_dir) if cfg.payload_mode == \"copy\" else None\n\n    # 2) payload (links to the verification page or the raw image)\n    payload = build_payload(\n        path, mode=cfg.payload_mode, base_url=cfg.base_url,\n        extension=link_ext, text_prefix=cfg.text_prefix,\n        csv_map=csv_map, old_data=old_data)\n    res.payload = payload\n\n    # 3) generate fresh QR at high resolution.\n    #    Optional AI \"artistic QR\" backend (HF QR Code Monster ControlNet):\n    #    returns an AI-styled QR that decodes to `payload`, else None -> we\n    #    transparently fall back to the deterministic standard QR.\n    qr_img = None\n    if cfg.qr_backend == \"artistic\":\n        from .qr_art import CLEAN_PROMPT\n        qr_img = build_artistic_qr(\n            payload,\n            prompt=cfg.art_prompt or CLEAN_PROMPT,\n            controlnet_scale=cfg.art_controlnet_scale,\n            size=cfg.art_size, max_tries=cfg.art_tries)\n        if qr_img is not None:\n            res.note = \"AI artistic QR (verified)\"\n    if qr_img is None:\n        qr_img = build_qr(payload, box_size=cfg.qr_box_size,\n                          border=cfg.qr_border,\n                          error_correction=cfg.qr_error_correction)\n\n    # 4) REPLACE the old QR.\n    #    - AI path: masked diffusion inpainting (prompt-driven), then the\n    #      real QR is composited so it is guaranteed scannable.\n    #    - deterministic path: exact perspective warp.\n    if cfg.use_ai:\n        out, ai_info = ai_replace_qr(\n            img, region, qr_img,\n            prompt=cfg.ai_prompt, negative_prompt=cfg.ai_negative,\n            model_id=cfg.ai_model, steps=cfg.ai_steps,\n            guidance=cfg.ai_guidance, seed=cfg.ai_seed,\n            margin=cfg.ai_margin, mode=cfg.ai_mode,\n            replace_mode=cfg.replace_mode,\n            cover_scale=cfg.qr_cover_scale)\n        res.ai_used = bool(ai_info.get(\"ran\"))\n        if not ai_info.get(\"ran\"):\n            res.method += \"+ai-fallback(warp)\"\n    else:\n        out = replace_qr(img, region, qr_img, mode=cfg.replace_mode,\n                         cover_scale=cfg.qr_cover_scale)\n        ai_info = {}\n\n    # 5) verify the new QR decodes on the output\n    decoded = _decode_any_qr(out, cfg.model_dir)\n    res.verified = bool(decoded)\n    art_tag = \"AI artistic QR; \" if (cfg.qr_backend == \"artistic\"\n                                    and \"artistic\" in res.note) else \"\"\n    if not decoded:\n        res.status = \"recheck\"\n        res.note = art_tag + \"new QR not auto-verified (usually still scans; check debug)\"\n    else:\n        res.status = \"ok\"\n        if res.ai_used:\n            res.note = art_tag + \"verified scannable (AI inpaint+QR)\"\n        else:\n            res.note = art_tag + \"verified scannable\"\n\n    return out, res\n\n\ndef run(cfg: Config) -> dict:\n    os.makedirs(cfg.output_dir, exist_ok=True)\n    debug_dir = os.path.join(cfg.output_dir, \"_debug\")\n    if cfg.save_debug:\n        os.makedirs(debug_dir, exist_ok=True)\n\n    csv_map = None\n    if cfg.payload_mode == \"csv\" and cfg.csv_path:\n        csv_map = load_csv_map(cfg.csv_path)\n\n    certs = list_certificates(cfg.input_dir)\n\n    if cfg.payload_mode == \"url\":\n        placeholder_markers = (\"YOUR-USERNAME\", \"example.github.io\",\n                               \"example.com\", \"REPLACE\", \"your-username\")\n        if any(m.lower() in (cfg.base_url or \"\").lower()\n               for m in placeholder_markers):\n            print(\"=\" * 70)\n            print(\"\u26a0\ufe0f  WARNING: BASE_URL still looks like a PLACEHOLDER.\")\n            print(f\"    base_url = {cfg.base_url}\")\n            print(\"    The QR codes WILL scan, but they point at a URL that\")\n            print(\"    does not exist yet -> your phone shows a 404 page.\")\n            print(\"    1) Set base_url to the address where you will upload the\")\n            print(\"       OUTPUT images (e.g. GitHub Pages: create a repo, turn\")\n            print(\"       on Settings->Pages, upload the finished PNGs).\")\n            print(\"    2) Re-run. See README 'Hosting' / notebook section 8.\")\n            print(\"=\" * 70)\n\n    # ---- detection pass: locate the QR on every certificate so that a cert\n    # where detection fails can inherit the position from its template (all\n    # certificates in a batch share a layout).\n    detected: dict[str, object] = {}\n    template: list = []\n    for path in certs:\n        img = cv2.imread(path, cv2.IMREAD_COLOR)\n        if img is None:\n            continue\n        reg = detect_qr(img, model_dir=cfg.model_dir, use_cnn=cfg.use_cnn)\n        h, w = img.shape[:2]\n        detected[path] = reg\n        if reg is not None and \"fallback\" not in reg.method:\n            template.append((reg, h, w))\n\n    results: list[Result] = []\n    ok = recheck = failed = 0\n    ai_ok = ai_fail = 0\n\n    zip_path = cfg.zip_path\n    with zipfile.ZipFile(zip_path, \"w\", zipfile.ZIP_DEFLATED) as zf:\n        for path in certs:\n            img = cv2.imread(path, cv2.IMREAD_COLOR)\n            override = None\n            if detected.get(path) is None and cfg.fallback_placement:\n                # inherit QR position from the other certificates / template\n                override = fallback_region_from_regions(img, template)\n                if override is None:\n                    override = fallback_region_default(img)\n            out, res = process_one(path, cfg, csv_map, region_override=override)\n            if res.extract_method == \"AI-VLM\":\n                ai_ok += 1\n            else:\n                ai_fail += 1\n            if override is not None and res.status in (\"ok\", \"recheck\"):\n                res.note += f\" | QR placed via {override.method}\"\n            results.append(res)\n            if res.status == \"ok\":\n                ok += 1\n            elif res.status == \"recheck\":\n                recheck += 1\n            else:\n                failed += 1\n\n            if out is not None and res.status in (\"ok\", \"recheck\"):\n                out_name = res.out_name\n                out_path = os.path.join(cfg.output_dir, out_name)\n                _write_image(out, out_path, cfg)\n                zf.write(out_path, arcname=\"img/\" + out_name)\n\n                # Per-candidate VERIFICATION PAGE (self-contained HTML).\n                if cfg.verification_page:\n                    page = build_page(res.fields or {}, out, out_name)\n                    page_name = res.slug + \".html\"\n                    page_path = os.path.join(cfg.output_dir, page_name)\n                    with open(page_path, \"w\", encoding=\"utf-8\") as fh:\n                        fh.write(page)\n                    zf.write(page_path, arcname=page_name)\n\n                if cfg.save_debug and res.status == \"recheck\":\n                    reg = detect_qr(out, model_dir=cfg.model_dir,\n                                    use_cnn=False)\n                    dbg = draw_debug(out, reg) if reg else out\n                    cv2.imwrite(os.path.join(debug_dir, \"dbg_\" + out_name), dbg)\n            elif out is not None and cfg.save_debug:\n                cv2.imwrite(os.path.join(\n                    debug_dir, \"FAILED_\" + safe_slug(path) + cfg.output_ext), out)\n\n            tag = \"AI\" if res.ai_used else (\"ai-fb\" if \"+ai-fallback\" in res.method else \"  \")\n            print(f\"[{res.status:7s}] {tag} {res.filename:36s} \"\n                  f\"{res.method:24s} {res.qr_side_px:6.1f}px  {res.payload}\")\n\n    # manifest\n    manifest = os.path.join(cfg.output_dir, \"manifest.csv\")\n    with open(manifest, \"w\", newline=\"\", encoding=\"utf-8\") as f:\n        wr = csv.writer(f)\n        wr.writerow([\"filename\", \"status\", \"ai_used\", \"method\", \"qr_side_px\",\n                     \"verified\", \"qr_payload\", \"name\", \"parent\", \"aadhar\",\n                     \"course\", \"date_from\", \"date_to\", \"institute\",\n                     \"extracted_by\", \"note\"])\n        for r in results:\n            f = r.fields or {}\n            wr.writerow([r.filename, r.status, r.ai_used, r.method,\n                         r.qr_side_px, r.verified, r.payload,\n                         f.get(\"name\", \"\"), f.get(\"parent\", \"\"),\n                         f.get(\"aadhar\", \"\"), f.get(\"course\", \"\"),\n                         f.get(\"date_from\", \"\"), f.get(\"date_to\", \"\"),\n                         f.get(\"institute\", \"\"), r.extract_method, r.note])\n\n    summary = {\"total\": len(certs), \"ok\": ok, \"recheck\": recheck,\n               \"failed\": failed, \"fields_extracted\": ai_ok,\n               \"fields_failed\": ai_fail,\n               \"zip\": os.path.abspath(zip_path),\n               \"manifest\": os.path.abspath(manifest)}\n    print(\"\\n===== SUMMARY =====\")\n    for k, v in summary.items():\n        print(f\"{k:16s}: {v}\")\n    if ai_fail:\n        print(\"\\n\u26a0\ufe0f  The AI could NOT read fields from \"\n              f\"{ai_fail} certificate(s) -> those pages show '\u2014'.\")\n        print(\"   Check: Accelerator = GPU T4; and run the 'Test AI extraction'\")\n        print(\"   cell to see the raw model output / error.\")\n    return summary\n\n\ndef _write_image(img: np.ndarray, path: str, cfg: Config):\n    ext = cfg.output_ext.lower()\n    if ext in (\".jpg\", \".jpeg\"):\n        cv2.imwrite(path, img, [cv2.IMWRITE_JPEG_QUALITY, cfg.jpeg_quality])\n    else:\n        cv2.imwrite(path, img)\n"
    }
    for name, content in SRC.items():
        with open(os.path.join("/kaggle/working/certbot/src", name), "w",
                  encoding="utf-8") as f:
            f.write(content)
        print("wrote", name, len(content), "chars")
    print("pipeline has qr_cover_scale:", "qr_cover_scale" in SRC["pipeline.py"])

    # publish helper (GitHub Pages)
    with open("/kaggle/working/certbot/publish_to_github_pages.py", "w",
              encoding="utf-8") as f:
        f.write("#!/usr/bin/env python3\n\"\"\"\npublish_to_github_pages.py\n--------------------------\nUpload the finished certificate images to a GitHub Pages repository so the QR\nlinks resolve (scan -> certificate opens in the browser). Self-verifying:\nafter each upload it polls the public Pages URL until the image is live.\n\nToken: a GitHub Personal Access Token (classic) with the `repo` scope.\nProvided via GITHUB_TOKEN / GH_TOKEN env var (on Kaggle, add a notebook Secret\nnamed GITHUB_TOKEN), or pasted at the prompt.\n\nCreate a token: https://github.com/settings/tokens  (Generate new token\n(classic) -> tick \"repo\").\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport base64\nimport os\nimport time\nimport zipfile\n\ntry:\n    import requests\nexcept Exception:  # pragma: no cover\n    requests = None\n\nAPI = \"https://api.github.com\"\n\n\ndef _get_token() -> str:\n    tok = os.environ.get(\"GITHUB_TOKEN\") or os.environ.get(\"GH_TOKEN\") or \"\"\n    if not tok:\n        tok = input(\"Paste your GitHub token (repo scope): \").strip()\n    return tok\n\n\ndef publish(zip_path: str, owner: str, repo: str, token: str,\n            subdir: str = \"img\", branch: str = \"main\",\n            verify: bool = True, verbose: bool = True):\n    if requests is None:\n        raise SystemExit(\"'requests' is required (pip install requests).\")\n\n    files = []   # (name, content, upload_dir)\n    with zipfile.ZipFile(zip_path) as z:\n        for name in z.namelist():\n            base_name = os.path.basename(name)\n            low = name.lower()\n            if low.endswith((\".png\", \".jpg\", \".jpeg\", \".webp\")):\n                # images stay under img/ (whether the zip root or img/ prefix)\n                files.append((base_name, z.read(name), subdir))\n            elif low.endswith(\".html\"):\n                # verification pages live at the repo root (QR URLs point there)\n                files.append((base_name, z.read(name), \"\"))\n\n    headers = {\"Authorization\": f\"Bearer {token}\",\n               \"Accept\": \"application/vnd.github+json\",\n               \"User-Agent\": \"cert-qr-bot-publisher\"}\n    api_base = f\"{API}/repos/{owner}/{repo}/contents\"\n    live_root = f\"https://{owner}.github.io/{repo}/\"\n    ok = fail = upload_fail = 0\n    uploaded_files = []   # (name, live_url, is_html)\n\n    if verbose:\n        print(f\"Publishing {len(files)} file(s) to {owner}/{repo} ...\")\n\n    for i, (name, content, upload_dir) in enumerate(files, 1):\n        path = f\"{api_base}/{upload_dir}/{name}\" if upload_dir \\\n            else f\"{api_base}/{name}\"\n        # find existing sha (to update rather than conflict)\n        sha = None\n        try:\n            r = requests.get(path, headers=headers,\n                             params={\"ref\": branch}, timeout=30)\n            if r.status_code == 200:\n                sha = r.json().get(\"sha\")\n        except Exception:\n            pass\n\n        body = {\"message\": f\"Publish {name}\",\n                \"content\": base64.b64encode(content).decode(),\n                \"branch\": branch}\n        if sha:\n            body[\"sha\"] = sha\n\n        uploaded = False\n        for attempt in range(3):\n            try:\n                r = requests.put(path, headers=headers, json=body, timeout=60)\n                if r.status_code in (200, 201):\n                    uploaded = True\n                    break\n                if verbose:\n                    print(f\"  ! {name} upload attempt {attempt+1}: \"\n                          f\"HTTP {r.status_code} {r.text[:160]}\")\n            except Exception as e:\n                if verbose:\n                    print(f\"  ! {name} attempt {attempt+1} error: {e}\")\n            time.sleep(2)\n\n        live_url = (f\"https://{owner}.github.io/{repo}/{upload_dir}/{name}\"\n                    if upload_dir else f\"https://{owner}.github.io/{repo}/{name}\")\n        is_img = name.lower().endswith((\".png\", \".jpg\", \".jpeg\", \".webp\"))\n        is_html = name.lower().endswith(\".html\")\n        if uploaded:\n            uploaded_files.append((name, live_url, is_html))\n        # Per-file quick check (short); the final sweep below re-checks any\n        # that weren't live yet, so Pages propagation lag is not fatal here.\n        if uploaded and verify and (is_img or is_html):\n            live = _wait_live(live_url, expect_html=is_html,\n                              tries=3, delay=4)\n        else:\n            live = uploaded\n\n        if uploaded and live:\n            ok += 1\n            if verbose:\n                print(f\"  [{i}/{len(files)}] \u2705 {name} -> {live_url}\")\n        elif uploaded:\n            # uploaded but not live yet \u2014 final sweep will re-check it\n            if verbose:\n                print(f\"  [{i}/{len(files)}] \u23f3 {name} uploaded, waiting for Pages ...\")\n        else:\n            upload_fail += 1\n            if verbose:\n                print(f\"  [{i}/{len(files)}] \u274c {name} (upload failed)\")\n        time.sleep(0.4)\n\n    # ---- Final verification sweep: keep re-checking every uploaded file until\n    # each is live on GitHub Pages (handles propagation lag for image AND html).\n    if verify and uploaded_files:\n        pending = [(n, u, h) for (n, u, h) in uploaded_files]\n        if verbose and pending:\n            print(f\"\\nVerifying {len(pending)} page(s)/image(s) are live ...\")\n        for round_no in range(1, 13):   # up to ~12 * 10s = 2 minutes\n            still = []\n            for name, live_url, is_html in pending:\n                if _check_one(live_url, is_html):\n                    ok += 0  # already counted as uploaded\n                    if verbose:\n                        print(f\"  \u2705 live: {name}\")\n                else:\n                    still.append((name, live_url, is_html))\n            pending = still\n            if not pending:\n                break\n            if verbose:\n                print(f\"  ... {len(pending)} not live yet, re-checking \"\n                      f\"(round {round_no})\")\n            time.sleep(10)\n        # anything still pending after the sweep is reported for a re-run\n        for name, _u, _h in pending:\n            if verbose:\n                print(f\"  \u274c {name} still not live after waiting (re-run \"\n                      f\"section 8 in ~1 min)\")\n        not_live = len(pending)\n    else:\n        not_live = 0\n    fail = not_live + upload_fail\n\n    live_count = len(uploaded_files) - not_live\n    if verbose:\n        print(f\"\\nDone: {live_count} live, {fail} not yet live \"\n              f\"({upload_fail} upload failure(s)).\")\n        if fail == 0:\n            print(\"\u2705 Every image and verification page is uploaded AND live.\")\n        print(\"Pages (QR base URL): \" + live_root)\n        print(\"Images under:        \" + live_root + subdir + \"/\")\n    return {\"ok\": live_count, \"fail\": fail, \"base\": live_root}\n\n\ndef _check_one(url: str, expect_html: bool) -> bool:\n    import random\n    bust = url + (\"&\" if \"?\" in url else \"?\") + \"cb=\" + str(random.randint(0, 10**9))\n    try:\n        r = requests.get(bust, timeout=30,\n                         headers={\"Cache-Control\": \"no-cache\",\n                                  \"User-Agent\": \"cert-qr-bot-publisher\"})\n        ctype = r.headers.get(\"content-type\", \"\")\n        if r.status_code != 200:\n            return False\n        if expect_html:\n            return \"html\" in ctype\n        return \"image\" in ctype\n    except Exception:\n        return False\n\n\ndef _wait_live(url: str, tries: int = 14, delay: int = 5,\n               expect_html: bool = False) -> bool:\n    for _ in range(tries):\n        if _check_one(url, expect_html):\n            return True\n        time.sleep(delay)\n    return False\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--zip\", default=\"certificates_with_new_qr.zip\")\n    ap.add_argument(\"--owner\", default=\"Naserkhan07\")\n    ap.add_argument(\"--repo\", default=\"certificate-qr\")\n    ap.add_argument(\"--subdir\", default=\"img\")\n    ap.add_argument(\"--branch\", default=\"main\")\n    ap.add_argument(\"--no-verify\", action=\"store_true\")\n    args = ap.parse_args()\n    token = _get_token()\n    if not token:\n        raise SystemExit(\"No GitHub token provided.\")\n    publish(args.zip, args.owner, args.repo, token, args.subdir,\n            args.branch, verify=not args.no_verify)\n\n\nif __name__ == \"__main__\":\n    main()\n")
    print("wrote publish_to_github_pages.py")
import sys, glob, json
import cv2
sys.path.insert(0, "/kaggle/working/certbot")
for _m in list(sys.modules):
    if _m == "src" or _m.startswith("src."):
        del sys.modules[_m]
from src.ai_extract import ai_extract_fields, runtime_status

print(runtime_status())
_test_imgs = [p for p in CANDIDATES if "/kaggle/working/" not in p]
if not _test_imgs:
    print("No input images found yet (run section 1 first).")
else:
    _tp = _test_imgs[0]
    print("\nReading:", _tp)
    _img = cv2.imread(_tp, cv2.IMREAD_COLOR)
    print("image size:", None if _img is None else _img.shape)
    _fields = ai_extract_fields(_img, model_id=AI_EXTRACT_MODEL)
    print("\n----- EXTRACTED FIELDS -----")
    for _k in ["name", "parent", "relation", "aadhar", "course",
               "date_from", "date_to", "institute"]:
        print(f"  {_k:10s}: {_fields.get(_k, '')}")
    if _fields.get("_error"):
        print("\n❌ ERROR:", _fields["_error"])
    if _fields.get("_raw_ai"):
        print("\n----- RAW MODEL OUTPUT -----")
        print(_fields["_raw_ai"][:1500])


## ▶️ 5. Run the batch

In [ ]:
import os as _os
if not _os.path.isfile("/kaggle/working/certbot/src/ai_extract.py"):
    print("Pipeline modules not written in this session yet — auto-running the section 4 writer first ...")
    import os, shutil
    os.makedirs("/kaggle/working/certbot/src", exist_ok=True)
    # clear stale bytecode from previous runs
    _pyc = "/kaggle/working/certbot/src/__pycache__"
    if os.path.isdir(_pyc): shutil.rmtree(_pyc)
    SRC = {
      "__init__.py": "from .pipeline import Config, run, process_one  # noqa\n",
      "detector.py": "\"\"\"\ndetector.py\n-----------\nLocate the OLD QR code on a certificate image.\n\nDetection cascade (tries each, returns first high-confidence hit):\n  1. WeChat CNN QR detector  (Deep-CV model downloaded from Hugging Face)\n  2. OpenCV Aruco QR detector\n  3. Geometric contour finder (finds the thick black square FRAME around a QR\n     even when the QR itself is too dense/small to decode -- works on certs\n     whose old QR is decorative or damaged)\n\nEvery detector returns the quadrangle of the *outer QR box* (including the\nblack border frame when present), so the new QR can be warped to exactly that\nsize / position / rotation.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport os\nfrom dataclasses import dataclass\n\nimport cv2\nimport numpy as np\n\n\n@dataclass\nclass QRRegion:\n    \"\"\"The 4 corners of the QR's outer box, in order TL, TR, BR, BL (px).\"\"\"\n    quad: np.ndarray            # shape (4, 2), float32\n    method: str                 # which detector found it\n    side_px: float              # average side length in pixels\n    score: float                # confidence 0..1\n\n\n# ---------------------------------------------------------------------------\n# Helpers\n# ---------------------------------------------------------------------------\ndef _order_points(pts: np.ndarray) -> np.ndarray:\n    \"\"\"Order 4 points as TL, TR, BR, BL.\"\"\"\n    pts = np.asarray(pts, dtype=np.float32).reshape(4, 2)\n    s = pts.sum(axis=1)\n    d = np.diff(pts, axis=1).reshape(-1)\n    tl = pts[np.argmin(s)]\n    br = pts[np.argmax(s)]\n    tr = pts[np.argmin(d)]\n    bl = pts[np.argmax(d)]\n    return np.array([tl, tr, br, bl], dtype=np.float32)\n\n\ndef _side_lengths(quad: np.ndarray) -> tuple[float, float, float, float]:\n    tl, tr, br, bl = quad\n    return (\n        float(np.linalg.norm(tr - tl)),\n        float(np.linalg.norm(br - tr)),\n        float(np.linalg.norm(bl - br)),\n        float(np.linalg.norm(tl - bl)),\n    )\n\n\ndef _valid_quad(gray: np.ndarray, quad: np.ndarray,\n                min_finders: int = 2) -> bool:\n    \"\"\"\n    Reject false-positive quads from the ML/OpenCV detectors.\n    A real QR quad: roughly square, on-canvas, convex, and contains >=2 of the\n    three characteristic finder patterns (nested squares).\n    \"\"\"\n    h, w = gray.shape[:2]\n    try:\n        quad = _order_points(quad)\n    except Exception:\n        return False\n    sides = _side_lengths(quad)\n    if min(sides) < max(8.0, 0.04 * min(h, w)):\n        return False\n    if max(sides) > max(h, w) * 1.15:\n        return False\n    if min(sides) / max(sides) < 0.7:\n        return False\n    # convexity (cross products same sign)\n    edges = np.roll(quad, -1, axis=0) - quad\n    crosses = edges[:, 0] * np.roll(edges[:, 1], -1) - \\\n        edges[:, 1] * np.roll(edges[:, 0], -1)\n    if not (np.all(crosses >= -1e-3) or np.all(crosses <= 1e-3)):\n        return False\n    # mostly on canvas\n    x0, y0 = quad.min(axis=0)\n    x1, y1 = quad.max(axis=0)\n    if x1 < -0.05 * w or x0 > 1.05 * w or y1 < -0.05 * h or y0 > 1.05 * h:\n        return False\n    if _count_finder_patterns(gray, quad) < min_finders:\n        return False\n    return True\n\n\ndef _count_finder_patterns(gray: np.ndarray, quad: np.ndarray) -> int:\n    \"\"\"\n    Count finder-like nested squares inside a quad. A QR has 3 finder patterns\n    (TL, TR, BL corners). Used to score contour candidates that look like a QR.\n    \"\"\"\n    tl, tr, br, bl = quad\n    w = int(max(np.linalg.norm(tr - tl), np.linalg.norm(br - bl)))\n    h = int(max(np.linalg.norm(bl - tl), np.linalg.norm(br - tr)))\n    if w < 10 or h < 10:\n        return 0\n    M = cv2.getPerspectiveTransform(\n        quad.astype(np.float32),\n        np.array([[0, 0], [w, 0], [w, h], [0, h]], dtype=np.float32),\n    )\n    patch = cv2.warpPerspective(gray, M, (w, h))\n    _, bw = cv2.threshold(patch, 0, 255,\n                          cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)\n    contours, hier = cv2.findContours(bw, cv2.RETR_TREE,\n                                      cv2.CHAIN_APPROX_SIMPLE)\n    if hier is None:\n        return 0\n    hier = hier[0]\n\n    def nested_depth(i: int) -> int:\n        depth = 0\n        parent = hier[i][3]\n        while parent != -1:\n            depth += 1\n            parent = hier[parent][3]\n        return depth\n\n    finders = 0\n    for i, c in enumerate(contours):\n        area = cv2.contourArea(c)\n        if area == 0:\n            continue\n        x, y, ww, hh = cv2.boundingRect(c)\n        ratio = ww / float(hh)\n        area_ratio = area / float(ww * hh)\n        # finder outer ring: roughly square, moderately sized, >=2 nestings\n        if (0.7 < ratio < 1.35 and area_ratio > 0.55\n                and ww > w * 0.08 and ww < w * 0.6 and nested_depth(i) >= 2):\n            finders += 1\n    return finders\n\n\n# ---------------------------------------------------------------------------\n# Detector 1: WeChat CNN (model from Hugging Face)\n# ---------------------------------------------------------------------------\n_WECHAT = None\n\n\ndef _get_wechat_detector(model_dir: str):\n    \"\"\"Lazy-load the WeChat QR CNN model if the 4 files are present.\"\"\"\n    global _WECHAT\n    if _WECHAT is not None:\n        return _WECHAT\n    if not hasattr(cv2, \"wechat_qrcode_WeChatQRCode\"):\n        return None\n    files = [\"detect.prototxt\", \"detect.caffemodel\",\n             \"sr.prototxt\", \"sr.caffemodel\"]\n    paths = [os.path.join(model_dir, f) for f in files]\n    if not all(os.path.exists(p) for p in paths):\n        return None\n    try:\n        _WECHAT = cv2.wechat_qrcode_WeChatQRCode(*paths)\n    except Exception:\n        _WECHAT = None\n    return _WECHAT\n\n\ndef _detect_wechat(img: np.ndarray, model_dir: str):\n    det = _get_wechat_detector(model_dir)\n    if det is None:\n        return None\n    h, w = img.shape[:2]\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img\n    for scale in (1.0, 2.0, 3.0):\n        src = gray if scale == 1.0 else cv2.resize(\n            gray, (int(w * scale), int(h * scale)),\n            interpolation=cv2.INTER_CUBIC)\n        try:\n            res, points = det.detectAndDecode(src)\n        except Exception:\n            res, points = [], None\n        if points is not None and len(points) > 0:\n            for p in points:\n                quad = _order_points(np.asarray(p, dtype=np.float32) / scale)\n                if _valid_quad(gray, quad, min_finders=2):\n                    return quad\n    return None\n\n\n# ---------------------------------------------------------------------------\n# Detector 2: OpenCV ArUco / classic QR detector\n# ---------------------------------------------------------------------------\ndef _detect_opencv(img: np.ndarray):\n    h, w = img.shape[:2]\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img\n    detectors = []\n    if hasattr(cv2, \"QRCodeDetectorAruco\"):\n        try:\n            detectors.append(cv2.QRCodeDetectorAruco())\n        except Exception:\n            pass\n    detectors.append(cv2.QRCodeDetector())\n    for scale in (1.0, 2.0, 3.0, 4.0):\n        src = gray if scale == 1.0 else cv2.resize(\n            gray, (int(w * scale), int(h * scale)),\n            interpolation=cv2.INTER_CUBIC)\n        for det in detectors:\n            ok, pts = det.detect(src)\n            if ok and pts is not None:\n                quad = _order_points(pts.reshape(4, 2) / scale)\n                if _valid_quad(gray, quad, min_finders=2):\n                    return quad\n    return None\n\n\n# ---------------------------------------------------------------------------\n# Detector 3: geometric contour finder (thick black frame around QR)\n# ---------------------------------------------------------------------------\ndef _detect_by_contours(img: np.ndarray):\n    h, w = img.shape[:2]\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img\n    img_area = h * w\n\n    best = None\n    best_score = -1.0\n\n    for thresh in (90, 120, 150, 180):\n        _, bw = cv2.threshold(gray, thresh, 255, cv2.THRESH_BINARY_INV)\n        # close small gaps inside the frame\n        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))\n        bw = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel)\n        contours, _ = cv2.findContours(bw, cv2.RETR_LIST,\n                                       cv2.CHAIN_APPROX_SIMPLE)\n        for c in contours:\n            area = cv2.contourArea(c)\n            if area < img_area * 0.0008 or area > img_area * 0.25:\n                continue\n            peri = cv2.arcLength(c, True)\n            approx = cv2.approxPolyDP(c, 0.04 * peri, True)\n            if len(approx) != 4:\n                # also accept rotated rect\n                rrect = cv2.minAreaRect(c)\n                (cx, cy), (rw, rh), _ = rrect\n                if rw == 0 or rh == 0:\n                    continue\n                sq = min(rw, rh) / max(rw, rh)\n                if sq < 0.82:\n                    continue\n                box = cv2.boxPoints(rrect)\n            else:\n                box = approx.reshape(4, 2).astype(np.float32)\n                (cx, cy), (rw, rh), _ = cv2.minAreaRect(c)\n                sq = min(rw, rh) / max(rw, rh) if rw and rh else 0\n                if sq < 0.82:\n                    continue\n            quad = _order_points(box)\n            sides = _side_lengths(quad)\n            side = float(np.mean(sides))\n            # reject wildly non-square quads\n            if min(sides) / max(sides) < 0.75:\n                continue\n            finders = _count_finder_patterns(gray, quad)\n            if finders < 2:\n                continue\n            # score: finder count first, then size (QRs on certs are big)\n            score = finders * 10 + (side / max(h, w))\n            if score > best_score:\n                best_score = score\n                best = (quad, finders, side)\n    if best is not None:\n        quad, finders, side = best\n        return QRRegion(quad=quad, method=\"contour\", side_px=side,\n                        score=min(1.0, finders / 3.0))\n    return None\n\n\n# ---------------------------------------------------------------------------\n# Detector 3b: finder-pattern locator (works WITHOUT a black frame)\n# ---------------------------------------------------------------------------\ndef _chain_depth(i: int, hier: np.ndarray, need: int = 2) -> bool:\n    \"\"\"True if contour `i` encloses >=`need` levels of descendants. A finder\n    pattern's outer black ring contains a white gap then the black centre, so\n    its first-child chain reaches depth 2 (the gap can merge with background,\n    which is why depth 4 is never seen on real scans).\"\"\"\n    cur = hier[i][2]\n    d = 0\n    while cur != -1:\n        d += 1\n        if d >= need:\n            return True\n        cur = hier[cur][2]\n    return False\n\n\ndef _find_finder_centers(gray: np.ndarray) -> list[tuple[np.ndarray, float]]:\n    \"\"\"Locate QR finder patterns (the 3 corner 'nested squares'). Returns a\n    list of (centre[x,y], outer-ring size px). Works for framed OR\n    frameless/plain QRs, since finders are always present.\"\"\"\n    h, w = gray.shape[:2]\n    found: list[tuple[np.ndarray, float]] = []\n\n    for thresh in (80, 110, 140, 170, 200):\n        _, bw = cv2.threshold(gray, thresh, 255, cv2.THRESH_BINARY_INV)\n        # No morphology here: closing merges the thin rings of tiny\n        # (sub-100px) QRs and destroys the nesting we are looking for.\n        contours, hier = cv2.findContours(bw, cv2.RETR_TREE,\n                                          cv2.CHAIN_APPROX_SIMPLE)\n        if hier is None:\n            continue\n        hier = hier[0]\n        cands = []\n        for i, c in enumerate(contours):\n            x, y, ww, hh = cv2.boundingRect(c)\n            if ww == 0 or hh == 0:\n                continue\n            side_min = float(min(h, w))\n            if not (side_min * 0.012 < ww < side_min * 0.35 and\n                    side_min * 0.012 < hh < side_min * 0.35):\n                continue\n            if not (0.65 < ww / float(hh) < 1.5):\n                continue\n            area = cv2.contourArea(c)\n            if area / float(ww * hh) < 0.5:\n                continue\n            if not _chain_depth(i, hier, need=2):\n                continue\n            cands.append((x + ww / 2.0, y + hh / 2.0, (ww + hh) / 2.0))\n        # within each threshold, collapse nested/contained candidates to the\n        # OUTERMOST ring (largest), since the centre dot also appears.\n        cands.sort(key=lambda t: -t[2])\n        for cx, cy, size in cands:\n            if not any(np.hypot(cx - fc[0][0], cy - fc[0][1]) < fc[1] * 0.7\n                       for fc in found):\n                found.append((np.array([cx, cy], dtype=np.float32),\n                              float(size)))\n    return found\n\n\ndef _quad_from_finders(finders: list[tuple[np.ndarray, float]]):\n    \"\"\"Given >=3 finder centres, build the QR outer quad (incl. quiet zone).\"\"\"\n    n = len(finders)\n    best = None\n    best_err = 1e9\n    for a in range(n):\n        for b in range(n):\n            if b == a:\n                continue\n            for c in range(n):\n                if c == a or c == b:\n                    continue\n                A, rA = finders[a]   # candidate elbow (TL)\n                B, _ = finders[b]    # candidate TR\n                C, _ = finders[c]    # candidate BL\n                vx = B - A\n                vy = C - A\n                lx = float(np.linalg.norm(vx))\n                ly = float(np.linalg.norm(vy))\n                if lx < 5 or ly < 5:\n                    continue\n                if min(lx, ly) / max(lx, ly) < 0.6:\n                    continue\n                cosang = float(np.dot(vx, vy) / (lx * ly))\n                if abs(cosang) > 0.45:   # want ~90 deg\n                    continue\n                ux = vx / lx\n                uy = vy / ly\n                r = float(np.mean([fr[1] for fr in (finders[a], finders[b],\n                                                    finders[c])]))\n                module = r / 7.0\n                ext = 7.5 * module      # ring centre (3.5 mod) + quiet (4 mod)\n                TL = A - ext * (ux + uy)\n                TR = TL + ux * (lx + 2 * ext)\n                BL = TL + uy * (ly + 2 * ext)\n                BR = TL + ux * (lx + 2 * ext) + uy * (ly + 2 * ext)\n                err = abs(cosang) + abs(lx - ly) / max(lx, ly)\n                if err < best_err:\n                    best_err = err\n                    best = _order_points(np.array([TL, TR, BR, BL],\n                                                  dtype=np.float32))\n    return best\n\n\ndef _detect_by_finders(img: np.ndarray):\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img\n    h, w = gray.shape[:2]\n    finders = _find_finder_centers(gray)\n    if len(finders) < 3:\n        return None\n    quad = _quad_from_finders(finders)\n    if quad is None:\n        return None\n    sides = _side_lengths(quad)\n    side = float(np.mean(sides))\n    if min(sides) < max(8.0, 0.04 * min(h, w)):\n        return None\n    if min(sides) / max(sides) < 0.7:\n        return None\n    return QRRegion(quad=quad, method=\"finders\", side_px=side, score=0.95)\n\n\n# ---------------------------------------------------------------------------\n# Public API\n# ---------------------------------------------------------------------------\ndef detect_qr(img: np.ndarray, model_dir: str = \"models/wechat\",\n              use_cnn: bool = True):\n    \"\"\"\n    Find the old QR on a certificate.\n\n    Returns QRRegion or None.\n    \"\"\"\n    if use_cnn:\n        quad = _detect_wechat(img, model_dir)\n        if quad is not None:\n            sides = _side_lengths(quad)\n            return QRRegion(quad=quad, method=\"wechat-cnn\",\n                            side_px=float(np.mean(sides)), score=0.99)\n\n    quad = _detect_opencv(img)\n    if quad is not None:\n        sides = _side_lengths(quad)\n        return QRRegion(quad=quad, method=\"opencv\",\n                        side_px=float(np.mean(sides)), score=0.9)\n\n    reg = _detect_by_contours(img)\n    if reg is not None:\n        return reg\n\n    # last resort: locate the three finder patterns (no frame needed)\n    return _detect_by_finders(img)\n\n\n# ---------------------------------------------------------------------------\n# Fallback placement (used when a QR cannot be located on one cert in a batch\n# of SAME-TEMPLATE certificates -- the QR is always in the same spot).\n# ---------------------------------------------------------------------------\ndef normalize_region(region: QRRegion, h: int, w: int) -> np.ndarray:\n    \"\"\"Quad coordinates as fractions of image width/height.\"\"\"\n    q = region.quad.astype(np.float32).copy()\n    q[:, 0] /= float(w)\n    q[:, 1] /= float(h)\n    return q\n\n\ndef fallback_region_from_regions(img: np.ndarray,\n                                 regions: list) -> QRRegion | None:\n    \"\"\"Place the new QR using the MEDIAN normalized position of the QRs that\n    WERE detected on other certificates in the batch. Works because all\n    certificates share one template. `regions` = list of (QRRegion, h, w).\n\n    Skips images whose aspect ratio differs from the certificates (e.g. a wide\n    16:9 screenshot that isn't a certificate) so a QR is never pasted onto a\n    non-certificate image.\"\"\"\n    h, w = img.shape[:2]\n    tmpl_aspect = float(np.median([ww / max(1, hh) for (_r, hh, ww) in regions]))\n    if abs((w / max(1, h)) - tmpl_aspect) / tmpl_aspect > 0.15:\n        return None\n    norms = [normalize_region(r, hh, ww) for (r, hh, ww) in regions]\n    if not norms:\n        return None\n    med = np.median(np.stack(norms, axis=0), axis=0).astype(np.float32)\n    quad = med.copy()\n    quad[:, 0] *= float(w)\n    quad[:, 1] *= float(h)\n    quad = _order_points(quad)\n    sides = _side_lengths(quad)\n    return QRRegion(quad=quad, method=\"batch-template-fallback\",\n                    side_px=float(np.mean(sides)), score=0.5)\n\n\ndef fallback_region_default(img: np.ndarray) -> QRRegion | None:\n    \"\"\"Last-resort placement for the TGMFC certificate template (QR sits in\n    the upper-left, under the red certificate number). Coordinates are\n    fractions of (width, height). Returns None for images whose aspect ratio\n    is a wide landscape (e.g. a 16:9 screenshot) rather than a certificate\n    page, so a QR is never pasted onto the wrong image.\"\"\"\n    h, w = img.shape[:2]\n    if w / max(1, h) >= 1.45:      # wide landscape screenshot, not a cert page\n        return None\n    n = np.array([[0.095, 0.345], [0.205, 0.345],\n                  [0.205, 0.515], [0.095, 0.515]], dtype=np.float32)\n    quad = n.copy()\n    quad[:, 0] *= float(w)\n    quad[:, 1] *= float(h)\n    quad = _order_points(quad)\n    sides = _side_lengths(quad)\n    return QRRegion(quad=quad, method=\"template-default-fallback\",\n                    side_px=float(np.mean(sides)), score=0.3)\n\n\ndef draw_debug(img: np.ndarray, region: QRRegion) -> np.ndarray:\n    \"\"\"Overlay the detected quad on an image (for review of failures).\"\"\"\n    out = img.copy()\n    q = region.quad.astype(np.int32)\n    cv2.polylines(out, [q], True, (0, 0, 255), 3)\n    cv2.putText(out, f\"{region.method} {region.side_px:.0f}px\",\n                (q[0][0], max(15, q[0][1] - 8)),\n                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)\n    return out\n",
      "qr_generator.py": "\"\"\"\nqr_generator.py\n---------------\nGenerate a fresh, crisp QR code image for a given payload (the URL/text that\na phone will read when scanning the certificate).\n\nThe QR is rendered at HIGH pixel resolution so that when it is perspective-\nwarped down onto the small QR region of the certificate, every module stays\nsharp and black/white (which is what makes it scan reliably).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport io\n\nimport cv2\nimport numpy as np\nimport qrcode\nfrom qrcode.constants import (\n    ERROR_CORRECT_L, ERROR_CORRECT_M, ERROR_CORRECT_Q, ERROR_CORRECT_H,\n)\n\n_EC = {\n    \"L\": ERROR_CORRECT_L,   # ~7%\n    \"M\": ERROR_CORRECT_M,   # ~15%\n    \"Q\": ERROR_CORRECT_Q,   # ~25%\n    \"H\": ERROR_CORRECT_H,   # ~30%\n}\n\n\ndef build_qr(payload: str,\n             box_size: int = 20,\n             border: int = 4,\n             error_correction: str = \"M\",\n             fg: tuple[int, int, int] = (0, 0, 0),\n             bg: tuple[int, int, int] = (255, 255, 255)) -> np.ndarray:\n    \"\"\"\n    Return a square BGR uint8 image (white background) containing the QR.\n\n    box_size: pixels per module at render time (large = crisp after warping)\n    border:   quiet-zone width in modules (standard 4)\n    \"\"\"\n    qr = qrcode.QRCode(\n        version=None,                       # auto-size to payload\n        error_correction=_EC[error_correction.upper()],\n        box_size=box_size,\n        border=border,\n    )\n    qr.add_data(payload)\n    qr.make(fit=True)\n\n    img = qr.make_image(fill_color=fg, back_color=bg).convert(\"RGB\")\n    arr = np.array(img)[:, :, ::-1].copy()  # RGB -> BGR\n    return arr\n\n\ndef qr_png_bytes(payload: str, **kwargs) -> bytes:\n    \"\"\"Render the QR and return PNG bytes (e.g. for saving a standalone file).\"\"\"\n    arr = build_qr(payload, **kwargs)\n    ok, buf = cv2.imencode(\".png\", arr)\n    return buf.tobytes() if ok else b\"\"\n",
      "qr_art.py": "\"\"\"\nqr_art.py\n---------\nOPTIONAL AI \"artistic QR\" generation using the Hugging Face ControlNet model\n**QR Code Monster** (`monster-labs/control_v1p_sd15_qrcode_monster`, v2) with a\nStable Diffusion 1.5 base.\n\nThis is the popular \"image/text -> QR code\" diffusion model family (QR Code\nMonster, IllusionDiffusion, etc.): a normal QR matrix is fed to ControlNet as\nthe conditioning image together with a text prompt, and the model produces a\ncreative QR in which art is blended into the modules.\n\nImportant facts (from the model cards):\n  * Not every generation scans -- typical scannability is 50-80%, so the\n    official workflow is \"generate several seeds and keep a readable one\".\n  * High `controlnet_conditioning_scale` (1.2-1.8) => more readable; low =>\n    more artistic. We default high because the QR must work on a certificate.\n  * These codes are usually colourful/textured -- great for marketing, but for\n    an official certificate the plain black-and-white `qr_generator.build_qr`\n    (100% reliable) is recommended.\n\nTherefore `build_artistic_qr()`:\n  1. renders the EXACT, verified QR matrix (error-correction H),\n  2. runs the ControlNet pipeline over several seeds,\n  3. DECODES every candidate and keeps one that reads back as `payload`,\n  4. returns None if none scan within `max_tries` (caller falls back to the\n     deterministic QR, so a batch never breaks).\n\nRuns on a CUDA GPU (Kaggle free T4). torch/diffusers are imported lazily, so\nimporting this module on a CPU-only box is harmless.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport cv2\nimport numpy as np\n\nfrom .qr_generator import build_qr\n\n# A prompt that pushes the model toward a clean, document-friendly result while\n# still using the AI generator. Override for creative/artistic codes.\nCLEAN_PROMPT = (\n    \"a clean flat black and white QR code on a plain white background, \"\n    \"crisp high contrast modules, minimalist, sharp edges, no decoration\"\n)\nCLEAN_NEGATIVE = (\n    \"colorful, painting, landscape, portrait, photo, texture, blurry, \"\n    \"watermark, text, logo, distorted, low contrast, cluttered background\"\n)\n\n_ART_PIPE = None\n_ART_DEVICE = None\n\n\ndef _device() -> str:\n    global _ART_DEVICE\n    if _ART_DEVICE is not None:\n        return _ART_DEVICE\n    try:\n        import torch\n        _ART_DEVICE = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    except Exception:\n        _ART_DEVICE = \"cpu\"\n    return _ART_DEVICE\n\n\ndef art_available() -> bool:\n    try:\n        import torch  # noqa: F401\n        import diffusers  # noqa: F401\n        return True\n    except Exception:\n        return False\n\n\ndef _condition_image(payload: str, module_px: int = 16,\n                     gray_bg: bool = False) -> np.ndarray:\n    \"\"\"Render the exact QR matrix as the ControlNet conditioning image.\n\n    QR Monster expects ~16px modules. v2 blends better on a gray (#808080)\n    background; for maximum readability we use white (gray_bg=False).\n    \"\"\"\n    qr = build_qr(payload, box_size=module_px, border=0,\n                  error_correction=\"H\")\n    if gray_bg:\n        # QR has white bg; replace outer non-module area with gray is complex,\n        # so we tint the whole background gray while keeping modules black.\n        gray = cv2.cvtColor(qr, cv2.COLOR_BGR2GRAY)\n        out = np.full_like(qr, 128)\n        dark = gray < 128\n        out[dark] = (0, 0, 0)\n        return out\n    return qr\n\n\ndef load_art_pipeline(\n        controlnet_id: str = \"monster-labs/control_v1p_sd15_qrcode_monster\",\n        base_model: str = \"stable-diffusion-v1-5/stable-diffusion-v1-5\"):\n    \"\"\"Lazily load and cache the ControlNet QR pipeline (fp16 on GPU).\"\"\"\n    global _ART_PIPE\n    if _ART_PIPE is not None:\n        return _ART_PIPE\n    import torch\n    from diffusers import (ControlNetModel,\n                           StableDiffusionControlNetPipeline,\n                           DPMSolverMultistepScheduler)\n\n    dtype = torch.float16 if _device() == \"cuda\" else torch.float32\n    controlnet = ControlNetModel.from_pretrained(\n        controlnet_id, torch_dtype=dtype)\n    pipe = StableDiffusionControlNetPipeline.from_pretrained(\n        base_model, controlnet=controlnet, torch_dtype=dtype,\n        safety_checker=None)\n    pipe.scheduler = DPMSolverMultistepScheduler.from_config(\n        pipe.scheduler.config)\n    if _device() == \"cuda\":\n        try:\n            pipe.enable_xformers_memory_efficient_attention()\n        except Exception:\n            pass\n        try:\n            pipe.enable_model_cpu_offload()\n        except Exception:\n            pipe.to(\"cuda\")\n    else:\n        pipe.to(\"cpu\")\n    _ART_PIPE = pipe\n    return pipe\n\n\ndef _decode_qr(img_bgr: np.ndarray, want: str) -> bool:\n    \"\"\"True if img contains a QR that decodes exactly to `want`.\"\"\"\n    from .pipeline import _decode_any_qr  # reuse strong verifier\n    h, w = img_bgr.shape[:2]\n    for scale in (1, 2):\n        im = img_bgr if scale == 1 else cv2.resize(\n            img_bgr, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)\n        got = _decode_any_qr(im)\n        if got and got.strip() == want.strip():\n            return True\n    return False\n\n\ndef build_artistic_qr(payload: str,\n                      prompt: str = CLEAN_PROMPT,\n                      negative_prompt: str = CLEAN_NEGATIVE,\n                      size: int = 768,\n                      controlnet_scale: float = 1.5,\n                      guidance_scale: float = 7.5,\n                      steps: int = 30,\n                      max_tries: int = 6,\n                      seed0: int = 1000,\n                      gray_bg: bool = False):\n    \"\"\"\n    Generate an AI (ControlNet) artistic QR for `payload`.\n\n    Returns a square BGR uint8 image that DECODES to `payload`, or None if no\n    seed produced a scannable code within `max_tries` (caller then falls back\n    to the deterministic QR).\n    \"\"\"\n    if not art_available() or _device() != \"cuda\":\n        return None\n    try:\n        import torch\n        from PIL import Image\n        pipe = load_art_pipeline()\n    except Exception as e:  # model download/OOM\n        print(f\"[qr_art] pipeline unavailable ({e}); using standard QR\")\n        return None\n\n    cond = _condition_image(payload, module_px=16, gray_bg=gray_bg)\n    cond_pil = Image.fromarray(cv2.cvtColor(cond, cv2.COLOR_BGR2RGB))\n    # make conditioning square at requested size\n    cond_pil = cond_pil.resize((size, size), Image.NEAREST)\n\n    for i in range(max_tries):\n        seed = seed0 + i\n        g = torch.Generator(device=\"cpu\").manual_seed(seed)\n        try:\n            result = pipe(\n                prompt=prompt, negative_prompt=negative_prompt,\n                image=cond_pil, width=size, height=size,\n                num_inference_steps=steps,\n                guidance_scale=guidance_scale,\n                controlnet_conditioning_scale=controlnet_scale,\n                generator=g,\n            ).images[0]\n        except Exception as e:\n            print(f\"[qr_art] generation failed (seed {seed}): {e}\")\n            continue\n        bgr = cv2.cvtColor(np.array(result), cv2.COLOR_RGB2BGR)\n        if _decode_qr(bgr, payload):\n            print(f\"[qr_art] scannable artistic QR at seed {seed} \"\n                  f\"(try {i+1}/{max_tries})\")\n            return bgr\n        print(f\"[qr_art] seed {seed} did not scan; retrying...\")\n    print(\"[qr_art] no scannable variant found; falling back to standard QR\")\n    return None\n",
      "replacer.py": "\"\"\"\nreplacer.py\n-----------\nPaste the freshly generated QR onto the certificate at the EXACT location,\nsize and rotation of the old QR.\n\n- Uses a perspective warp so rotated/skewed old QRs are matched exactly.\n- Uses NEAREST-neighbour rendering so QR modules stay hard-edged (critical for\n  scanning) -- no anti-aliased grey fuzz.\n- `mode`:\n    \"full_replace\" -> new QR (with white quiet zone) covers the entire old QR\n                      box including its black frame. Cleanest, most reliable.\n    \"keep_frame\"   -> the old black border is preserved; only the inner QR\n                      area is replaced (the new QR is shrunk inside the frame).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport cv2\nimport numpy as np\n\nfrom .detector import QRRegion, _order_points\n\n\ndef _scale_quad(quad: np.ndarray, scale: float) -> np.ndarray:\n    \"\"\"Scale a quad about its centre.\"\"\"\n    c = quad.mean(axis=0)\n    return c + (quad - c) * scale\n\n\ndef _warp_qr_to_cert(cert: np.ndarray, qr_img: np.ndarray,\n                     dst_quad: np.ndarray, supersample: int = 4) -> np.ndarray:\n    \"\"\"Perspective-warp qr_img onto cert at dst_quad (TL,TR,BR,BL).\n\n    The QR is first re-rendered at ~`supersample`x the destination size, then\n    warped with bilinear sampling (a good approximation of area-averaged\n    downscaling) and finally re-binarised to pure black/white inside the box.\n    This keeps every module hard-edged and scannable even when the on-cert\n    QR is very small (avoids the aliasing of a single huge->tiny NEAREST warp).\n    \"\"\"\n    h, w = cert.shape[:2]\n    ss = int(supersample)\n    side = float(np.mean([\n        np.linalg.norm(dst_quad[1] - dst_quad[0]),\n        np.linalg.norm(dst_quad[2] - dst_quad[1]),\n        np.linalg.norm(dst_quad[3] - dst_quad[2]),\n        np.linalg.norm(dst_quad[0] - dst_quad[3]),\n    ]))\n    target = max(8, int(round(side * ss)))\n    if qr_img.shape[0] != target:\n        interp = cv2.INTER_AREA if qr_img.shape[0] > target else cv2.INTER_NEAREST\n        qr_img = cv2.resize(qr_img, (target, target), interpolation=interp)\n\n    qh, qw = qr_img.shape[:2]\n    # Warp onto an ss-times-oversized overlay at the ss-scaled quad, then pull\n    # back down to cert resolution with INTER_AREA -> a proper area-averaged\n    # anti-aliased downscale of the perspective warp (keeps modules legible at\n    # small on-cert sizes).\n    big_W, big_H = w * ss, h * ss\n    big_quad = dst_quad.astype(np.float32) * ss\n    src = np.array([[0, 0], [qw - 1, 0], [qw - 1, qh - 1], [0, qh - 1]],\n                   dtype=np.float32)\n    M = cv2.getPerspectiveTransform(src, big_quad)\n    warped_big = cv2.warpPerspective(\n        qr_img, M, (big_W, big_H),\n        flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT,\n        borderValue=(255, 255, 255))\n    warped = cv2.resize(warped_big, (w, h), interpolation=cv2.INTER_AREA)\n\n    # mask of the QR box\n    mask = np.zeros((h, w), dtype=np.uint8)\n    cv2.fillConvexPoly(mask, dst_quad.astype(np.int32), 255)\n    mask = cv2.erode(mask, np.ones((3, 3), np.uint8), iterations=1)\n\n    # re-binarise the warped patch to pure black / white (hard module edges)\n    warped_gray = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)\n    _, binary = cv2.threshold(warped_gray, 160, 255, cv2.THRESH_BINARY)\n    binary_bgr = cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)\n\n    out = cert.copy()\n    m3 = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR) > 0\n    out[m3] = binary_bgr[m3]\n    return out\n\n\ndef replace_qr(cert: np.ndarray, region: QRRegion, qr_img: np.ndarray,\n               mode: str = \"full_replace\",\n               inner_margin_frac: float = 0.10,\n               cover_scale: float = 1.08) -> np.ndarray:\n    \"\"\"\n    Return a new certificate image with the new QR in place of the old one.\n\n    Everything outside the QR quad is left byte-for-byte untouched.\n\n    cover_scale (full_replace only): the new QR's white box is pasted over a\n    quad scaled outward by this factor so it reliably covers the ENTIRE old\n    black frame plus any dark halo / AI-inpaint remnant at the frame edge.\n    1.0 = exact old-quad footprint; ~1.08 fully hides the old frame (the white\n    background looks natural against the certificate and keeps a valid quiet\n    zone).\n    \"\"\"\n    quad = _order_points(region.quad)\n\n    if mode == \"keep_frame\":\n        # Replace only the inside of the black frame.\n        dst = _scale_quad(quad, 1.0 - 2 * inner_margin_frac)\n        # Give the pasted QR its own thin white border so modules never touch\n        # the black frame (keeps quiet-zone scanning rules).\n        qr = _add_white_border(qr_img, frac=0.06)\n    else:\n        # Full replace: scale the footprint outward a little so the new QR's\n        # white background fully hides the old frame + any edge halo.\n        dst = _scale_quad(quad, float(cover_scale))\n        qr = qr_img\n\n    return _warp_qr_to_cert(cert, qr, dst)\n\n\ndef _add_white_border(img: np.ndarray, frac: float = 0.08) -> np.ndarray:\n    h, w = img.shape[:2]\n    b = int(round(min(h, w) * frac))\n    return cv2.copyMakeBorder(img, b, b, b, b, cv2.BORDER_CONSTANT,\n                              value=(255, 255, 255))\n",
      "payload.py": "\"\"\"\npayload.py\n----------\nDecide what text/URL each certificate's NEW QR code encodes.\n\nSupported modes (set in config / notebook):\n\n  url   : base_url + slug + extension      -> e.g.\n          https://<your-username>.github.io/certificates/031-2024-25.png\n          (this is the recommended mode: host the OUTPUT images online --\n           e.g. GitHub Pages / your server / cloud storage -- and scanning the\n           QR opens that certificate in the phone's browser)\n\n  csv   : read a spreadsheet with columns: filename, qr_data\n          (per-certificate exact URL or text)\n\n  text  : fixed prefix + slug\n\n  copy  : copy whatever the old QR decoded to (requires the old QR to be\n          readable; falls back to url mode when it isn't)\n\n`slug` defaults to the certificate file's stem (filename without extension).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport csv\nimport os\n\n\ndef slug_from_filename(path: str) -> str:\n    return os.path.splitext(os.path.basename(path))[0]\n\n\ndef safe_slug(path: str) -> str:\n    \"\"\"URL/filename-safe stem. Replaces spaces, parentheses and other chars that\n    break URLs (e.g. 'image (1)' -> 'image_1') so the QR link and the saved\n    output file always match and resolve.\"\"\"\n    import re\n    stem = slug_from_filename(path)\n    stem = re.sub(r\"[^A-Za-z0-9._-]+\", \"_\", stem)\n    stem = re.sub(r\"_+\", \"_\", stem).strip(\"._-\")\n    return stem or \"certificate\"\n\n\ndef build_payload(path: str,\n                  mode: str = \"url\",\n                  base_url: str = \"\",\n                  extension: str = \".png\",\n                  text_prefix: str = \"\",\n                  csv_map: dict[str, str] | None = None,\n                  old_data: str | None = None) -> str:\n    slug = safe_slug(path)\n\n    if mode == \"csv\":\n        if csv_map and (os.path.basename(path) in csv_map or slug in csv_map):\n            return csv_map.get(os.path.basename(path)) or csv_map[slug]\n        # fall back to URL if the cert is missing from the spreadsheet\n        mode = \"url\"\n\n    if mode == \"copy\" and old_data:\n        return old_data\n\n    if mode == \"text\":\n        return f\"{text_prefix}{slug}\"\n\n    # default: url\n    base = base_url.rstrip(\"/\") + \"/\"\n    return f\"{base}{slug}{extension}\"\n\n\ndef load_csv_map(csv_path: str) -> dict[str, str]:\n    \"\"\"Read filename->qr_data mapping from a CSV (headers: filename,qr_data).\"\"\"\n    mapping: dict[str, str] = {}\n    with open(csv_path, newline=\"\", encoding=\"utf-8-sig\") as f:\n        for row in csv.DictReader(f):\n            fn = (row.get(\"filename\") or row.get(\"file\") or \"\").strip()\n            data = (row.get(\"qr_data\") or row.get(\"url\") or\n                    row.get(\"payload\") or \"\").strip()\n            if fn and data:\n                mapping[fn] = data\n                mapping[os.path.splitext(fn)[0]] = data\n    return mapping\n",
      "ai_extract.py": "\"\"\"\nai_extract.py\n-------------\nRead the candidate's details DIRECTLY FROM THE CERTIFICATE IMAGE using an AI\nvision-language model from Hugging Face (no OCR, no CSV, no hand-written rules).\n\nModel (runs on a free Kaggle T4 GPU; no num2words / tesseract needed):\n    Qwen/Qwen2-VL-2B-Instruct   (default) -- strong document/JSON reader\n    Qwen/Qwen2.5-VL-3B-Instruct (optional) -- slightly larger, also fine on T4\n\nThe model is shown the certificate and asked to return ONLY JSON:\n    name, parent, relation, aadhar, course, date_from, date_to, institute.\n\nFailures are NEVER silent: the returned dict carries an \"_error\" / \"_raw_ai\"\nkey so the notebook log / manifest show exactly what happened.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport re\nimport os\nimport gc\nimport importlib\nimport subprocess\nimport sys\n\n# Reduce CUDA fragmentation / OOM on the 16 GB T4 when a model is loaded more\n# than once in a session (must be set before torch first allocates).\nos.environ.setdefault(\"PYTORCH_CUDA_ALLOC_CONF\", \"expandable_segments:True\")\n\n_PROMPT = (\n    \"You are reading an Indian training-completion certificate IMAGE. \"\n    \"Look carefully at the whole image and extract these SIX fields. \"\n    \"Return ONLY a JSON object (no prose, no markdown, no code fence) with \"\n    \"exactly these keys:\\n\"\n    '{\"name\": \"<the candidate OWN name only: the words printed after Mr./Ms. '\n    'and BEFORE the token S/o or D/o or W/o. Do NOT include S/o, D/o, W/o, or '\n    'any name after them.>\", '\n    '\"parent\": \"<the father/mother name ONLY: the person name printed '\n    'immediately AFTER S/o (son of -> father), D/o (daughter of -> father) or '\n    'W/o (wife of -> husband), up to the next comma or bracket.>\", '\n    '\"relation\": \"<exactly S/o if the text says S/o, D/o if it says D/o, or '\n    'W/o if it says W/o>\", '\n    '\"aadhar\": \"<the candidate 12-digit Aadhaar number, digits only>\", '\n    '\"course\": \"<the full course/training name the candidate completed, e.g. '\n    'the text after Training in / Trainings in / Course>\", '\n    '\"date_from\": \"<course START date formatted dd/mm/yyyy>\", '\n    '\"date_to\": \"<course END date formatted dd/mm/yyyy>\", '\n    '\"institute\": \"<the COMPLETE institute name printed anywhere on the '\n    'certificate, written out in full exactly as shown (spell out every word; '\n    'do NOT shorten to initials - e.g. write Falih Consultancy Services (FCS) '\n    'Training Institute, not just FCS). Read the org name LETTER BY LETTER: '\n    'the printed letters l, i, s and h look alike - Falih is F-a-l-i-h (NOT '\n    'Falis, NOT Fahis). It differs on every certificate; never '\n    'assume or default to a name.>\"}\\n'\n    \"CRITICAL: 'S/o' = Son of (FATHER), 'D/o' = Daughter of (FATHER), 'W/o' = \"\n    \"Wife of (HUSBAND). These give the parent name. 'R/o' = Resident of (the \"\n    \"HOME ADDRESS / place, e.g. Rajendra Nager) \u2014 R/o is NOT a person and must \"\n    \"NEVER be put in name or parent. Read from the image only; do not invent; \"\n    \"use \\\"\\\" for anything unreadable; dates as dd/mm/yyyy.\"\n)\n\n# Small helper packages (NOT torch/transformers). If one is missing in the\n# kernel we self-heal by pip-installing it at runtime.\n_REQUIRED = [\n    (\"PIL\", \"pillow\"),\n    (\"accelerate\", \"accelerate\"),\n    (\"torchvision\", \"torchvision\"),\n]\n\n_MODEL = None\n_PROC = None\n_DEVICE = None\n_MODEL_ID = None\n\n\ndef _pip_install(pkg: str) -> bool:\n    try:\n        subprocess.check_call(\n            [sys.executable, \"-m\", \"pip\", \"install\", \"-q\", pkg])\n        return True\n    except Exception as e:\n        print(f\"[ai_extract] pip install {pkg} failed: {e}\")\n        return False\n\n\ndef _ensure_deps():\n    # Only self-heal where torch/transformers already exist (the GPU notebook).\n    try:\n        importlib.import_module(\"torch\")\n        importlib.import_module(\"transformers\")\n    except Exception:\n        return\n    for import_name, pip_name in _REQUIRED:\n        try:\n            importlib.import_module(import_name)\n        except Exception:\n            print(f\"[ai_extract] installing missing dependency '{pip_name}' ...\")\n            if _pip_install(pip_name):\n                try:\n                    importlib.import_module(import_name)\n                except Exception:\n                    pass\n\n\ndef device() -> str:\n    global _DEVICE\n    if _DEVICE is not None:\n        return _DEVICE\n    try:\n        import torch\n        _DEVICE = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    except Exception:\n        _DEVICE = \"cpu\"\n    return _DEVICE\n\n\ndef runtime_status() -> str:\n    try:\n        import torch  # noqa\n    except Exception as e:\n        return f\"torch NOT importable: {e}\"\n    try:\n        import transformers\n        tv = transformers.__version__\n    except Exception as e:\n        return f\"transformers NOT importable: {e}\"\n    dev = device()\n    gpu = \"\"\n    if dev == \"cuda\":\n        try:\n            import torch\n            gpu = f\" | GPU: {torch.cuda.get_device_name(0)}\"\n        except Exception:\n            pass\n    warn = \"\" if dev == \"cuda\" else \" | \u26a0\ufe0f CPU only \u2014 set Accelerator to GPU T4\"\n    return f\"torch ok | transformers {tv} | device={dev}{gpu}{warn}\"\n\n\ndef release_model():\n    \"\"\"Free the cached vision model and its CUDA memory. Call before a fresh\n    module load in the same session (e.g. after the test cell) to avoid OOM.\"\"\"\n    global _MODEL, _PROC, _MODEL_ID\n    _MODEL = None\n    _PROC = None\n    _MODEL_ID = None\n    gc.collect()\n    try:\n        import torch\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n            torch.cuda.ipc_collect()\n    except Exception:\n        pass\n\n\ndef _load(model_id: str):\n    \"\"\"Load (once) and cache the Qwen2-VL model + processor. Frees any stale\n    model first and retries once after emptying the CUDA cache on OOM.\"\"\"\n    global _MODEL, _PROC, _MODEL_ID\n    if _MODEL is not None and _MODEL_ID == model_id:\n        return _MODEL, _PROC\n    release_model()  # ensure no previous model (e.g. from the test cell) lingers\n    import torch\n    from transformers import AutoProcessor\n\n    def _build():\n        ModelCls = None\n        if \"Qwen2.5\" in model_id:\n            try:\n                from transformers import Qwen2_5_VLForConditionalGeneration as ModelCls\n            except Exception:\n                ModelCls = None\n        if ModelCls is None:\n            try:\n                from transformers import Qwen2VLForConditionalGeneration as ModelCls\n            except Exception:\n                ModelCls = None\n        if ModelCls is None:\n            from transformers import AutoModelForImageTextToText as ModelCls\n        dtype = torch.float16 if device() == \"cuda\" else torch.float32\n        proc = AutoProcessor.from_pretrained(model_id)\n        kwargs = dict(torch_dtype=dtype, low_cpu_mem_usage=True)\n        if device() == \"cuda\":\n            kwargs.update(device_map=\"auto\", offload_folder=\"/kaggle/working/offload\")\n        model = ModelCls.from_pretrained(model_id, **kwargs)\n        if \"device_map\" not in kwargs:\n            model = model.to(device())\n        model.eval()\n        return model, proc\n\n    try:\n        _MODEL, _PROC = _build()\n    except (RuntimeError, MemoryError) as e:\n        if \"out of memory\" in str(e).lower() or \"CUDA\" in str(e):\n            print(\"[ai_extract] GPU out of memory while loading \u2014 freeing cache \"\n                  \"and retrying once...\")\n            release_model()\n            try:\n                _MODEL, _PROC = _build()\n            except Exception as e2:\n                raise RuntimeError(\n                    f\"GPU out of memory even after freeing cache ({e2}). \"\n                    \"Restart the kernel (Run -> Restart) and run section 5 only, \"\n                    \"without running the section 4b test cell first.\") from e2\n        else:\n            raise\n    _MODEL_ID = model_id\n    return _MODEL, _PROC\n\n\ndef _norm_date(v: str) -> str:\n    if not v:\n        return \"\"\n    m = re.search(r\"(\\d{1,2})[^\\d]?(\\d{1,2})[^\\d]?(\\d{4})\", str(v))\n    if not m:\n        return str(v).strip()\n    try:\n        return f\"{int(m.group(1)):02d}/{int(m.group(2)):02d}/{m.group(3)}\"\n    except Exception:\n        return str(v).strip()\n\n\ndef _digits_aadhar(v: str) -> str:\n    if not v:\n        return \"\"\n    d = re.sub(r\"\\D\", \"\", str(v))\n    if len(d) > 12:\n        d = d[-12:]\n    if len(d) == 12:\n        return f\"{d[0:4]} {d[4:8]} {d[8:12]}\"\n    return str(v).strip()\n\n\ndef _clean_person_fields(d: dict) -> dict:\n    \"\"\"Deterministically fix name/parent/relation, regardless of model slips:\n    split at the S/o / D/o / W/o marker and never confuse it with R/o (address).\"\"\"\n    name = str(d.get(\"name\", \"\") or \"\").strip()\n    parent = str(d.get(\"parent\", \"\") or \"\").strip()\n    rel = str(d.get(\"relation\", \"\") or \"\").strip().upper()\n\n    relmap = {\"S\": \"S/o\", \"D\": \"D/o\", \"W\": \"W/o\"}\n    rel_letter = rel[0] if (rel and rel[0] in \"SDW\") else \"\"\n    if not rel_letter:\n        m = re.search(r\"\\b([SDW])\\s*/?\\s*[oO0]\\b\", f\"{name} {parent}\")\n        rel_letter = m.group(1) if m else \"\"\n\n    # If the relation marker (S/o, D/o, W/o) ended up INSIDE the name, split it\n    # and trust the marker found in the name (it directly introduces the parent).\n    m = re.search(r\"^(.*?)\\b([SDW])\\s*/?\\s*[oO0]\\s+(.*)$\", name, flags=re.S)\n    if m:\n        name = m.group(1).strip(\" ,./-\")\n        rel_letter = m.group(2)\n        tail = m.group(3)\n        # parent = up to the next R/o (address), Aadhaar, bracket, comma, newline\n        tail = re.split(r\"\\bR\\s*/\\s*o\\b|\\(?Aadha\", tail, flags=re.I)[0]\n        cand = tail.split(\",\")[0].split(\"(\")[0].strip(\" ,./-\")\n        if cand:\n            parent = cand\n\n    # If parent wrongly contains the R/o (residence/address) marker, cut there.\n    parent = re.split(r\"\\bR\\s*/\\s*o\\b\", parent, flags=re.I)[0].strip(\" ,./-\")\n    # Strip a leaked relation PREFIX from parent only if it is the actual\n    # \"S/o\" / \"D/o\" token (never strip a bare initial like \"D.\" in \"D. Jahangeen\").\n    parent = re.sub(r\"^[SDW]\\s*/\\s*[oO0]\\s*\", \"\", parent, flags=re.I).strip()\n    # Parent shouldn't contain Aadhaar / trailing bracket junk.\n    parent = re.split(r\"\\(?Aadha|\\d{4}\\s*\\d{4}\\s*\\d{4}\", parent)[0].strip(\" ,./-()\")\n\n    # Name must not keep a trailing relation marker either.\n    name = re.split(r\"\\b[SDW]\\s*/?\\s*[oO0]\\b\", name)[0].strip(\" ,./-\")\n    # Strip leading honorifics the model may keep (requires the dot, so real\n    # names like \"Msmena\" are never cut). Handles \"Mr./Ms.\", \"Ms.\", \"Dr.\" etc.\n    name = re.sub(\n        r\"^(?:Mr|Mrs|Ms|Miss|Dr|Prof)\\.\\s*(?:/\\s*(?:Mrs?|Ms|Miss|Dr|Prof)\\.\\s*)*\",\n        \"\", name).strip(\" ,./-\")\n\n    relation = relmap.get(rel_letter, \"\")\n    d[\"name\"] = name\n    d[\"parent\"] = parent\n    d[\"relation\"] = relation\n    return d\n\n\ndef _parse_json(text: str) -> dict:\n    if not text:\n        return {}\n    t = text.strip()\n    t = re.sub(r\"^```(?:json)?\\s*|\\s*```$\", \"\", t, flags=re.I).strip()\n    m = re.search(r\"\\{.*\\}\", t, re.S)\n    if not m:\n        return {}\n    data = None\n    for candidate in (m.group(0), \"{\" + m.group(0).strip(\"{} \") + \"}\"):\n        try:\n            data = json.loads(candidate)\n            break\n        except Exception:\n            data = None\n    if not isinstance(data, dict):\n        return {}\n    rel = str(data.get(\"relation\", \"\") or \"\").strip().upper()\n    rel = rel.replace(\"SON OF\", \"S/O\").replace(\"DAUGHTER OF\", \"D/O\")\n    rel = rel.replace(\"WIFE OF\", \"W/O\")\n    if rel and rel[0] in \"SDW\":\n        rel = rel[0] + \"/O\"\n    out = {\n        \"name\": str(data.get(\"name\", \"\") or \"\").strip(),\n        \"parent\": str(data.get(\"parent\", \"\") or \"\").strip(),\n        \"relation\": rel,\n        \"aadhar\": _digits_aadhar(data.get(\"aadhar\", \"\")),\n        \"course\": str(data.get(\"course\", \"\") or \"\").strip(),\n        \"date_from\": _norm_date(data.get(\"date_from\", \"\")),\n        \"date_to\": _norm_date(data.get(\"date_to\", \"\")),\n        \"institute\": str(data.get(\"institute\", \"\") or \"\").strip(),\n        \"_raw_ai\": text.strip(),\n    }\n    return _clean_person_fields(out)\n\n\ndef _generate(pil, model, proc, prompt: str | None = None,\n              max_new_tokens: int = 512) -> str:\n    import torch\n    messages = [{\"role\": \"user\", \"content\": [\n        {\"type\": \"image\"}, {\"type\": \"text\", \"text\": prompt or _PROMPT}]}]\n    text = proc.apply_chat_template(\n        messages, tokenize=False, add_generation_prompt=True)\n    inputs = proc(text=[text], images=[pil], padding=True,\n                  return_tensors=\"pt\").to(model.device)\n    with torch.no_grad():\n        gen = model.generate(**inputs, max_new_tokens=max_new_tokens,\n                             do_sample=False)\n    trimmed = gen[:, inputs[\"input_ids\"].shape[1]:]\n    return proc.batch_decode(trimmed, skip_special_tokens=True)[0]\n\n\n_VERIFY_PROMPT = (\n    \"Look ONLY at the name of the organization/institute that ISSUED this \"\n    \"certificate (printed in words, usually under or around the logo at the \"\n    \"top, and/or near the signatory at the bottom).\\n\"\n    \"A previous reading reported the institute as: \\\"{got}\\\"\\n\"\n    \"Re-read the printed name CAREFULLY, letter by letter, and fix any \"\n    \"misread letters. Printed 'l', 'i', 's' and 'h' look alike: for example \"\n    \"'Falih' is F-a-l-i-h (NOT Falis, NOT Fahis, NOT Faris). Also write the \"\n    \"name COMPLETE, exactly as printed \u2014 spell out every word; never shorten \"\n    \"to initials.\\n\"\n    \"Return ONLY a JSON object with exactly this key (no prose, no code \"\n    \"fence):\\n\"\n    '{\"institute\": \"<the complete, letter-checked correct institute name, '\n    'or \\\\\"\\\\\" if not readable>\"}'\n)\n\n\ndef _clean_institute(v: str) -> str:\n    v = str(v or \"\").strip().strip(\".,;\\\"'` \")\n    if v.lower() in (\"\", \"n/a\", \"na\", \"null\", \"none\", \"-\"):\n        return \"\"\n    return v\n\n\ndef _parse_verify(text: str) -> str:\n    \"\"\"Extract the corrected institute string from the verify-pass output.\"\"\"\n    if not text:\n        return \"\"\n    t = re.sub(r\"^```(?:json)?\\s*|\\s*```$\", \"\", text.strip(),\n               flags=re.I).strip()\n    if \"{\" in t:  # model answered in JSON (possibly slightly malformed)\n        m = re.search(r\"\\{.*\\}\", t, re.S)\n        blob = m.group(0) if m else t\n        try:\n            obj = json.loads(blob)\n            return _clean_institute(obj.get(\"institute\", \"\"))\n        except Exception:\n            mm = re.search(r'\"institute\"\\s*:\\s*\"([^\"]*)\"', blob)\n            if mm:\n                return _clean_institute(mm.group(1))\n            return \"\"\n    # plain-text answer: prefer the longest line (model may add a short\n    # preamble like \"The institute is:\" before the actual name)\n    best = \"\"\n    for line in t.splitlines():\n        line = line.strip().strip(\"{}\\\"'` \")\n        line = re.sub(r\"^institute\\s*:?\", \"\", line, flags=re.I).strip()\n        line = re.sub(r\"^(the\\s+)?(name|institute)\\s+(is|of)\\s*:?\\s*$\",\n                      \"\", line, flags=re.I).strip()\n        if _clean_institute(line) and len(line.strip()) >= 3 \\\n                and len(line) > len(best):\n            best = line\n    return _clean_institute(best)\n\n\ndef ai_extract_fields(img_bgr,\n                      model_id: str = \"Qwen/Qwen2-VL-2B-Instruct\"\n                      ) -> dict:\n    \"\"\"Run the AI vision model on one certificate (BGR ndarray) and return a\n    fields dict. On any failure returns a dict with '_error' (never raises).\"\"\"\n    try:\n        import numpy as np  # noqa\n        import cv2\n        from PIL import Image\n    except Exception as e:\n        return {\"_error\": f\"deps missing (numpy/Pillow/cv2): {e}\"}\n\n    global _MODEL, _PROC\n    model = proc = None\n    try:\n        _ensure_deps()\n        model, proc = _load(model_id)\n    except Exception as e:\n        name = getattr(e, \"name\", None)\n        m = re.search(r\"No module named ['\\\"]([a-zA-Z0-9_\\-]+)\", str(e))\n        pkg = name or (m.group(1) if m else None)\n        pip_map = {\"PIL\": \"pillow\", \"torchvision\": \"torchvision\",\n                   \"accelerate\": \"accelerate\"}\n        pip = pip_map.get(pkg)\n        if pip:\n            print(f\"[ai_extract] model needs '{pip}'; installing and retrying...\")\n            if _pip_install(pip):\n                _MODEL, _PROC = None, None\n                try:\n                    _ensure_deps()\n                    model, proc = _load(model_id)\n                except Exception as e2:\n                    return {\"_error\": f\"model load failed after installing \"\n                                      f\"{pip}: {e2}\"}\n            else:\n                return {\"_error\": f\"model load failed (missing {pip}): {e}\"}\n        else:\n            return {\"_error\": f\"model load failed: {e}\"}\n\n    rgb = img_bgr[:, :, ::-1]\n    # Up-scale small/phone screenshots a bit so the VLM has more pixels to read.\n    if rgb.shape[1] < 1000:\n        scale = 1000 / rgb.shape[1]\n        rgb = cv2.resize(rgb, None, fx=scale, fy=scale,\n                         interpolation=cv2.INTER_CUBIC)\n    import numpy as np\n    pil = Image.fromarray(np.ascontiguousarray(rgb)).convert(\"RGB\")\n\n    out = \"\"\n    try:\n        out = _generate(pil, model, proc)\n    except Exception as e:\n        try:  # one retry slightly larger\n            bigger = cv2.resize(rgb, None, fx=1.4, fy=1.4,\n                                interpolation=cv2.INTER_CUBIC)\n            pil = Image.fromarray(np.ascontiguousarray(bigger)).convert(\"RGB\")\n            out = _generate(pil, model, proc)\n        except Exception as e2:\n            return {\"_error\": f\"generation failed: {e2}\"}\n\n    fields = _parse_json(out)\n    if not fields:\n        return {\"_error\": \"model output was not JSON\", \"_raw_ai\": out.strip()}\n\n    # ---- Institute spelling check: one extra focused look so names like\n    # 'Falih' are not misread as 'Falis' (l/i/s confusion). Never fatal.\n    got_inst = str(fields.get(\"institute\", \"\") or \"\").strip()\n    if got_inst:\n        try:\n            v_raw = _generate(pil, model, proc,\n                              prompt=_VERIFY_PROMPT.replace(\"{got}\", got_inst),\n                              max_new_tokens=96)\n            fixed = _parse_verify(v_raw)\n            if fixed:\n                fields[\"institute_verified\"] = fixed\n                if fixed.strip().lower() != got_inst.strip().lower():\n                    print(f\"[ai_extract] institute spelling check: \"\n                          f\"'{got_inst}' -> '{fixed}'\")\n                    fields[\"institute\"] = fixed\n            fields[\"_raw_ai\"] = out.strip() + \"\\n\\n[institute re-check] \" \\\n                + v_raw.strip()[:220]\n        except Exception as e:\n            print(f\"[ai_extract] institute re-check skipped ({e})\")\n    return fields\n",
      "verification_page.py": "\"\"\"\nverification_page.py\n--------------------\nBuild a self-contained HTML certificate-verification page for each candidate.\nThe certificate image is EMBEDDED (base64), so uploading the single .html file\nis enough \u2014 when a user scans the QR (which links to this page) they see the\ncertificate image with the candidate's details beneath it.\n\nDetails (read off the certificate by the AI vision model):\n    Name of the candidate, Father/Mother Name, Aadhar No., Course Name,\n    Course Duration (dd/mm/yyyy -> dd/mm/yyyy), Institute.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport base64\nimport html as _html\n\n\ndef _b64_png(img_bgr) -> str:\n    import cv2\n    ok, buf = cv2.imencode(\".png\", img_bgr)\n    return base64.b64encode(buf.tobytes()).decode() if ok else \"\"\n\n\ndef _esc(x) -> str:\n    return _html.escape(str(x if x not in (None, \"\") else \"\u2014\"))\n\n\ndef build_page(fields: dict, img_bgr, cert_filename: str) -> str:\n    name = fields.get(\"name\", \"\")\n    parent = fields.get(\"parent\", \"\")\n    aadhar = fields.get(\"aadhar\", \"\")\n    course = fields.get(\"course\", \"\")\n    dfrom = fields.get(\"date_from\", \"\")\n    dto = fields.get(\"date_to\", \"\")\n    dates = f\"{_esc(dfrom)} &nbsp;to&nbsp; {_esc(dto)}\" if (dfrom or dto) else \"\u2014\"\n\n    img_b64 = _b64_png(img_bgr)\n\n    institute = fields.get(\"institute\", \"\")\n    rows = [\n        (\"Name of the candidate\", _esc(name)),\n        (\"Father/Mother Name\", _esc(parent)),\n        (\"Aadhar No.\", _esc(aadhar)),\n        (\"Course Name\", _esc(course)),\n        (\"Course Duration\", dates),\n        (\"Institute\", _esc(institute)),\n    ]\n    rows_html = \"\\n\".join(\n        f'<tr><th>{label} :</th><td>{val}</td></tr>' for label, val in rows)\n\n    cert_html = f'''\n    <div class=\"cert\">\n      <img alt=\"Certificate for {_esc(name or cert_filename)}\"\n           src=\"data:image/png;base64,{img_b64}\">\n      <div><span class=\"badge\">&#10003; Verified certificate</span></div>\n    </div>'''\n\n    info_html = f'''\n    <h2 class=\"sechd\">Candidate Information</h2>\n    <table>\n{rows_html}\n    </table>'''\n\n    return f\"\"\"<!DOCTYPE html>\n<html lang=\"en\">\n<head>\n<meta charset=\"utf-8\">\n<meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">\n<title>Certificate Verification - {_esc(name or cert_filename)}</title>\n<style>\n  *{{box-sizing:border-box}}\n  body{{font-family:Georgia,'Times New Roman',serif;background:#eef1f5;margin:0;\n       padding:24px 12px;color:#1b1b1b}}\n  .wrap{{max-width:760px;margin:0 auto}}\n  .head{{text-align:center;margin-bottom:16px}}\n  .head .org{{font-size:13px;letter-spacing:2px;color:#0b6b3a;font-weight:bold;\n       text-transform:uppercase}}\n  .head h1{{font-size:22px;margin:6px 0 2px;color:#10345e}}\n  .head .sub{{color:#666;font-size:13px}}\n  .card{{background:#fff;border:1px solid #d9dee6;border-radius:14px;\n       box-shadow:0 10px 30px rgba(0,0,0,.08);padding:22px 24px}}\n  table{{width:100%;border-collapse:collapse;font-size:16px}}\n  th{{text-align:left;width:46%;color:#333;font-weight:normal;padding:10px 8px;\n     border-bottom:1px solid #eef0f3;vertical-align:top}}\n  td{{padding:10px 8px;border-bottom:1px solid #eef0f3;font-weight:bold;\n     color:#111;word-break:break-word}}\n  .fcs{{margin:6px 0 10px;text-align:center;font-weight:bold;letter-spacing:1px;\n       color:#0b6b3a;font-size:14px;text-transform:uppercase}}\n  .sechd{{font-size:16px;color:#10345e;border-bottom:2px solid #e5e9f0;\n       padding-bottom:6px;margin:18px 0 8px}}\n  .cert{{margin-top:16px;text-align:center}}\n  .cert img{{max-width:100%;border:1px solid #d9dee6;border-radius:8px;\n       box-shadow:0 4px 14px rgba(0,0,0,.10)}}\n  .badge{{display:inline-block;background:#e8f6ee;color:#0b6b3a;border:1px solid #bfe6cf;\n       border-radius:999px;padding:5px 14px;font-size:12px;margin-top:8px}}\n  .foot{{text-align:center;color:#8a93a0;font-size:12px;margin-top:16px}}\n</style>\n</head>\n<body>\n<div class=\"wrap\">\n  <div class=\"head\">\n    <div class=\"org\">Government of Telangana &middot; TGMFC</div>\n    <h1>Certificate Verification</h1>\n    <div class=\"sub\">Skill Development Training certificate</div>\n  </div>\n  <div class=\"card\">\n{cert_html}\n{info_html}\n  </div>\n  <div class=\"foot\">Government of Telangana &middot; TGMFC &mdash;\n       Skill Development Training certificate verification</div>\n</div>\n</body>\n</html>\n\"\"\"\n",
      "ai_inpaint.py": "\"\"\"\nai_inpaint.py\n-------------\nAI image-generator step (diffusion INPAINTING) for the QR replacement.\n\nWhy inpainting + compositing instead of pure text-to-image?\n  * A generative model that \"draws\" a QR from a text prompt produces a\n    plausible-looking but INVALID QR (it cannot reproduce the exact\n    error-corrected module grid), and free-form editing can subtly alter\n    names/faces.\n  * Masked inpainting is given the certificate, a MASK limited to the old-QR\n    square, and your natural-language prompt. The model may only regenerate\n    pixels inside the mask, so every other pixel (design, text, face, seals)\n    is provably unchanged.\n  * We then composite the REAL, freshly computed QR (perspective-warped to the\n    exact quad) on top of the inpainted region -> the edit is AI-generated and\n    blended as requested, AND the QR is guaranteed crisp & scannable.\n\nModel (Hugging Face, runs on a Kaggle T4 16GB):\n    diffusers/stable-diffusion-xl-1.0-inpainting-0.1   (default, fp16)\n    stabilityai/stable-diffusion-2-inpainting         (lighter fallback)\n\nEverything is lazy: torch/diffusers are only imported when you actually run\nthe AI step, so the deterministic warp path needs no GPU and no big download.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport os\n\nimport cv2\nimport numpy as np\n\nfrom .replacer import replace_qr\n\n# The exact prompt requested by the user.\nDEFAULT_PROMPT = (\n    \"Generate an image of this certificate replacing only the old QR code \"\n    \"with a new crisp black and white QR code in the same position. \"\n    \"Do not change any design, do not change any information, do not change \"\n    \"the face or anything else. Just replace the old QR code with the new \"\n    \"QR code, keeping the identical certificate layout, colors, and text.\"\n)\nDEFAULT_NEGATIVE = (\n    \"invalid qr code, blurry qr, distorted text, changed name, altered face, \"\n    \"modified design, extra logo, watermark, deformed, low quality, artifacts\"\n)\n\n_PIPE = None\n_DEVICE = None\n\n\n# --------------------------------------------------------------------------- #\n# Model loading\n# --------------------------------------------------------------------------- #\ndef _device() -> str:\n    global _DEVICE\n    if _DEVICE is not None:\n        return _DEVICE\n    try:\n        import torch\n        _DEVICE = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    except Exception:\n        _DEVICE = \"cpu\"\n    return _DEVICE\n\n\ndef load_pipeline(model_id: str = \"diffusers/stable-diffusion-xl-1.0-inpainting-0.1\"):\n    \"\"\"Lazily load (and cache) the HF inpainting pipeline.\"\"\"\n    global _PIPE\n    if _PIPE is not None:\n        return _PIPE\n    import torch\n    from diffusers import AutoPipelineForInpainting\n\n    dtype = torch.float16 if _device() == \"cuda\" else torch.float32\n    kw = dict(torch_dtype=dtype)\n    try:\n        pipe = AutoPipelineForInpainting.from_pretrained(model_id, **kw)\n    except Exception:\n        # some repos need the variant flag only on GPU fp16\n        kw[\"variant\"] = \"fp16\"\n        pipe = AutoPipelineForInpainting.from_pretrained(model_id, **kw)\n\n    if _device() == \"cuda\":\n        try:\n            pipe.enable_xformers_memory_efficient_attention()\n        except Exception:\n            pass\n        try:\n            pipe.enable_model_cpu_offload()\n        except Exception:\n            pipe.to(\"cuda\")\n    else:\n        pipe.to(\"cpu\")\n    pipe.set_progress_bar_config(disable=False)\n    _PIPE = pipe\n    return pipe\n\n\ndef ai_available() -> bool:\n    try:\n        import torch  # noqa: F401\n        import diffusers  # noqa: F401\n        return True\n    except Exception:\n        return False\n\n\n# --------------------------------------------------------------------------- #\n# Geometry helpers\n# --------------------------------------------------------------------------- #\ndef _mask_from_quad(shape_hw: tuple[int, int], quad: np.ndarray,\n                    dilate_px: int = 6) -> np.ndarray:\n    h, w = shape_hw\n    mask = np.zeros((h, w), np.uint8)\n    cv2.fillConvexPoly(mask, quad.astype(np.int32), 255)\n    if dilate_px:\n        k = cv2.getStructuringElement(\n            cv2.MORPH_RECT, (2 * dilate_px + 1, 2 * dilate_px + 1))\n        mask = cv2.dilate(mask, k)\n    return mask\n\n\ndef _crop_bbox(quad: np.ndarray, h: int, w: int, margin: int):\n    x0, y0 = quad.min(axis=0)\n    x1, y1 = quad.max(axis=0)\n    x0 = int(max(0, x0 - margin)); y0 = int(max(0, y0 - margin))\n    x1 = int(min(w, x1 + margin)); y1 = int(min(h, y1 + margin))\n    return x0, y0, x1, y1\n\n\ndef _round8(v: int) -> int:\n    return max(8, int(round(v / 8.0)) * 8)\n\n\n# --------------------------------------------------------------------------- #\n# Main entry\n# --------------------------------------------------------------------------- #\ndef ai_replace_qr(cert_bgr: np.ndarray, region, qr_bgr: np.ndarray,\n                  prompt: str = DEFAULT_PROMPT,\n                  negative_prompt: str = DEFAULT_NEGATIVE,\n                  model_id: str = \"diffusers/stable-diffusion-xl-1.0-inpainting-0.1\",\n                  steps: int = 30, guidance: float = 8.0, seed: int = 0,\n                  margin: int = 120, mode: str = \"inpaint_then_paste\",\n                  replace_mode: str = \"full_replace\",\n                  cover_scale: float = 1.08):\n    \"\"\"\n    Run AI inpainting over the old-QR mask with `prompt`, then (default)\n    composite the real QR so it scans.\n\n    mode:\n      \"inpaint_then_paste\" (default, recommended):\n            AI regenerates/blends the QR region per the prompt, then the\n            deterministic QR is warped on top -> scannable + AI-edited.\n      \"generative_only\":\n            return the raw model output (NOT guaranteed to scan -- for\n            comparison/experimentation only).\n\n    Returns (out_bgr, info_dict). If the model can't load, falls back to the\n    deterministic warp and info[\"fallback\"]=True.\n    \"\"\"\n    h, w = cert_bgr.shape[:2]\n    quad = region.quad.astype(np.float32)\n\n    if not ai_available():\n        out = replace_qr(cert_bgr, region, qr_bgr, mode=replace_mode,\n                         cover_scale=cover_scale)\n        return out, {\"ran\": False, \"fallback\": \"diffusers/torch not installed\"}\n\n    # The SDXL inpainting model is GPU-only in practice: never attempt it on\n    # CPU (multi-GB download, minutes/image, likely OOM). Fall back instantly.\n    if _device() != \"cuda\":\n        out = replace_qr(cert_bgr, region, qr_bgr, mode=replace_mode,\n                         cover_scale=cover_scale)\n        return out, {\"ran\": False,\n                     \"fallback\": \"AI inpainting needs a CUDA GPU; used warp\"}\n\n    try:\n        import torch\n    except Exception:\n        torch = None\n    try:\n        from PIL import Image\n    except Exception as e:\n        out = replace_qr(cert_bgr, region, qr_bgr, mode=replace_mode,\n                         cover_scale=cover_scale)\n        return out, {\"ran\": False, \"fallback\": f\"PIL missing: {e}\"}\n\n    try:\n        pipe = load_pipeline(model_id)\n    except Exception as e:  # download/OOM -> graceful fallback\n        out = replace_qr(cert_bgr, region, qr_bgr, mode=replace_mode,\n                         cover_scale=cover_scale)\n        return out, {\"ran\": False, \"fallback\": f\"model load failed: {e}\"}\n\n    mask = _mask_from_quad((h, w), quad, dilate_px=8)\n    x0, y0, x1, y1 = _crop_bbox(quad, h, w, margin)\n    crop = cert_bgr[y0:y1, x0:x1].copy()\n    cmask = mask[y0:y1, x0:x1].copy()\n\n    ch, cw = crop.shape[:2]\n    W8, H8 = _round8(cw), _round8(ch)\n\n    pil_img = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)).resize((W8, H8))\n    pil_mask = Image.fromarray(cmask).resize((W8, H8), Image.NEAREST)\n\n    gen = torch.Generator(device=\"cpu\").manual_seed(seed) if torch is not None else None\n    result = pipe(\n        prompt=prompt, negative_prompt=negative_prompt,\n        image=pil_img, mask_image=pil_mask,\n        height=H8, width=W8,\n        guidance_scale=guidance, num_inference_steps=steps,\n        generator=gen,\n    ).images[0]\n    result = result.resize((cw, ch), Image.LANCZOS)\n    inp_crop = cv2.cvtColor(np.array(result), cv2.COLOR_RGB2BGR)\n\n    # Strictly apply ONLY inside the mask -> rest of certificate untouched.\n    m3 = cv2.cvtColor(cmask, cv2.COLOR_GRAY2BGR) > 0\n    edited_crop = crop.copy()\n    edited_crop[m3] = inp_crop[m3]\n\n    edited = cert_bgr.copy()\n    edited[y0:y1, x0:x1] = edited_crop\n\n    info = {\"ran\": True, \"model\": model_id, \"device\": _device(),\n            \"steps\": steps, \"crop\": (x0, y0, x1, y1), \"mode\": mode}\n\n    if mode == \"generative_only\":\n        return edited, info\n\n    # Composite the REAL QR (guaranteed scannable) over the AI region.\n    final = replace_qr(edited, region, qr_bgr, mode=replace_mode,\n                            cover_scale=cover_scale)\n    info[\"qr_composited\"] = True\n    return final, info\n",
      "pipeline.py": "\"\"\"\npipeline.py\n-----------\nEnd-to-end batch workflow:\n\n  for every certificate image in an input folder:\n     1. DETECT  the old QR  (WeChat CNN model from Hugging Face -> OpenCV ->\n                geometric contour finder)\n     2. BUILD   the payload (URL/text/CSV that scanning should open)\n     3. GENERATE a brand-new crisp QR code\n     4. REPLACE the old QR with the new one at the exact same place/size/angle\n                (nothing else on the certificate is touched)\n     5. VERIFY  the new QR is actually decodable on the output image\n     6. SAVE    the new certificate\n\n  then: write a manifest.csv and ZIP all outputs.\n\nRuns on CPU or GPU identically (the geometry work is CPU; the CNN detector is\ntiny). Designed for 200+ certificates in one pass on Kaggle.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport csv\nimport io\nimport os\nimport zipfile\nfrom dataclasses import dataclass, field\n\nimport cv2\nimport numpy as np\n\nfrom .detector import (detect_qr, draw_debug,\n                       fallback_region_from_regions, fallback_region_default)\nfrom .payload import build_payload, load_csv_map, slug_from_filename, safe_slug\nfrom .qr_generator import build_qr\nfrom .qr_art import build_artistic_qr\nfrom .replacer import replace_qr\nfrom .ai_extract import ai_extract_fields\nfrom .verification_page import build_page\nfrom .ai_inpaint import (\n    ai_replace_qr, ai_available, DEFAULT_PROMPT, DEFAULT_NEGATIVE,\n)\n\nIMG_EXTS = (\".png\", \".jpg\", \".jpeg\", \".webp\", \".bmp\", \".tif\", \".tiff\")\n\n\n@dataclass\nclass Result:\n    filename: str\n    status: str                       # \"ok\" | \"recheck\" | \"failed\"\n    payload: str = \"\"\n    method: str = \"\"\n    qr_side_px: float = 0.0\n    verified: bool = False\n    ai_used: bool = False\n    out_name: str = \"\"\n    slug: str = \"\"\n    fields: dict = None\n    extract_method: str = \"\"\n    note: str = \"\"\n\n\n@dataclass\nclass Config:\n    input_dir: str = \"input\"\n    output_dir: str = \"output\"\n    model_dir: str = \"models/wechat\"\n    zip_path: str = \"certificates_with_new_qr.zip\"\n    payload_mode: str = \"url\"          # url | text | csv | copy\n    base_url: str = \"https://example.github.io/certificates/\"\n    url_extension: str = \".png\"\n    # QR links to a per-candidate VERIFICATION PAGE (.html) that shows the\n    # certificate image with the candidate details (read off the certificate by\n    # the AI vision model) beneath it. False -> QR links directly to the image.\n    verification_page: bool = True\n    # AI vision-language model that READS every field off the certificate image\n    # (no OCR / no CSV). Requires a GPU on Kaggle (T4). Empty fields if it fails.\n    ai_extract_model: str = \"Qwen/Qwen2-VL-2B-Instruct\"\n    text_prefix: str = \"\"\n    csv_path: str | None = None\n    qr_error_correction: str = \"M\"\n    qr_box_size: int = 20\n    qr_border: int = 4\n    replace_mode: str = \"full_replace\"  # full_replace | keep_frame\n    qr_cover_scale: float = 1.08        # >1 = white box grows to hide old frame\n    output_ext: str = \".png\"\n    jpeg_quality: int = 95\n    save_debug: bool = True\n    use_cnn: bool = True\n    # ---- QR generator backend ----\n    # \"standard\"  -> plain deterministic qrcode (always scans; recommended for\n    #                official certificates)\n    # \"artistic\"  -> HF QR Code Monster ControlNet (AI art-QR; auto-verified and\n    #                falls back to standard if no seed scans). GPU recommended.\n    qr_backend: str = \"standard\"\n    art_prompt: str = \"\"       # \"\" -> qr_art.CLEAN_PROMPT\n    art_controlnet_scale: float = 1.5\n    art_size: int = 768\n    art_tries: int = 6\n    # ---- AI image-generator (diffusion inpainting) ----\n    use_ai: bool = False\n    ai_model: str = \"diffusers/stable-diffusion-xl-1.0-inpainting-0.1\"\n    ai_prompt: str = DEFAULT_PROMPT\n    ai_negative: str = DEFAULT_NEGATIVE\n    ai_steps: int = 30\n    ai_guidance: float = 8.0\n    ai_seed: int = 0\n    ai_margin: int = 120\n    ai_mode: str = \"inpaint_then_paste\"   # inpaint_then_paste | generative_only\n    # If a QR can't be located on one certificate, place the new QR at the\n    # batch-derived template position (works because all certs share a layout).\n    fallback_placement: bool = True\n\n\ndef list_certificates(input_dir: str) -> list[str]:\n    out = []\n    for root, _dirs, files in os.walk(input_dir):\n        for f in sorted(files):\n            if f.lower().endswith(IMG_EXTS):\n                out.append(os.path.join(root, f))\n    return sorted(out)\n\n\n_WECHAT_DECODER = None\n\n\ndef _get_wechat_decoder(model_dir: str):\n    \"\"\"A WeChat CNN decoder is far stronger on small/dense QRs (used both to\n    read old QRs and to verify new ones).\"\"\"\n    global _WECHAT_DECODER\n    if _WECHAT_DECODER is not None:\n        return _WECHAT_DECODER\n    if not hasattr(cv2, \"wechat_qrcode_WeChatQRCode\"):\n        return None\n    paths = [os.path.join(model_dir, f) for f in\n             (\"detect.prototxt\", \"detect.caffemodel\",\n              \"sr.prototxt\", \"sr.caffemodel\")]\n    if not all(os.path.exists(p) for p in paths):\n        return None\n    try:\n        _WECHAT_DECODER = cv2.wechat_qrcode_WeChatQRCode(*paths)\n    except Exception:\n        _WECHAT_DECODER = None\n    return _WECHAT_DECODER\n\n\ndef _decode_any_qr(img: np.ndarray, model_dir: str = \"models/wechat\") -> str:\n    \"\"\"Best-effort decode to VERIFY the new QR scans. Returns text or ''.\n\n    Tries the strong WeChat CNN decoder first (it reads small/dense QRs that\n    OpenCV's classic decoder misses), then OpenCV ArUco/classic at several\n    scales.\n    \"\"\"\n    h, w = img.shape[:2]\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img\n\n    wd = _get_wechat_decoder(model_dir)\n    if wd is not None:\n        for scale in (1, 2, 3):\n            src = gray if scale == 1 else cv2.resize(\n                gray, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)\n            try:\n                res, _pts = wd.detectAndDecode(src)\n                if len(res) > 0 and res[0]:\n                    return res[0]\n            except Exception:\n                pass\n\n    dets = []\n    if hasattr(cv2, \"QRCodeDetectorAruco\"):\n        try:\n            dets.append(cv2.QRCodeDetectorAruco())\n        except Exception:\n            pass\n    dets.append(cv2.QRCodeDetector())\n    for scale in (1, 2, 3, 4):\n        src = gray if scale == 1 else cv2.resize(\n            gray, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)\n        for d in dets:\n            try:\n                data, _, _ = d.detectAndDecode(src)\n            except Exception:\n                data = \"\"\n            if data:\n                return data\n    return \"\"\n\n\ndef process_one(path: str, cfg: Config,\n                csv_map: dict | None = None,\n                region_override=None) -> tuple[np.ndarray | None, Result]:\n    name = os.path.basename(path)\n    slug = safe_slug(path)\n    res = Result(filename=name, status=\"failed\", slug=slug,\n                 out_name=slug + cfg.output_ext)\n    img = cv2.imread(path, cv2.IMREAD_COLOR)\n    if img is None:\n        res.note = \"could not read image\"\n        return None, res\n\n    # 0) The AI vision model READS all candidate details off the certificate\n    #    image (name, parent, aadhar, course, dates, institute). No OCR / CSV.\n    fields = {}\n    extract_used = \"\"\n    try:\n        fields = ai_extract_fields(img, model_id=cfg.ai_extract_model)\n        if fields and not fields.get(\"_error\"):\n            extract_used = \"AI-VLM\"\n        elif fields.get(\"_error\"):\n            extract_used = \"AI-FAILED\"\n            print(f\"   \u26a0\ufe0f  AI extraction failed for {name}: {fields['_error']}\")\n    except Exception as e:\n        fields = {\"_error\": str(e)}\n        extract_used = \"AI-FAILED\"\n        print(f\"   \u26a0\ufe0f  AI extraction crashed for {name}: {e}\")\n    res.fields = fields\n    res.extract_method = extract_used\n\n    # The QR links to the verification PAGE (.html) when enabled, else the raw\n    # image (cfg.url_extension).\n    link_ext = \".html\" if cfg.verification_page else cfg.url_extension\n\n    # 1) locate the old QR: detect it, or use a provided fallback placement.\n    region = region_override\n    if region is None:\n        region = detect_qr(img, model_dir=cfg.model_dir, use_cnn=cfg.use_cnn)\n    if region is None:\n        res.note = \"old QR not found\"\n        return img, res\n\n    res.method = region.method\n    res.qr_side_px = round(region.side_px, 1)\n\n    # old payload (only needed for 'copy' mode)\n    old_data = _decode_any_qr(img, cfg.model_dir) if cfg.payload_mode == \"copy\" else None\n\n    # 2) payload (links to the verification page or the raw image)\n    payload = build_payload(\n        path, mode=cfg.payload_mode, base_url=cfg.base_url,\n        extension=link_ext, text_prefix=cfg.text_prefix,\n        csv_map=csv_map, old_data=old_data)\n    res.payload = payload\n\n    # 3) generate fresh QR at high resolution.\n    #    Optional AI \"artistic QR\" backend (HF QR Code Monster ControlNet):\n    #    returns an AI-styled QR that decodes to `payload`, else None -> we\n    #    transparently fall back to the deterministic standard QR.\n    qr_img = None\n    if cfg.qr_backend == \"artistic\":\n        from .qr_art import CLEAN_PROMPT\n        qr_img = build_artistic_qr(\n            payload,\n            prompt=cfg.art_prompt or CLEAN_PROMPT,\n            controlnet_scale=cfg.art_controlnet_scale,\n            size=cfg.art_size, max_tries=cfg.art_tries)\n        if qr_img is not None:\n            res.note = \"AI artistic QR (verified)\"\n    if qr_img is None:\n        qr_img = build_qr(payload, box_size=cfg.qr_box_size,\n                          border=cfg.qr_border,\n                          error_correction=cfg.qr_error_correction)\n\n    # 4) REPLACE the old QR.\n    #    - AI path: masked diffusion inpainting (prompt-driven), then the\n    #      real QR is composited so it is guaranteed scannable.\n    #    - deterministic path: exact perspective warp.\n    if cfg.use_ai:\n        out, ai_info = ai_replace_qr(\n            img, region, qr_img,\n            prompt=cfg.ai_prompt, negative_prompt=cfg.ai_negative,\n            model_id=cfg.ai_model, steps=cfg.ai_steps,\n            guidance=cfg.ai_guidance, seed=cfg.ai_seed,\n            margin=cfg.ai_margin, mode=cfg.ai_mode,\n            replace_mode=cfg.replace_mode,\n            cover_scale=cfg.qr_cover_scale)\n        res.ai_used = bool(ai_info.get(\"ran\"))\n        if not ai_info.get(\"ran\"):\n            res.method += \"+ai-fallback(warp)\"\n    else:\n        out = replace_qr(img, region, qr_img, mode=cfg.replace_mode,\n                         cover_scale=cfg.qr_cover_scale)\n        ai_info = {}\n\n    # 5) verify the new QR decodes on the output\n    decoded = _decode_any_qr(out, cfg.model_dir)\n    res.verified = bool(decoded)\n    art_tag = \"AI artistic QR; \" if (cfg.qr_backend == \"artistic\"\n                                    and \"artistic\" in res.note) else \"\"\n    if not decoded:\n        res.status = \"recheck\"\n        res.note = art_tag + \"new QR not auto-verified (usually still scans; check debug)\"\n    else:\n        res.status = \"ok\"\n        if res.ai_used:\n            res.note = art_tag + \"verified scannable (AI inpaint+QR)\"\n        else:\n            res.note = art_tag + \"verified scannable\"\n\n    return out, res\n\n\ndef run(cfg: Config) -> dict:\n    os.makedirs(cfg.output_dir, exist_ok=True)\n    debug_dir = os.path.join(cfg.output_dir, \"_debug\")\n    if cfg.save_debug:\n        os.makedirs(debug_dir, exist_ok=True)\n\n    csv_map = None\n    if cfg.payload_mode == \"csv\" and cfg.csv_path:\n        csv_map = load_csv_map(cfg.csv_path)\n\n    certs = list_certificates(cfg.input_dir)\n\n    if cfg.payload_mode == \"url\":\n        placeholder_markers = (\"YOUR-USERNAME\", \"example.github.io\",\n                               \"example.com\", \"REPLACE\", \"your-username\")\n        if any(m.lower() in (cfg.base_url or \"\").lower()\n               for m in placeholder_markers):\n            print(\"=\" * 70)\n            print(\"\u26a0\ufe0f  WARNING: BASE_URL still looks like a PLACEHOLDER.\")\n            print(f\"    base_url = {cfg.base_url}\")\n            print(\"    The QR codes WILL scan, but they point at a URL that\")\n            print(\"    does not exist yet -> your phone shows a 404 page.\")\n            print(\"    1) Set base_url to the address where you will upload the\")\n            print(\"       OUTPUT images (e.g. GitHub Pages: create a repo, turn\")\n            print(\"       on Settings->Pages, upload the finished PNGs).\")\n            print(\"    2) Re-run. See README 'Hosting' / notebook section 8.\")\n            print(\"=\" * 70)\n\n    # ---- detection pass: locate the QR on every certificate so that a cert\n    # where detection fails can inherit the position from its template (all\n    # certificates in a batch share a layout).\n    detected: dict[str, object] = {}\n    template: list = []\n    for path in certs:\n        img = cv2.imread(path, cv2.IMREAD_COLOR)\n        if img is None:\n            continue\n        reg = detect_qr(img, model_dir=cfg.model_dir, use_cnn=cfg.use_cnn)\n        h, w = img.shape[:2]\n        detected[path] = reg\n        if reg is not None and \"fallback\" not in reg.method:\n            template.append((reg, h, w))\n\n    results: list[Result] = []\n    ok = recheck = failed = 0\n    ai_ok = ai_fail = 0\n\n    zip_path = cfg.zip_path\n    with zipfile.ZipFile(zip_path, \"w\", zipfile.ZIP_DEFLATED) as zf:\n        for path in certs:\n            img = cv2.imread(path, cv2.IMREAD_COLOR)\n            override = None\n            if detected.get(path) is None and cfg.fallback_placement:\n                # inherit QR position from the other certificates / template\n                override = fallback_region_from_regions(img, template)\n                if override is None:\n                    override = fallback_region_default(img)\n            out, res = process_one(path, cfg, csv_map, region_override=override)\n            if res.extract_method == \"AI-VLM\":\n                ai_ok += 1\n            else:\n                ai_fail += 1\n            if override is not None and res.status in (\"ok\", \"recheck\"):\n                res.note += f\" | QR placed via {override.method}\"\n            results.append(res)\n            if res.status == \"ok\":\n                ok += 1\n            elif res.status == \"recheck\":\n                recheck += 1\n            else:\n                failed += 1\n\n            if out is not None and res.status in (\"ok\", \"recheck\"):\n                out_name = res.out_name\n                out_path = os.path.join(cfg.output_dir, out_name)\n                _write_image(out, out_path, cfg)\n                zf.write(out_path, arcname=\"img/\" + out_name)\n\n                # Per-candidate VERIFICATION PAGE (self-contained HTML).\n                if cfg.verification_page:\n                    page = build_page(res.fields or {}, out, out_name)\n                    page_name = res.slug + \".html\"\n                    page_path = os.path.join(cfg.output_dir, page_name)\n                    with open(page_path, \"w\", encoding=\"utf-8\") as fh:\n                        fh.write(page)\n                    zf.write(page_path, arcname=page_name)\n\n                if cfg.save_debug and res.status == \"recheck\":\n                    reg = detect_qr(out, model_dir=cfg.model_dir,\n                                    use_cnn=False)\n                    dbg = draw_debug(out, reg) if reg else out\n                    cv2.imwrite(os.path.join(debug_dir, \"dbg_\" + out_name), dbg)\n            elif out is not None and cfg.save_debug:\n                cv2.imwrite(os.path.join(\n                    debug_dir, \"FAILED_\" + safe_slug(path) + cfg.output_ext), out)\n\n            tag = \"AI\" if res.ai_used else (\"ai-fb\" if \"+ai-fallback\" in res.method else \"  \")\n            print(f\"[{res.status:7s}] {tag} {res.filename:36s} \"\n                  f\"{res.method:24s} {res.qr_side_px:6.1f}px  {res.payload}\")\n\n    # manifest\n    manifest = os.path.join(cfg.output_dir, \"manifest.csv\")\n    with open(manifest, \"w\", newline=\"\", encoding=\"utf-8\") as f:\n        wr = csv.writer(f)\n        wr.writerow([\"filename\", \"status\", \"ai_used\", \"method\", \"qr_side_px\",\n                     \"verified\", \"qr_payload\", \"name\", \"parent\", \"aadhar\",\n                     \"course\", \"date_from\", \"date_to\", \"institute\",\n                     \"extracted_by\", \"note\"])\n        for r in results:\n            f = r.fields or {}\n            wr.writerow([r.filename, r.status, r.ai_used, r.method,\n                         r.qr_side_px, r.verified, r.payload,\n                         f.get(\"name\", \"\"), f.get(\"parent\", \"\"),\n                         f.get(\"aadhar\", \"\"), f.get(\"course\", \"\"),\n                         f.get(\"date_from\", \"\"), f.get(\"date_to\", \"\"),\n                         f.get(\"institute\", \"\"), r.extract_method, r.note])\n\n    summary = {\"total\": len(certs), \"ok\": ok, \"recheck\": recheck,\n               \"failed\": failed, \"fields_extracted\": ai_ok,\n               \"fields_failed\": ai_fail,\n               \"zip\": os.path.abspath(zip_path),\n               \"manifest\": os.path.abspath(manifest)}\n    print(\"\\n===== SUMMARY =====\")\n    for k, v in summary.items():\n        print(f\"{k:16s}: {v}\")\n    if ai_fail:\n        print(\"\\n\u26a0\ufe0f  The AI could NOT read fields from \"\n              f\"{ai_fail} certificate(s) -> those pages show '\u2014'.\")\n        print(\"   Check: Accelerator = GPU T4; and run the 'Test AI extraction'\")\n        print(\"   cell to see the raw model output / error.\")\n    return summary\n\n\ndef _write_image(img: np.ndarray, path: str, cfg: Config):\n    ext = cfg.output_ext.lower()\n    if ext in (\".jpg\", \".jpeg\"):\n        cv2.imwrite(path, img, [cv2.IMWRITE_JPEG_QUALITY, cfg.jpeg_quality])\n    else:\n        cv2.imwrite(path, img)\n"
    }
    for name, content in SRC.items():
        with open(os.path.join("/kaggle/working/certbot/src", name), "w",
                  encoding="utf-8") as f:
            f.write(content)
        print("wrote", name, len(content), "chars")
    print("pipeline has qr_cover_scale:", "qr_cover_scale" in SRC["pipeline.py"])

    # publish helper (GitHub Pages)
    with open("/kaggle/working/certbot/publish_to_github_pages.py", "w",
              encoding="utf-8") as f:
        f.write("#!/usr/bin/env python3\n\"\"\"\npublish_to_github_pages.py\n--------------------------\nUpload the finished certificate images to a GitHub Pages repository so the QR\nlinks resolve (scan -> certificate opens in the browser). Self-verifying:\nafter each upload it polls the public Pages URL until the image is live.\n\nToken: a GitHub Personal Access Token (classic) with the `repo` scope.\nProvided via GITHUB_TOKEN / GH_TOKEN env var (on Kaggle, add a notebook Secret\nnamed GITHUB_TOKEN), or pasted at the prompt.\n\nCreate a token: https://github.com/settings/tokens  (Generate new token\n(classic) -> tick \"repo\").\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport base64\nimport os\nimport time\nimport zipfile\n\ntry:\n    import requests\nexcept Exception:  # pragma: no cover\n    requests = None\n\nAPI = \"https://api.github.com\"\n\n\ndef _get_token() -> str:\n    tok = os.environ.get(\"GITHUB_TOKEN\") or os.environ.get(\"GH_TOKEN\") or \"\"\n    if not tok:\n        tok = input(\"Paste your GitHub token (repo scope): \").strip()\n    return tok\n\n\ndef publish(zip_path: str, owner: str, repo: str, token: str,\n            subdir: str = \"img\", branch: str = \"main\",\n            verify: bool = True, verbose: bool = True):\n    if requests is None:\n        raise SystemExit(\"'requests' is required (pip install requests).\")\n\n    files = []   # (name, content, upload_dir)\n    with zipfile.ZipFile(zip_path) as z:\n        for name in z.namelist():\n            base_name = os.path.basename(name)\n            low = name.lower()\n            if low.endswith((\".png\", \".jpg\", \".jpeg\", \".webp\")):\n                # images stay under img/ (whether the zip root or img/ prefix)\n                files.append((base_name, z.read(name), subdir))\n            elif low.endswith(\".html\"):\n                # verification pages live at the repo root (QR URLs point there)\n                files.append((base_name, z.read(name), \"\"))\n\n    headers = {\"Authorization\": f\"Bearer {token}\",\n               \"Accept\": \"application/vnd.github+json\",\n               \"User-Agent\": \"cert-qr-bot-publisher\"}\n    api_base = f\"{API}/repos/{owner}/{repo}/contents\"\n    live_root = f\"https://{owner}.github.io/{repo}/\"\n    ok = fail = upload_fail = 0\n    uploaded_files = []   # (name, live_url, is_html)\n\n    if verbose:\n        print(f\"Publishing {len(files)} file(s) to {owner}/{repo} ...\")\n\n    for i, (name, content, upload_dir) in enumerate(files, 1):\n        path = f\"{api_base}/{upload_dir}/{name}\" if upload_dir \\\n            else f\"{api_base}/{name}\"\n        # find existing sha (to update rather than conflict)\n        sha = None\n        try:\n            r = requests.get(path, headers=headers,\n                             params={\"ref\": branch}, timeout=30)\n            if r.status_code == 200:\n                sha = r.json().get(\"sha\")\n        except Exception:\n            pass\n\n        body = {\"message\": f\"Publish {name}\",\n                \"content\": base64.b64encode(content).decode(),\n                \"branch\": branch}\n        if sha:\n            body[\"sha\"] = sha\n\n        uploaded = False\n        for attempt in range(3):\n            try:\n                r = requests.put(path, headers=headers, json=body, timeout=60)\n                if r.status_code in (200, 201):\n                    uploaded = True\n                    break\n                if verbose:\n                    print(f\"  ! {name} upload attempt {attempt+1}: \"\n                          f\"HTTP {r.status_code} {r.text[:160]}\")\n            except Exception as e:\n                if verbose:\n                    print(f\"  ! {name} attempt {attempt+1} error: {e}\")\n            time.sleep(2)\n\n        live_url = (f\"https://{owner}.github.io/{repo}/{upload_dir}/{name}\"\n                    if upload_dir else f\"https://{owner}.github.io/{repo}/{name}\")\n        is_img = name.lower().endswith((\".png\", \".jpg\", \".jpeg\", \".webp\"))\n        is_html = name.lower().endswith(\".html\")\n        if uploaded:\n            uploaded_files.append((name, live_url, is_html))\n        # Per-file quick check (short); the final sweep below re-checks any\n        # that weren't live yet, so Pages propagation lag is not fatal here.\n        if uploaded and verify and (is_img or is_html):\n            live = _wait_live(live_url, expect_html=is_html,\n                              tries=3, delay=4)\n        else:\n            live = uploaded\n\n        if uploaded and live:\n            ok += 1\n            if verbose:\n                print(f\"  [{i}/{len(files)}] \u2705 {name} -> {live_url}\")\n        elif uploaded:\n            # uploaded but not live yet \u2014 final sweep will re-check it\n            if verbose:\n                print(f\"  [{i}/{len(files)}] \u23f3 {name} uploaded, waiting for Pages ...\")\n        else:\n            upload_fail += 1\n            if verbose:\n                print(f\"  [{i}/{len(files)}] \u274c {name} (upload failed)\")\n        time.sleep(0.4)\n\n    # ---- Final verification sweep: keep re-checking every uploaded file until\n    # each is live on GitHub Pages (handles propagation lag for image AND html).\n    if verify and uploaded_files:\n        pending = [(n, u, h) for (n, u, h) in uploaded_files]\n        if verbose and pending:\n            print(f\"\\nVerifying {len(pending)} page(s)/image(s) are live ...\")\n        for round_no in range(1, 13):   # up to ~12 * 10s = 2 minutes\n            still = []\n            for name, live_url, is_html in pending:\n                if _check_one(live_url, is_html):\n                    ok += 0  # already counted as uploaded\n                    if verbose:\n                        print(f\"  \u2705 live: {name}\")\n                else:\n                    still.append((name, live_url, is_html))\n            pending = still\n            if not pending:\n                break\n            if verbose:\n                print(f\"  ... {len(pending)} not live yet, re-checking \"\n                      f\"(round {round_no})\")\n            time.sleep(10)\n        # anything still pending after the sweep is reported for a re-run\n        for name, _u, _h in pending:\n            if verbose:\n                print(f\"  \u274c {name} still not live after waiting (re-run \"\n                      f\"section 8 in ~1 min)\")\n        not_live = len(pending)\n    else:\n        not_live = 0\n    fail = not_live + upload_fail\n\n    live_count = len(uploaded_files) - not_live\n    if verbose:\n        print(f\"\\nDone: {live_count} live, {fail} not yet live \"\n              f\"({upload_fail} upload failure(s)).\")\n        if fail == 0:\n            print(\"\u2705 Every image and verification page is uploaded AND live.\")\n        print(\"Pages (QR base URL): \" + live_root)\n        print(\"Images under:        \" + live_root + subdir + \"/\")\n    return {\"ok\": live_count, \"fail\": fail, \"base\": live_root}\n\n\ndef _check_one(url: str, expect_html: bool) -> bool:\n    import random\n    bust = url + (\"&\" if \"?\" in url else \"?\") + \"cb=\" + str(random.randint(0, 10**9))\n    try:\n        r = requests.get(bust, timeout=30,\n                         headers={\"Cache-Control\": \"no-cache\",\n                                  \"User-Agent\": \"cert-qr-bot-publisher\"})\n        ctype = r.headers.get(\"content-type\", \"\")\n        if r.status_code != 200:\n            return False\n        if expect_html:\n            return \"html\" in ctype\n        return \"image\" in ctype\n    except Exception:\n        return False\n\n\ndef _wait_live(url: str, tries: int = 14, delay: int = 5,\n               expect_html: bool = False) -> bool:\n    for _ in range(tries):\n        if _check_one(url, expect_html):\n            return True\n        time.sleep(delay)\n    return False\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--zip\", default=\"certificates_with_new_qr.zip\")\n    ap.add_argument(\"--owner\", default=\"Naserkhan07\")\n    ap.add_argument(\"--repo\", default=\"certificate-qr\")\n    ap.add_argument(\"--subdir\", default=\"img\")\n    ap.add_argument(\"--branch\", default=\"main\")\n    ap.add_argument(\"--no-verify\", action=\"store_true\")\n    args = ap.parse_args()\n    token = _get_token()\n    if not token:\n        raise SystemExit(\"No GitHub token provided.\")\n    publish(args.zip, args.owner, args.repo, token, args.subdir,\n            args.branch, verify=not args.no_verify)\n\n\nif __name__ == \"__main__\":\n    main()\n")
    print("wrote publish_to_github_pages.py")
import sys, os, shutil

# Free any vision model the section 4b test cell left on the GPU, then force
# Python to pick up the freshly written modules (avoids CUDA OOM / stale code).
try:
    from src.ai_extract import release_model
    release_model()
except Exception:
    pass
for _m in list(sys.modules):
    if _m == "src" or _m.startswith("src."):
        del sys.modules[_m]
try:
    import gc, torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
except Exception:
    pass

sys.path.insert(0, "/kaggle/working/certbot")
from src.pipeline import Config, run

# Build an "input" folder view of the dataset images (non-destructive: copies)
import shutil
INPUT_DIR = "/kaggle/working/certbot/input"
OUTPUT_DIR = "/kaggle/working/certbot/output"
if os.path.exists(INPUT_DIR): shutil.rmtree(INPUT_DIR)
os.makedirs(INPUT_DIR, exist_ok=True)
for p in CANDIDATES:
    # skip anything inside our own working dir
    if "/kaggle/working/" in p: continue
    shutil.copy(p, os.path.join(INPUT_DIR, os.path.basename(p)))

try:
    import torch
    print("torch:", torch.__version__, "| CUDA GPU:", torch.cuda.is_available())
    if USE_AI and not torch.cuda.is_available():
        print("⚠️ USE_AI=True but no GPU — set the notebook accelerator to GPU "
              "(Settings → Accelerator → GPU P100/T4). Falling back to the "
              "deterministic warp if the model cannot run.")
except Exception as e:
    print("torch unavailable:", e)

from src.ai_inpaint import DEFAULT_PROMPT
print("AI prompt used:\n", DEFAULT_PROMPT)

cfg = Config(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    model_dir=MODEL_DIR,
    zip_path="/kaggle/working/certificates_with_new_qr.zip",
    payload_mode=PAYLOAD_MODE,
    base_url=BASE_URL,
    csv_path=CSV_PATH,
    text_prefix=TEXT_PREFIX,
    replace_mode=REPLACE_MODE,
    qr_cover_scale=QR_COVER_SCALE,
    qr_error_correction=QR_ERROR_CORRECTION,
    output_ext=OUTPUT_EXT,
    ai_extract_model=AI_EXTRACT_MODEL,
    qr_backend=QR_BACKEND,
    art_controlnet_scale=ART_CONTROLNET_SCALE,
    art_tries=ART_TRIES,
    art_size=ART_SIZE,
    use_cnn=True,
    save_debug=True,
    use_ai=USE_AI,
    ai_model=AI_MODEL,
    ai_steps=AI_STEPS,
    ai_guidance=AI_GUIDANCE,
    ai_seed=AI_SEED,
    ai_mode=AI_MODE,
)
summary = run(cfg)


## 🔍 6. Review the manifest & spot-check an output

In [ ]:
import pandas as pd
manifest = pd.read_csv(os.path.join(OUTPUT_DIR, "manifest.csv"))
display(manifest)
print("Counts:\n", manifest["status"].value_counts())
if (manifest["status"] != "ok").any():
    print("\n⚠️ Files needing a quick look (debug overlays in output/_debug/):")
    print(manifest[manifest["status"] != "ok"][["filename","status","note"]])

In [ ]:
# Visually spot-check the first output certificate
from PIL import Image
import glob
outs = sorted(glob.glob(OUTPUT_DIR + "/*" + OUTPUT_EXT))
if outs:
    im = Image.open(outs[0])
    print("Showing:", outs[0], im.size)
    display(im)

## 🗜️ 7. Get the ZIP
The zip is at `/kaggle/working/certificates_with_new_qr.zip` — download it from the Kaggle **Output** panel (it also persists as the notebook output).

In [ ]:
import zipfile, os
zp = "/kaggle/working/certificates_with_new_qr.zip"
print("zip exists:", os.path.exists(zp), "size:",
      round(os.path.getsize(zp)/1e6, 2), "MB")
with zipfile.ZipFile(zp) as z:
    names = z.namelist()
    print("files in zip:", len(names))
    print("\n".join(names[:10]), "..." if len(names) > 10 else "")

## 🚀 8. Auto-upload every certificate (so scanning opens it)

The QR stores a URL — the finished image must be uploaded for the link to
resolve. This cell uploads **all** certificates from the ZIP to your Pages
repo and verifies each one is live; a scan then opens that exact certificate.

One-time setup on Kaggle:
1. Create a token: https://github.com/settings/tokens → **Generate new token
   (classic)** → tick **`repo`** → copy it.
2. **Add-ons → Secrets → Add a new secret** named `GITHUB_TOKEN`, paste it.

The hosting repo (`certificate-qr`) is kept empty/clean and serves only the
certificate images. ⚠️ It is public — see section 9 for private options.


In [ ]:
import os as _os
if not _os.path.isfile("/kaggle/working/certbot/src/ai_extract.py"):
    print("Pipeline modules not written in this session yet — auto-running the section 4 writer first ...")
    import os, shutil
    os.makedirs("/kaggle/working/certbot/src", exist_ok=True)
    # clear stale bytecode from previous runs
    _pyc = "/kaggle/working/certbot/src/__pycache__"
    if os.path.isdir(_pyc): shutil.rmtree(_pyc)
    SRC = {
      "__init__.py": "from .pipeline import Config, run, process_one  # noqa\n",
      "detector.py": "\"\"\"\ndetector.py\n-----------\nLocate the OLD QR code on a certificate image.\n\nDetection cascade (tries each, returns first high-confidence hit):\n  1. WeChat CNN QR detector  (Deep-CV model downloaded from Hugging Face)\n  2. OpenCV Aruco QR detector\n  3. Geometric contour finder (finds the thick black square FRAME around a QR\n     even when the QR itself is too dense/small to decode -- works on certs\n     whose old QR is decorative or damaged)\n\nEvery detector returns the quadrangle of the *outer QR box* (including the\nblack border frame when present), so the new QR can be warped to exactly that\nsize / position / rotation.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport os\nfrom dataclasses import dataclass\n\nimport cv2\nimport numpy as np\n\n\n@dataclass\nclass QRRegion:\n    \"\"\"The 4 corners of the QR's outer box, in order TL, TR, BR, BL (px).\"\"\"\n    quad: np.ndarray            # shape (4, 2), float32\n    method: str                 # which detector found it\n    side_px: float              # average side length in pixels\n    score: float                # confidence 0..1\n\n\n# ---------------------------------------------------------------------------\n# Helpers\n# ---------------------------------------------------------------------------\ndef _order_points(pts: np.ndarray) -> np.ndarray:\n    \"\"\"Order 4 points as TL, TR, BR, BL.\"\"\"\n    pts = np.asarray(pts, dtype=np.float32).reshape(4, 2)\n    s = pts.sum(axis=1)\n    d = np.diff(pts, axis=1).reshape(-1)\n    tl = pts[np.argmin(s)]\n    br = pts[np.argmax(s)]\n    tr = pts[np.argmin(d)]\n    bl = pts[np.argmax(d)]\n    return np.array([tl, tr, br, bl], dtype=np.float32)\n\n\ndef _side_lengths(quad: np.ndarray) -> tuple[float, float, float, float]:\n    tl, tr, br, bl = quad\n    return (\n        float(np.linalg.norm(tr - tl)),\n        float(np.linalg.norm(br - tr)),\n        float(np.linalg.norm(bl - br)),\n        float(np.linalg.norm(tl - bl)),\n    )\n\n\ndef _valid_quad(gray: np.ndarray, quad: np.ndarray,\n                min_finders: int = 2) -> bool:\n    \"\"\"\n    Reject false-positive quads from the ML/OpenCV detectors.\n    A real QR quad: roughly square, on-canvas, convex, and contains >=2 of the\n    three characteristic finder patterns (nested squares).\n    \"\"\"\n    h, w = gray.shape[:2]\n    try:\n        quad = _order_points(quad)\n    except Exception:\n        return False\n    sides = _side_lengths(quad)\n    if min(sides) < max(8.0, 0.04 * min(h, w)):\n        return False\n    if max(sides) > max(h, w) * 1.15:\n        return False\n    if min(sides) / max(sides) < 0.7:\n        return False\n    # convexity (cross products same sign)\n    edges = np.roll(quad, -1, axis=0) - quad\n    crosses = edges[:, 0] * np.roll(edges[:, 1], -1) - \\\n        edges[:, 1] * np.roll(edges[:, 0], -1)\n    if not (np.all(crosses >= -1e-3) or np.all(crosses <= 1e-3)):\n        return False\n    # mostly on canvas\n    x0, y0 = quad.min(axis=0)\n    x1, y1 = quad.max(axis=0)\n    if x1 < -0.05 * w or x0 > 1.05 * w or y1 < -0.05 * h or y0 > 1.05 * h:\n        return False\n    if _count_finder_patterns(gray, quad) < min_finders:\n        return False\n    return True\n\n\ndef _count_finder_patterns(gray: np.ndarray, quad: np.ndarray) -> int:\n    \"\"\"\n    Count finder-like nested squares inside a quad. A QR has 3 finder patterns\n    (TL, TR, BL corners). Used to score contour candidates that look like a QR.\n    \"\"\"\n    tl, tr, br, bl = quad\n    w = int(max(np.linalg.norm(tr - tl), np.linalg.norm(br - bl)))\n    h = int(max(np.linalg.norm(bl - tl), np.linalg.norm(br - tr)))\n    if w < 10 or h < 10:\n        return 0\n    M = cv2.getPerspectiveTransform(\n        quad.astype(np.float32),\n        np.array([[0, 0], [w, 0], [w, h], [0, h]], dtype=np.float32),\n    )\n    patch = cv2.warpPerspective(gray, M, (w, h))\n    _, bw = cv2.threshold(patch, 0, 255,\n                          cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)\n    contours, hier = cv2.findContours(bw, cv2.RETR_TREE,\n                                      cv2.CHAIN_APPROX_SIMPLE)\n    if hier is None:\n        return 0\n    hier = hier[0]\n\n    def nested_depth(i: int) -> int:\n        depth = 0\n        parent = hier[i][3]\n        while parent != -1:\n            depth += 1\n            parent = hier[parent][3]\n        return depth\n\n    finders = 0\n    for i, c in enumerate(contours):\n        area = cv2.contourArea(c)\n        if area == 0:\n            continue\n        x, y, ww, hh = cv2.boundingRect(c)\n        ratio = ww / float(hh)\n        area_ratio = area / float(ww * hh)\n        # finder outer ring: roughly square, moderately sized, >=2 nestings\n        if (0.7 < ratio < 1.35 and area_ratio > 0.55\n                and ww > w * 0.08 and ww < w * 0.6 and nested_depth(i) >= 2):\n            finders += 1\n    return finders\n\n\n# ---------------------------------------------------------------------------\n# Detector 1: WeChat CNN (model from Hugging Face)\n# ---------------------------------------------------------------------------\n_WECHAT = None\n\n\ndef _get_wechat_detector(model_dir: str):\n    \"\"\"Lazy-load the WeChat QR CNN model if the 4 files are present.\"\"\"\n    global _WECHAT\n    if _WECHAT is not None:\n        return _WECHAT\n    if not hasattr(cv2, \"wechat_qrcode_WeChatQRCode\"):\n        return None\n    files = [\"detect.prototxt\", \"detect.caffemodel\",\n             \"sr.prototxt\", \"sr.caffemodel\"]\n    paths = [os.path.join(model_dir, f) for f in files]\n    if not all(os.path.exists(p) for p in paths):\n        return None\n    try:\n        _WECHAT = cv2.wechat_qrcode_WeChatQRCode(*paths)\n    except Exception:\n        _WECHAT = None\n    return _WECHAT\n\n\ndef _detect_wechat(img: np.ndarray, model_dir: str):\n    det = _get_wechat_detector(model_dir)\n    if det is None:\n        return None\n    h, w = img.shape[:2]\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img\n    for scale in (1.0, 2.0, 3.0):\n        src = gray if scale == 1.0 else cv2.resize(\n            gray, (int(w * scale), int(h * scale)),\n            interpolation=cv2.INTER_CUBIC)\n        try:\n            res, points = det.detectAndDecode(src)\n        except Exception:\n            res, points = [], None\n        if points is not None and len(points) > 0:\n            for p in points:\n                quad = _order_points(np.asarray(p, dtype=np.float32) / scale)\n                if _valid_quad(gray, quad, min_finders=2):\n                    return quad\n    return None\n\n\n# ---------------------------------------------------------------------------\n# Detector 2: OpenCV ArUco / classic QR detector\n# ---------------------------------------------------------------------------\ndef _detect_opencv(img: np.ndarray):\n    h, w = img.shape[:2]\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img\n    detectors = []\n    if hasattr(cv2, \"QRCodeDetectorAruco\"):\n        try:\n            detectors.append(cv2.QRCodeDetectorAruco())\n        except Exception:\n            pass\n    detectors.append(cv2.QRCodeDetector())\n    for scale in (1.0, 2.0, 3.0, 4.0):\n        src = gray if scale == 1.0 else cv2.resize(\n            gray, (int(w * scale), int(h * scale)),\n            interpolation=cv2.INTER_CUBIC)\n        for det in detectors:\n            ok, pts = det.detect(src)\n            if ok and pts is not None:\n                quad = _order_points(pts.reshape(4, 2) / scale)\n                if _valid_quad(gray, quad, min_finders=2):\n                    return quad\n    return None\n\n\n# ---------------------------------------------------------------------------\n# Detector 3: geometric contour finder (thick black frame around QR)\n# ---------------------------------------------------------------------------\ndef _detect_by_contours(img: np.ndarray):\n    h, w = img.shape[:2]\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img\n    img_area = h * w\n\n    best = None\n    best_score = -1.0\n\n    for thresh in (90, 120, 150, 180):\n        _, bw = cv2.threshold(gray, thresh, 255, cv2.THRESH_BINARY_INV)\n        # close small gaps inside the frame\n        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))\n        bw = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel)\n        contours, _ = cv2.findContours(bw, cv2.RETR_LIST,\n                                       cv2.CHAIN_APPROX_SIMPLE)\n        for c in contours:\n            area = cv2.contourArea(c)\n            if area < img_area * 0.0008 or area > img_area * 0.25:\n                continue\n            peri = cv2.arcLength(c, True)\n            approx = cv2.approxPolyDP(c, 0.04 * peri, True)\n            if len(approx) != 4:\n                # also accept rotated rect\n                rrect = cv2.minAreaRect(c)\n                (cx, cy), (rw, rh), _ = rrect\n                if rw == 0 or rh == 0:\n                    continue\n                sq = min(rw, rh) / max(rw, rh)\n                if sq < 0.82:\n                    continue\n                box = cv2.boxPoints(rrect)\n            else:\n                box = approx.reshape(4, 2).astype(np.float32)\n                (cx, cy), (rw, rh), _ = cv2.minAreaRect(c)\n                sq = min(rw, rh) / max(rw, rh) if rw and rh else 0\n                if sq < 0.82:\n                    continue\n            quad = _order_points(box)\n            sides = _side_lengths(quad)\n            side = float(np.mean(sides))\n            # reject wildly non-square quads\n            if min(sides) / max(sides) < 0.75:\n                continue\n            finders = _count_finder_patterns(gray, quad)\n            if finders < 2:\n                continue\n            # score: finder count first, then size (QRs on certs are big)\n            score = finders * 10 + (side / max(h, w))\n            if score > best_score:\n                best_score = score\n                best = (quad, finders, side)\n    if best is not None:\n        quad, finders, side = best\n        return QRRegion(quad=quad, method=\"contour\", side_px=side,\n                        score=min(1.0, finders / 3.0))\n    return None\n\n\n# ---------------------------------------------------------------------------\n# Detector 3b: finder-pattern locator (works WITHOUT a black frame)\n# ---------------------------------------------------------------------------\ndef _chain_depth(i: int, hier: np.ndarray, need: int = 2) -> bool:\n    \"\"\"True if contour `i` encloses >=`need` levels of descendants. A finder\n    pattern's outer black ring contains a white gap then the black centre, so\n    its first-child chain reaches depth 2 (the gap can merge with background,\n    which is why depth 4 is never seen on real scans).\"\"\"\n    cur = hier[i][2]\n    d = 0\n    while cur != -1:\n        d += 1\n        if d >= need:\n            return True\n        cur = hier[cur][2]\n    return False\n\n\ndef _find_finder_centers(gray: np.ndarray) -> list[tuple[np.ndarray, float]]:\n    \"\"\"Locate QR finder patterns (the 3 corner 'nested squares'). Returns a\n    list of (centre[x,y], outer-ring size px). Works for framed OR\n    frameless/plain QRs, since finders are always present.\"\"\"\n    h, w = gray.shape[:2]\n    found: list[tuple[np.ndarray, float]] = []\n\n    for thresh in (80, 110, 140, 170, 200):\n        _, bw = cv2.threshold(gray, thresh, 255, cv2.THRESH_BINARY_INV)\n        # No morphology here: closing merges the thin rings of tiny\n        # (sub-100px) QRs and destroys the nesting we are looking for.\n        contours, hier = cv2.findContours(bw, cv2.RETR_TREE,\n                                          cv2.CHAIN_APPROX_SIMPLE)\n        if hier is None:\n            continue\n        hier = hier[0]\n        cands = []\n        for i, c in enumerate(contours):\n            x, y, ww, hh = cv2.boundingRect(c)\n            if ww == 0 or hh == 0:\n                continue\n            side_min = float(min(h, w))\n            if not (side_min * 0.012 < ww < side_min * 0.35 and\n                    side_min * 0.012 < hh < side_min * 0.35):\n                continue\n            if not (0.65 < ww / float(hh) < 1.5):\n                continue\n            area = cv2.contourArea(c)\n            if area / float(ww * hh) < 0.5:\n                continue\n            if not _chain_depth(i, hier, need=2):\n                continue\n            cands.append((x + ww / 2.0, y + hh / 2.0, (ww + hh) / 2.0))\n        # within each threshold, collapse nested/contained candidates to the\n        # OUTERMOST ring (largest), since the centre dot also appears.\n        cands.sort(key=lambda t: -t[2])\n        for cx, cy, size in cands:\n            if not any(np.hypot(cx - fc[0][0], cy - fc[0][1]) < fc[1] * 0.7\n                       for fc in found):\n                found.append((np.array([cx, cy], dtype=np.float32),\n                              float(size)))\n    return found\n\n\ndef _quad_from_finders(finders: list[tuple[np.ndarray, float]]):\n    \"\"\"Given >=3 finder centres, build the QR outer quad (incl. quiet zone).\"\"\"\n    n = len(finders)\n    best = None\n    best_err = 1e9\n    for a in range(n):\n        for b in range(n):\n            if b == a:\n                continue\n            for c in range(n):\n                if c == a or c == b:\n                    continue\n                A, rA = finders[a]   # candidate elbow (TL)\n                B, _ = finders[b]    # candidate TR\n                C, _ = finders[c]    # candidate BL\n                vx = B - A\n                vy = C - A\n                lx = float(np.linalg.norm(vx))\n                ly = float(np.linalg.norm(vy))\n                if lx < 5 or ly < 5:\n                    continue\n                if min(lx, ly) / max(lx, ly) < 0.6:\n                    continue\n                cosang = float(np.dot(vx, vy) / (lx * ly))\n                if abs(cosang) > 0.45:   # want ~90 deg\n                    continue\n                ux = vx / lx\n                uy = vy / ly\n                r = float(np.mean([fr[1] for fr in (finders[a], finders[b],\n                                                    finders[c])]))\n                module = r / 7.0\n                ext = 7.5 * module      # ring centre (3.5 mod) + quiet (4 mod)\n                TL = A - ext * (ux + uy)\n                TR = TL + ux * (lx + 2 * ext)\n                BL = TL + uy * (ly + 2 * ext)\n                BR = TL + ux * (lx + 2 * ext) + uy * (ly + 2 * ext)\n                err = abs(cosang) + abs(lx - ly) / max(lx, ly)\n                if err < best_err:\n                    best_err = err\n                    best = _order_points(np.array([TL, TR, BR, BL],\n                                                  dtype=np.float32))\n    return best\n\n\ndef _detect_by_finders(img: np.ndarray):\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img\n    h, w = gray.shape[:2]\n    finders = _find_finder_centers(gray)\n    if len(finders) < 3:\n        return None\n    quad = _quad_from_finders(finders)\n    if quad is None:\n        return None\n    sides = _side_lengths(quad)\n    side = float(np.mean(sides))\n    if min(sides) < max(8.0, 0.04 * min(h, w)):\n        return None\n    if min(sides) / max(sides) < 0.7:\n        return None\n    return QRRegion(quad=quad, method=\"finders\", side_px=side, score=0.95)\n\n\n# ---------------------------------------------------------------------------\n# Public API\n# ---------------------------------------------------------------------------\ndef detect_qr(img: np.ndarray, model_dir: str = \"models/wechat\",\n              use_cnn: bool = True):\n    \"\"\"\n    Find the old QR on a certificate.\n\n    Returns QRRegion or None.\n    \"\"\"\n    if use_cnn:\n        quad = _detect_wechat(img, model_dir)\n        if quad is not None:\n            sides = _side_lengths(quad)\n            return QRRegion(quad=quad, method=\"wechat-cnn\",\n                            side_px=float(np.mean(sides)), score=0.99)\n\n    quad = _detect_opencv(img)\n    if quad is not None:\n        sides = _side_lengths(quad)\n        return QRRegion(quad=quad, method=\"opencv\",\n                        side_px=float(np.mean(sides)), score=0.9)\n\n    reg = _detect_by_contours(img)\n    if reg is not None:\n        return reg\n\n    # last resort: locate the three finder patterns (no frame needed)\n    return _detect_by_finders(img)\n\n\n# ---------------------------------------------------------------------------\n# Fallback placement (used when a QR cannot be located on one cert in a batch\n# of SAME-TEMPLATE certificates -- the QR is always in the same spot).\n# ---------------------------------------------------------------------------\ndef normalize_region(region: QRRegion, h: int, w: int) -> np.ndarray:\n    \"\"\"Quad coordinates as fractions of image width/height.\"\"\"\n    q = region.quad.astype(np.float32).copy()\n    q[:, 0] /= float(w)\n    q[:, 1] /= float(h)\n    return q\n\n\ndef fallback_region_from_regions(img: np.ndarray,\n                                 regions: list) -> QRRegion | None:\n    \"\"\"Place the new QR using the MEDIAN normalized position of the QRs that\n    WERE detected on other certificates in the batch. Works because all\n    certificates share one template. `regions` = list of (QRRegion, h, w).\n\n    Skips images whose aspect ratio differs from the certificates (e.g. a wide\n    16:9 screenshot that isn't a certificate) so a QR is never pasted onto a\n    non-certificate image.\"\"\"\n    h, w = img.shape[:2]\n    tmpl_aspect = float(np.median([ww / max(1, hh) for (_r, hh, ww) in regions]))\n    if abs((w / max(1, h)) - tmpl_aspect) / tmpl_aspect > 0.15:\n        return None\n    norms = [normalize_region(r, hh, ww) for (r, hh, ww) in regions]\n    if not norms:\n        return None\n    med = np.median(np.stack(norms, axis=0), axis=0).astype(np.float32)\n    quad = med.copy()\n    quad[:, 0] *= float(w)\n    quad[:, 1] *= float(h)\n    quad = _order_points(quad)\n    sides = _side_lengths(quad)\n    return QRRegion(quad=quad, method=\"batch-template-fallback\",\n                    side_px=float(np.mean(sides)), score=0.5)\n\n\ndef fallback_region_default(img: np.ndarray) -> QRRegion | None:\n    \"\"\"Last-resort placement for the TGMFC certificate template (QR sits in\n    the upper-left, under the red certificate number). Coordinates are\n    fractions of (width, height). Returns None for images whose aspect ratio\n    is a wide landscape (e.g. a 16:9 screenshot) rather than a certificate\n    page, so a QR is never pasted onto the wrong image.\"\"\"\n    h, w = img.shape[:2]\n    if w / max(1, h) >= 1.45:      # wide landscape screenshot, not a cert page\n        return None\n    n = np.array([[0.095, 0.345], [0.205, 0.345],\n                  [0.205, 0.515], [0.095, 0.515]], dtype=np.float32)\n    quad = n.copy()\n    quad[:, 0] *= float(w)\n    quad[:, 1] *= float(h)\n    quad = _order_points(quad)\n    sides = _side_lengths(quad)\n    return QRRegion(quad=quad, method=\"template-default-fallback\",\n                    side_px=float(np.mean(sides)), score=0.3)\n\n\ndef draw_debug(img: np.ndarray, region: QRRegion) -> np.ndarray:\n    \"\"\"Overlay the detected quad on an image (for review of failures).\"\"\"\n    out = img.copy()\n    q = region.quad.astype(np.int32)\n    cv2.polylines(out, [q], True, (0, 0, 255), 3)\n    cv2.putText(out, f\"{region.method} {region.side_px:.0f}px\",\n                (q[0][0], max(15, q[0][1] - 8)),\n                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)\n    return out\n",
      "qr_generator.py": "\"\"\"\nqr_generator.py\n---------------\nGenerate a fresh, crisp QR code image for a given payload (the URL/text that\na phone will read when scanning the certificate).\n\nThe QR is rendered at HIGH pixel resolution so that when it is perspective-\nwarped down onto the small QR region of the certificate, every module stays\nsharp and black/white (which is what makes it scan reliably).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport io\n\nimport cv2\nimport numpy as np\nimport qrcode\nfrom qrcode.constants import (\n    ERROR_CORRECT_L, ERROR_CORRECT_M, ERROR_CORRECT_Q, ERROR_CORRECT_H,\n)\n\n_EC = {\n    \"L\": ERROR_CORRECT_L,   # ~7%\n    \"M\": ERROR_CORRECT_M,   # ~15%\n    \"Q\": ERROR_CORRECT_Q,   # ~25%\n    \"H\": ERROR_CORRECT_H,   # ~30%\n}\n\n\ndef build_qr(payload: str,\n             box_size: int = 20,\n             border: int = 4,\n             error_correction: str = \"M\",\n             fg: tuple[int, int, int] = (0, 0, 0),\n             bg: tuple[int, int, int] = (255, 255, 255)) -> np.ndarray:\n    \"\"\"\n    Return a square BGR uint8 image (white background) containing the QR.\n\n    box_size: pixels per module at render time (large = crisp after warping)\n    border:   quiet-zone width in modules (standard 4)\n    \"\"\"\n    qr = qrcode.QRCode(\n        version=None,                       # auto-size to payload\n        error_correction=_EC[error_correction.upper()],\n        box_size=box_size,\n        border=border,\n    )\n    qr.add_data(payload)\n    qr.make(fit=True)\n\n    img = qr.make_image(fill_color=fg, back_color=bg).convert(\"RGB\")\n    arr = np.array(img)[:, :, ::-1].copy()  # RGB -> BGR\n    return arr\n\n\ndef qr_png_bytes(payload: str, **kwargs) -> bytes:\n    \"\"\"Render the QR and return PNG bytes (e.g. for saving a standalone file).\"\"\"\n    arr = build_qr(payload, **kwargs)\n    ok, buf = cv2.imencode(\".png\", arr)\n    return buf.tobytes() if ok else b\"\"\n",
      "qr_art.py": "\"\"\"\nqr_art.py\n---------\nOPTIONAL AI \"artistic QR\" generation using the Hugging Face ControlNet model\n**QR Code Monster** (`monster-labs/control_v1p_sd15_qrcode_monster`, v2) with a\nStable Diffusion 1.5 base.\n\nThis is the popular \"image/text -> QR code\" diffusion model family (QR Code\nMonster, IllusionDiffusion, etc.): a normal QR matrix is fed to ControlNet as\nthe conditioning image together with a text prompt, and the model produces a\ncreative QR in which art is blended into the modules.\n\nImportant facts (from the model cards):\n  * Not every generation scans -- typical scannability is 50-80%, so the\n    official workflow is \"generate several seeds and keep a readable one\".\n  * High `controlnet_conditioning_scale` (1.2-1.8) => more readable; low =>\n    more artistic. We default high because the QR must work on a certificate.\n  * These codes are usually colourful/textured -- great for marketing, but for\n    an official certificate the plain black-and-white `qr_generator.build_qr`\n    (100% reliable) is recommended.\n\nTherefore `build_artistic_qr()`:\n  1. renders the EXACT, verified QR matrix (error-correction H),\n  2. runs the ControlNet pipeline over several seeds,\n  3. DECODES every candidate and keeps one that reads back as `payload`,\n  4. returns None if none scan within `max_tries` (caller falls back to the\n     deterministic QR, so a batch never breaks).\n\nRuns on a CUDA GPU (Kaggle free T4). torch/diffusers are imported lazily, so\nimporting this module on a CPU-only box is harmless.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport cv2\nimport numpy as np\n\nfrom .qr_generator import build_qr\n\n# A prompt that pushes the model toward a clean, document-friendly result while\n# still using the AI generator. Override for creative/artistic codes.\nCLEAN_PROMPT = (\n    \"a clean flat black and white QR code on a plain white background, \"\n    \"crisp high contrast modules, minimalist, sharp edges, no decoration\"\n)\nCLEAN_NEGATIVE = (\n    \"colorful, painting, landscape, portrait, photo, texture, blurry, \"\n    \"watermark, text, logo, distorted, low contrast, cluttered background\"\n)\n\n_ART_PIPE = None\n_ART_DEVICE = None\n\n\ndef _device() -> str:\n    global _ART_DEVICE\n    if _ART_DEVICE is not None:\n        return _ART_DEVICE\n    try:\n        import torch\n        _ART_DEVICE = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    except Exception:\n        _ART_DEVICE = \"cpu\"\n    return _ART_DEVICE\n\n\ndef art_available() -> bool:\n    try:\n        import torch  # noqa: F401\n        import diffusers  # noqa: F401\n        return True\n    except Exception:\n        return False\n\n\ndef _condition_image(payload: str, module_px: int = 16,\n                     gray_bg: bool = False) -> np.ndarray:\n    \"\"\"Render the exact QR matrix as the ControlNet conditioning image.\n\n    QR Monster expects ~16px modules. v2 blends better on a gray (#808080)\n    background; for maximum readability we use white (gray_bg=False).\n    \"\"\"\n    qr = build_qr(payload, box_size=module_px, border=0,\n                  error_correction=\"H\")\n    if gray_bg:\n        # QR has white bg; replace outer non-module area with gray is complex,\n        # so we tint the whole background gray while keeping modules black.\n        gray = cv2.cvtColor(qr, cv2.COLOR_BGR2GRAY)\n        out = np.full_like(qr, 128)\n        dark = gray < 128\n        out[dark] = (0, 0, 0)\n        return out\n    return qr\n\n\ndef load_art_pipeline(\n        controlnet_id: str = \"monster-labs/control_v1p_sd15_qrcode_monster\",\n        base_model: str = \"stable-diffusion-v1-5/stable-diffusion-v1-5\"):\n    \"\"\"Lazily load and cache the ControlNet QR pipeline (fp16 on GPU).\"\"\"\n    global _ART_PIPE\n    if _ART_PIPE is not None:\n        return _ART_PIPE\n    import torch\n    from diffusers import (ControlNetModel,\n                           StableDiffusionControlNetPipeline,\n                           DPMSolverMultistepScheduler)\n\n    dtype = torch.float16 if _device() == \"cuda\" else torch.float32\n    controlnet = ControlNetModel.from_pretrained(\n        controlnet_id, torch_dtype=dtype)\n    pipe = StableDiffusionControlNetPipeline.from_pretrained(\n        base_model, controlnet=controlnet, torch_dtype=dtype,\n        safety_checker=None)\n    pipe.scheduler = DPMSolverMultistepScheduler.from_config(\n        pipe.scheduler.config)\n    if _device() == \"cuda\":\n        try:\n            pipe.enable_xformers_memory_efficient_attention()\n        except Exception:\n            pass\n        try:\n            pipe.enable_model_cpu_offload()\n        except Exception:\n            pipe.to(\"cuda\")\n    else:\n        pipe.to(\"cpu\")\n    _ART_PIPE = pipe\n    return pipe\n\n\ndef _decode_qr(img_bgr: np.ndarray, want: str) -> bool:\n    \"\"\"True if img contains a QR that decodes exactly to `want`.\"\"\"\n    from .pipeline import _decode_any_qr  # reuse strong verifier\n    h, w = img_bgr.shape[:2]\n    for scale in (1, 2):\n        im = img_bgr if scale == 1 else cv2.resize(\n            img_bgr, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)\n        got = _decode_any_qr(im)\n        if got and got.strip() == want.strip():\n            return True\n    return False\n\n\ndef build_artistic_qr(payload: str,\n                      prompt: str = CLEAN_PROMPT,\n                      negative_prompt: str = CLEAN_NEGATIVE,\n                      size: int = 768,\n                      controlnet_scale: float = 1.5,\n                      guidance_scale: float = 7.5,\n                      steps: int = 30,\n                      max_tries: int = 6,\n                      seed0: int = 1000,\n                      gray_bg: bool = False):\n    \"\"\"\n    Generate an AI (ControlNet) artistic QR for `payload`.\n\n    Returns a square BGR uint8 image that DECODES to `payload`, or None if no\n    seed produced a scannable code within `max_tries` (caller then falls back\n    to the deterministic QR).\n    \"\"\"\n    if not art_available() or _device() != \"cuda\":\n        return None\n    try:\n        import torch\n        from PIL import Image\n        pipe = load_art_pipeline()\n    except Exception as e:  # model download/OOM\n        print(f\"[qr_art] pipeline unavailable ({e}); using standard QR\")\n        return None\n\n    cond = _condition_image(payload, module_px=16, gray_bg=gray_bg)\n    cond_pil = Image.fromarray(cv2.cvtColor(cond, cv2.COLOR_BGR2RGB))\n    # make conditioning square at requested size\n    cond_pil = cond_pil.resize((size, size), Image.NEAREST)\n\n    for i in range(max_tries):\n        seed = seed0 + i\n        g = torch.Generator(device=\"cpu\").manual_seed(seed)\n        try:\n            result = pipe(\n                prompt=prompt, negative_prompt=negative_prompt,\n                image=cond_pil, width=size, height=size,\n                num_inference_steps=steps,\n                guidance_scale=guidance_scale,\n                controlnet_conditioning_scale=controlnet_scale,\n                generator=g,\n            ).images[0]\n        except Exception as e:\n            print(f\"[qr_art] generation failed (seed {seed}): {e}\")\n            continue\n        bgr = cv2.cvtColor(np.array(result), cv2.COLOR_RGB2BGR)\n        if _decode_qr(bgr, payload):\n            print(f\"[qr_art] scannable artistic QR at seed {seed} \"\n                  f\"(try {i+1}/{max_tries})\")\n            return bgr\n        print(f\"[qr_art] seed {seed} did not scan; retrying...\")\n    print(\"[qr_art] no scannable variant found; falling back to standard QR\")\n    return None\n",
      "replacer.py": "\"\"\"\nreplacer.py\n-----------\nPaste the freshly generated QR onto the certificate at the EXACT location,\nsize and rotation of the old QR.\n\n- Uses a perspective warp so rotated/skewed old QRs are matched exactly.\n- Uses NEAREST-neighbour rendering so QR modules stay hard-edged (critical for\n  scanning) -- no anti-aliased grey fuzz.\n- `mode`:\n    \"full_replace\" -> new QR (with white quiet zone) covers the entire old QR\n                      box including its black frame. Cleanest, most reliable.\n    \"keep_frame\"   -> the old black border is preserved; only the inner QR\n                      area is replaced (the new QR is shrunk inside the frame).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport cv2\nimport numpy as np\n\nfrom .detector import QRRegion, _order_points\n\n\ndef _scale_quad(quad: np.ndarray, scale: float) -> np.ndarray:\n    \"\"\"Scale a quad about its centre.\"\"\"\n    c = quad.mean(axis=0)\n    return c + (quad - c) * scale\n\n\ndef _warp_qr_to_cert(cert: np.ndarray, qr_img: np.ndarray,\n                     dst_quad: np.ndarray, supersample: int = 4) -> np.ndarray:\n    \"\"\"Perspective-warp qr_img onto cert at dst_quad (TL,TR,BR,BL).\n\n    The QR is first re-rendered at ~`supersample`x the destination size, then\n    warped with bilinear sampling (a good approximation of area-averaged\n    downscaling) and finally re-binarised to pure black/white inside the box.\n    This keeps every module hard-edged and scannable even when the on-cert\n    QR is very small (avoids the aliasing of a single huge->tiny NEAREST warp).\n    \"\"\"\n    h, w = cert.shape[:2]\n    ss = int(supersample)\n    side = float(np.mean([\n        np.linalg.norm(dst_quad[1] - dst_quad[0]),\n        np.linalg.norm(dst_quad[2] - dst_quad[1]),\n        np.linalg.norm(dst_quad[3] - dst_quad[2]),\n        np.linalg.norm(dst_quad[0] - dst_quad[3]),\n    ]))\n    target = max(8, int(round(side * ss)))\n    if qr_img.shape[0] != target:\n        interp = cv2.INTER_AREA if qr_img.shape[0] > target else cv2.INTER_NEAREST\n        qr_img = cv2.resize(qr_img, (target, target), interpolation=interp)\n\n    qh, qw = qr_img.shape[:2]\n    # Warp onto an ss-times-oversized overlay at the ss-scaled quad, then pull\n    # back down to cert resolution with INTER_AREA -> a proper area-averaged\n    # anti-aliased downscale of the perspective warp (keeps modules legible at\n    # small on-cert sizes).\n    big_W, big_H = w * ss, h * ss\n    big_quad = dst_quad.astype(np.float32) * ss\n    src = np.array([[0, 0], [qw - 1, 0], [qw - 1, qh - 1], [0, qh - 1]],\n                   dtype=np.float32)\n    M = cv2.getPerspectiveTransform(src, big_quad)\n    warped_big = cv2.warpPerspective(\n        qr_img, M, (big_W, big_H),\n        flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT,\n        borderValue=(255, 255, 255))\n    warped = cv2.resize(warped_big, (w, h), interpolation=cv2.INTER_AREA)\n\n    # mask of the QR box\n    mask = np.zeros((h, w), dtype=np.uint8)\n    cv2.fillConvexPoly(mask, dst_quad.astype(np.int32), 255)\n    mask = cv2.erode(mask, np.ones((3, 3), np.uint8), iterations=1)\n\n    # re-binarise the warped patch to pure black / white (hard module edges)\n    warped_gray = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)\n    _, binary = cv2.threshold(warped_gray, 160, 255, cv2.THRESH_BINARY)\n    binary_bgr = cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)\n\n    out = cert.copy()\n    m3 = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR) > 0\n    out[m3] = binary_bgr[m3]\n    return out\n\n\ndef replace_qr(cert: np.ndarray, region: QRRegion, qr_img: np.ndarray,\n               mode: str = \"full_replace\",\n               inner_margin_frac: float = 0.10,\n               cover_scale: float = 1.08) -> np.ndarray:\n    \"\"\"\n    Return a new certificate image with the new QR in place of the old one.\n\n    Everything outside the QR quad is left byte-for-byte untouched.\n\n    cover_scale (full_replace only): the new QR's white box is pasted over a\n    quad scaled outward by this factor so it reliably covers the ENTIRE old\n    black frame plus any dark halo / AI-inpaint remnant at the frame edge.\n    1.0 = exact old-quad footprint; ~1.08 fully hides the old frame (the white\n    background looks natural against the certificate and keeps a valid quiet\n    zone).\n    \"\"\"\n    quad = _order_points(region.quad)\n\n    if mode == \"keep_frame\":\n        # Replace only the inside of the black frame.\n        dst = _scale_quad(quad, 1.0 - 2 * inner_margin_frac)\n        # Give the pasted QR its own thin white border so modules never touch\n        # the black frame (keeps quiet-zone scanning rules).\n        qr = _add_white_border(qr_img, frac=0.06)\n    else:\n        # Full replace: scale the footprint outward a little so the new QR's\n        # white background fully hides the old frame + any edge halo.\n        dst = _scale_quad(quad, float(cover_scale))\n        qr = qr_img\n\n    return _warp_qr_to_cert(cert, qr, dst)\n\n\ndef _add_white_border(img: np.ndarray, frac: float = 0.08) -> np.ndarray:\n    h, w = img.shape[:2]\n    b = int(round(min(h, w) * frac))\n    return cv2.copyMakeBorder(img, b, b, b, b, cv2.BORDER_CONSTANT,\n                              value=(255, 255, 255))\n",
      "payload.py": "\"\"\"\npayload.py\n----------\nDecide what text/URL each certificate's NEW QR code encodes.\n\nSupported modes (set in config / notebook):\n\n  url   : base_url + slug + extension      -> e.g.\n          https://<your-username>.github.io/certificates/031-2024-25.png\n          (this is the recommended mode: host the OUTPUT images online --\n           e.g. GitHub Pages / your server / cloud storage -- and scanning the\n           QR opens that certificate in the phone's browser)\n\n  csv   : read a spreadsheet with columns: filename, qr_data\n          (per-certificate exact URL or text)\n\n  text  : fixed prefix + slug\n\n  copy  : copy whatever the old QR decoded to (requires the old QR to be\n          readable; falls back to url mode when it isn't)\n\n`slug` defaults to the certificate file's stem (filename without extension).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport csv\nimport os\n\n\ndef slug_from_filename(path: str) -> str:\n    return os.path.splitext(os.path.basename(path))[0]\n\n\ndef safe_slug(path: str) -> str:\n    \"\"\"URL/filename-safe stem. Replaces spaces, parentheses and other chars that\n    break URLs (e.g. 'image (1)' -> 'image_1') so the QR link and the saved\n    output file always match and resolve.\"\"\"\n    import re\n    stem = slug_from_filename(path)\n    stem = re.sub(r\"[^A-Za-z0-9._-]+\", \"_\", stem)\n    stem = re.sub(r\"_+\", \"_\", stem).strip(\"._-\")\n    return stem or \"certificate\"\n\n\ndef build_payload(path: str,\n                  mode: str = \"url\",\n                  base_url: str = \"\",\n                  extension: str = \".png\",\n                  text_prefix: str = \"\",\n                  csv_map: dict[str, str] | None = None,\n                  old_data: str | None = None) -> str:\n    slug = safe_slug(path)\n\n    if mode == \"csv\":\n        if csv_map and (os.path.basename(path) in csv_map or slug in csv_map):\n            return csv_map.get(os.path.basename(path)) or csv_map[slug]\n        # fall back to URL if the cert is missing from the spreadsheet\n        mode = \"url\"\n\n    if mode == \"copy\" and old_data:\n        return old_data\n\n    if mode == \"text\":\n        return f\"{text_prefix}{slug}\"\n\n    # default: url\n    base = base_url.rstrip(\"/\") + \"/\"\n    return f\"{base}{slug}{extension}\"\n\n\ndef load_csv_map(csv_path: str) -> dict[str, str]:\n    \"\"\"Read filename->qr_data mapping from a CSV (headers: filename,qr_data).\"\"\"\n    mapping: dict[str, str] = {}\n    with open(csv_path, newline=\"\", encoding=\"utf-8-sig\") as f:\n        for row in csv.DictReader(f):\n            fn = (row.get(\"filename\") or row.get(\"file\") or \"\").strip()\n            data = (row.get(\"qr_data\") or row.get(\"url\") or\n                    row.get(\"payload\") or \"\").strip()\n            if fn and data:\n                mapping[fn] = data\n                mapping[os.path.splitext(fn)[0]] = data\n    return mapping\n",
      "ai_extract.py": "\"\"\"\nai_extract.py\n-------------\nRead the candidate's details DIRECTLY FROM THE CERTIFICATE IMAGE using an AI\nvision-language model from Hugging Face (no OCR, no CSV, no hand-written rules).\n\nModel (runs on a free Kaggle T4 GPU; no num2words / tesseract needed):\n    Qwen/Qwen2-VL-2B-Instruct   (default) -- strong document/JSON reader\n    Qwen/Qwen2.5-VL-3B-Instruct (optional) -- slightly larger, also fine on T4\n\nThe model is shown the certificate and asked to return ONLY JSON:\n    name, parent, relation, aadhar, course, date_from, date_to, institute.\n\nFailures are NEVER silent: the returned dict carries an \"_error\" / \"_raw_ai\"\nkey so the notebook log / manifest show exactly what happened.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport re\nimport os\nimport gc\nimport importlib\nimport subprocess\nimport sys\n\n# Reduce CUDA fragmentation / OOM on the 16 GB T4 when a model is loaded more\n# than once in a session (must be set before torch first allocates).\nos.environ.setdefault(\"PYTORCH_CUDA_ALLOC_CONF\", \"expandable_segments:True\")\n\n_PROMPT = (\n    \"You are reading an Indian training-completion certificate IMAGE. \"\n    \"Look carefully at the whole image and extract these SIX fields. \"\n    \"Return ONLY a JSON object (no prose, no markdown, no code fence) with \"\n    \"exactly these keys:\\n\"\n    '{\"name\": \"<the candidate OWN name only: the words printed after Mr./Ms. '\n    'and BEFORE the token S/o or D/o or W/o. Do NOT include S/o, D/o, W/o, or '\n    'any name after them.>\", '\n    '\"parent\": \"<the father/mother name ONLY: the person name printed '\n    'immediately AFTER S/o (son of -> father), D/o (daughter of -> father) or '\n    'W/o (wife of -> husband), up to the next comma or bracket.>\", '\n    '\"relation\": \"<exactly S/o if the text says S/o, D/o if it says D/o, or '\n    'W/o if it says W/o>\", '\n    '\"aadhar\": \"<the candidate 12-digit Aadhaar number, digits only>\", '\n    '\"course\": \"<the full course/training name the candidate completed, e.g. '\n    'the text after Training in / Trainings in / Course>\", '\n    '\"date_from\": \"<course START date formatted dd/mm/yyyy>\", '\n    '\"date_to\": \"<course END date formatted dd/mm/yyyy>\", '\n    '\"institute\": \"<the COMPLETE institute name printed anywhere on the '\n    'certificate, written out in full exactly as shown (spell out every word; '\n    'do NOT shorten to initials - e.g. write Falih Consultancy Services (FCS) '\n    'Training Institute, not just FCS). Read the org name LETTER BY LETTER: '\n    'the printed letters l, i, s and h look alike - Falih is F-a-l-i-h (NOT '\n    'Falis, NOT Fahis). It differs on every certificate; never '\n    'assume or default to a name.>\"}\\n'\n    \"CRITICAL: 'S/o' = Son of (FATHER), 'D/o' = Daughter of (FATHER), 'W/o' = \"\n    \"Wife of (HUSBAND). These give the parent name. 'R/o' = Resident of (the \"\n    \"HOME ADDRESS / place, e.g. Rajendra Nager) \u2014 R/o is NOT a person and must \"\n    \"NEVER be put in name or parent. Read from the image only; do not invent; \"\n    \"use \\\"\\\" for anything unreadable; dates as dd/mm/yyyy.\"\n)\n\n# Small helper packages (NOT torch/transformers). If one is missing in the\n# kernel we self-heal by pip-installing it at runtime.\n_REQUIRED = [\n    (\"PIL\", \"pillow\"),\n    (\"accelerate\", \"accelerate\"),\n    (\"torchvision\", \"torchvision\"),\n]\n\n_MODEL = None\n_PROC = None\n_DEVICE = None\n_MODEL_ID = None\n\n\ndef _pip_install(pkg: str) -> bool:\n    try:\n        subprocess.check_call(\n            [sys.executable, \"-m\", \"pip\", \"install\", \"-q\", pkg])\n        return True\n    except Exception as e:\n        print(f\"[ai_extract] pip install {pkg} failed: {e}\")\n        return False\n\n\ndef _ensure_deps():\n    # Only self-heal where torch/transformers already exist (the GPU notebook).\n    try:\n        importlib.import_module(\"torch\")\n        importlib.import_module(\"transformers\")\n    except Exception:\n        return\n    for import_name, pip_name in _REQUIRED:\n        try:\n            importlib.import_module(import_name)\n        except Exception:\n            print(f\"[ai_extract] installing missing dependency '{pip_name}' ...\")\n            if _pip_install(pip_name):\n                try:\n                    importlib.import_module(import_name)\n                except Exception:\n                    pass\n\n\ndef device() -> str:\n    global _DEVICE\n    if _DEVICE is not None:\n        return _DEVICE\n    try:\n        import torch\n        _DEVICE = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    except Exception:\n        _DEVICE = \"cpu\"\n    return _DEVICE\n\n\ndef runtime_status() -> str:\n    try:\n        import torch  # noqa\n    except Exception as e:\n        return f\"torch NOT importable: {e}\"\n    try:\n        import transformers\n        tv = transformers.__version__\n    except Exception as e:\n        return f\"transformers NOT importable: {e}\"\n    dev = device()\n    gpu = \"\"\n    if dev == \"cuda\":\n        try:\n            import torch\n            gpu = f\" | GPU: {torch.cuda.get_device_name(0)}\"\n        except Exception:\n            pass\n    warn = \"\" if dev == \"cuda\" else \" | \u26a0\ufe0f CPU only \u2014 set Accelerator to GPU T4\"\n    return f\"torch ok | transformers {tv} | device={dev}{gpu}{warn}\"\n\n\ndef release_model():\n    \"\"\"Free the cached vision model and its CUDA memory. Call before a fresh\n    module load in the same session (e.g. after the test cell) to avoid OOM.\"\"\"\n    global _MODEL, _PROC, _MODEL_ID\n    _MODEL = None\n    _PROC = None\n    _MODEL_ID = None\n    gc.collect()\n    try:\n        import torch\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n            torch.cuda.ipc_collect()\n    except Exception:\n        pass\n\n\ndef _load(model_id: str):\n    \"\"\"Load (once) and cache the Qwen2-VL model + processor. Frees any stale\n    model first and retries once after emptying the CUDA cache on OOM.\"\"\"\n    global _MODEL, _PROC, _MODEL_ID\n    if _MODEL is not None and _MODEL_ID == model_id:\n        return _MODEL, _PROC\n    release_model()  # ensure no previous model (e.g. from the test cell) lingers\n    import torch\n    from transformers import AutoProcessor\n\n    def _build():\n        ModelCls = None\n        if \"Qwen2.5\" in model_id:\n            try:\n                from transformers import Qwen2_5_VLForConditionalGeneration as ModelCls\n            except Exception:\n                ModelCls = None\n        if ModelCls is None:\n            try:\n                from transformers import Qwen2VLForConditionalGeneration as ModelCls\n            except Exception:\n                ModelCls = None\n        if ModelCls is None:\n            from transformers import AutoModelForImageTextToText as ModelCls\n        dtype = torch.float16 if device() == \"cuda\" else torch.float32\n        proc = AutoProcessor.from_pretrained(model_id)\n        kwargs = dict(torch_dtype=dtype, low_cpu_mem_usage=True)\n        if device() == \"cuda\":\n            kwargs.update(device_map=\"auto\", offload_folder=\"/kaggle/working/offload\")\n        model = ModelCls.from_pretrained(model_id, **kwargs)\n        if \"device_map\" not in kwargs:\n            model = model.to(device())\n        model.eval()\n        return model, proc\n\n    try:\n        _MODEL, _PROC = _build()\n    except (RuntimeError, MemoryError) as e:\n        if \"out of memory\" in str(e).lower() or \"CUDA\" in str(e):\n            print(\"[ai_extract] GPU out of memory while loading \u2014 freeing cache \"\n                  \"and retrying once...\")\n            release_model()\n            try:\n                _MODEL, _PROC = _build()\n            except Exception as e2:\n                raise RuntimeError(\n                    f\"GPU out of memory even after freeing cache ({e2}). \"\n                    \"Restart the kernel (Run -> Restart) and run section 5 only, \"\n                    \"without running the section 4b test cell first.\") from e2\n        else:\n            raise\n    _MODEL_ID = model_id\n    return _MODEL, _PROC\n\n\ndef _norm_date(v: str) -> str:\n    if not v:\n        return \"\"\n    m = re.search(r\"(\\d{1,2})[^\\d]?(\\d{1,2})[^\\d]?(\\d{4})\", str(v))\n    if not m:\n        return str(v).strip()\n    try:\n        return f\"{int(m.group(1)):02d}/{int(m.group(2)):02d}/{m.group(3)}\"\n    except Exception:\n        return str(v).strip()\n\n\ndef _digits_aadhar(v: str) -> str:\n    if not v:\n        return \"\"\n    d = re.sub(r\"\\D\", \"\", str(v))\n    if len(d) > 12:\n        d = d[-12:]\n    if len(d) == 12:\n        return f\"{d[0:4]} {d[4:8]} {d[8:12]}\"\n    return str(v).strip()\n\n\ndef _clean_person_fields(d: dict) -> dict:\n    \"\"\"Deterministically fix name/parent/relation, regardless of model slips:\n    split at the S/o / D/o / W/o marker and never confuse it with R/o (address).\"\"\"\n    name = str(d.get(\"name\", \"\") or \"\").strip()\n    parent = str(d.get(\"parent\", \"\") or \"\").strip()\n    rel = str(d.get(\"relation\", \"\") or \"\").strip().upper()\n\n    relmap = {\"S\": \"S/o\", \"D\": \"D/o\", \"W\": \"W/o\"}\n    rel_letter = rel[0] if (rel and rel[0] in \"SDW\") else \"\"\n    if not rel_letter:\n        m = re.search(r\"\\b([SDW])\\s*/?\\s*[oO0]\\b\", f\"{name} {parent}\")\n        rel_letter = m.group(1) if m else \"\"\n\n    # If the relation marker (S/o, D/o, W/o) ended up INSIDE the name, split it\n    # and trust the marker found in the name (it directly introduces the parent).\n    m = re.search(r\"^(.*?)\\b([SDW])\\s*/?\\s*[oO0]\\s+(.*)$\", name, flags=re.S)\n    if m:\n        name = m.group(1).strip(\" ,./-\")\n        rel_letter = m.group(2)\n        tail = m.group(3)\n        # parent = up to the next R/o (address), Aadhaar, bracket, comma, newline\n        tail = re.split(r\"\\bR\\s*/\\s*o\\b|\\(?Aadha\", tail, flags=re.I)[0]\n        cand = tail.split(\",\")[0].split(\"(\")[0].strip(\" ,./-\")\n        if cand:\n            parent = cand\n\n    # If parent wrongly contains the R/o (residence/address) marker, cut there.\n    parent = re.split(r\"\\bR\\s*/\\s*o\\b\", parent, flags=re.I)[0].strip(\" ,./-\")\n    # Strip a leaked relation PREFIX from parent only if it is the actual\n    # \"S/o\" / \"D/o\" token (never strip a bare initial like \"D.\" in \"D. Jahangeen\").\n    parent = re.sub(r\"^[SDW]\\s*/\\s*[oO0]\\s*\", \"\", parent, flags=re.I).strip()\n    # Parent shouldn't contain Aadhaar / trailing bracket junk.\n    parent = re.split(r\"\\(?Aadha|\\d{4}\\s*\\d{4}\\s*\\d{4}\", parent)[0].strip(\" ,./-()\")\n\n    # Name must not keep a trailing relation marker either.\n    name = re.split(r\"\\b[SDW]\\s*/?\\s*[oO0]\\b\", name)[0].strip(\" ,./-\")\n    # Strip leading honorifics the model may keep (requires the dot, so real\n    # names like \"Msmena\" are never cut). Handles \"Mr./Ms.\", \"Ms.\", \"Dr.\" etc.\n    name = re.sub(\n        r\"^(?:Mr|Mrs|Ms|Miss|Dr|Prof)\\.\\s*(?:/\\s*(?:Mrs?|Ms|Miss|Dr|Prof)\\.\\s*)*\",\n        \"\", name).strip(\" ,./-\")\n\n    relation = relmap.get(rel_letter, \"\")\n    d[\"name\"] = name\n    d[\"parent\"] = parent\n    d[\"relation\"] = relation\n    return d\n\n\ndef _parse_json(text: str) -> dict:\n    if not text:\n        return {}\n    t = text.strip()\n    t = re.sub(r\"^```(?:json)?\\s*|\\s*```$\", \"\", t, flags=re.I).strip()\n    m = re.search(r\"\\{.*\\}\", t, re.S)\n    if not m:\n        return {}\n    data = None\n    for candidate in (m.group(0), \"{\" + m.group(0).strip(\"{} \") + \"}\"):\n        try:\n            data = json.loads(candidate)\n            break\n        except Exception:\n            data = None\n    if not isinstance(data, dict):\n        return {}\n    rel = str(data.get(\"relation\", \"\") or \"\").strip().upper()\n    rel = rel.replace(\"SON OF\", \"S/O\").replace(\"DAUGHTER OF\", \"D/O\")\n    rel = rel.replace(\"WIFE OF\", \"W/O\")\n    if rel and rel[0] in \"SDW\":\n        rel = rel[0] + \"/O\"\n    out = {\n        \"name\": str(data.get(\"name\", \"\") or \"\").strip(),\n        \"parent\": str(data.get(\"parent\", \"\") or \"\").strip(),\n        \"relation\": rel,\n        \"aadhar\": _digits_aadhar(data.get(\"aadhar\", \"\")),\n        \"course\": str(data.get(\"course\", \"\") or \"\").strip(),\n        \"date_from\": _norm_date(data.get(\"date_from\", \"\")),\n        \"date_to\": _norm_date(data.get(\"date_to\", \"\")),\n        \"institute\": str(data.get(\"institute\", \"\") or \"\").strip(),\n        \"_raw_ai\": text.strip(),\n    }\n    return _clean_person_fields(out)\n\n\ndef _generate(pil, model, proc, prompt: str | None = None,\n              max_new_tokens: int = 512) -> str:\n    import torch\n    messages = [{\"role\": \"user\", \"content\": [\n        {\"type\": \"image\"}, {\"type\": \"text\", \"text\": prompt or _PROMPT}]}]\n    text = proc.apply_chat_template(\n        messages, tokenize=False, add_generation_prompt=True)\n    inputs = proc(text=[text], images=[pil], padding=True,\n                  return_tensors=\"pt\").to(model.device)\n    with torch.no_grad():\n        gen = model.generate(**inputs, max_new_tokens=max_new_tokens,\n                             do_sample=False)\n    trimmed = gen[:, inputs[\"input_ids\"].shape[1]:]\n    return proc.batch_decode(trimmed, skip_special_tokens=True)[0]\n\n\n_VERIFY_PROMPT = (\n    \"Look ONLY at the name of the organization/institute that ISSUED this \"\n    \"certificate (printed in words, usually under or around the logo at the \"\n    \"top, and/or near the signatory at the bottom).\\n\"\n    \"A previous reading reported the institute as: \\\"{got}\\\"\\n\"\n    \"Re-read the printed name CAREFULLY, letter by letter, and fix any \"\n    \"misread letters. Printed 'l', 'i', 's' and 'h' look alike: for example \"\n    \"'Falih' is F-a-l-i-h (NOT Falis, NOT Fahis, NOT Faris). Also write the \"\n    \"name COMPLETE, exactly as printed \u2014 spell out every word; never shorten \"\n    \"to initials.\\n\"\n    \"Return ONLY a JSON object with exactly this key (no prose, no code \"\n    \"fence):\\n\"\n    '{\"institute\": \"<the complete, letter-checked correct institute name, '\n    'or \\\\\"\\\\\" if not readable>\"}'\n)\n\n\ndef _clean_institute(v: str) -> str:\n    v = str(v or \"\").strip().strip(\".,;\\\"'` \")\n    if v.lower() in (\"\", \"n/a\", \"na\", \"null\", \"none\", \"-\"):\n        return \"\"\n    return v\n\n\ndef _parse_verify(text: str) -> str:\n    \"\"\"Extract the corrected institute string from the verify-pass output.\"\"\"\n    if not text:\n        return \"\"\n    t = re.sub(r\"^```(?:json)?\\s*|\\s*```$\", \"\", text.strip(),\n               flags=re.I).strip()\n    if \"{\" in t:  # model answered in JSON (possibly slightly malformed)\n        m = re.search(r\"\\{.*\\}\", t, re.S)\n        blob = m.group(0) if m else t\n        try:\n            obj = json.loads(blob)\n            return _clean_institute(obj.get(\"institute\", \"\"))\n        except Exception:\n            mm = re.search(r'\"institute\"\\s*:\\s*\"([^\"]*)\"', blob)\n            if mm:\n                return _clean_institute(mm.group(1))\n            return \"\"\n    # plain-text answer: prefer the longest line (model may add a short\n    # preamble like \"The institute is:\" before the actual name)\n    best = \"\"\n    for line in t.splitlines():\n        line = line.strip().strip(\"{}\\\"'` \")\n        line = re.sub(r\"^institute\\s*:?\", \"\", line, flags=re.I).strip()\n        line = re.sub(r\"^(the\\s+)?(name|institute)\\s+(is|of)\\s*:?\\s*$\",\n                      \"\", line, flags=re.I).strip()\n        if _clean_institute(line) and len(line.strip()) >= 3 \\\n                and len(line) > len(best):\n            best = line\n    return _clean_institute(best)\n\n\ndef ai_extract_fields(img_bgr,\n                      model_id: str = \"Qwen/Qwen2-VL-2B-Instruct\"\n                      ) -> dict:\n    \"\"\"Run the AI vision model on one certificate (BGR ndarray) and return a\n    fields dict. On any failure returns a dict with '_error' (never raises).\"\"\"\n    try:\n        import numpy as np  # noqa\n        import cv2\n        from PIL import Image\n    except Exception as e:\n        return {\"_error\": f\"deps missing (numpy/Pillow/cv2): {e}\"}\n\n    global _MODEL, _PROC\n    model = proc = None\n    try:\n        _ensure_deps()\n        model, proc = _load(model_id)\n    except Exception as e:\n        name = getattr(e, \"name\", None)\n        m = re.search(r\"No module named ['\\\"]([a-zA-Z0-9_\\-]+)\", str(e))\n        pkg = name or (m.group(1) if m else None)\n        pip_map = {\"PIL\": \"pillow\", \"torchvision\": \"torchvision\",\n                   \"accelerate\": \"accelerate\"}\n        pip = pip_map.get(pkg)\n        if pip:\n            print(f\"[ai_extract] model needs '{pip}'; installing and retrying...\")\n            if _pip_install(pip):\n                _MODEL, _PROC = None, None\n                try:\n                    _ensure_deps()\n                    model, proc = _load(model_id)\n                except Exception as e2:\n                    return {\"_error\": f\"model load failed after installing \"\n                                      f\"{pip}: {e2}\"}\n            else:\n                return {\"_error\": f\"model load failed (missing {pip}): {e}\"}\n        else:\n            return {\"_error\": f\"model load failed: {e}\"}\n\n    rgb = img_bgr[:, :, ::-1]\n    # Up-scale small/phone screenshots a bit so the VLM has more pixels to read.\n    if rgb.shape[1] < 1000:\n        scale = 1000 / rgb.shape[1]\n        rgb = cv2.resize(rgb, None, fx=scale, fy=scale,\n                         interpolation=cv2.INTER_CUBIC)\n    import numpy as np\n    pil = Image.fromarray(np.ascontiguousarray(rgb)).convert(\"RGB\")\n\n    out = \"\"\n    try:\n        out = _generate(pil, model, proc)\n    except Exception as e:\n        try:  # one retry slightly larger\n            bigger = cv2.resize(rgb, None, fx=1.4, fy=1.4,\n                                interpolation=cv2.INTER_CUBIC)\n            pil = Image.fromarray(np.ascontiguousarray(bigger)).convert(\"RGB\")\n            out = _generate(pil, model, proc)\n        except Exception as e2:\n            return {\"_error\": f\"generation failed: {e2}\"}\n\n    fields = _parse_json(out)\n    if not fields:\n        return {\"_error\": \"model output was not JSON\", \"_raw_ai\": out.strip()}\n\n    # ---- Institute spelling check: one extra focused look so names like\n    # 'Falih' are not misread as 'Falis' (l/i/s confusion). Never fatal.\n    got_inst = str(fields.get(\"institute\", \"\") or \"\").strip()\n    if got_inst:\n        try:\n            v_raw = _generate(pil, model, proc,\n                              prompt=_VERIFY_PROMPT.replace(\"{got}\", got_inst),\n                              max_new_tokens=96)\n            fixed = _parse_verify(v_raw)\n            if fixed:\n                fields[\"institute_verified\"] = fixed\n                if fixed.strip().lower() != got_inst.strip().lower():\n                    print(f\"[ai_extract] institute spelling check: \"\n                          f\"'{got_inst}' -> '{fixed}'\")\n                    fields[\"institute\"] = fixed\n            fields[\"_raw_ai\"] = out.strip() + \"\\n\\n[institute re-check] \" \\\n                + v_raw.strip()[:220]\n        except Exception as e:\n            print(f\"[ai_extract] institute re-check skipped ({e})\")\n    return fields\n",
      "verification_page.py": "\"\"\"\nverification_page.py\n--------------------\nBuild a self-contained HTML certificate-verification page for each candidate.\nThe certificate image is EMBEDDED (base64), so uploading the single .html file\nis enough \u2014 when a user scans the QR (which links to this page) they see the\ncertificate image with the candidate's details beneath it.\n\nDetails (read off the certificate by the AI vision model):\n    Name of the candidate, Father/Mother Name, Aadhar No., Course Name,\n    Course Duration (dd/mm/yyyy -> dd/mm/yyyy), Institute.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport base64\nimport html as _html\n\n\ndef _b64_png(img_bgr) -> str:\n    import cv2\n    ok, buf = cv2.imencode(\".png\", img_bgr)\n    return base64.b64encode(buf.tobytes()).decode() if ok else \"\"\n\n\ndef _esc(x) -> str:\n    return _html.escape(str(x if x not in (None, \"\") else \"\u2014\"))\n\n\ndef build_page(fields: dict, img_bgr, cert_filename: str) -> str:\n    name = fields.get(\"name\", \"\")\n    parent = fields.get(\"parent\", \"\")\n    aadhar = fields.get(\"aadhar\", \"\")\n    course = fields.get(\"course\", \"\")\n    dfrom = fields.get(\"date_from\", \"\")\n    dto = fields.get(\"date_to\", \"\")\n    dates = f\"{_esc(dfrom)} &nbsp;to&nbsp; {_esc(dto)}\" if (dfrom or dto) else \"\u2014\"\n\n    img_b64 = _b64_png(img_bgr)\n\n    institute = fields.get(\"institute\", \"\")\n    rows = [\n        (\"Name of the candidate\", _esc(name)),\n        (\"Father/Mother Name\", _esc(parent)),\n        (\"Aadhar No.\", _esc(aadhar)),\n        (\"Course Name\", _esc(course)),\n        (\"Course Duration\", dates),\n        (\"Institute\", _esc(institute)),\n    ]\n    rows_html = \"\\n\".join(\n        f'<tr><th>{label} :</th><td>{val}</td></tr>' for label, val in rows)\n\n    cert_html = f'''\n    <div class=\"cert\">\n      <img alt=\"Certificate for {_esc(name or cert_filename)}\"\n           src=\"data:image/png;base64,{img_b64}\">\n      <div><span class=\"badge\">&#10003; Verified certificate</span></div>\n    </div>'''\n\n    info_html = f'''\n    <h2 class=\"sechd\">Candidate Information</h2>\n    <table>\n{rows_html}\n    </table>'''\n\n    return f\"\"\"<!DOCTYPE html>\n<html lang=\"en\">\n<head>\n<meta charset=\"utf-8\">\n<meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">\n<title>Certificate Verification - {_esc(name or cert_filename)}</title>\n<style>\n  *{{box-sizing:border-box}}\n  body{{font-family:Georgia,'Times New Roman',serif;background:#eef1f5;margin:0;\n       padding:24px 12px;color:#1b1b1b}}\n  .wrap{{max-width:760px;margin:0 auto}}\n  .head{{text-align:center;margin-bottom:16px}}\n  .head .org{{font-size:13px;letter-spacing:2px;color:#0b6b3a;font-weight:bold;\n       text-transform:uppercase}}\n  .head h1{{font-size:22px;margin:6px 0 2px;color:#10345e}}\n  .head .sub{{color:#666;font-size:13px}}\n  .card{{background:#fff;border:1px solid #d9dee6;border-radius:14px;\n       box-shadow:0 10px 30px rgba(0,0,0,.08);padding:22px 24px}}\n  table{{width:100%;border-collapse:collapse;font-size:16px}}\n  th{{text-align:left;width:46%;color:#333;font-weight:normal;padding:10px 8px;\n     border-bottom:1px solid #eef0f3;vertical-align:top}}\n  td{{padding:10px 8px;border-bottom:1px solid #eef0f3;font-weight:bold;\n     color:#111;word-break:break-word}}\n  .fcs{{margin:6px 0 10px;text-align:center;font-weight:bold;letter-spacing:1px;\n       color:#0b6b3a;font-size:14px;text-transform:uppercase}}\n  .sechd{{font-size:16px;color:#10345e;border-bottom:2px solid #e5e9f0;\n       padding-bottom:6px;margin:18px 0 8px}}\n  .cert{{margin-top:16px;text-align:center}}\n  .cert img{{max-width:100%;border:1px solid #d9dee6;border-radius:8px;\n       box-shadow:0 4px 14px rgba(0,0,0,.10)}}\n  .badge{{display:inline-block;background:#e8f6ee;color:#0b6b3a;border:1px solid #bfe6cf;\n       border-radius:999px;padding:5px 14px;font-size:12px;margin-top:8px}}\n  .foot{{text-align:center;color:#8a93a0;font-size:12px;margin-top:16px}}\n</style>\n</head>\n<body>\n<div class=\"wrap\">\n  <div class=\"head\">\n    <div class=\"org\">Government of Telangana &middot; TGMFC</div>\n    <h1>Certificate Verification</h1>\n    <div class=\"sub\">Skill Development Training certificate</div>\n  </div>\n  <div class=\"card\">\n{cert_html}\n{info_html}\n  </div>\n  <div class=\"foot\">Government of Telangana &middot; TGMFC &mdash;\n       Skill Development Training certificate verification</div>\n</div>\n</body>\n</html>\n\"\"\"\n",
      "ai_inpaint.py": "\"\"\"\nai_inpaint.py\n-------------\nAI image-generator step (diffusion INPAINTING) for the QR replacement.\n\nWhy inpainting + compositing instead of pure text-to-image?\n  * A generative model that \"draws\" a QR from a text prompt produces a\n    plausible-looking but INVALID QR (it cannot reproduce the exact\n    error-corrected module grid), and free-form editing can subtly alter\n    names/faces.\n  * Masked inpainting is given the certificate, a MASK limited to the old-QR\n    square, and your natural-language prompt. The model may only regenerate\n    pixels inside the mask, so every other pixel (design, text, face, seals)\n    is provably unchanged.\n  * We then composite the REAL, freshly computed QR (perspective-warped to the\n    exact quad) on top of the inpainted region -> the edit is AI-generated and\n    blended as requested, AND the QR is guaranteed crisp & scannable.\n\nModel (Hugging Face, runs on a Kaggle T4 16GB):\n    diffusers/stable-diffusion-xl-1.0-inpainting-0.1   (default, fp16)\n    stabilityai/stable-diffusion-2-inpainting         (lighter fallback)\n\nEverything is lazy: torch/diffusers are only imported when you actually run\nthe AI step, so the deterministic warp path needs no GPU and no big download.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport os\n\nimport cv2\nimport numpy as np\n\nfrom .replacer import replace_qr\n\n# The exact prompt requested by the user.\nDEFAULT_PROMPT = (\n    \"Generate an image of this certificate replacing only the old QR code \"\n    \"with a new crisp black and white QR code in the same position. \"\n    \"Do not change any design, do not change any information, do not change \"\n    \"the face or anything else. Just replace the old QR code with the new \"\n    \"QR code, keeping the identical certificate layout, colors, and text.\"\n)\nDEFAULT_NEGATIVE = (\n    \"invalid qr code, blurry qr, distorted text, changed name, altered face, \"\n    \"modified design, extra logo, watermark, deformed, low quality, artifacts\"\n)\n\n_PIPE = None\n_DEVICE = None\n\n\n# --------------------------------------------------------------------------- #\n# Model loading\n# --------------------------------------------------------------------------- #\ndef _device() -> str:\n    global _DEVICE\n    if _DEVICE is not None:\n        return _DEVICE\n    try:\n        import torch\n        _DEVICE = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    except Exception:\n        _DEVICE = \"cpu\"\n    return _DEVICE\n\n\ndef load_pipeline(model_id: str = \"diffusers/stable-diffusion-xl-1.0-inpainting-0.1\"):\n    \"\"\"Lazily load (and cache) the HF inpainting pipeline.\"\"\"\n    global _PIPE\n    if _PIPE is not None:\n        return _PIPE\n    import torch\n    from diffusers import AutoPipelineForInpainting\n\n    dtype = torch.float16 if _device() == \"cuda\" else torch.float32\n    kw = dict(torch_dtype=dtype)\n    try:\n        pipe = AutoPipelineForInpainting.from_pretrained(model_id, **kw)\n    except Exception:\n        # some repos need the variant flag only on GPU fp16\n        kw[\"variant\"] = \"fp16\"\n        pipe = AutoPipelineForInpainting.from_pretrained(model_id, **kw)\n\n    if _device() == \"cuda\":\n        try:\n            pipe.enable_xformers_memory_efficient_attention()\n        except Exception:\n            pass\n        try:\n            pipe.enable_model_cpu_offload()\n        except Exception:\n            pipe.to(\"cuda\")\n    else:\n        pipe.to(\"cpu\")\n    pipe.set_progress_bar_config(disable=False)\n    _PIPE = pipe\n    return pipe\n\n\ndef ai_available() -> bool:\n    try:\n        import torch  # noqa: F401\n        import diffusers  # noqa: F401\n        return True\n    except Exception:\n        return False\n\n\n# --------------------------------------------------------------------------- #\n# Geometry helpers\n# --------------------------------------------------------------------------- #\ndef _mask_from_quad(shape_hw: tuple[int, int], quad: np.ndarray,\n                    dilate_px: int = 6) -> np.ndarray:\n    h, w = shape_hw\n    mask = np.zeros((h, w), np.uint8)\n    cv2.fillConvexPoly(mask, quad.astype(np.int32), 255)\n    if dilate_px:\n        k = cv2.getStructuringElement(\n            cv2.MORPH_RECT, (2 * dilate_px + 1, 2 * dilate_px + 1))\n        mask = cv2.dilate(mask, k)\n    return mask\n\n\ndef _crop_bbox(quad: np.ndarray, h: int, w: int, margin: int):\n    x0, y0 = quad.min(axis=0)\n    x1, y1 = quad.max(axis=0)\n    x0 = int(max(0, x0 - margin)); y0 = int(max(0, y0 - margin))\n    x1 = int(min(w, x1 + margin)); y1 = int(min(h, y1 + margin))\n    return x0, y0, x1, y1\n\n\ndef _round8(v: int) -> int:\n    return max(8, int(round(v / 8.0)) * 8)\n\n\n# --------------------------------------------------------------------------- #\n# Main entry\n# --------------------------------------------------------------------------- #\ndef ai_replace_qr(cert_bgr: np.ndarray, region, qr_bgr: np.ndarray,\n                  prompt: str = DEFAULT_PROMPT,\n                  negative_prompt: str = DEFAULT_NEGATIVE,\n                  model_id: str = \"diffusers/stable-diffusion-xl-1.0-inpainting-0.1\",\n                  steps: int = 30, guidance: float = 8.0, seed: int = 0,\n                  margin: int = 120, mode: str = \"inpaint_then_paste\",\n                  replace_mode: str = \"full_replace\",\n                  cover_scale: float = 1.08):\n    \"\"\"\n    Run AI inpainting over the old-QR mask with `prompt`, then (default)\n    composite the real QR so it scans.\n\n    mode:\n      \"inpaint_then_paste\" (default, recommended):\n            AI regenerates/blends the QR region per the prompt, then the\n            deterministic QR is warped on top -> scannable + AI-edited.\n      \"generative_only\":\n            return the raw model output (NOT guaranteed to scan -- for\n            comparison/experimentation only).\n\n    Returns (out_bgr, info_dict). If the model can't load, falls back to the\n    deterministic warp and info[\"fallback\"]=True.\n    \"\"\"\n    h, w = cert_bgr.shape[:2]\n    quad = region.quad.astype(np.float32)\n\n    if not ai_available():\n        out = replace_qr(cert_bgr, region, qr_bgr, mode=replace_mode,\n                         cover_scale=cover_scale)\n        return out, {\"ran\": False, \"fallback\": \"diffusers/torch not installed\"}\n\n    # The SDXL inpainting model is GPU-only in practice: never attempt it on\n    # CPU (multi-GB download, minutes/image, likely OOM). Fall back instantly.\n    if _device() != \"cuda\":\n        out = replace_qr(cert_bgr, region, qr_bgr, mode=replace_mode,\n                         cover_scale=cover_scale)\n        return out, {\"ran\": False,\n                     \"fallback\": \"AI inpainting needs a CUDA GPU; used warp\"}\n\n    try:\n        import torch\n    except Exception:\n        torch = None\n    try:\n        from PIL import Image\n    except Exception as e:\n        out = replace_qr(cert_bgr, region, qr_bgr, mode=replace_mode,\n                         cover_scale=cover_scale)\n        return out, {\"ran\": False, \"fallback\": f\"PIL missing: {e}\"}\n\n    try:\n        pipe = load_pipeline(model_id)\n    except Exception as e:  # download/OOM -> graceful fallback\n        out = replace_qr(cert_bgr, region, qr_bgr, mode=replace_mode,\n                         cover_scale=cover_scale)\n        return out, {\"ran\": False, \"fallback\": f\"model load failed: {e}\"}\n\n    mask = _mask_from_quad((h, w), quad, dilate_px=8)\n    x0, y0, x1, y1 = _crop_bbox(quad, h, w, margin)\n    crop = cert_bgr[y0:y1, x0:x1].copy()\n    cmask = mask[y0:y1, x0:x1].copy()\n\n    ch, cw = crop.shape[:2]\n    W8, H8 = _round8(cw), _round8(ch)\n\n    pil_img = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)).resize((W8, H8))\n    pil_mask = Image.fromarray(cmask).resize((W8, H8), Image.NEAREST)\n\n    gen = torch.Generator(device=\"cpu\").manual_seed(seed) if torch is not None else None\n    result = pipe(\n        prompt=prompt, negative_prompt=negative_prompt,\n        image=pil_img, mask_image=pil_mask,\n        height=H8, width=W8,\n        guidance_scale=guidance, num_inference_steps=steps,\n        generator=gen,\n    ).images[0]\n    result = result.resize((cw, ch), Image.LANCZOS)\n    inp_crop = cv2.cvtColor(np.array(result), cv2.COLOR_RGB2BGR)\n\n    # Strictly apply ONLY inside the mask -> rest of certificate untouched.\n    m3 = cv2.cvtColor(cmask, cv2.COLOR_GRAY2BGR) > 0\n    edited_crop = crop.copy()\n    edited_crop[m3] = inp_crop[m3]\n\n    edited = cert_bgr.copy()\n    edited[y0:y1, x0:x1] = edited_crop\n\n    info = {\"ran\": True, \"model\": model_id, \"device\": _device(),\n            \"steps\": steps, \"crop\": (x0, y0, x1, y1), \"mode\": mode}\n\n    if mode == \"generative_only\":\n        return edited, info\n\n    # Composite the REAL QR (guaranteed scannable) over the AI region.\n    final = replace_qr(edited, region, qr_bgr, mode=replace_mode,\n                            cover_scale=cover_scale)\n    info[\"qr_composited\"] = True\n    return final, info\n",
      "pipeline.py": "\"\"\"\npipeline.py\n-----------\nEnd-to-end batch workflow:\n\n  for every certificate image in an input folder:\n     1. DETECT  the old QR  (WeChat CNN model from Hugging Face -> OpenCV ->\n                geometric contour finder)\n     2. BUILD   the payload (URL/text/CSV that scanning should open)\n     3. GENERATE a brand-new crisp QR code\n     4. REPLACE the old QR with the new one at the exact same place/size/angle\n                (nothing else on the certificate is touched)\n     5. VERIFY  the new QR is actually decodable on the output image\n     6. SAVE    the new certificate\n\n  then: write a manifest.csv and ZIP all outputs.\n\nRuns on CPU or GPU identically (the geometry work is CPU; the CNN detector is\ntiny). Designed for 200+ certificates in one pass on Kaggle.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport csv\nimport io\nimport os\nimport zipfile\nfrom dataclasses import dataclass, field\n\nimport cv2\nimport numpy as np\n\nfrom .detector import (detect_qr, draw_debug,\n                       fallback_region_from_regions, fallback_region_default)\nfrom .payload import build_payload, load_csv_map, slug_from_filename, safe_slug\nfrom .qr_generator import build_qr\nfrom .qr_art import build_artistic_qr\nfrom .replacer import replace_qr\nfrom .ai_extract import ai_extract_fields\nfrom .verification_page import build_page\nfrom .ai_inpaint import (\n    ai_replace_qr, ai_available, DEFAULT_PROMPT, DEFAULT_NEGATIVE,\n)\n\nIMG_EXTS = (\".png\", \".jpg\", \".jpeg\", \".webp\", \".bmp\", \".tif\", \".tiff\")\n\n\n@dataclass\nclass Result:\n    filename: str\n    status: str                       # \"ok\" | \"recheck\" | \"failed\"\n    payload: str = \"\"\n    method: str = \"\"\n    qr_side_px: float = 0.0\n    verified: bool = False\n    ai_used: bool = False\n    out_name: str = \"\"\n    slug: str = \"\"\n    fields: dict = None\n    extract_method: str = \"\"\n    note: str = \"\"\n\n\n@dataclass\nclass Config:\n    input_dir: str = \"input\"\n    output_dir: str = \"output\"\n    model_dir: str = \"models/wechat\"\n    zip_path: str = \"certificates_with_new_qr.zip\"\n    payload_mode: str = \"url\"          # url | text | csv | copy\n    base_url: str = \"https://example.github.io/certificates/\"\n    url_extension: str = \".png\"\n    # QR links to a per-candidate VERIFICATION PAGE (.html) that shows the\n    # certificate image with the candidate details (read off the certificate by\n    # the AI vision model) beneath it. False -> QR links directly to the image.\n    verification_page: bool = True\n    # AI vision-language model that READS every field off the certificate image\n    # (no OCR / no CSV). Requires a GPU on Kaggle (T4). Empty fields if it fails.\n    ai_extract_model: str = \"Qwen/Qwen2-VL-2B-Instruct\"\n    text_prefix: str = \"\"\n    csv_path: str | None = None\n    qr_error_correction: str = \"M\"\n    qr_box_size: int = 20\n    qr_border: int = 4\n    replace_mode: str = \"full_replace\"  # full_replace | keep_frame\n    qr_cover_scale: float = 1.08        # >1 = white box grows to hide old frame\n    output_ext: str = \".png\"\n    jpeg_quality: int = 95\n    save_debug: bool = True\n    use_cnn: bool = True\n    # ---- QR generator backend ----\n    # \"standard\"  -> plain deterministic qrcode (always scans; recommended for\n    #                official certificates)\n    # \"artistic\"  -> HF QR Code Monster ControlNet (AI art-QR; auto-verified and\n    #                falls back to standard if no seed scans). GPU recommended.\n    qr_backend: str = \"standard\"\n    art_prompt: str = \"\"       # \"\" -> qr_art.CLEAN_PROMPT\n    art_controlnet_scale: float = 1.5\n    art_size: int = 768\n    art_tries: int = 6\n    # ---- AI image-generator (diffusion inpainting) ----\n    use_ai: bool = False\n    ai_model: str = \"diffusers/stable-diffusion-xl-1.0-inpainting-0.1\"\n    ai_prompt: str = DEFAULT_PROMPT\n    ai_negative: str = DEFAULT_NEGATIVE\n    ai_steps: int = 30\n    ai_guidance: float = 8.0\n    ai_seed: int = 0\n    ai_margin: int = 120\n    ai_mode: str = \"inpaint_then_paste\"   # inpaint_then_paste | generative_only\n    # If a QR can't be located on one certificate, place the new QR at the\n    # batch-derived template position (works because all certs share a layout).\n    fallback_placement: bool = True\n\n\ndef list_certificates(input_dir: str) -> list[str]:\n    out = []\n    for root, _dirs, files in os.walk(input_dir):\n        for f in sorted(files):\n            if f.lower().endswith(IMG_EXTS):\n                out.append(os.path.join(root, f))\n    return sorted(out)\n\n\n_WECHAT_DECODER = None\n\n\ndef _get_wechat_decoder(model_dir: str):\n    \"\"\"A WeChat CNN decoder is far stronger on small/dense QRs (used both to\n    read old QRs and to verify new ones).\"\"\"\n    global _WECHAT_DECODER\n    if _WECHAT_DECODER is not None:\n        return _WECHAT_DECODER\n    if not hasattr(cv2, \"wechat_qrcode_WeChatQRCode\"):\n        return None\n    paths = [os.path.join(model_dir, f) for f in\n             (\"detect.prototxt\", \"detect.caffemodel\",\n              \"sr.prototxt\", \"sr.caffemodel\")]\n    if not all(os.path.exists(p) for p in paths):\n        return None\n    try:\n        _WECHAT_DECODER = cv2.wechat_qrcode_WeChatQRCode(*paths)\n    except Exception:\n        _WECHAT_DECODER = None\n    return _WECHAT_DECODER\n\n\ndef _decode_any_qr(img: np.ndarray, model_dir: str = \"models/wechat\") -> str:\n    \"\"\"Best-effort decode to VERIFY the new QR scans. Returns text or ''.\n\n    Tries the strong WeChat CNN decoder first (it reads small/dense QRs that\n    OpenCV's classic decoder misses), then OpenCV ArUco/classic at several\n    scales.\n    \"\"\"\n    h, w = img.shape[:2]\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img\n\n    wd = _get_wechat_decoder(model_dir)\n    if wd is not None:\n        for scale in (1, 2, 3):\n            src = gray if scale == 1 else cv2.resize(\n                gray, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)\n            try:\n                res, _pts = wd.detectAndDecode(src)\n                if len(res) > 0 and res[0]:\n                    return res[0]\n            except Exception:\n                pass\n\n    dets = []\n    if hasattr(cv2, \"QRCodeDetectorAruco\"):\n        try:\n            dets.append(cv2.QRCodeDetectorAruco())\n        except Exception:\n            pass\n    dets.append(cv2.QRCodeDetector())\n    for scale in (1, 2, 3, 4):\n        src = gray if scale == 1 else cv2.resize(\n            gray, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)\n        for d in dets:\n            try:\n                data, _, _ = d.detectAndDecode(src)\n            except Exception:\n                data = \"\"\n            if data:\n                return data\n    return \"\"\n\n\ndef process_one(path: str, cfg: Config,\n                csv_map: dict | None = None,\n                region_override=None) -> tuple[np.ndarray | None, Result]:\n    name = os.path.basename(path)\n    slug = safe_slug(path)\n    res = Result(filename=name, status=\"failed\", slug=slug,\n                 out_name=slug + cfg.output_ext)\n    img = cv2.imread(path, cv2.IMREAD_COLOR)\n    if img is None:\n        res.note = \"could not read image\"\n        return None, res\n\n    # 0) The AI vision model READS all candidate details off the certificate\n    #    image (name, parent, aadhar, course, dates, institute). No OCR / CSV.\n    fields = {}\n    extract_used = \"\"\n    try:\n        fields = ai_extract_fields(img, model_id=cfg.ai_extract_model)\n        if fields and not fields.get(\"_error\"):\n            extract_used = \"AI-VLM\"\n        elif fields.get(\"_error\"):\n            extract_used = \"AI-FAILED\"\n            print(f\"   \u26a0\ufe0f  AI extraction failed for {name}: {fields['_error']}\")\n    except Exception as e:\n        fields = {\"_error\": str(e)}\n        extract_used = \"AI-FAILED\"\n        print(f\"   \u26a0\ufe0f  AI extraction crashed for {name}: {e}\")\n    res.fields = fields\n    res.extract_method = extract_used\n\n    # The QR links to the verification PAGE (.html) when enabled, else the raw\n    # image (cfg.url_extension).\n    link_ext = \".html\" if cfg.verification_page else cfg.url_extension\n\n    # 1) locate the old QR: detect it, or use a provided fallback placement.\n    region = region_override\n    if region is None:\n        region = detect_qr(img, model_dir=cfg.model_dir, use_cnn=cfg.use_cnn)\n    if region is None:\n        res.note = \"old QR not found\"\n        return img, res\n\n    res.method = region.method\n    res.qr_side_px = round(region.side_px, 1)\n\n    # old payload (only needed for 'copy' mode)\n    old_data = _decode_any_qr(img, cfg.model_dir) if cfg.payload_mode == \"copy\" else None\n\n    # 2) payload (links to the verification page or the raw image)\n    payload = build_payload(\n        path, mode=cfg.payload_mode, base_url=cfg.base_url,\n        extension=link_ext, text_prefix=cfg.text_prefix,\n        csv_map=csv_map, old_data=old_data)\n    res.payload = payload\n\n    # 3) generate fresh QR at high resolution.\n    #    Optional AI \"artistic QR\" backend (HF QR Code Monster ControlNet):\n    #    returns an AI-styled QR that decodes to `payload`, else None -> we\n    #    transparently fall back to the deterministic standard QR.\n    qr_img = None\n    if cfg.qr_backend == \"artistic\":\n        from .qr_art import CLEAN_PROMPT\n        qr_img = build_artistic_qr(\n            payload,\n            prompt=cfg.art_prompt or CLEAN_PROMPT,\n            controlnet_scale=cfg.art_controlnet_scale,\n            size=cfg.art_size, max_tries=cfg.art_tries)\n        if qr_img is not None:\n            res.note = \"AI artistic QR (verified)\"\n    if qr_img is None:\n        qr_img = build_qr(payload, box_size=cfg.qr_box_size,\n                          border=cfg.qr_border,\n                          error_correction=cfg.qr_error_correction)\n\n    # 4) REPLACE the old QR.\n    #    - AI path: masked diffusion inpainting (prompt-driven), then the\n    #      real QR is composited so it is guaranteed scannable.\n    #    - deterministic path: exact perspective warp.\n    if cfg.use_ai:\n        out, ai_info = ai_replace_qr(\n            img, region, qr_img,\n            prompt=cfg.ai_prompt, negative_prompt=cfg.ai_negative,\n            model_id=cfg.ai_model, steps=cfg.ai_steps,\n            guidance=cfg.ai_guidance, seed=cfg.ai_seed,\n            margin=cfg.ai_margin, mode=cfg.ai_mode,\n            replace_mode=cfg.replace_mode,\n            cover_scale=cfg.qr_cover_scale)\n        res.ai_used = bool(ai_info.get(\"ran\"))\n        if not ai_info.get(\"ran\"):\n            res.method += \"+ai-fallback(warp)\"\n    else:\n        out = replace_qr(img, region, qr_img, mode=cfg.replace_mode,\n                         cover_scale=cfg.qr_cover_scale)\n        ai_info = {}\n\n    # 5) verify the new QR decodes on the output\n    decoded = _decode_any_qr(out, cfg.model_dir)\n    res.verified = bool(decoded)\n    art_tag = \"AI artistic QR; \" if (cfg.qr_backend == \"artistic\"\n                                    and \"artistic\" in res.note) else \"\"\n    if not decoded:\n        res.status = \"recheck\"\n        res.note = art_tag + \"new QR not auto-verified (usually still scans; check debug)\"\n    else:\n        res.status = \"ok\"\n        if res.ai_used:\n            res.note = art_tag + \"verified scannable (AI inpaint+QR)\"\n        else:\n            res.note = art_tag + \"verified scannable\"\n\n    return out, res\n\n\ndef run(cfg: Config) -> dict:\n    os.makedirs(cfg.output_dir, exist_ok=True)\n    debug_dir = os.path.join(cfg.output_dir, \"_debug\")\n    if cfg.save_debug:\n        os.makedirs(debug_dir, exist_ok=True)\n\n    csv_map = None\n    if cfg.payload_mode == \"csv\" and cfg.csv_path:\n        csv_map = load_csv_map(cfg.csv_path)\n\n    certs = list_certificates(cfg.input_dir)\n\n    if cfg.payload_mode == \"url\":\n        placeholder_markers = (\"YOUR-USERNAME\", \"example.github.io\",\n                               \"example.com\", \"REPLACE\", \"your-username\")\n        if any(m.lower() in (cfg.base_url or \"\").lower()\n               for m in placeholder_markers):\n            print(\"=\" * 70)\n            print(\"\u26a0\ufe0f  WARNING: BASE_URL still looks like a PLACEHOLDER.\")\n            print(f\"    base_url = {cfg.base_url}\")\n            print(\"    The QR codes WILL scan, but they point at a URL that\")\n            print(\"    does not exist yet -> your phone shows a 404 page.\")\n            print(\"    1) Set base_url to the address where you will upload the\")\n            print(\"       OUTPUT images (e.g. GitHub Pages: create a repo, turn\")\n            print(\"       on Settings->Pages, upload the finished PNGs).\")\n            print(\"    2) Re-run. See README 'Hosting' / notebook section 8.\")\n            print(\"=\" * 70)\n\n    # ---- detection pass: locate the QR on every certificate so that a cert\n    # where detection fails can inherit the position from its template (all\n    # certificates in a batch share a layout).\n    detected: dict[str, object] = {}\n    template: list = []\n    for path in certs:\n        img = cv2.imread(path, cv2.IMREAD_COLOR)\n        if img is None:\n            continue\n        reg = detect_qr(img, model_dir=cfg.model_dir, use_cnn=cfg.use_cnn)\n        h, w = img.shape[:2]\n        detected[path] = reg\n        if reg is not None and \"fallback\" not in reg.method:\n            template.append((reg, h, w))\n\n    results: list[Result] = []\n    ok = recheck = failed = 0\n    ai_ok = ai_fail = 0\n\n    zip_path = cfg.zip_path\n    with zipfile.ZipFile(zip_path, \"w\", zipfile.ZIP_DEFLATED) as zf:\n        for path in certs:\n            img = cv2.imread(path, cv2.IMREAD_COLOR)\n            override = None\n            if detected.get(path) is None and cfg.fallback_placement:\n                # inherit QR position from the other certificates / template\n                override = fallback_region_from_regions(img, template)\n                if override is None:\n                    override = fallback_region_default(img)\n            out, res = process_one(path, cfg, csv_map, region_override=override)\n            if res.extract_method == \"AI-VLM\":\n                ai_ok += 1\n            else:\n                ai_fail += 1\n            if override is not None and res.status in (\"ok\", \"recheck\"):\n                res.note += f\" | QR placed via {override.method}\"\n            results.append(res)\n            if res.status == \"ok\":\n                ok += 1\n            elif res.status == \"recheck\":\n                recheck += 1\n            else:\n                failed += 1\n\n            if out is not None and res.status in (\"ok\", \"recheck\"):\n                out_name = res.out_name\n                out_path = os.path.join(cfg.output_dir, out_name)\n                _write_image(out, out_path, cfg)\n                zf.write(out_path, arcname=\"img/\" + out_name)\n\n                # Per-candidate VERIFICATION PAGE (self-contained HTML).\n                if cfg.verification_page:\n                    page = build_page(res.fields or {}, out, out_name)\n                    page_name = res.slug + \".html\"\n                    page_path = os.path.join(cfg.output_dir, page_name)\n                    with open(page_path, \"w\", encoding=\"utf-8\") as fh:\n                        fh.write(page)\n                    zf.write(page_path, arcname=page_name)\n\n                if cfg.save_debug and res.status == \"recheck\":\n                    reg = detect_qr(out, model_dir=cfg.model_dir,\n                                    use_cnn=False)\n                    dbg = draw_debug(out, reg) if reg else out\n                    cv2.imwrite(os.path.join(debug_dir, \"dbg_\" + out_name), dbg)\n            elif out is not None and cfg.save_debug:\n                cv2.imwrite(os.path.join(\n                    debug_dir, \"FAILED_\" + safe_slug(path) + cfg.output_ext), out)\n\n            tag = \"AI\" if res.ai_used else (\"ai-fb\" if \"+ai-fallback\" in res.method else \"  \")\n            print(f\"[{res.status:7s}] {tag} {res.filename:36s} \"\n                  f\"{res.method:24s} {res.qr_side_px:6.1f}px  {res.payload}\")\n\n    # manifest\n    manifest = os.path.join(cfg.output_dir, \"manifest.csv\")\n    with open(manifest, \"w\", newline=\"\", encoding=\"utf-8\") as f:\n        wr = csv.writer(f)\n        wr.writerow([\"filename\", \"status\", \"ai_used\", \"method\", \"qr_side_px\",\n                     \"verified\", \"qr_payload\", \"name\", \"parent\", \"aadhar\",\n                     \"course\", \"date_from\", \"date_to\", \"institute\",\n                     \"extracted_by\", \"note\"])\n        for r in results:\n            f = r.fields or {}\n            wr.writerow([r.filename, r.status, r.ai_used, r.method,\n                         r.qr_side_px, r.verified, r.payload,\n                         f.get(\"name\", \"\"), f.get(\"parent\", \"\"),\n                         f.get(\"aadhar\", \"\"), f.get(\"course\", \"\"),\n                         f.get(\"date_from\", \"\"), f.get(\"date_to\", \"\"),\n                         f.get(\"institute\", \"\"), r.extract_method, r.note])\n\n    summary = {\"total\": len(certs), \"ok\": ok, \"recheck\": recheck,\n               \"failed\": failed, \"fields_extracted\": ai_ok,\n               \"fields_failed\": ai_fail,\n               \"zip\": os.path.abspath(zip_path),\n               \"manifest\": os.path.abspath(manifest)}\n    print(\"\\n===== SUMMARY =====\")\n    for k, v in summary.items():\n        print(f\"{k:16s}: {v}\")\n    if ai_fail:\n        print(\"\\n\u26a0\ufe0f  The AI could NOT read fields from \"\n              f\"{ai_fail} certificate(s) -> those pages show '\u2014'.\")\n        print(\"   Check: Accelerator = GPU T4; and run the 'Test AI extraction'\")\n        print(\"   cell to see the raw model output / error.\")\n    return summary\n\n\ndef _write_image(img: np.ndarray, path: str, cfg: Config):\n    ext = cfg.output_ext.lower()\n    if ext in (\".jpg\", \".jpeg\"):\n        cv2.imwrite(path, img, [cv2.IMWRITE_JPEG_QUALITY, cfg.jpeg_quality])\n    else:\n        cv2.imwrite(path, img)\n"
    }
    for name, content in SRC.items():
        with open(os.path.join("/kaggle/working/certbot/src", name), "w",
                  encoding="utf-8") as f:
            f.write(content)
        print("wrote", name, len(content), "chars")
    print("pipeline has qr_cover_scale:", "qr_cover_scale" in SRC["pipeline.py"])

    # publish helper (GitHub Pages)
    with open("/kaggle/working/certbot/publish_to_github_pages.py", "w",
              encoding="utf-8") as f:
        f.write("#!/usr/bin/env python3\n\"\"\"\npublish_to_github_pages.py\n--------------------------\nUpload the finished certificate images to a GitHub Pages repository so the QR\nlinks resolve (scan -> certificate opens in the browser). Self-verifying:\nafter each upload it polls the public Pages URL until the image is live.\n\nToken: a GitHub Personal Access Token (classic) with the `repo` scope.\nProvided via GITHUB_TOKEN / GH_TOKEN env var (on Kaggle, add a notebook Secret\nnamed GITHUB_TOKEN), or pasted at the prompt.\n\nCreate a token: https://github.com/settings/tokens  (Generate new token\n(classic) -> tick \"repo\").\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport base64\nimport os\nimport time\nimport zipfile\n\ntry:\n    import requests\nexcept Exception:  # pragma: no cover\n    requests = None\n\nAPI = \"https://api.github.com\"\n\n\ndef _get_token() -> str:\n    tok = os.environ.get(\"GITHUB_TOKEN\") or os.environ.get(\"GH_TOKEN\") or \"\"\n    if not tok:\n        tok = input(\"Paste your GitHub token (repo scope): \").strip()\n    return tok\n\n\ndef publish(zip_path: str, owner: str, repo: str, token: str,\n            subdir: str = \"img\", branch: str = \"main\",\n            verify: bool = True, verbose: bool = True):\n    if requests is None:\n        raise SystemExit(\"'requests' is required (pip install requests).\")\n\n    files = []   # (name, content, upload_dir)\n    with zipfile.ZipFile(zip_path) as z:\n        for name in z.namelist():\n            base_name = os.path.basename(name)\n            low = name.lower()\n            if low.endswith((\".png\", \".jpg\", \".jpeg\", \".webp\")):\n                # images stay under img/ (whether the zip root or img/ prefix)\n                files.append((base_name, z.read(name), subdir))\n            elif low.endswith(\".html\"):\n                # verification pages live at the repo root (QR URLs point there)\n                files.append((base_name, z.read(name), \"\"))\n\n    headers = {\"Authorization\": f\"Bearer {token}\",\n               \"Accept\": \"application/vnd.github+json\",\n               \"User-Agent\": \"cert-qr-bot-publisher\"}\n    api_base = f\"{API}/repos/{owner}/{repo}/contents\"\n    live_root = f\"https://{owner}.github.io/{repo}/\"\n    ok = fail = upload_fail = 0\n    uploaded_files = []   # (name, live_url, is_html)\n\n    if verbose:\n        print(f\"Publishing {len(files)} file(s) to {owner}/{repo} ...\")\n\n    for i, (name, content, upload_dir) in enumerate(files, 1):\n        path = f\"{api_base}/{upload_dir}/{name}\" if upload_dir \\\n            else f\"{api_base}/{name}\"\n        # find existing sha (to update rather than conflict)\n        sha = None\n        try:\n            r = requests.get(path, headers=headers,\n                             params={\"ref\": branch}, timeout=30)\n            if r.status_code == 200:\n                sha = r.json().get(\"sha\")\n        except Exception:\n            pass\n\n        body = {\"message\": f\"Publish {name}\",\n                \"content\": base64.b64encode(content).decode(),\n                \"branch\": branch}\n        if sha:\n            body[\"sha\"] = sha\n\n        uploaded = False\n        for attempt in range(3):\n            try:\n                r = requests.put(path, headers=headers, json=body, timeout=60)\n                if r.status_code in (200, 201):\n                    uploaded = True\n                    break\n                if verbose:\n                    print(f\"  ! {name} upload attempt {attempt+1}: \"\n                          f\"HTTP {r.status_code} {r.text[:160]}\")\n            except Exception as e:\n                if verbose:\n                    print(f\"  ! {name} attempt {attempt+1} error: {e}\")\n            time.sleep(2)\n\n        live_url = (f\"https://{owner}.github.io/{repo}/{upload_dir}/{name}\"\n                    if upload_dir else f\"https://{owner}.github.io/{repo}/{name}\")\n        is_img = name.lower().endswith((\".png\", \".jpg\", \".jpeg\", \".webp\"))\n        is_html = name.lower().endswith(\".html\")\n        if uploaded:\n            uploaded_files.append((name, live_url, is_html))\n        # Per-file quick check (short); the final sweep below re-checks any\n        # that weren't live yet, so Pages propagation lag is not fatal here.\n        if uploaded and verify and (is_img or is_html):\n            live = _wait_live(live_url, expect_html=is_html,\n                              tries=3, delay=4)\n        else:\n            live = uploaded\n\n        if uploaded and live:\n            ok += 1\n            if verbose:\n                print(f\"  [{i}/{len(files)}] \u2705 {name} -> {live_url}\")\n        elif uploaded:\n            # uploaded but not live yet \u2014 final sweep will re-check it\n            if verbose:\n                print(f\"  [{i}/{len(files)}] \u23f3 {name} uploaded, waiting for Pages ...\")\n        else:\n            upload_fail += 1\n            if verbose:\n                print(f\"  [{i}/{len(files)}] \u274c {name} (upload failed)\")\n        time.sleep(0.4)\n\n    # ---- Final verification sweep: keep re-checking every uploaded file until\n    # each is live on GitHub Pages (handles propagation lag for image AND html).\n    if verify and uploaded_files:\n        pending = [(n, u, h) for (n, u, h) in uploaded_files]\n        if verbose and pending:\n            print(f\"\\nVerifying {len(pending)} page(s)/image(s) are live ...\")\n        for round_no in range(1, 13):   # up to ~12 * 10s = 2 minutes\n            still = []\n            for name, live_url, is_html in pending:\n                if _check_one(live_url, is_html):\n                    ok += 0  # already counted as uploaded\n                    if verbose:\n                        print(f\"  \u2705 live: {name}\")\n                else:\n                    still.append((name, live_url, is_html))\n            pending = still\n            if not pending:\n                break\n            if verbose:\n                print(f\"  ... {len(pending)} not live yet, re-checking \"\n                      f\"(round {round_no})\")\n            time.sleep(10)\n        # anything still pending after the sweep is reported for a re-run\n        for name, _u, _h in pending:\n            if verbose:\n                print(f\"  \u274c {name} still not live after waiting (re-run \"\n                      f\"section 8 in ~1 min)\")\n        not_live = len(pending)\n    else:\n        not_live = 0\n    fail = not_live + upload_fail\n\n    live_count = len(uploaded_files) - not_live\n    if verbose:\n        print(f\"\\nDone: {live_count} live, {fail} not yet live \"\n              f\"({upload_fail} upload failure(s)).\")\n        if fail == 0:\n            print(\"\u2705 Every image and verification page is uploaded AND live.\")\n        print(\"Pages (QR base URL): \" + live_root)\n        print(\"Images under:        \" + live_root + subdir + \"/\")\n    return {\"ok\": live_count, \"fail\": fail, \"base\": live_root}\n\n\ndef _check_one(url: str, expect_html: bool) -> bool:\n    import random\n    bust = url + (\"&\" if \"?\" in url else \"?\") + \"cb=\" + str(random.randint(0, 10**9))\n    try:\n        r = requests.get(bust, timeout=30,\n                         headers={\"Cache-Control\": \"no-cache\",\n                                  \"User-Agent\": \"cert-qr-bot-publisher\"})\n        ctype = r.headers.get(\"content-type\", \"\")\n        if r.status_code != 200:\n            return False\n        if expect_html:\n            return \"html\" in ctype\n        return \"image\" in ctype\n    except Exception:\n        return False\n\n\ndef _wait_live(url: str, tries: int = 14, delay: int = 5,\n               expect_html: bool = False) -> bool:\n    for _ in range(tries):\n        if _check_one(url, expect_html):\n            return True\n        time.sleep(delay)\n    return False\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--zip\", default=\"certificates_with_new_qr.zip\")\n    ap.add_argument(\"--owner\", default=\"Naserkhan07\")\n    ap.add_argument(\"--repo\", default=\"certificate-qr\")\n    ap.add_argument(\"--subdir\", default=\"img\")\n    ap.add_argument(\"--branch\", default=\"main\")\n    ap.add_argument(\"--no-verify\", action=\"store_true\")\n    args = ap.parse_args()\n    token = _get_token()\n    if not token:\n        raise SystemExit(\"No GitHub token provided.\")\n    publish(args.zip, args.owner, args.repo, token, args.subdir,\n            args.branch, verify=not args.no_verify)\n\n\nif __name__ == \"__main__\":\n    main()\n")
    print("wrote publish_to_github_pages.py")
# AUTO-PUBLISH: upload every finished certificate to the public GitHub Pages
# repo so each QR link opens the certificate. Each upload is verified live.
PUBLISH = True       # keep True so all certs are uploaded (scanning then works)
if PUBLISH:
    import os, sys
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        token = os.environ.get("GITHUB_TOKEN", "")
    if not token:
        print("❌ GITHUB_TOKEN secret not found. Add it once: Kaggle Add-ons ->")
        print("   Secrets -> add secret named GITHUB_TOKEN (a classic GitHub")
        print("   token with the 'repo' scope from https://github.com/settings/tokens).")
        print("   Then re-run this cell. Certificates were still saved to the ZIP.")
    else:
        os.environ["GITHUB_TOKEN"] = token
        sys.path.insert(0, "/kaggle/working/certbot")
        _pub = open("/kaggle/working/certbot/publish_to_github_pages.py").read()
        _pub = _pub.replace('if __name__ == "__main__":\n    main()', "")
        g = {"__name__": "publisher"}
        exec(compile(_pub, "publish_to_github_pages.py", "exec"), g)
        _res = g["publish"](
            zip_path="/kaggle/working/certificates_with_new_qr.zip",
            owner="Naserkhan07", repo="certificate-qr",
            token=token, subdir="img", branch="main")
        if _res["fail"] == 0:
            print("\n✅ All certificates live. Scanning any QR now opens the")
            print("   verification page at https://naserkhan07.github.io/certificate-qr/<file>.html")
        else:
            print(f"\n⚠️ {_res['fail']} file(s) failed verification - re-run this cell.")
else:
    print("PUBLISH=False: certificates only saved to the ZIP (QR links won't resolve).")


## 🌐 9. Notes / other hosts

The QR encodes a **URL** — hosting is unavoidable (this is exactly how every
"image QR" website works; they just host the file on *their* server). The
publisher above puts them on your own GitHub Pages.

Other hosts work too: your own server, S3/Cloudflare R2, Netlify, or image-host
APIs (imgbb/Cloudinary). For **private** data (Aadhaar), use a private host and
`PAYLOAD_MODE="csv"` with a `filename,qr_data` mapping of direct links.

### Tuning
- **QR won't scan on paper?** It's verified digitally; for very small print
  raise `QR_ERROR_CORRECTION="H"` and print at ≥ ~150 DPI.
- **Keep the old black frame line?** Set `REPLACE_MODE="keep_frame"`.
- **Old QR never detected?** Check `output/_debug/FAILED_*` overlays. Detection
  now uses WeChat CNN → OpenCV → frame contours → finder patterns, so it works
  for framed AND frameless QRs.
